# data loader

In [1]:
!pip install torch_geometric
!pip install gdown
import gdown
import pandas as pd

file_id = '1Q0EdTGj1AvneaYib29d-KDulpL6qZ_qe'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'Crimes_2001_to_Present.csv'
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
df.head()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00a 0:00:01


Downloading...
From (original): https://drive.google.com/uc?id=1Q0EdTGj1AvneaYib29d-KDulpL6qZ_qe
From (redirected): https://drive.google.com/uc?id=1Q0EdTGj1AvneaYib29d-KDulpL6qZ_qe&confirm=t&uuid=bc154f7c-ef3b-473b-be00-421188390215
To: /kaggle/working/Crimes_2001_to_Present.csv
100%|██████████| 2.33G/2.33G [00:21<00:00, 106MB/s] 


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,13940083,JJ381831,08/20/2025 12:00:00 AM,051XX N SIMONDS DR,5002,OTHER OFFENSE,OTHER VEHICLE OFFENSE,PARK PROPERTY,False,False,...,48.0,3.0,26,1170788.0,1934634.0,2025,08/27/2025 03:42:42 PM,41.976117,-87.647318,"(41.976117409, -87.647317624)"
1,13940166,JJ381863,08/20/2025 12:00:00 AM,043XX S MARSHFIELD AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,APARTMENT,False,True,...,15.0,61.0,14,1166064.0,1875915.0,2025,08/27/2025 03:42:42 PM,41.815091,-87.666368,"(41.815090766, -87.666368145)"
2,13942926,JJ385476,08/20/2025 12:00:00 AM,019XX N CICERO AVE,1150,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,COMMERCIAL / BUSINESS OFFICE,False,False,...,36.0,19.0,11,1144091.0,1912424.0,2025,08/27/2025 03:42:42 PM,41.915716,-87.746053,"(41.915715874, -87.74605323)"
3,13940457,JJ382312,08/20/2025 12:00:00 AM,050XX N HERMITAGE AVE,0810,THEFT,OVER $500,RESIDENCE - YARD (FRONT / BACK),False,False,...,47.0,3.0,06,1163892.0,1933382.0,2025,08/27/2025 03:42:42 PM,41.972831,-87.672712,"(41.972830613, -87.67271233)"
4,13940146,JJ381842,08/20/2025 12:00:00 AM,0000X E ROOSEVELT RD,0460,BATTERY,SIMPLE,SIDEWALK,False,False,...,4.0,32.0,08B,1176901.0,1895075.0,2025,08/27/2025 03:42:42 PM,41.867430,-87.626038,"(41.867429555, -87.626037832)"


In [3]:
# =============================================================================
# LOAD PRETRAINED MODELS + RUN 10.6 (BOOTSTRAP CI) AND 10.7 (PAIRED BOOTSTRAP)
# ONLY — standalone, no training of anything.
#
# This script:
#   1) runs the data pipeline (needed so tensors/scaler/splits match exactly)
#   2) defines the HAST_GNN and BaselineLSTMGAT model classes
#   3) LOADS both checkpoints (no training, no seed loop, no re-fit of
#      anything) from the paths below and evaluates each once on the test set
#   4) computes the persistence (10.1) and seasonal (10.2) naive baselines,
#      needed only because 10.7 compares HAST-GNN against them too
#   5) runs Point 10.6 (bootstrap 95% CIs for the loaded HAST-GNN) and
#      Point 10.7 (paired bootstrap: HAST-GNN vs persistence / seasonal /
#      LSTM+GATv2) and prints the results
#
# Edit the two paths below if your checkpoints are somewhere else.
# =============================================================================

PRIMARY_CKPT  = '/kaggle/input/models/itsami12/modee/pytorch/default/1/hast_gnn_best (1).pth'
BASELINE_CKPT = '/kaggle/working/baseline_lstm_gat_best.pth'

# =============================================================================
# HAST-GNN — Chicago Crime Prediction  v14
#
# INHERITS v13 fully:
#   Architecture:  ARATA + DWConv1d | CTIG | AHG+residual | MAMTO
#   Features:      hour-bucket feats, rolling type counts, loc-type feats
#   v13 changes:   CrimeType -> 10 fine-grained classes, CrimeHour -> 8 buckets
#
# CHANGES in v14 (this pass — preprocessing/pipeline fixes only, no architecture
# or loss changes):
#
#   FIX-1  train_mask referenced before assignment (real bug in v13)
#   FIX-2  Explicit, non-ambiguous StandardScaler fit/transform on train/val/test
#   FIX-3  CrimeDS split boundary indices fixed (N_TRAIN_DAYS/N_VAL_DAYS/N_TEST_DAYS)
#   FIX-4  Sequence-level temporal audit with hard assertions
#   GPU-1  Rolling-feature block runs on cuDF when available, pandas fallback
#   GPU-2  Vectorized precompute + native agg for the daily-grid build
#
# CHANGES in this pass (review-doc "Point 10" additions):
#
#   REFACTOR-1  evaluate() now accepts an optional `eval_model` argument so it
#     can score any model (HAST-GNN or a baseline) that returns the same
#     6-tuple (hot, loc, typ, per, cnt, trd). Falls back to the global `model`
#     when eval_model is omitted, so all existing call sites keep working
#     unmodified.
#
#   REFACTOR-2  The main training loop is now the reusable function
#     `train_hast_gnn(...)`. The primary run below calls it once; Point 10.4
#     (multi-seed) and Point 10.3 (baseline) call the SAME function instead of
#     duplicating the 120-epoch loop. Behavior/printed metrics for the primary
#     run are unchanged.
#
#   10.1  Persistence baseline (lag-1 "predict tomorrow = today")
#   10.2  Seasonal 7-day baseline ("predict day t = day t-7")
#   10.3  Compact LSTM + GATv2 spatio-temporal baseline, same inputs/graph/
#         split/targets/loss, trained and evaluated separately from HAST-GNN.
#   10.4  Multi-seed HAST-GNN runs (reuses train_hast_gnn, no code duplication)
#   10.5  Multi-seed mean +/- SD summary
#   10.6  Bootstrap 95% CIs for the single best HAST-GNN run
#   10.7  Paired bootstrap: HAST-GNN vs persistence / seasonal / LSTM-GAT
#
#   ALIGNMENT FIX (flagged by the review doc as something to double check
#   before running 10.7): the doc's example 10.1/10.2 code slices
#   log_cnt_np[test_start+T : NUM_DAYS], which is off by one relative to the
#   actual number of test sequences produced by CrimeDS. CrimeDS.__len__ is
#   max(0, (e-s)-T-1) -- note the extra "-1" -- so the last usable target day
#   is NUM_DAYS-2, not NUM_DAYS-1. Using NUM_DAYS as the upper bound gives the
#   baseline arrays one MORE row than s["cr"]/s["ht"] coming out of
#   evaluate(test_loader), which would silently misalign every paired
#   comparison in 10.7. Below, the baseline target window is derived directly
#   from len(test_ds) instead of hand re-deriving the slice bounds, so it is
#   guaranteed to match the HAST-GNN test predictions row-for-row.
#
#   PROGRESS-1 (this pass): tqdm progress bars added throughout so long,
#   quiet stretches (verbose=False multi-seed / baseline runs) still show
#   live progress instead of going silent for the whole run.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    roc_auc_score, precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# GPU-1: optional cuDF backend for the rolling-feature block
# =============================================================================
HAS_CUDF = False
_cudf_err = None
if torch.cuda.is_available():
    try:
        import cudf
        import cupy as cp
        _probe = cudf.DataFrame({"a": [1, 2, 3]})
        _ = _probe.groupby("a").size()
        cp.array([1.0]).sum()
        HAS_CUDF = True
    except Exception as e:
        HAS_CUDF = False
        _cudf_err = e
if HAS_CUDF:
    print("cuDF GPU feature engineering: ENABLED")
else:
    print("cuDF GPU feature engineering: DISABLED (pandas fallback)")
    if _cudf_err is not None:
        print(f"  cudf/cupy import/probe failed ({type(_cudf_err).__name__}: {_cudf_err})")
        print("  This is usually a broken cupy/cudf install, not a code bug — common causes:")
        print("    - mismatched cupy-cudaXX wheel vs the container's CUDA version")
        print("    - a stray local file/folder named cupy.py or cudf.py shadowing the real package")
        print("    - a half-completed `pip install`/kernel restart needed")
        print("  Falling back to pandas; the pipeline will still run correctly, just on CPU for this block.")


def gpu_shift_rolling(df, group_col, value_col, window, min_periods, shift=1,
                       agg="mean"):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values,
                               value_col: df[value_col].values})
        gdf["_shifted"] = gdf.groupby(group_col)[value_col].shift(shift)
        roll = gdf.groupby(group_col)["_shifted"].rolling(window, min_periods=min_periods)
        if agg == "mean":
            out = roll.mean()
        elif agg == "sum":
            out = roll.sum()
        elif agg == "std":
            out = roll.std()
        else:
            raise ValueError(agg)
        out = out.reset_index(drop=True)
        result = out.to_pandas().values.astype(np.float32)
        if agg == "std":
            result = np.nan_to_num(result, nan=0.0)
        return result
    else:
        if agg == "mean":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).mean())
        elif agg == "sum":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).sum())
        elif agg == "std":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).std().fillna(0))
        else:
            raise ValueError(agg)
        return s.values.astype(np.float32)


def gpu_shift(df, group_col, value_col, lag):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values, value_col: df[value_col].values})
        out = gdf.groupby(group_col)[value_col].shift(lag)
        return out.reset_index(drop=True).to_pandas().fillna(0).values.astype(np.float32)
    else:
        return df.groupby(group_col)[value_col].transform(lambda x: x.shift(lag).fillna(0)).values.astype(np.float32)


def gpu_build_daily_grid(crimes_df, group_cols, agg_dict, has_cudf):
    needed_cols = list(group_cols) + list(agg_dict.keys())
    sub = crimes_df[needed_cols]

    if has_cudf:
        gdf = cudf.DataFrame.from_pandas(sub)
        out = gdf.groupby(group_cols, as_index=False).agg(agg_dict)
        out = out.to_pandas()
    else:
        out = sub.groupby(group_cols, as_index=False).agg(agg_dict)

    new_cols = []
    for c in out.columns:
        if isinstance(c, tuple):
            base, aggname = c
            new_cols.append(base if aggname in ("", None) else f"{base}__{aggname}")
        else:
            new_cols.append(c)
    out.columns = new_cols
    return out


def gpu_rolling_argmax_sum(df, group_col, value_cols, window, min_periods, shift=1):
    rolled = np.zeros((len(df), len(value_cols)), dtype=np.float32)
    for k, col in enumerate(value_cols):
        r = gpu_shift_rolling(df, group_col, col, window, min_periods, shift=shift, agg="sum")
        nan_mask = np.isnan(r)
        if nan_mask.any():
            exp = df.groupby(group_col)[col].transform(
                lambda x: x.shift(shift).expanding(min_periods=1).sum()).values
            r = np.where(nan_mask, exp, r)
        rolled[:, k] = np.nan_to_num(r, nan=0.0)
    argmax = np.argmax(rolled, axis=1).astype(np.int64)
    return rolled, argmax


# =============================================================================
# CONFIG
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9          # top-9 raw types + "OTHER" = 10 classes
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

    # Point 10.3 — compact baseline hyperparams (kept separate from HAST-GNN)
    BASELINE_HIDDEN   = 64
    BASELINE_HEADS    = 4
    BASELINE_EPOCHS   = 60
    BASELINE_PATIENCE = 15
    BASELINE_LR       = 3e-4

    # Point 10.4 — multi-seed experiment
    MULTI_SEEDS = [42, 123, 2024]

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("HAST-GNN v14 | CrimeType=10-class | CrimeHour=8-bucket | split/scaler/dataset fixes | GPU rolling feats")

# =============================================================================
# ADJACENCY GRAPHS  (unchanged)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# LOAD DATA  (unchanged — one-time pass with arbitrary lambdas, stays on pandas)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
print("Reading individual crime events (8.3M rows)...")

chunk_dfs = []
for i, chunk in enumerate(tqdm(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"]),
        desc="Reading CSV chunks", unit="chunk")):

    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    ad = chunk.groupby(["date","ca"]).size()
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows → {len(ad):,} area-days")

print("Combining all chunks...")
crimes = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

# =============================================================================
# TEMPORAL SPLIT DEFINITIONS
# =============================================================================
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")

print(
    f"Temporal split definitions: "
    f"Train <= {split_train.date()} | "
    f"Validation <= {split_val.date()} | "
    f"Test > {split_val.date()}"
)

print(f"\n✓ Community areas: {crimes['ca'].nunique()}  |  Days: {NUM_DAYS}")
print(f"  Date range: {all_dates.iloc[0].date()} → {all_dates.iloc[-1].date()}")
print(f"  Total crime events: {len(crimes):,}")

type_counts  = crimes["Primary Type"].value_counts()
top_types    = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"\nCrime types ({N_TYPES}): {TYPE_CLASSES}")

# =============================================================================
# BUILD DAILY AGGREGATED GRID
# =============================================================================
print("\nComputing features...")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]

crimes["_one"]     = np.int8(1)
crimes["is_eve"]   = (crimes["hour"] >= 18).astype(np.int8)
crimes["is_night"] = (crimes["hour"] < 6).astype(np.int8)
crimes["arrest_i"] = crimes["Arrest"].astype(bool).astype(np.int8)

type_onehot_cols = []
for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    crimes[col] = (crimes["Primary Type"] == t).astype(np.int8)
    type_onehot_cols.append(col)

for s, e in HOUR_BUCKETS:
    col = f"hour_{s}_{e}"
    crimes[col] = ((crimes["hour"] >= s) & (crimes["hour"] < e)).astype(np.int8)

agg_dict = {
    "_one":     ["sum"],
    "hour":     ["mean"],
    "is_eve":   ["sum"],
    "is_night": ["sum"],
    "Latitude":  ["mean", "std"],
    "Longitude": ["mean", "std"],
    "arrest_i": ["mean"],
}
for col in type_onehot_cols:
    agg_dict[col] = ["sum"]
for col in HOUR_BUCKET_COLS:
    agg_dict[col] = ["sum"]

grid = gpu_build_daily_grid(crimes, ["date","ca"], agg_dict, HAS_CUDF)

rename_map = {
    "_one__sum": "crime_count",
    "hour__mean": "avg_hour",
    "is_eve__sum": "eve_cnt",
    "is_night__sum": "night_cnt",
    "Latitude__mean": "mean_lat",
    "Latitude__std": "std_lat",
    "Longitude__mean": "mean_lon",
    "Longitude__std": "std_lon",
    "arrest_i__mean": "arrest_rate",
}
for col in type_onehot_cols:
    rename_map[f"{col}__sum"] = col
for col in HOUR_BUCKET_COLS:
    rename_map[f"{col}__sum"] = col
grid = grid.rename(columns=rename_map)

crimes.drop(columns=["_one","is_eve","is_night","arrest_i"] + type_onehot_cols + HOUR_BUCKET_COLS,
            inplace=True)

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)

RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)

for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"])
            .reindex(full_idx, fill_value=0)
            .reset_index()
            .sort_values(["date","ca"])
            .reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

# ── GPU-1: rolling temporal features on crime_count ───────────────────────────
grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["roll7"]  = gpu_shift_rolling(grid, "ca", "crime_count", 7,  1, agg="mean")
grid["roll14"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 1, agg="mean")
grid["roll30"] = gpu_shift_rolling(grid, "ca", "crime_count", 30, 1, agg="mean")
grid["roll90"] = gpu_shift_rolling(grid, "ca", "crime_count", 90, 1, agg="mean")
grid["std7"]   = gpu_shift_rolling(grid, "ca", "crime_count", 7,  2, agg="std")
grid["std30"]  = gpu_shift_rolling(grid, "ca", "crime_count", 30, 2, agg="std")
grid["lag1"]   = gpu_shift(grid, "ca", "crime_count", 1)
grid["lag7"]   = gpu_shift(grid, "ca", "crime_count", 7)
grid["lag14"]  = gpu_shift(grid, "ca", "crime_count", 14)
grid["lag28"]  = gpu_shift(grid, "ca", "crime_count", 28)
grid["wow"] = (
    grid.groupby("ca")["roll7"]
        .transform(lambda x: (x / x.shift(7).clip(lower=1)).fillna(1))
)
grid["mom"] = (
    grid.groupby("ca")["roll14"]
        .transform(lambda x: (x / x.shift(14).clip(lower=1)).fillna(1))
)
grid["roll_hour7"]  = gpu_shift_rolling(grid, "ca", "avg_hour", 7,  1, agg="mean")
grid["roll_hour30"] = gpu_shift_rolling(grid, "ca", "avg_hour", 30, 1, agg="mean")

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col  = f"r7_{col}"
    r30_col = f"r30_{col}"
    grid[r7_col]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
    grid[r30_col] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)

# =============================================================================
# TRAINING-ONLY GEOGRAPHIC AREA STATISTICS
# =============================================================================
train_rows_geo = grid["date"] <= split_train

train_area_lat = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lat"]
        .mean()
)
train_area_lon = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lon"]
        .mean()
)
grid["area_lat"] = grid["ca"].map(train_area_lat)
grid["area_lon"] = grid["ca"].map(train_area_lon)
grid["area_lat"] = grid["area_lat"].fillna(grid["mean_lat"])
grid["area_lon"] = grid["area_lon"].fillna(grid["mean_lon"])

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
        grid[f"r30_{col}"] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")

loc_feats = []
for lt in tqdm(cfg.TOP_LOC_TYPES, desc="Building location-type features"):
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1)
               .reindex(full_idx, fill_value=0)
               .reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom",
              "roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]

FEAT_COLS = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
             HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT    = len(FEAT_COLS)
print(f"Features: {N_FEAT}  (temporal:{len(TEMP_COLS)} cal:{len(CAL_COLS)} "
      f"type_r:{len(TYPE_RCOLS)} misc:{len(MISC_COLS)} "
      f"hour_buckets:{len(HOUR_BUCKET_COLS)+len(HOUR_BUCKET_ROLL_COLS)} loc:{len(LOC_COLS)})")

grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# =============================================================================
# FIX-1: FINAL CHRONOLOGICAL SPLIT MASKS
# =============================================================================
train_mask = grid["date"].le(split_train)
val_mask   = grid["date"].gt(split_train) & grid["date"].le(split_val)
test_mask  = grid["date"].gt(split_val)

train_dates = all_dates[all_dates <= split_train]
val_dates   = all_dates[(all_dates > split_train) & (all_dates <= split_val)]
test_dates  = all_dates[all_dates > split_val]

n_train = len(train_dates)
n_val   = len(val_dates)

print("\n========== FINAL TEMPORAL SPLIT ==========")
print(
    f"TRAIN:      {train_dates.iloc[0].date()} → "
    f"{train_dates.iloc[-1].date()} | {len(train_dates):,} days"
)
print(
    f"VALIDATION: {val_dates.iloc[0].date()} → "
    f"{val_dates.iloc[-1].date()} | {len(val_dates):,} days"
)
print(
    f"TEST:       {test_dates.iloc[0].date()} → "
    f"{test_dates.iloc[-1].date()} | {len(test_dates):,} days"
)
print("\nGrid rows:")
print(f"  Train:      {train_mask.sum():,}")
print(f"  Validation: {val_mask.sum():,}")
print(f"  Test:       {test_mask.sum():,}")

assert train_mask.sum() == n_train * N_COMM
assert val_mask.sum()   == len(val_dates) * N_COMM
assert test_mask.sum()  == len(test_dates) * N_COMM
assert grid.loc[train_mask, "date"].max() == split_train
assert grid.loc[val_mask, "date"].min() > split_train
assert grid.loc[val_mask, "date"].max() == split_val
assert grid.loc[test_mask, "date"].min() > split_val
assert grid.loc[test_mask, "date"].max() == all_dates.iloc[-1]

print("\n✓ Temporal split masks are correctly aligned.")
print("============================================\n")

# =============================================================================
# LABELS
# =============================================================================
print("Computing labels...")

grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)
print(f"  Hotspot rate: {grid.loc[train_mask,'is_hotspot'].mean():.3f}")

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lat centroids")], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lon centroids")], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std
print(f"  Lat: mean={lat_mean:.5f}  std={lat_std:.5f}")
print(f"  Lon: mean={lon_mean:.5f}  std={lon_std:.5f}")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
cnt_cols_fine = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in TYPE_CLASSES
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]

top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)

all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER"
                for t in TYPE_CLASSES]

rolled_type, dominant_type_argmax = gpu_rolling_argmax_sum(
    grid, "ca", all_cnt_cols, window=14, min_periods=3, shift=1)
grid["dominant_type"] = dominant_type_argmax
zero_mask = rolled_type.sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

type_dist_tr = pd.Series(grid.loc[train_mask,"dominant_type"].values).value_counts().sort_index()
type_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_type"].values).value_counts().sort_index()
print(f"  Type label (10-class fine-grained, 14d rolling)  [train% / test%]:")
for k, name in enumerate(TYPE_CLASSES):
    ptr = type_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = type_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:30}: {ptr:5.1f}% / {pte:5.1f}%")

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS         = 8

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
rolled_bucket, dominant_period_argmax = gpu_rolling_argmax_sum(
    grid, "ca", RAW_BUCKET_COLS, window=14, min_periods=3, shift=1)
grid["dominant_period"] = dominant_period_argmax
zero_mask_b = rolled_bucket.sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

per_dist_tr = pd.Series(grid.loc[train_mask,"dominant_period"].values).value_counts().sort_index()
per_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_period"].values).value_counts().sort_index()
print(f"  Hour bucket label (8-class dynamic, 14d rolling)  [train% / test%]:")
for k, name in enumerate(HOUR_BUCKET_NAMES):
    ptr = per_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = per_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:10}: {ptr:5.1f}% / {pte:5.1f}%")

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area log-count means")], dtype=np.float32)
grid["cnt_z"] = grid["log_cnt"]
print(f"  Log count range: [{grid.loc[train_mask,'log_cnt'].min():.2f}, "
      f"{grid.loc[train_mask,'log_cnt'].max():.2f}]")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["wk2_now"]   = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=1,  agg="mean")
grid["wk2_prior"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=15, agg="mean")
grid["wk2_prior"] = grid["wk2_prior"].clip(1)
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)
tr_dist = grid.loc[train_mask,"trend"].value_counts().sort_index()
print(f"  Trend — Fall:{tr_dist.get(0,0)/train_mask.sum()*100:.1f}%  "
      f"Stable:{tr_dist.get(1,0)/train_mask.sum()*100:.1f}%  "
      f"Rise:{tr_dist.get(2,0)/train_mask.sum()*100:.1f}%")

print("\n========== LEAKAGE / MASK AUDIT ==========")
print("Current grid first date:", grid["date"].iloc[0])
print("Current grid last date :", grid["date"].iloc[-1])
print("\nMask counts:")
print("train_mask:", train_mask.sum())
print("val_mask:  ", val_mask.sum())
print("test_mask: ", test_mask.sum())
print("\nDates selected by masks:")
print("TRAIN:", grid.loc[train_mask, "date"].min(), "→", grid.loc[train_mask, "date"].max())
print("VAL:", grid.loc[val_mask, "date"].min(), "→", grid.loc[val_mask, "date"].max())
print("TEST:", grid.loc[test_mask, "date"].min(), "→", grid.loc[test_mask, "date"].max())
print("============================================")
print("============================================")

# =============================================================================
# TEMPORAL FEATURE LEAKAGE AUDIT
# =============================================================================
print("\n========== TEMPORAL FEATURE LEAKAGE AUDIT ==========")

TEMPORAL_AUDIT_COLS = [
    "roll7", "roll14", "roll30", "roll90",
    "std7", "std30",
    "lag1", "lag7", "lag14", "lag28",
    "wow", "mom",
    "roll_hour7", "roll_hour30",
]

for col in tqdm(TEMPORAL_AUDIT_COLS, desc="Auditing temporal features"):
    assert col in grid.columns, f"Missing temporal feature: {col}"
    assert np.isfinite(
        grid[col].to_numpy(dtype=np.float64)
    ).all(), f"NaN/Inf detected in temporal feature: {col}"

print(f"✓ {len(TEMPORAL_AUDIT_COLS)} temporal features exist and are finite.")

audit_grid = grid[["ca", "day", "crime_count"]].copy()
audit_grid = audit_grid.sort_values(["ca", "day"]).reset_index(drop=True)

audit_grid["expected_lag1"]  = audit_grid.groupby("ca")["crime_count"].shift(1).fillna(0)
audit_grid["expected_lag7"]  = audit_grid.groupby("ca")["crime_count"].shift(7).fillna(0)
audit_grid["expected_lag14"] = audit_grid.groupby("ca")["crime_count"].shift(14).fillna(0)
audit_grid["expected_lag28"] = audit_grid.groupby("ca")["crime_count"].shift(28).fillna(0)

stored = grid.sort_values(["ca", "day"]).reset_index(drop=True)

assert np.allclose(stored["lag1"].to_numpy(), audit_grid["expected_lag1"].to_numpy(), atol=1e-5), "❌ lag1 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag7"].to_numpy(), audit_grid["expected_lag7"].to_numpy(), atol=1e-5), "❌ lag7 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag14"].to_numpy(), audit_grid["expected_lag14"].to_numpy(), atol=1e-5), "❌ lag14 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag28"].to_numpy(), audit_grid["expected_lag28"].to_numpy(), atol=1e-5), "❌ lag28 temporal leakage/mismatch detected!"

print("✓ Lag features verified: current-day crime counts are excluded.")

expected_roll7 = (
    audit_grid.groupby("ca")["crime_count"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    .fillna(0)
    .to_numpy()
)
stored_roll7 = stored["roll7"].to_numpy()
mature = stored["day"].to_numpy() >= 7
assert np.allclose(stored_roll7[mature], expected_roll7[mature], atol=1e-4, rtol=1e-4), "❌ roll7 temporal leakage/mismatch detected!"
print("✓ roll7 verified: only previous 7 days are used.")

area_first = stored.groupby("ca", sort=False).head(1)
assert np.allclose(area_first["lag1"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag1!"
assert np.allclose(area_first["lag7"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag7!"
assert np.allclose(area_first["lag14"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag14!"
assert np.allclose(area_first["lag28"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag28!"
print("✓ Cross-area temporal leakage check passed.")

print("✓ Crime-type labels use lagged 14-day rolling history.")
print("✓ Hour-bucket labels use lagged 14-day rolling history.")
print("✓ Trend labels use lagged rolling windows.")

print("\n✓ TEMPORAL FEATURE LEAKAGE AUDIT PASSED")
print("====================================================")

# =============================================================================
# FIX-2: TRAINING-ONLY FEATURE NORMALIZATION (explicit train/val/test)
# =============================================================================
scaler = StandardScaler()
train_features = grid.loc[train_mask, FEAT_COLS]
scaler.fit(train_features)

grid.loc[train_mask, FEAT_COLS] = scaler.transform(grid.loc[train_mask, FEAT_COLS])
grid.loc[val_mask,   FEAT_COLS] = scaler.transform(grid.loc[val_mask,   FEAT_COLS])
grid.loc[test_mask,  FEAT_COLS] = scaler.transform(grid.loc[test_mask,  FEAT_COLS])

grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)
print("✓ StandardScaler fitted using training data only.")

feat_np    = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np     = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np     = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np     = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw    = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw    = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np    = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np     = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np   = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np   = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)

assert not np.isnan(feat_np).any(), "NaN in features!"
print(f"\nTensors OK: feat{feat_np.shape}  type{type_np.shape}  period{per_np.shape}")
print(f"  N_TYPES={N_TYPES}  N_PERIODS={N_PERIODS}")

# =============================================================================
# GRAPH EDGES  (unchanged)
# =============================================================================
def build_ei(adj, id2node):
    src,dst=[],[]
    for n,nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src,dst],dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1,N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1,N_COMM+1)],
                            dtype=torch.long).to(DEVICE)
print(f"Graphs: fine={fine_edge.shape[1]}e  coarse={coarse_edge.shape[1]}e")

# =============================================================================
# DATASET
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T=T; sl=slice(s,e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n=max(0,(e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t=idx+self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]),
                )

# =============================================================================
# FIX-3: correct absolute day-index boundaries for train/val/test CrimeDS
# =============================================================================
T=cfg.TIMESTEPS
N_TRAIN_DAYS = len(train_dates)
N_VAL_DAYS   = len(val_dates)
N_TEST_DAYS  = len(test_dates)

train_ds = CrimeDS(0, N_TRAIN_DAYS, T)
val_ds   = CrimeDS(N_TRAIN_DAYS, N_TRAIN_DAYS + N_VAL_DAYS, T)
test_ds  = CrimeDS(N_TRAIN_DAYS + N_VAL_DAYS, NUM_DAYS, T)

dl_kw=dict(batch_size=cfg.BATCH,num_workers=cfg.NUM_WORKERS,pin_memory=True,
           persistent_workers=cfg.NUM_WORKERS>0)
train_loader=DataLoader(train_ds,shuffle=True,**dl_kw)
val_loader  =DataLoader(val_ds,  shuffle=False,**dl_kw)
test_loader =DataLoader(test_ds, shuffle=False,**dl_kw)
print(f"Sequences — Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}")

# =============================================================================
# FIX-4: SEQUENCE TEMPORAL AUDIT
# =============================================================================
print("\n========== SEQUENCE TEMPORAL AUDIT ==========")
print("Training period:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("Validation period:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("Test period:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())

print("\nExpected sequence counts:")
print("  Train:", max(0, N_TRAIN_DAYS - T - 1))
print("  Val:  ", max(0, N_VAL_DAYS - T - 1))
print("  Test: ", max(0, N_TEST_DAYS - T - 1))

assert len(train_ds) == max(0, N_TRAIN_DAYS - T - 1)
assert len(val_ds)   == max(0, N_VAL_DAYS - T - 1)
assert len(test_ds)  == max(0, N_TEST_DAYS - T - 1)

print("\n✓ Sequence counts and chronological boundaries verified.")
print("===============================================\n")

print("\n========== DATASET AUDIT ==========")
print("Raw records:                    ", len(crimes))
print("Earliest date:                  ", crimes["date"].min())
print("Latest date:                    ", crimes["date"].max())
print("Unique dates:                   ", crimes["date"].nunique())
print("Cleaned grid rows:              ", len(grid))
print("Number of community areas:      ", N_COMM)
print("Number of daily graph snapshots:", NUM_DAYS)

print("\n--- EXACT SPLITS ---")
print("TRAIN:")
print("  Date range:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("  Number of days:", len(train_dates))
print("VALIDATION:")
print("  Date range:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("  Number of days:", len(val_dates))
print("TEST:")
print("  Date range:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())
print("  Number of days:", len(test_dates))

print("\n--- GRID ROW COUNTS ---")
print("Train grid rows:       ", len(train_dates) * N_COMM)
print("Validation grid rows:  ", len(val_dates) * N_COMM)
print("Test grid rows:        ", len(test_dates) * N_COMM)

print("\n--- SEQUENCES ---")
print("Train sequences:       ", len(train_ds))
print("Validation sequences:  ", len(val_ds))
print("Test sequences:        ", len(test_ds))
print("====================================\n")

# MODEL: HAST-GNN v14  (architecture IDENTICAL to v13 — no changes here)
# =============================================================================

class ARATABlock(nn.Module):
    """[N1] ARATA + DWConv1d (unchanged)."""
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H=n_heads; self.dh=d//n_heads
        self.qkv=nn.Linear(d,3*d,bias=False); self.out=nn.Linear(d,d,bias=False)
        self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)
        self.log_lam_bwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam)))
        self.log_lam_fwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam*0.5)))
        self.n_causal=n_heads//2
        self.tconv     =nn.Conv1d(d,d,kernel_size=3,padding=1,groups=d,bias=False)
        self.tconv_norm=nn.LayerNorm(d)

    def forward(self,x):
        B,T,NF,d=x.shape
        xc=x.permute(0,2,3,1).reshape(B*NF,d,T)
        xc=self.tconv(xc)
        xc=xc.reshape(B,NF,d,T).permute(0,3,1,2)
        x=self.tconv_norm(x+xc)
        x2=x.reshape(B*NF,T,d)
        Q,K,V=self.qkv(x2).chunk(3,dim=-1)
        Q=Q.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        K=K.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        V=V.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        pi=torch.arange(T,device=x.device).float().unsqueeze(1)
        pj=torch.arange(T,device=x.device).float().unsqueeze(0)
        diff=pi-pj
        lb=torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf=torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias=(-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0)+
              -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal>0:
            cm=torch.triu(torch.full((T,T),float("-inf"),device=x.device),diagonal=1)
            bias[:self.n_causal]+=cm
        attn=F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0),dim=-1)
        out=(self.drop(attn)@V).transpose(1,2).reshape(B*NF,T,d)
        return self.norm(x+self.out(out).reshape(B,NF,T,d).permute(0,2,1,3))


class CTIGBlock(nn.Module):
    """[N2] Dynamic Crime-Type Interaction Graph (unchanged)."""
    def __init__(self,H,n_types,dropout):
        super().__init__()
        self.K=n_types
        self.proj=nn.Linear(H,H)
        self.eg=nn.Sequential(nn.Linear(2*H,H),nn.GELU(),nn.Linear(H,1))
        self.op=nn.Linear(H,H); self.norm=nn.LayerNorm(H); self.drop=nn.Dropout(dropout)
    def forward(self,tr,hp):
        B,NF,H=hp.shape
        tf=self.proj(tr.unsqueeze(-1)*hp.unsqueeze(2))
        tm=tf.mean(1)
        ti=tm.unsqueeze(2).expand(-1,-1,self.K,-1)
        tj=tm.unsqueeze(1).expand(-1,self.K,-1,-1)
        ew=torch.sigmoid(self.eg(torch.cat([ti,tj],dim=-1))).squeeze(-1)
        k3=min(3,self.K-1)
        tv,tix=ew.topk(k3,dim=-1)
        es=torch.zeros_like(ew).scatter_(2,tix,tv)
        msg=(torch.einsum("bkj,bjh->bkh",es,tm).unsqueeze(1).expand(-1,NF,-1,-1))
        ht=(tr.unsqueeze(-1)*self.op(msg)).sum(2)
        return self.norm(hp+self.drop(ht))


class AHGLayer(nn.Module):
    """[N3] AHG + explicit hf residual (unchanged)."""
    def __init__(self,H,n_fine,n_coarse,heads,dropout):
        super().__init__()
        self.gf=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gc=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gate=nn.Linear(H*2,1); self.norm=nn.LayerNorm(H)
    def forward(self,hf,hc,fei,cei,c2dn):
        B,NF,H=hf.shape
        hf2=torch.stack([self.gf(hf[b],fei) for b in range(B)])
        hc2=torch.stack([self.gc(hc[b],cei) for b in range(B)])
        hcca=hc2[:,c2dn,:]
        alpha=torch.sigmoid(self.gate(torch.cat([hf2,hcca],dim=-1)))
        return self.norm(hf+alpha*hf2+(1-alpha)*hcca), hc2


class HAST_GNN(nn.Module):
    def __init__(self,n_feat,n_fine,n_coarse,H,heads,n_layers,dropout,n_types,n_periods):
        super().__init__()
        self.nf=n_fine; self.nc=n_coarse
        self.pf=nn.Sequential(nn.Linear(n_feat,H),nn.LayerNorm(H),nn.GELU())
        self.pc=nn.Sequential(nn.Linear(H,H),nn.LayerNorm(H),nn.GELU())
        self.arata=nn.ModuleList([ARATABlock(H,heads,dropout) for _ in range(n_layers)])
        self.ahg  =nn.ModuleList([AHGLayer(H,n_fine,n_coarse,heads,dropout) for _ in range(n_layers)])
        self.ctig =CTIGBlock(H,cfg.TOP_N_TYPES,dropout)
        self.mp   =nn.Sequential(nn.Linear(H,H),nn.GELU(),nn.LayerNorm(H))
        self.n_rec=3

        def _h(i,m,o):
            blk=nn.Sequential(nn.Linear(i,m),nn.GELU(),nn.Dropout(dropout),nn.Linear(m,o))
            for l in blk:
                if isinstance(l,nn.Linear): nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
            return blk

        self.hot_h         = _h(H*2, 64, 1)
        self.loc_h         = _h(H+32, 64, 2)
        self.area_lat_bias = nn.Parameter(torch.zeros(n_fine))
        self.area_lon_bias = nn.Parameter(torch.zeros(n_fine))

        self.type_h = _h(H, 128, n_types)
        self.per_h  = _h(H, 64, n_periods)

        self.area_emb      = nn.Embedding(n_fine, 32)
        self.cnt_h         = _h(H+32, 64, 1)
        self.area_log_bias = nn.Parameter(torch.zeros(n_fine))
        self.trend_h       = _h(H*2, 64, 3)

    def forward(self,xf,fei,cei):
        B,T,NF,_=xf.shape
        H=self.pf[0].out_features; DEV=xf.device
        hf=self.pf(xf)
        for a in self.arata: hf=a(hf)
        hp=hf[:,-1,:,:]
        c2dn=CA_TO_DN.to(DEV)
        hc=torch.zeros(B,self.nc,H,device=DEV)
        hc.scatter_reduce_(1,c2dn.unsqueeze(0).unsqueeze(-1).expand(B,-1,H),
                           hp,reduce="mean",include_self=True)
        hc=self.pc(hc)
        for a in self.ahg: hp,hc=a(hp,hc,fei,cei,c2dn)
        hr=hf[:,-self.n_rec:,:,:].mean(1)
        hl=hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom=self.mp(hr-hl)
        hm=torch.cat([hp,mom],dim=-1)
        trs=len(TEMP_COLS)+len(CAL_COLS)
        ntr=min(cfg.TOP_N_TYPES,xf.shape[-1]-trs)
        tr=(xf[:,-1,:,trs:trs+ntr] if ntr>0
            else torch.zeros(B,NF,cfg.TOP_N_TYPES,device=DEV))
        hp=self.ctig(tr,hp)
        area_idx=torch.arange(self.nf,device=DEV).unsqueeze(0).expand(B,-1)
        ae=self.area_emb(area_idx)
        hp_ae=torch.cat([hp,ae],dim=-1)

        hot     =self.hot_h(hm).squeeze(-1)
        loc_res =self.loc_h(hp_ae)
        lat_pred=loc_res[...,0]+self.area_lat_bias.unsqueeze(0)
        lon_pred=loc_res[...,1]+self.area_lon_bias.unsqueeze(0)
        loc     =torch.stack([lat_pred,lon_pred],dim=-1)
        typ     =self.type_h(hp)
        per     =self.per_h(hp)
        cnt     =(self.cnt_h(hp_ae).squeeze(-1)+self.area_log_bias.unsqueeze(0)).clamp(0,6)
        trd     =self.trend_h(hm)
        return hot,loc,typ,per,cnt,trd
# POINT 10.3 — COMPACT LSTM + GATv2 SPATIO-TEMPORAL BASELINE
# =============================================================================
# Same inputs (60-day window, 77 areas, N_FEAT features), same fine
# community-area graph, same train/val/test split, same targets, and it is
# scored with the SAME evaluate()/compute_loss() as HAST-GNN. This class does
# NOT touch or modify HAST_GNN above; it's a separate, smaller model trained
# and evaluated independently, purely as a stronger baseline than persistence
# / seasonal-7 for the paired comparison in Point 10.7.
class BaselineLSTMGAT(nn.Module):
    def __init__(self, n_feat, n_fine, H, heads, n_types, n_periods, dropout=0.2):
        super().__init__()
        self.nf = n_fine
        self.H = H
        self.lstm = nn.LSTM(n_feat, H, batch_first=True)
        self.gat  = GATv2Conv(H, H // heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.norm = nn.LayerNorm(H)
        self.drop = nn.Dropout(dropout)

        def _h(i, m, o):
            blk = nn.Sequential(nn.Linear(i, m), nn.GELU(), nn.Dropout(dropout), nn.Linear(m, o))
            for l in blk:
                if isinstance(l, nn.Linear):
                    nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
            return blk

        self.hot_h   = _h(H, 32, 1)
        self.loc_h   = _h(H, 32, 2)
        self.type_h  = _h(H, 64, n_types)
        self.per_h   = _h(H, 32, n_periods)
        self.cnt_h   = _h(H, 32, 1)
        self.trend_h = _h(H, 32, 3)

    def forward(self, xf, fei, cei=None):
        # xf: (B, T, NF, n_feat) — identical layout to HAST-GNN's input.
        B, T, NF, Fdim = xf.shape
        x = xf.permute(0, 2, 1, 3).reshape(B * NF, T, Fdim)
        out, _ = self.lstm(x)
        h_last = out[:, -1, :].reshape(B, NF, self.H)
        h_gat = torch.stack([self.gat(h_last[b], fei) for b in range(B)])
        h = self.norm(h_last + self.drop(h_gat))

        hot = self.hot_h(h).squeeze(-1)
        loc = self.loc_h(h)
        typ = self.type_h(h)
        per = self.per_h(h)
        cnt = self.cnt_h(h).squeeze(-1).clamp(0, 6)
        trd = self.trend_h(h)
        return hot, loc, typ, per, cnt, trd
hot_rate=grid.loc[train_mask,"is_hotspot"].mean()
pos_wt=torch.tensor((1-hot_rate)/hot_rate,dtype=torch.float32).to(DEVICE)
print(f"Hotspot pos_weight: {pos_wt.item():.3f}")

tc=np.bincount(type_np[:n_train].reshape(-1),minlength=N_TYPES).astype(float)
tc/=tc.sum()
typw=torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES),
                   dtype=torch.float32).to(DEVICE)
print(f"Type weights ({N_TYPES} classes): {typw.cpu().numpy().round(2)}")

pc=np.bincount(per_np[:n_train].reshape(-1),minlength=N_PERIODS).astype(float)
pc/=pc.sum()
perw=torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS),
                   dtype=torch.float32).to(DEVICE)
print(f"Hour bucket weights ({N_PERIODS} classes): {perw.cpu().numpy().round(2)}")

trc=np.bincount(trend_np[:n_train].reshape(-1),minlength=3).astype(float)
trc/=trc.sum()
tw=torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3),dtype=torch.float32).to(DEVICE)
print(f"Trend weights: {tw.cpu().numpy().round(2)}")
# =============================================================================
# LOSS FUNCTIONS  (unchanged)
# =============================================================================

def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls=logits.shape[-1]
    log_p=F.log_softmax(logits,dim=-1)
    p=torch.exp(log_p)
    with torch.no_grad():
        smooth_t=torch.full_like(logits,smoothing/(n_cls-1))
        smooth_t.scatter_(-1,targets.unsqueeze(-1),1.0-smoothing)
    ce_smooth=-(smooth_t*log_p).sum(-1)
    p_t=p.gather(-1,targets.unsqueeze(-1)).squeeze(-1)
    focal_w=(1.0-p_t)**gamma
    if weight is not None:
        focal_w=focal_w*weight[targets]
    focal=(focal_w*ce_smooth).mean()
    entropy=-(p*log_p).sum(-1).mean()
    return focal-entropy_reg*entropy


def compute_loss(hot_l,loc_p,typ_l,per_l,cnt_p,trd_l,
                 hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t):
    l_hot=F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1),hot_t.reshape(-1),pos_weight=pos_wt)
    valid=(lat_t.abs()>0.01)|(lon_t.abs()>0.01)
    if valid.any():
        l_loc=(F.huber_loss(loc_p[...,0][valid],lat_t[valid],delta=1.0)+
               F.huber_loss(loc_p[...,1][valid],lon_t[valid],delta=1.0))
    else:
        l_loc=torch.tensor(0.0,device=hot_l.device)

    l_typ=focal_loss_with_smooth(
        typ_l.reshape(-1,N_TYPES),typ_t.reshape(-1),
        weight=typw,gamma=cfg.FOCAL_GAMMA,
        smoothing=cfg.LABEL_SMOOTH,entropy_reg=cfg.ENTROPY_REG)

    l_per=F.cross_entropy(per_l.reshape(-1,N_PERIODS),per_t.reshape(-1),weight=perw)

    l_cnt=F.huber_loss(cnt_p.reshape(-1),cnt_t.reshape(-1),delta=0.5)
    l_trd=F.cross_entropy(trd_l.reshape(-1,3),trd_t.reshape(-1),weight=tw)

    total=1.0*l_hot+1.0*l_loc+1.5*l_typ+0.6*l_per+1.2*l_cnt+0.5*l_trd
    return total,l_hot,l_loc,l_typ,l_per,l_cnt,l_trd

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1,lo1,la2,lo2):
    R=6371.0
    dlat=np.radians(la2-la1); dlon=np.radians(lo2-lo1)
    a=np.sin(dlat/2)**2+np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a,dtype=np.float32))

# =============================================================================
# EVALUATE
# =============================================================================
# REFACTOR-1: `eval_model` lets this score ANY model with the HAST-GNN output
# signature (hot, loc, typ, per, cnt, trd), e.g. the Point 10.3 baseline or
# a Point 10.4 multi-seed model, without duplicating this function.
def evaluate(loader, eval_model=None, desc="Evaluating", show_pbar=True):
    m = eval_model if eval_model is not None else model
    m.eval()
    ht,hs=[],[]
    latt,lont,latr,lonr,latp,lonp=[],[],[],[],[],[]
    tytt,tytp=[],[]
    pert,perp=[],[]
    cntr,cntp=[],[]
    trt_,trp_=[],[]
    tot=n=0

    iterator = tqdm(loader, desc=desc, leave=False, unit="batch") if show_pbar else loader

    with torch.no_grad():
        for batch in iterator:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=m(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                      hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
            if not torch.isnan(loss):
                tot+=loss.item()*x.shape[0]; n+=x.shape[0]
                if show_pbar:
                    iterator.set_postfix(loss=f"{tot/max(1,n):.4f}")

    ht=np.array(ht,dtype=np.int32); hs=san(hs)
    pr,re,thr=precision_recall_curve(ht,hs)
    f1c=2*pr*re/(pr+re+1e-8)
    bt=float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp=(hs>=bt).astype(int)
    try: rauc=roc_auc_score(ht,hs)
    except: rauc=0.0
    hot_f1=f1_score(ht,hp,zero_division=0)

    lp_raw=san(latp)*lat_std+lat_mean; lr_raw=np.array(latr)
    op_raw=san(lonp)*lon_std+lon_mean; or_raw=np.array(lonr)
    valid=(lr_raw>40)&(or_raw<-80)
    if valid.sum()>0:
        hav=haversine_km(lr_raw[valid],or_raw[valid],lp_raw[valid],op_raw[valid])
        loc_mae=float(hav.mean()); l1=float((hav<1).mean()); l05=float((hav<0.5).mean())
    else:
        loc_mae=99.9; l1=0.0; l05=0.0

    ta=accuracy_score(np.array(tytt),np.array(tytp))
    tf=f1_score(np.array(tytt),np.array(tytp),average="weighted",zero_division=0)
    pa=accuracy_score(np.array(pert),np.array(perp))
    pf=f1_score(np.array(pert),np.array(perp),average="weighted",zero_division=0)
    cr=san(cntr); cp=san(cntp)
    cnt_corr=float(np.corrcoef(cr,cp)[0,1]) if len(cr)>1 else 0.0
    cnt_mae=float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp.clip(0,10))).mean())
    trend_acc=accuracy_score(np.array(trt_),np.array(trp_))

    loc_score=max(0,1-loc_mae/10)
    primary=np.mean([rauc,ta,pa,max(cnt_corr,0),loc_score])

    return dict(
        loss=tot/max(1,n),primary=primary,
        hot_auc=rauc,hot_f1=hot_f1,best_thr=bt,ht=ht,hs=hs,hp=hp,
        loc_mae=loc_mae,loc_1km=l1,loc_05km=l05,
        lat_p=lp_raw,lon_p=op_raw,lat_t=lr_raw,lon_t=or_raw,
        type_acc=ta,type_f1=tf,typ_t=np.array(tytt),typ_p=np.array(tytp),
        per_acc=pa,per_f1=pf,per_t=np.array(pert),per_p=np.array(perp),
        cnt_corr=cnt_corr,cnt_mae=cnt_mae,cr=cr,cp=cp,
        trend_acc=trend_acc,trt=np.array(trt_),trp=np.array(trp_),
    )

# =============================================================================

# =============================================================================
# LOAD CHECKPOINTS + EVALUATE (no training)
# =============================================================================
for _p, _what in [(PRIMARY_CKPT, "primary HAST-GNN"), (BASELINE_CKPT, "LSTM+GATv2 baseline")]:
    if not os.path.exists(_p):
        raise FileNotFoundError(f"Checkpoint for {_what} not found at: {_p}")

print(f"\nLoading primary HAST-GNN from: {PRIMARY_CKPT}")
model = HAST_GNN(N_FEAT, N_COMM, N_DIST, cfg.HIDDEN_DIM, cfg.GAT_HEADS,
                  cfg.N_LAYERS, cfg.DROPOUT, N_TYPES, N_PERIODS).to(DEVICE)
model.load_state_dict(torch.load(PRIMARY_CKPT, map_location=DEVICE))
model.eval()

print(f"Loading LSTM+GATv2 baseline from: {BASELINE_CKPT}")
baseline_model = BaselineLSTMGAT(
    N_FEAT, N_COMM, cfg.BASELINE_HIDDEN, cfg.BASELINE_HEADS, N_TYPES, N_PERIODS
).to(DEVICE)
baseline_model.load_state_dict(torch.load(BASELINE_CKPT, map_location=DEVICE))
baseline_model.eval()

print("\n"+"="*100)
print("  FINAL TEST EVAL — loaded HAST-GNN")
print("="*100)
s = evaluate(test_loader, eval_model=model, desc="Final test eval (HAST-GNN, loaded)")
print(f"HotAUC={s['hot_auc']:.4f}  LocMAE={s['loc_mae']:.2f}km  "
      f"TypeAcc={s['type_acc']:.4f}  HourAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}")

print("\n"+"="*100)
print("  FINAL TEST EVAL — loaded LSTM+GATv2 baseline")
print("="*100)
s_bl = evaluate(test_loader, eval_model=baseline_model, desc="Final test eval (LSTM+GATv2, loaded)")
print(f"HotAUC={s_bl['hot_auc']:.4f}  LocMAE={s_bl['loc_mae']:.2f}km  "
      f"TypeAcc={s_bl['type_acc']:.4f}  HourAcc={s_bl['per_acc']:.4f}  CntR={s_bl['cnt_corr']:.4f}")

baseline_cnt_flat = s_bl["cp"]   # already row-aligned to s["cr"] (same test_loader, same order)
baseline_hot_flat = s_bl["hs"]

# =============================================================================
# POINT 10.1 — PERSISTENCE BASELINE
# =============================================================================
# Predict day t using the most recently OBSERVED day (t-1).
#
# ALIGNMENT: target days must match exactly what evaluate(test_loader)
# produced in `s["cr"]`/`s["ht"]` above. CrimeDS.__len__ = max(0,(e-s)-T-1),
# and test_loader is shuffle=False, so sample idx (0-indexed, in order)
# corresponds to absolute day = test_start + T + idx, for idx in
# range(len(test_ds)). We derive the slice from len(test_ds) directly instead
# of re-deriving it from NUM_DAYS, so this can't silently drift by one row.
test_start = n_train + n_val               # == N_TRAIN_DAYS + N_VAL_DAYS
n_test_seq = len(test_ds)                  # exact row count evaluate() produced
tgt0 = test_start + T                      # absolute day of the FIRST test target

persist_cnt = log_cnt_np[tgt0 - 1 : tgt0 - 1 + n_test_seq]   # day t-1, for each target day t
persist_hot = hot_np[tgt0 - 1 : tgt0 - 1 + n_test_seq]

true_cnt = log_cnt_np[tgt0 : tgt0 + n_test_seq]
true_hot = hot_np[tgt0 : tgt0 + n_test_seq]

assert true_cnt.shape[0] * N_COMM == s["cr"].shape[0], \
    f"Persistence baseline misaligned: {true_cnt.shape[0]*N_COMM} vs {s['cr'].shape[0]}"

cnt_r = np.corrcoef(true_cnt.reshape(-1), persist_cnt.reshape(-1))[0, 1]
hot_auc_persist = roc_auc_score(true_hot.reshape(-1), persist_hot.reshape(-1))
hot_f1_persist = f1_score(true_hot.reshape(-1), (persist_hot.reshape(-1) >= 0.5).astype(int), zero_division=0)

print("\n========== POINT 10.1 — PERSISTENCE ==========")
print(f"Crime Count Pearson r: {cnt_r:.4f}")
print(f"Hotspot ROC-AUC:       {hot_auc_persist:.4f}")
print(f"Hotspot F1:            {hot_f1_persist:.4f}")

# =============================================================================
# POINT 10.2 — SEASONAL (7-DAY) BASELINE
# =============================================================================
season_cnt = log_cnt_np[tgt0 - 7 : tgt0 - 7 + n_test_seq]
season_hot = hot_np[tgt0 - 7 : tgt0 - 7 + n_test_seq]

cnt_r_7 = np.corrcoef(true_cnt.reshape(-1), season_cnt.reshape(-1))[0, 1]
hot_auc_7 = roc_auc_score(true_hot.reshape(-1), season_hot.reshape(-1))
hot_f1_7 = f1_score(true_hot.reshape(-1), (season_hot.reshape(-1) >= 0.5).astype(int), zero_division=0)

print("\n========== POINT 10.2 — 7-DAY SEASONAL ==========")
print(f"Crime Count Pearson r: {cnt_r_7:.4f}")
print(f"Hotspot ROC-AUC:       {hot_auc_7:.4f}")
print(f"Hotspot F1:            {hot_f1_7:.4f}")

# Flat, row-aligned versions of the two naive baselines, ready for 10.7.
persist_cnt_flat = persist_cnt.reshape(-1)
persist_hot_flat = persist_hot.reshape(-1)
season_cnt_flat  = season_cnt.reshape(-1)
season_hot_flat  = season_hot.reshape(-1)

# =============================================================================
# POINT 10.6 — BOOTSTRAP 95% CONFIDENCE INTERVALS
# =============================================================================
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, seed=42, desc="Bootstrapping"):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []
    for _ in tqdm(range(n_boot), desc=desc, leave=False, unit="resample"):
        idx = rng.integers(0, n, n)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            continue
    scores = np.asarray(scores)
    return (float(np.mean(scores)), float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5)))

hot_ci  = bootstrap_ci(s["ht"], s["hs"], roc_auc_score, desc="Bootstrap: hotspot AUC")
cnt_ci  = bootstrap_ci(s["cr"], s["cp"], lambda y, p: np.corrcoef(y, p)[0, 1], desc="Bootstrap: count r")
type_ci = bootstrap_ci(s["typ_t"], s["typ_p"], lambda y, p: accuracy_score(y, p), desc="Bootstrap: type acc")
hour_ci = bootstrap_ci(s["per_t"], s["per_p"], lambda y, p: accuracy_score(y, p), desc="Bootstrap: hour acc")

print("\n========== POINT 10.6 — 95% BOOTSTRAP CI (HAST-GNN v14 primary run) ==========")
print(f"Hotspot AUC : {hot_ci[0]:.4f} [{hot_ci[1]:.4f}, {hot_ci[2]:.4f}]")
print(f"Count r     : {cnt_ci[0]:.4f} [{cnt_ci[1]:.4f}, {cnt_ci[2]:.4f}]")
print(f"Type Acc    : {type_ci[0]:.4f} [{type_ci[1]:.4f}, {type_ci[2]:.4f}]")
print(f"Hour Acc    : {hour_ci[0]:.4f} [{hour_ci[1]:.4f}, {hour_ci[2]:.4f}]")

# =============================================================================
# POINT 10.7 — PAIRED BOOTSTRAP: HAST-GNN vs BASELINES
# =============================================================================
def paired_bootstrap(y_true, pred_a, pred_b, metric_fn, n_boot=2000, seed=42, desc="Paired bootstrapping"):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = []
    for _ in tqdm(range(n_boot), desc=desc, leave=False, unit="resample"):
        idx = rng.integers(0, n, n)
        try:
            a = metric_fn(y_true[idx], pred_a[idx])
            b = metric_fn(y_true[idx], pred_b[idx])
            diffs.append(a - b)
        except Exception:
            continue
    diffs = np.asarray(diffs)
    return (float(np.mean(diffs)), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5)))

# Sanity checks — every comparison array must be exactly the same length and
# ordering as s["cr"]/s["ht"] (which is what makes this a valid PAIRED test).
assert s["cr"].shape[0] == persist_cnt_flat.shape[0] == season_cnt_flat.shape[0] == baseline_cnt_flat.shape[0], \
    "Row-count mismatch between HAST-GNN test predictions and baseline arrays — do not run paired bootstrap on misaligned arrays."
assert s["ht"].shape[0] == persist_hot_flat.shape[0] == season_hot_flat.shape[0] == baseline_hot_flat.shape[0], \
    "Row-count mismatch between HAST-GNN hotspot labels and baseline hotspot arrays."

print("\n========== POINT 10.7 — PAIRED BOOTSTRAP (HAST-GNN vs baselines) ==========")

# --- Crime count (Pearson r) ---
delta_cnt_persist = paired_bootstrap(
    s["cr"], s["cp"], persist_cnt_flat, lambda y, p: np.corrcoef(y, p)[0, 1],
    desc="Paired bootstrap: count r vs persistence")
delta_cnt_season = paired_bootstrap(
    s["cr"], s["cp"], season_cnt_flat, lambda y, p: np.corrcoef(y, p)[0, 1],
    desc="Paired bootstrap: count r vs seasonal")
delta_cnt_baseline = paired_bootstrap(
    s["cr"], s["cp"], baseline_cnt_flat, lambda y, p: np.corrcoef(y, p)[0, 1],
    desc="Paired bootstrap: count r vs LSTM+GATv2")

print("HAST-GNN vs Persistence   — Count r difference: "
      f"{delta_cnt_persist[0]:.4f} [{delta_cnt_persist[1]:.4f}, {delta_cnt_persist[2]:.4f}]")
print("HAST-GNN vs 7-day Seasonal — Count r difference: "
      f"{delta_cnt_season[0]:.4f} [{delta_cnt_season[1]:.4f}, {delta_cnt_season[2]:.4f}]")
print("HAST-GNN vs LSTM+GATv2     — Count r difference: "
      f"{delta_cnt_baseline[0]:.4f} [{delta_cnt_baseline[1]:.4f}, {delta_cnt_baseline[2]:.4f}]")

# --- Hotspot ROC-AUC ---
delta_hot_persist = paired_bootstrap(s["ht"], s["hs"], persist_hot_flat, roc_auc_score,
                                      desc="Paired bootstrap: hotspot AUC vs persistence")
delta_hot_season = paired_bootstrap(s["ht"], s["hs"], season_hot_flat, roc_auc_score,
                                     desc="Paired bootstrap: hotspot AUC vs seasonal")
delta_hot_baseline = paired_bootstrap(s["ht"], s["hs"], baseline_hot_flat, roc_auc_score,
                                       desc="Paired bootstrap: hotspot AUC vs LSTM+GATv2")

print("\nHAST-GNN vs Persistence   — Hotspot AUC difference: "
      f"{delta_hot_persist[0]:.4f} [{delta_hot_persist[1]:.4f}, {delta_hot_persist[2]:.4f}]")
print("HAST-GNN vs 7-day Seasonal — Hotspot AUC difference: "
      f"{delta_hot_season[0]:.4f} [{delta_hot_season[1]:.4f}, {delta_hot_season[2]:.4f}]")
print("HAST-GNN vs LSTM+GATv2     — Hotspot AUC difference: "
      f"{delta_hot_baseline[0]:.4f} [{delta_hot_baseline[1]:.4f}, {delta_hot_baseline[2]:.4f}]")

# --- Crime Type / Hour Accuracy vs the LSTM+GATv2 baseline (naive baselines
#     don't produce type/hour predictions, so only the learned baseline is
#     comparable here) ---
delta_type_baseline = paired_bootstrap(
    s["typ_t"], s["typ_p"], s_bl["typ_p"], lambda y, p: accuracy_score(y, p),
    desc="Paired bootstrap: type acc vs LSTM+GATv2")
delta_hour_baseline = paired_bootstrap(
    s["per_t"], s["per_p"], s_bl["per_p"], lambda y, p: accuracy_score(y, p),
    desc="Paired bootstrap: hour acc vs LSTM+GATv2")

print("\nHAST-GNN vs LSTM+GATv2 — Type Acc difference: "
      f"{delta_type_baseline[0]:.4f} [{delta_type_baseline[1]:.4f}, {delta_type_baseline[2]:.4f}]")
print("HAST-GNN vs LSTM+GATv2 — Hour Acc difference: "
      f"{delta_hour_baseline[0]:.4f} [{delta_hour_baseline[1]:.4f}, {delta_hour_baseline[2]:.4f}]")

print("\n✅ Point 10 (baselines, multi-seed, bootstrap CIs, paired comparison) complete.")

cuDF GPU feature engineering: ENABLED
Device: cuda
HAST-GNN v14 | CrimeType=10-class | CrimeHour=8-bucket | split/scaler/dataset fixes | GPU rolling feats

Loading: /kaggle/working/Crimes_2001_to_Present.csv
Reading individual crime events (8.3M rows)...


Reading CSV chunks: 0chunk [00:00, ?chunk/s]

  Chunk 1: 988,558 rows → 108,560 area-days
  Chunk 2: 981,239 rows → 109,260 area-days
  Chunk 3: 986,295 rows → 99,874 area-days
  Chunk 4: 997,020 rows → 81,668 area-days
  Chunk 5: 987,125 rows → 68,734 area-days
  Chunk 6: 993,794 rows → 63,518 area-days
  Chunk 7: 993,241 rows → 60,356 area-days
  Chunk 8: 758,808 rows → 47,635 area-days
  Chunk 9: 4,322 rows → 2,360 area-days
Combining all chunks...
Temporal split definitions: Train <= 2018-03-30 | Validation <= 2021-12-09 | Test > 2021-12-09

✓ Community areas: 77  |  Days: 8998
  Date range: 2001-01-01 → 2025-08-20
  Total crime events: 7,690,402

Crime types (10): ['ASSAULT', 'BATTERY', 'BURGLARY', 'CRIMINAL DAMAGE', 'DECEPTIVE PRACTICE', 'MOTOR VEHICLE THEFT', 'NARCOTICS', 'OTHER OFFENSE', 'THEFT', 'OTHER']

Computing features...


Building location-type features:   0%|          | 0/15 [00:00<?, ?it/s]

Features: 88  (temporal:14 cal:11 type_r:18 misc:6 hour_buckets:24 loc:15)

========== FINAL TEMPORAL SPLIT ==========
TRAIN:      2001-01-01 → 2018-03-30 | 6,298 days
VALIDATION: 2018-03-31 → 2021-12-09 | 1,350 days
TEST:       2021-12-10 → 2025-08-20 | 1,350 days

Grid rows:
  Train:      484,946
  Validation: 103,950
  Test:       103,950

✓ Temporal split masks are correctly aligned.

Computing labels...
  Hotspot rate: 0.247


...

Paired bootstrap: hotspot AUC vs LSTM+GATv2:   0%|          | 0/2000 [00:00<?, ?resample/s]


HAST-GNN vs Persistence   — Hotspot AUC difference: 0.1243 [0.1215, 0.1270]
HAST-GNN vs 7-day Seasonal — Hotspot AUC difference: 0.1278 [0.1249, 0.1305]
HAST-GNN vs LSTM+GATv2     — Hotspot AUC difference: -0.0110 [-0.0118, -0.0101]


Paired bootstrap: type acc vs LSTM+GATv2:   0%|          | 0/2000 [00:00<?, ?resample/s]

Paired bootstrap: hour acc vs LSTM+GATv2:   0%|          | 0/2000 [00:00<?, ?resample/s]


HAST-GNN vs LSTM+GATv2 — Type Acc difference: -0.0794 [-0.0822, -0.0766]
HAST-GNN vs LSTM+GATv2 — Hour Acc difference: -0.1799 [-0.1830, -0.1764]

✅ Point 10 (baselines, multi-seed, bootstrap CIs, paired comparison) complete.


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("Crimes_2001_to_Present.csv")

# Count unique crime types
num_crime_types = df["Primary Type"].nunique()

print("Total unique crime types:", num_crime_types)

# Show all crime types
crime_types = sorted(df["Primary Type"].unique())

print("\nCrime Types:\n")
for i, crime in enumerate(crime_types, 1):
    print(f"{i}. {crime}")

# test new 2

In [ ]:
# =============================================================================
# HAST-GNN — Chicago Crime Prediction  v14
#
# INHERITS v13 fully:
#   Architecture:  ARATA + DWConv1d | CTIG | AHG+residual | MAMTO
#   Features:      hour-bucket feats, rolling type counts, loc-type feats
#   v13 changes:   CrimeType -> 10 fine-grained classes, CrimeHour -> 8 buckets
#
# CHANGES in v14 (this pass — preprocessing/pipeline fixes only, no architecture
# or loss changes):
#
#   FIX-1  train_mask referenced before assignment (real bug in v13)
#   FIX-2  Explicit, non-ambiguous StandardScaler fit/transform on train/val/test
#   FIX-3  CrimeDS split boundary indices fixed (N_TRAIN_DAYS/N_VAL_DAYS/N_TEST_DAYS)
#   FIX-4  Sequence-level temporal audit with hard assertions
#   GPU-1  Rolling-feature block runs on cuDF when available, pandas fallback
#   GPU-2  Vectorized precompute + native agg for the daily-grid build
#
# CHANGES in this pass (review-doc "Point 10" additions):
#
#   REFACTOR-1  evaluate() now accepts an optional `eval_model` argument so it
#     can score any model (HAST-GNN or a baseline) that returns the same
#     6-tuple (hot, loc, typ, per, cnt, trd). Falls back to the global `model`
#     when eval_model is omitted, so all existing call sites keep working
#     unmodified.
#
#   REFACTOR-2  The main training loop is now the reusable function
#     `train_hast_gnn(...)`. The primary run below calls it once; Point 10.4
#     (multi-seed) and Point 10.3 (baseline) call the SAME function instead of
#     duplicating the 120-epoch loop. Behavior/printed metrics for the primary
#     run are unchanged.
#
#   10.1  Persistence baseline (lag-1 "predict tomorrow = today")
#   10.2  Seasonal 7-day baseline ("predict day t = day t-7")
#   10.3  Compact LSTM + GATv2 spatio-temporal baseline, same inputs/graph/
#         split/targets/loss, trained and evaluated separately from HAST-GNN.
#   10.4  Multi-seed HAST-GNN runs (reuses train_hast_gnn, no code duplication)
#   10.5  Multi-seed mean +/- SD summary
#   10.6  Bootstrap 95% CIs for the single best HAST-GNN run
#   10.7  Paired bootstrap: HAST-GNN vs persistence / seasonal / LSTM-GAT
#
#   ALIGNMENT FIX (flagged by the review doc as something to double check
#   before running 10.7): the doc's example 10.1/10.2 code slices
#   log_cnt_np[test_start+T : NUM_DAYS], which is off by one relative to the
#   actual number of test sequences produced by CrimeDS. CrimeDS.__len__ is
#   max(0, (e-s)-T-1) -- note the extra "-1" -- so the last usable target day
#   is NUM_DAYS-2, not NUM_DAYS-1. Using NUM_DAYS as the upper bound gives the
#   baseline arrays one MORE row than s["cr"]/s["ht"] coming out of
#   evaluate(test_loader), which would silently misalign every paired
#   comparison in 10.7. Below, the baseline target window is derived directly
#   from len(test_ds) instead of hand re-deriving the slice bounds, so it is
#   guaranteed to match the HAST-GNN test predictions row-for-row.
#
#   PROGRESS-1 (this pass): tqdm progress bars added throughout so long,
#   quiet stretches (verbose=False multi-seed / baseline runs) still show
#   live progress instead of going silent for the whole run.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    roc_auc_score, precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# GPU-1: optional cuDF backend for the rolling-feature block
# =============================================================================
HAS_CUDF = False
_cudf_err = None
if torch.cuda.is_available():
    try:
        import cudf
        import cupy as cp
        _probe = cudf.DataFrame({"a": [1, 2, 3]})
        _ = _probe.groupby("a").size()
        cp.array([1.0]).sum()
        HAS_CUDF = True
    except Exception as e:
        HAS_CUDF = False
        _cudf_err = e
if HAS_CUDF:
    print("cuDF GPU feature engineering: ENABLED")
else:
    print("cuDF GPU feature engineering: DISABLED (pandas fallback)")
    if _cudf_err is not None:
        print(f"  cudf/cupy import/probe failed ({type(_cudf_err).__name__}: {_cudf_err})")
        print("  This is usually a broken cupy/cudf install, not a code bug — common causes:")
        print("    - mismatched cupy-cudaXX wheel vs the container's CUDA version")
        print("    - a stray local file/folder named cupy.py or cudf.py shadowing the real package")
        print("    - a half-completed `pip install`/kernel restart needed")
        print("  Falling back to pandas; the pipeline will still run correctly, just on CPU for this block.")


def gpu_shift_rolling(df, group_col, value_col, window, min_periods, shift=1,
                       agg="mean"):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values,
                               value_col: df[value_col].values})
        gdf["_shifted"] = gdf.groupby(group_col)[value_col].shift(shift)
        roll = gdf.groupby(group_col)["_shifted"].rolling(window, min_periods=min_periods)
        if agg == "mean":
            out = roll.mean()
        elif agg == "sum":
            out = roll.sum()
        elif agg == "std":
            out = roll.std()
        else:
            raise ValueError(agg)
        out = out.reset_index(drop=True)
        result = out.to_pandas().values.astype(np.float32)
        if agg == "std":
            result = np.nan_to_num(result, nan=0.0)
        return result
    else:
        if agg == "mean":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).mean())
        elif agg == "sum":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).sum())
        elif agg == "std":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).std().fillna(0))
        else:
            raise ValueError(agg)
        return s.values.astype(np.float32)


def gpu_shift(df, group_col, value_col, lag):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values, value_col: df[value_col].values})
        out = gdf.groupby(group_col)[value_col].shift(lag)
        return out.reset_index(drop=True).to_pandas().fillna(0).values.astype(np.float32)
    else:
        return df.groupby(group_col)[value_col].transform(lambda x: x.shift(lag).fillna(0)).values.astype(np.float32)


def gpu_build_daily_grid(crimes_df, group_cols, agg_dict, has_cudf):
    needed_cols = list(group_cols) + list(agg_dict.keys())
    sub = crimes_df[needed_cols]

    if has_cudf:
        gdf = cudf.DataFrame.from_pandas(sub)
        out = gdf.groupby(group_cols, as_index=False).agg(agg_dict)
        out = out.to_pandas()
    else:
        out = sub.groupby(group_cols, as_index=False).agg(agg_dict)

    new_cols = []
    for c in out.columns:
        if isinstance(c, tuple):
            base, aggname = c
            new_cols.append(base if aggname in ("", None) else f"{base}__{aggname}")
        else:
            new_cols.append(c)
    out.columns = new_cols
    return out


def gpu_rolling_argmax_sum(df, group_col, value_cols, window, min_periods, shift=1):
    rolled = np.zeros((len(df), len(value_cols)), dtype=np.float32)
    for k, col in enumerate(value_cols):
        r = gpu_shift_rolling(df, group_col, col, window, min_periods, shift=shift, agg="sum")
        nan_mask = np.isnan(r)
        if nan_mask.any():
            exp = df.groupby(group_col)[col].transform(
                lambda x: x.shift(shift).expanding(min_periods=1).sum()).values
            r = np.where(nan_mask, exp, r)
        rolled[:, k] = np.nan_to_num(r, nan=0.0)
    argmax = np.argmax(rolled, axis=1).astype(np.int64)
    return rolled, argmax


# =============================================================================
# CONFIG
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9          # top-9 raw types + "OTHER" = 10 classes
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

    # Point 10.3 — compact baseline hyperparams (kept separate from HAST-GNN)
    BASELINE_HIDDEN   = 64
    BASELINE_HEADS    = 4
    BASELINE_EPOCHS   = 60
    BASELINE_PATIENCE = 15
    BASELINE_LR       = 3e-4

    # Point 10.4 — multi-seed experiment
    MULTI_SEEDS = [42, 123, 2024]

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("HAST-GNN v14 | CrimeType=10-class | CrimeHour=8-bucket | split/scaler/dataset fixes | GPU rolling feats")

# =============================================================================
# ADJACENCY GRAPHS  (unchanged)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# LOAD DATA  (unchanged — one-time pass with arbitrary lambdas, stays on pandas)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
print("Reading individual crime events (8.3M rows)...")

chunk_dfs = []
for i, chunk in enumerate(tqdm(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"]),
        desc="Reading CSV chunks", unit="chunk")):

    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    ad = chunk.groupby(["date","ca"]).size()
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows → {len(ad):,} area-days")

print("Combining all chunks...")
crimes = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

# =============================================================================
# TEMPORAL SPLIT DEFINITIONS
# =============================================================================
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")

print(
    f"Temporal split definitions: "
    f"Train <= {split_train.date()} | "
    f"Validation <= {split_val.date()} | "
    f"Test > {split_val.date()}"
)

print(f"\n✓ Community areas: {crimes['ca'].nunique()}  |  Days: {NUM_DAYS}")
print(f"  Date range: {all_dates.iloc[0].date()} → {all_dates.iloc[-1].date()}")
print(f"  Total crime events: {len(crimes):,}")

type_counts  = crimes["Primary Type"].value_counts()
top_types    = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"\nCrime types ({N_TYPES}): {TYPE_CLASSES}")

# =============================================================================
# BUILD DAILY AGGREGATED GRID
# =============================================================================
print("\nComputing features...")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]

crimes["_one"]     = np.int8(1)
crimes["is_eve"]   = (crimes["hour"] >= 18).astype(np.int8)
crimes["is_night"] = (crimes["hour"] < 6).astype(np.int8)
crimes["arrest_i"] = crimes["Arrest"].astype(bool).astype(np.int8)

type_onehot_cols = []
for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    crimes[col] = (crimes["Primary Type"] == t).astype(np.int8)
    type_onehot_cols.append(col)

for s, e in HOUR_BUCKETS:
    col = f"hour_{s}_{e}"
    crimes[col] = ((crimes["hour"] >= s) & (crimes["hour"] < e)).astype(np.int8)

agg_dict = {
    "_one":     ["sum"],
    "hour":     ["mean"],
    "is_eve":   ["sum"],
    "is_night": ["sum"],
    "Latitude":  ["mean", "std"],
    "Longitude": ["mean", "std"],
    "arrest_i": ["mean"],
}
for col in type_onehot_cols:
    agg_dict[col] = ["sum"]
for col in HOUR_BUCKET_COLS:
    agg_dict[col] = ["sum"]

grid = gpu_build_daily_grid(crimes, ["date","ca"], agg_dict, HAS_CUDF)

rename_map = {
    "_one__sum": "crime_count",
    "hour__mean": "avg_hour",
    "is_eve__sum": "eve_cnt",
    "is_night__sum": "night_cnt",
    "Latitude__mean": "mean_lat",
    "Latitude__std": "std_lat",
    "Longitude__mean": "mean_lon",
    "Longitude__std": "std_lon",
    "arrest_i__mean": "arrest_rate",
}
for col in type_onehot_cols:
    rename_map[f"{col}__sum"] = col
for col in HOUR_BUCKET_COLS:
    rename_map[f"{col}__sum"] = col
grid = grid.rename(columns=rename_map)

crimes.drop(columns=["_one","is_eve","is_night","arrest_i"] + type_onehot_cols + HOUR_BUCKET_COLS,
            inplace=True)

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)

RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)

for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"])
            .reindex(full_idx, fill_value=0)
            .reset_index()
            .sort_values(["date","ca"])
            .reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

# ── GPU-1: rolling temporal features on crime_count ───────────────────────────
grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["roll7"]  = gpu_shift_rolling(grid, "ca", "crime_count", 7,  1, agg="mean")
grid["roll14"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 1, agg="mean")
grid["roll30"] = gpu_shift_rolling(grid, "ca", "crime_count", 30, 1, agg="mean")
grid["roll90"] = gpu_shift_rolling(grid, "ca", "crime_count", 90, 1, agg="mean")
grid["std7"]   = gpu_shift_rolling(grid, "ca", "crime_count", 7,  2, agg="std")
grid["std30"]  = gpu_shift_rolling(grid, "ca", "crime_count", 30, 2, agg="std")
grid["lag1"]   = gpu_shift(grid, "ca", "crime_count", 1)
grid["lag7"]   = gpu_shift(grid, "ca", "crime_count", 7)
grid["lag14"]  = gpu_shift(grid, "ca", "crime_count", 14)
grid["lag28"]  = gpu_shift(grid, "ca", "crime_count", 28)
grid["wow"] = (
    grid.groupby("ca")["roll7"]
        .transform(lambda x: (x / x.shift(7).clip(lower=1)).fillna(1))
)
grid["mom"] = (
    grid.groupby("ca")["roll14"]
        .transform(lambda x: (x / x.shift(14).clip(lower=1)).fillna(1))
)
grid["roll_hour7"]  = gpu_shift_rolling(grid, "ca", "avg_hour", 7,  1, agg="mean")
grid["roll_hour30"] = gpu_shift_rolling(grid, "ca", "avg_hour", 30, 1, agg="mean")

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col  = f"r7_{col}"
    r30_col = f"r30_{col}"
    grid[r7_col]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
    grid[r30_col] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)

# =============================================================================
# TRAINING-ONLY GEOGRAPHIC AREA STATISTICS
# =============================================================================
train_rows_geo = grid["date"] <= split_train

train_area_lat = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lat"]
        .mean()
)
train_area_lon = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lon"]
        .mean()
)
grid["area_lat"] = grid["ca"].map(train_area_lat)
grid["area_lon"] = grid["ca"].map(train_area_lon)
grid["area_lat"] = grid["area_lat"].fillna(grid["mean_lat"])
grid["area_lon"] = grid["area_lon"].fillna(grid["mean_lon"])

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
        grid[f"r30_{col}"] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")

loc_feats = []
for lt in tqdm(cfg.TOP_LOC_TYPES, desc="Building location-type features"):
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1)
               .reindex(full_idx, fill_value=0)
               .reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom",
              "roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]

FEAT_COLS = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
             HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT    = len(FEAT_COLS)
print(f"Features: {N_FEAT}  (temporal:{len(TEMP_COLS)} cal:{len(CAL_COLS)} "
      f"type_r:{len(TYPE_RCOLS)} misc:{len(MISC_COLS)} "
      f"hour_buckets:{len(HOUR_BUCKET_COLS)+len(HOUR_BUCKET_ROLL_COLS)} loc:{len(LOC_COLS)})")

grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# =============================================================================
# FIX-1: FINAL CHRONOLOGICAL SPLIT MASKS
# =============================================================================
train_mask = grid["date"].le(split_train)
val_mask   = grid["date"].gt(split_train) & grid["date"].le(split_val)
test_mask  = grid["date"].gt(split_val)

train_dates = all_dates[all_dates <= split_train]
val_dates   = all_dates[(all_dates > split_train) & (all_dates <= split_val)]
test_dates  = all_dates[all_dates > split_val]

n_train = len(train_dates)
n_val   = len(val_dates)

print("\n========== FINAL TEMPORAL SPLIT ==========")
print(
    f"TRAIN:      {train_dates.iloc[0].date()} → "
    f"{train_dates.iloc[-1].date()} | {len(train_dates):,} days"
)
print(
    f"VALIDATION: {val_dates.iloc[0].date()} → "
    f"{val_dates.iloc[-1].date()} | {len(val_dates):,} days"
)
print(
    f"TEST:       {test_dates.iloc[0].date()} → "
    f"{test_dates.iloc[-1].date()} | {len(test_dates):,} days"
)
print("\nGrid rows:")
print(f"  Train:      {train_mask.sum():,}")
print(f"  Validation: {val_mask.sum():,}")
print(f"  Test:       {test_mask.sum():,}")

assert train_mask.sum() == n_train * N_COMM
assert val_mask.sum()   == len(val_dates) * N_COMM
assert test_mask.sum()  == len(test_dates) * N_COMM
assert grid.loc[train_mask, "date"].max() == split_train
assert grid.loc[val_mask, "date"].min() > split_train
assert grid.loc[val_mask, "date"].max() == split_val
assert grid.loc[test_mask, "date"].min() > split_val
assert grid.loc[test_mask, "date"].max() == all_dates.iloc[-1]

print("\n✓ Temporal split masks are correctly aligned.")
print("============================================\n")

# =============================================================================
# LABELS
# =============================================================================
print("Computing labels...")

grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)
print(f"  Hotspot rate: {grid.loc[train_mask,'is_hotspot'].mean():.3f}")

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lat centroids")], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lon centroids")], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std
print(f"  Lat: mean={lat_mean:.5f}  std={lat_std:.5f}")
print(f"  Lon: mean={lon_mean:.5f}  std={lon_std:.5f}")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
cnt_cols_fine = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in TYPE_CLASSES
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]

top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)

all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER"
                for t in TYPE_CLASSES]

rolled_type, dominant_type_argmax = gpu_rolling_argmax_sum(
    grid, "ca", all_cnt_cols, window=14, min_periods=3, shift=1)
grid["dominant_type"] = dominant_type_argmax
zero_mask = rolled_type.sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

type_dist_tr = pd.Series(grid.loc[train_mask,"dominant_type"].values).value_counts().sort_index()
type_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_type"].values).value_counts().sort_index()
print(f"  Type label (10-class fine-grained, 14d rolling)  [train% / test%]:")
for k, name in enumerate(TYPE_CLASSES):
    ptr = type_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = type_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:30}: {ptr:5.1f}% / {pte:5.1f}%")

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS         = 8

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
rolled_bucket, dominant_period_argmax = gpu_rolling_argmax_sum(
    grid, "ca", RAW_BUCKET_COLS, window=14, min_periods=3, shift=1)
grid["dominant_period"] = dominant_period_argmax
zero_mask_b = rolled_bucket.sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

per_dist_tr = pd.Series(grid.loc[train_mask,"dominant_period"].values).value_counts().sort_index()
per_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_period"].values).value_counts().sort_index()
print(f"  Hour bucket label (8-class dynamic, 14d rolling)  [train% / test%]:")
for k, name in enumerate(HOUR_BUCKET_NAMES):
    ptr = per_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = per_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:10}: {ptr:5.1f}% / {pte:5.1f}%")

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area log-count means")], dtype=np.float32)
grid["cnt_z"] = grid["log_cnt"]
print(f"  Log count range: [{grid.loc[train_mask,'log_cnt'].min():.2f}, "
      f"{grid.loc[train_mask,'log_cnt'].max():.2f}]")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["wk2_now"]   = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=1,  agg="mean")
grid["wk2_prior"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=15, agg="mean")
grid["wk2_prior"] = grid["wk2_prior"].clip(1)
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)
tr_dist = grid.loc[train_mask,"trend"].value_counts().sort_index()
print(f"  Trend — Fall:{tr_dist.get(0,0)/train_mask.sum()*100:.1f}%  "
      f"Stable:{tr_dist.get(1,0)/train_mask.sum()*100:.1f}%  "
      f"Rise:{tr_dist.get(2,0)/train_mask.sum()*100:.1f}%")

print("\n========== LEAKAGE / MASK AUDIT ==========")
print("Current grid first date:", grid["date"].iloc[0])
print("Current grid last date :", grid["date"].iloc[-1])
print("\nMask counts:")
print("train_mask:", train_mask.sum())
print("val_mask:  ", val_mask.sum())
print("test_mask: ", test_mask.sum())
print("\nDates selected by masks:")
print("TRAIN:", grid.loc[train_mask, "date"].min(), "→", grid.loc[train_mask, "date"].max())
print("VAL:", grid.loc[val_mask, "date"].min(), "→", grid.loc[val_mask, "date"].max())
print("TEST:", grid.loc[test_mask, "date"].min(), "→", grid.loc[test_mask, "date"].max())
print("============================================")
print("============================================")

# =============================================================================
# TEMPORAL FEATURE LEAKAGE AUDIT
# =============================================================================
print("\n========== TEMPORAL FEATURE LEAKAGE AUDIT ==========")

TEMPORAL_AUDIT_COLS = [
    "roll7", "roll14", "roll30", "roll90",
    "std7", "std30",
    "lag1", "lag7", "lag14", "lag28",
    "wow", "mom",
    "roll_hour7", "roll_hour30",
]

for col in tqdm(TEMPORAL_AUDIT_COLS, desc="Auditing temporal features"):
    assert col in grid.columns, f"Missing temporal feature: {col}"
    assert np.isfinite(
        grid[col].to_numpy(dtype=np.float64)
    ).all(), f"NaN/Inf detected in temporal feature: {col}"

print(f"✓ {len(TEMPORAL_AUDIT_COLS)} temporal features exist and are finite.")

audit_grid = grid[["ca", "day", "crime_count"]].copy()
audit_grid = audit_grid.sort_values(["ca", "day"]).reset_index(drop=True)

audit_grid["expected_lag1"]  = audit_grid.groupby("ca")["crime_count"].shift(1).fillna(0)
audit_grid["expected_lag7"]  = audit_grid.groupby("ca")["crime_count"].shift(7).fillna(0)
audit_grid["expected_lag14"] = audit_grid.groupby("ca")["crime_count"].shift(14).fillna(0)
audit_grid["expected_lag28"] = audit_grid.groupby("ca")["crime_count"].shift(28).fillna(0)

stored = grid.sort_values(["ca", "day"]).reset_index(drop=True)

assert np.allclose(stored["lag1"].to_numpy(), audit_grid["expected_lag1"].to_numpy(), atol=1e-5), "❌ lag1 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag7"].to_numpy(), audit_grid["expected_lag7"].to_numpy(), atol=1e-5), "❌ lag7 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag14"].to_numpy(), audit_grid["expected_lag14"].to_numpy(), atol=1e-5), "❌ lag14 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag28"].to_numpy(), audit_grid["expected_lag28"].to_numpy(), atol=1e-5), "❌ lag28 temporal leakage/mismatch detected!"

print("✓ Lag features verified: current-day crime counts are excluded.")

expected_roll7 = (
    audit_grid.groupby("ca")["crime_count"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    .fillna(0)
    .to_numpy()
)
stored_roll7 = stored["roll7"].to_numpy()
mature = stored["day"].to_numpy() >= 7
assert np.allclose(stored_roll7[mature], expected_roll7[mature], atol=1e-4, rtol=1e-4), "❌ roll7 temporal leakage/mismatch detected!"
print("✓ roll7 verified: only previous 7 days are used.")

area_first = stored.groupby("ca", sort=False).head(1)
assert np.allclose(area_first["lag1"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag1!"
assert np.allclose(area_first["lag7"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag7!"
assert np.allclose(area_first["lag14"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag14!"
assert np.allclose(area_first["lag28"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag28!"
print("✓ Cross-area temporal leakage check passed.")

print("✓ Crime-type labels use lagged 14-day rolling history.")
print("✓ Hour-bucket labels use lagged 14-day rolling history.")
print("✓ Trend labels use lagged rolling windows.")

print("\n✓ TEMPORAL FEATURE LEAKAGE AUDIT PASSED")
print("====================================================")

# =============================================================================
# FIX-2: TRAINING-ONLY FEATURE NORMALIZATION (explicit train/val/test)
# =============================================================================
scaler = StandardScaler()
train_features = grid.loc[train_mask, FEAT_COLS]
scaler.fit(train_features)

grid.loc[train_mask, FEAT_COLS] = scaler.transform(grid.loc[train_mask, FEAT_COLS])
grid.loc[val_mask,   FEAT_COLS] = scaler.transform(grid.loc[val_mask,   FEAT_COLS])
grid.loc[test_mask,  FEAT_COLS] = scaler.transform(grid.loc[test_mask,  FEAT_COLS])

grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)
print("✓ StandardScaler fitted using training data only.")

feat_np    = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np     = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np     = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np     = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw    = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw    = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np    = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np     = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np   = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np   = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)

assert not np.isnan(feat_np).any(), "NaN in features!"
print(f"\nTensors OK: feat{feat_np.shape}  type{type_np.shape}  period{per_np.shape}")
print(f"  N_TYPES={N_TYPES}  N_PERIODS={N_PERIODS}")

# =============================================================================
# GRAPH EDGES  (unchanged)
# =============================================================================
def build_ei(adj, id2node):
    src,dst=[],[]
    for n,nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src,dst],dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1,N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1,N_COMM+1)],
                            dtype=torch.long).to(DEVICE)
print(f"Graphs: fine={fine_edge.shape[1]}e  coarse={coarse_edge.shape[1]}e")

# =============================================================================
# DATASET
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T=T; sl=slice(s,e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n=max(0,(e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t=idx+self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]),
                )

# =============================================================================
# FIX-3: correct absolute day-index boundaries for train/val/test CrimeDS
# =============================================================================
T=cfg.TIMESTEPS
N_TRAIN_DAYS = len(train_dates)
N_VAL_DAYS   = len(val_dates)
N_TEST_DAYS  = len(test_dates)

train_ds = CrimeDS(0, N_TRAIN_DAYS, T)
val_ds   = CrimeDS(N_TRAIN_DAYS, N_TRAIN_DAYS + N_VAL_DAYS, T)
test_ds  = CrimeDS(N_TRAIN_DAYS + N_VAL_DAYS, NUM_DAYS, T)

dl_kw=dict(batch_size=cfg.BATCH,num_workers=cfg.NUM_WORKERS,pin_memory=True,
           persistent_workers=cfg.NUM_WORKERS>0)
train_loader=DataLoader(train_ds,shuffle=True,**dl_kw)
val_loader  =DataLoader(val_ds,  shuffle=False,**dl_kw)
test_loader =DataLoader(test_ds, shuffle=False,**dl_kw)
print(f"Sequences — Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}")

# =============================================================================
# FIX-4: SEQUENCE TEMPORAL AUDIT
# =============================================================================
print("\n========== SEQUENCE TEMPORAL AUDIT ==========")
print("Training period:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("Validation period:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("Test period:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())

print("\nExpected sequence counts:")
print("  Train:", max(0, N_TRAIN_DAYS - T - 1))
print("  Val:  ", max(0, N_VAL_DAYS - T - 1))
print("  Test: ", max(0, N_TEST_DAYS - T - 1))

assert len(train_ds) == max(0, N_TRAIN_DAYS - T - 1)
assert len(val_ds)   == max(0, N_VAL_DAYS - T - 1)
assert len(test_ds)  == max(0, N_TEST_DAYS - T - 1)

print("\n✓ Sequence counts and chronological boundaries verified.")
print("===============================================\n")

print("\n========== DATASET AUDIT ==========")
print("Raw records:                    ", len(crimes))
print("Earliest date:                  ", crimes["date"].min())
print("Latest date:                    ", crimes["date"].max())
print("Unique dates:                   ", crimes["date"].nunique())
print("Cleaned grid rows:              ", len(grid))
print("Number of community areas:      ", N_COMM)
print("Number of daily graph snapshots:", NUM_DAYS)

print("\n--- EXACT SPLITS ---")
print("TRAIN:")
print("  Date range:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("  Number of days:", len(train_dates))
print("VALIDATION:")
print("  Date range:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("  Number of days:", len(val_dates))
print("TEST:")
print("  Date range:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())
print("  Number of days:", len(test_dates))

print("\n--- GRID ROW COUNTS ---")
print("Train grid rows:       ", len(train_dates) * N_COMM)
print("Validation grid rows:  ", len(val_dates) * N_COMM)
print("Test grid rows:        ", len(test_dates) * N_COMM)

print("\n--- SEQUENCES ---")
print("Train sequences:       ", len(train_ds))
print("Validation sequences:  ", len(val_ds))
print("Test sequences:        ", len(test_ds))
print("====================================\n")

# =============================================================================
# MODEL: HAST-GNN v14  (architecture IDENTICAL to v13 — no changes here)
# =============================================================================

class ARATABlock(nn.Module):
    """[N1] ARATA + DWConv1d (unchanged)."""
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H=n_heads; self.dh=d//n_heads
        self.qkv=nn.Linear(d,3*d,bias=False); self.out=nn.Linear(d,d,bias=False)
        self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)
        self.log_lam_bwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam)))
        self.log_lam_fwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam*0.5)))
        self.n_causal=n_heads//2
        self.tconv     =nn.Conv1d(d,d,kernel_size=3,padding=1,groups=d,bias=False)
        self.tconv_norm=nn.LayerNorm(d)

    def forward(self,x):
        B,T,NF,d=x.shape
        xc=x.permute(0,2,3,1).reshape(B*NF,d,T)
        xc=self.tconv(xc)
        xc=xc.reshape(B,NF,d,T).permute(0,3,1,2)
        x=self.tconv_norm(x+xc)
        x2=x.reshape(B*NF,T,d)
        Q,K,V=self.qkv(x2).chunk(3,dim=-1)
        Q=Q.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        K=K.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        V=V.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        pi=torch.arange(T,device=x.device).float().unsqueeze(1)
        pj=torch.arange(T,device=x.device).float().unsqueeze(0)
        diff=pi-pj
        lb=torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf=torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias=(-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0)+
              -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal>0:
            cm=torch.triu(torch.full((T,T),float("-inf"),device=x.device),diagonal=1)
            bias[:self.n_causal]+=cm
        attn=F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0),dim=-1)
        out=(self.drop(attn)@V).transpose(1,2).reshape(B*NF,T,d)
        return self.norm(x+self.out(out).reshape(B,NF,T,d).permute(0,2,1,3))


class CTIGBlock(nn.Module):
    """[N2] Dynamic Crime-Type Interaction Graph (unchanged)."""
    def __init__(self,H,n_types,dropout):
        super().__init__()
        self.K=n_types
        self.proj=nn.Linear(H,H)
        self.eg=nn.Sequential(nn.Linear(2*H,H),nn.GELU(),nn.Linear(H,1))
        self.op=nn.Linear(H,H); self.norm=nn.LayerNorm(H); self.drop=nn.Dropout(dropout)
    def forward(self,tr,hp):
        B,NF,H=hp.shape
        tf=self.proj(tr.unsqueeze(-1)*hp.unsqueeze(2))
        tm=tf.mean(1)
        ti=tm.unsqueeze(2).expand(-1,-1,self.K,-1)
        tj=tm.unsqueeze(1).expand(-1,self.K,-1,-1)
        ew=torch.sigmoid(self.eg(torch.cat([ti,tj],dim=-1))).squeeze(-1)
        k3=min(3,self.K-1)
        tv,tix=ew.topk(k3,dim=-1)
        es=torch.zeros_like(ew).scatter_(2,tix,tv)
        msg=(torch.einsum("bkj,bjh->bkh",es,tm).unsqueeze(1).expand(-1,NF,-1,-1))
        ht=(tr.unsqueeze(-1)*self.op(msg)).sum(2)
        return self.norm(hp+self.drop(ht))


class AHGLayer(nn.Module):
    """[N3] AHG + explicit hf residual (unchanged)."""
    def __init__(self,H,n_fine,n_coarse,heads,dropout):
        super().__init__()
        self.gf=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gc=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gate=nn.Linear(H*2,1); self.norm=nn.LayerNorm(H)
    def forward(self,hf,hc,fei,cei,c2dn):
        B,NF,H=hf.shape
        hf2=torch.stack([self.gf(hf[b],fei) for b in range(B)])
        hc2=torch.stack([self.gc(hc[b],cei) for b in range(B)])
        hcca=hc2[:,c2dn,:]
        alpha=torch.sigmoid(self.gate(torch.cat([hf2,hcca],dim=-1)))
        return self.norm(hf+alpha*hf2+(1-alpha)*hcca), hc2


class HAST_GNN(nn.Module):
    def __init__(self,n_feat,n_fine,n_coarse,H,heads,n_layers,dropout,n_types,n_periods):
        super().__init__()
        self.nf=n_fine; self.nc=n_coarse
        self.pf=nn.Sequential(nn.Linear(n_feat,H),nn.LayerNorm(H),nn.GELU())
        self.pc=nn.Sequential(nn.Linear(H,H),nn.LayerNorm(H),nn.GELU())
        self.arata=nn.ModuleList([ARATABlock(H,heads,dropout) for _ in range(n_layers)])
        self.ahg  =nn.ModuleList([AHGLayer(H,n_fine,n_coarse,heads,dropout) for _ in range(n_layers)])
        self.ctig =CTIGBlock(H,cfg.TOP_N_TYPES,dropout)
        self.mp   =nn.Sequential(nn.Linear(H,H),nn.GELU(),nn.LayerNorm(H))
        self.n_rec=3

        def _h(i,m,o):
            blk=nn.Sequential(nn.Linear(i,m),nn.GELU(),nn.Dropout(dropout),nn.Linear(m,o))
            for l in blk:
                if isinstance(l,nn.Linear): nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
            return blk

        self.hot_h         = _h(H*2, 64, 1)
        self.loc_h         = _h(H+32, 64, 2)
        self.area_lat_bias = nn.Parameter(torch.zeros(n_fine))
        self.area_lon_bias = nn.Parameter(torch.zeros(n_fine))

        self.type_h = _h(H, 128, n_types)
        self.per_h  = _h(H, 64, n_periods)

        self.area_emb      = nn.Embedding(n_fine, 32)
        self.cnt_h         = _h(H+32, 64, 1)
        self.area_log_bias = nn.Parameter(torch.zeros(n_fine))
        self.trend_h       = _h(H*2, 64, 3)

    def forward(self,xf,fei,cei):
        B,T,NF,_=xf.shape
        H=self.pf[0].out_features; DEV=xf.device
        hf=self.pf(xf)
        for a in self.arata: hf=a(hf)
        hp=hf[:,-1,:,:]
        c2dn=CA_TO_DN.to(DEV)
        hc=torch.zeros(B,self.nc,H,device=DEV)
        hc.scatter_reduce_(1,c2dn.unsqueeze(0).unsqueeze(-1).expand(B,-1,H),
                           hp,reduce="mean",include_self=True)
        hc=self.pc(hc)
        for a in self.ahg: hp,hc=a(hp,hc,fei,cei,c2dn)
        hr=hf[:,-self.n_rec:,:,:].mean(1)
        hl=hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom=self.mp(hr-hl)
        hm=torch.cat([hp,mom],dim=-1)
        trs=len(TEMP_COLS)+len(CAL_COLS)
        ntr=min(cfg.TOP_N_TYPES,xf.shape[-1]-trs)
        tr=(xf[:,-1,:,trs:trs+ntr] if ntr>0
            else torch.zeros(B,NF,cfg.TOP_N_TYPES,device=DEV))
        hp=self.ctig(tr,hp)
        area_idx=torch.arange(self.nf,device=DEV).unsqueeze(0).expand(B,-1)
        ae=self.area_emb(area_idx)
        hp_ae=torch.cat([hp,ae],dim=-1)

        hot     =self.hot_h(hm).squeeze(-1)
        loc_res =self.loc_h(hp_ae)
        lat_pred=loc_res[...,0]+self.area_lat_bias.unsqueeze(0)
        lon_pred=loc_res[...,1]+self.area_lon_bias.unsqueeze(0)
        loc     =torch.stack([lat_pred,lon_pred],dim=-1)
        typ     =self.type_h(hp)
        per     =self.per_h(hp)
        cnt     =(self.cnt_h(hp_ae).squeeze(-1)+self.area_log_bias.unsqueeze(0)).clamp(0,6)
        trd     =self.trend_h(hm)
        return hot,loc,typ,per,cnt,trd


# =============================================================================
# POINT 10.3 — COMPACT LSTM + GATv2 SPATIO-TEMPORAL BASELINE
# =============================================================================
# Same inputs (60-day window, 77 areas, N_FEAT features), same fine
# community-area graph, same train/val/test split, same targets, and it is
# scored with the SAME evaluate()/compute_loss() as HAST-GNN. This class does
# NOT touch or modify HAST_GNN above; it's a separate, smaller model trained
# and evaluated independently, purely as a stronger baseline than persistence
# / seasonal-7 for the paired comparison in Point 10.7.
class BaselineLSTMGAT(nn.Module):
    def __init__(self, n_feat, n_fine, H, heads, n_types, n_periods, dropout=0.2):
        super().__init__()
        self.nf = n_fine
        self.H = H
        self.lstm = nn.LSTM(n_feat, H, batch_first=True)
        self.gat  = GATv2Conv(H, H // heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.norm = nn.LayerNorm(H)
        self.drop = nn.Dropout(dropout)

        def _h(i, m, o):
            blk = nn.Sequential(nn.Linear(i, m), nn.GELU(), nn.Dropout(dropout), nn.Linear(m, o))
            for l in blk:
                if isinstance(l, nn.Linear):
                    nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
            return blk

        self.hot_h   = _h(H, 32, 1)
        self.loc_h   = _h(H, 32, 2)
        self.type_h  = _h(H, 64, n_types)
        self.per_h   = _h(H, 32, n_periods)
        self.cnt_h   = _h(H, 32, 1)
        self.trend_h = _h(H, 32, 3)

    def forward(self, xf, fei, cei=None):
        # xf: (B, T, NF, n_feat) — identical layout to HAST-GNN's input.
        B, T, NF, Fdim = xf.shape
        x = xf.permute(0, 2, 1, 3).reshape(B * NF, T, Fdim)
        out, _ = self.lstm(x)
        h_last = out[:, -1, :].reshape(B, NF, self.H)
        h_gat = torch.stack([self.gat(h_last[b], fei) for b in range(B)])
        h = self.norm(h_last + self.drop(h_gat))

        hot = self.hot_h(h).squeeze(-1)
        loc = self.loc_h(h)
        typ = self.type_h(h)
        per = self.per_h(h)
        cnt = self.cnt_h(h).squeeze(-1).clamp(0, 6)
        trd = self.trend_h(h)
        return hot, loc, typ, per, cnt, trd


model=HAST_GNN(N_FEAT,N_COMM,N_DIST,cfg.HIDDEN_DIM,cfg.GAT_HEADS,cfg.N_LAYERS,
               cfg.DROPOUT,N_TYPES,N_PERIODS).to(DEVICE)
n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nHAST-GNN v14")
print(f"  [N1] ARATA + DWConv1d(k=3)")
print(f"  [N2] CTIG — dynamic crime-type interaction")
print(f"  [N3] AHG + explicit hf residual")
print(f"  [N4] MAMTO — momentum multi-task output")
print(f"  Parameters: {n_params:,} | Features: {N_FEAT}")
print(f"  type_h: H→128→{N_TYPES}  |  per_h: H→64→{N_PERIODS}")

# =============================================================================
# LOSS WEIGHTS + BIAS INIT
# =============================================================================
with torch.no_grad():
    model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
    model.area_lat_bias.copy_(torch.tensor(area_lat_z,dtype=torch.float32).to(DEVICE))
    model.area_lon_bias.copy_(torch.tensor(area_lon_z,dtype=torch.float32).to(DEVICE))

hot_rate=grid.loc[train_mask,"is_hotspot"].mean()
pos_wt=torch.tensor((1-hot_rate)/hot_rate,dtype=torch.float32).to(DEVICE)
print(f"Hotspot pos_weight: {pos_wt.item():.3f}")

tc=np.bincount(type_np[:n_train].reshape(-1),minlength=N_TYPES).astype(float)
tc/=tc.sum()
typw=torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES),
                   dtype=torch.float32).to(DEVICE)
print(f"Type weights ({N_TYPES} classes): {typw.cpu().numpy().round(2)}")

pc=np.bincount(per_np[:n_train].reshape(-1),minlength=N_PERIODS).astype(float)
pc/=pc.sum()
perw=torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS),
                   dtype=torch.float32).to(DEVICE)
print(f"Hour bucket weights ({N_PERIODS} classes): {perw.cpu().numpy().round(2)}")

trc=np.bincount(trend_np[:n_train].reshape(-1),minlength=3).astype(float)
trc/=trc.sum()
tw=torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3),dtype=torch.float32).to(DEVICE)
print(f"Trend weights: {tw.cpu().numpy().round(2)}")

# =============================================================================
# LOSS FUNCTIONS  (unchanged)
# =============================================================================

def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls=logits.shape[-1]
    log_p=F.log_softmax(logits,dim=-1)
    p=torch.exp(log_p)
    with torch.no_grad():
        smooth_t=torch.full_like(logits,smoothing/(n_cls-1))
        smooth_t.scatter_(-1,targets.unsqueeze(-1),1.0-smoothing)
    ce_smooth=-(smooth_t*log_p).sum(-1)
    p_t=p.gather(-1,targets.unsqueeze(-1)).squeeze(-1)
    focal_w=(1.0-p_t)**gamma
    if weight is not None:
        focal_w=focal_w*weight[targets]
    focal=(focal_w*ce_smooth).mean()
    entropy=-(p*log_p).sum(-1).mean()
    return focal-entropy_reg*entropy


def compute_loss(hot_l,loc_p,typ_l,per_l,cnt_p,trd_l,
                 hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t):
    l_hot=F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1),hot_t.reshape(-1),pos_weight=pos_wt)
    valid=(lat_t.abs()>0.01)|(lon_t.abs()>0.01)
    if valid.any():
        l_loc=(F.huber_loss(loc_p[...,0][valid],lat_t[valid],delta=1.0)+
               F.huber_loss(loc_p[...,1][valid],lon_t[valid],delta=1.0))
    else:
        l_loc=torch.tensor(0.0,device=hot_l.device)

    l_typ=focal_loss_with_smooth(
        typ_l.reshape(-1,N_TYPES),typ_t.reshape(-1),
        weight=typw,gamma=cfg.FOCAL_GAMMA,
        smoothing=cfg.LABEL_SMOOTH,entropy_reg=cfg.ENTROPY_REG)

    l_per=F.cross_entropy(per_l.reshape(-1,N_PERIODS),per_t.reshape(-1),weight=perw)

    l_cnt=F.huber_loss(cnt_p.reshape(-1),cnt_t.reshape(-1),delta=0.5)
    l_trd=F.cross_entropy(trd_l.reshape(-1,3),trd_t.reshape(-1),weight=tw)

    total=1.0*l_hot+1.0*l_loc+1.5*l_typ+0.6*l_per+1.2*l_cnt+0.5*l_trd
    return total,l_hot,l_loc,l_typ,l_per,l_cnt,l_trd

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1,lo1,la2,lo2):
    R=6371.0
    dlat=np.radians(la2-la1); dlon=np.radians(lo2-lo1)
    a=np.sin(dlat/2)**2+np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a,dtype=np.float32))

# =============================================================================
# EVALUATE
# =============================================================================
# REFACTOR-1: `eval_model` lets this score ANY model with the HAST-GNN output
# signature (hot, loc, typ, per, cnt, trd), e.g. the Point 10.3 baseline or
# a Point 10.4 multi-seed model, without duplicating this function.
def evaluate(loader, eval_model=None, desc="Evaluating", show_pbar=True):
    m = eval_model if eval_model is not None else model
    m.eval()
    ht,hs=[],[]
    latt,lont,latr,lonr,latp,lonp=[],[],[],[],[],[]
    tytt,tytp=[],[]
    pert,perp=[],[]
    cntr,cntp=[],[]
    trt_,trp_=[],[]
    tot=n=0

    iterator = tqdm(loader, desc=desc, leave=False, unit="batch") if show_pbar else loader

    with torch.no_grad():
        for batch in iterator:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=m(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                      hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
            if not torch.isnan(loss):
                tot+=loss.item()*x.shape[0]; n+=x.shape[0]
                if show_pbar:
                    iterator.set_postfix(loss=f"{tot/max(1,n):.4f}")

    ht=np.array(ht,dtype=np.int32); hs=san(hs)
    pr,re,thr=precision_recall_curve(ht,hs)
    f1c=2*pr*re/(pr+re+1e-8)
    bt=float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp=(hs>=bt).astype(int)
    try: rauc=roc_auc_score(ht,hs)
    except: rauc=0.0
    hot_f1=f1_score(ht,hp,zero_division=0)

    lp_raw=san(latp)*lat_std+lat_mean; lr_raw=np.array(latr)
    op_raw=san(lonp)*lon_std+lon_mean; or_raw=np.array(lonr)
    valid=(lr_raw>40)&(or_raw<-80)
    if valid.sum()>0:
        hav=haversine_km(lr_raw[valid],or_raw[valid],lp_raw[valid],op_raw[valid])
        loc_mae=float(hav.mean()); l1=float((hav<1).mean()); l05=float((hav<0.5).mean())
    else:
        loc_mae=99.9; l1=0.0; l05=0.0

    ta=accuracy_score(np.array(tytt),np.array(tytp))
    tf=f1_score(np.array(tytt),np.array(tytp),average="weighted",zero_division=0)
    pa=accuracy_score(np.array(pert),np.array(perp))
    pf=f1_score(np.array(pert),np.array(perp),average="weighted",zero_division=0)
    cr=san(cntr); cp=san(cntp)
    cnt_corr=float(np.corrcoef(cr,cp)[0,1]) if len(cr)>1 else 0.0
    cnt_mae=float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp.clip(0,10))).mean())
    trend_acc=accuracy_score(np.array(trt_),np.array(trp_))

    loc_score=max(0,1-loc_mae/10)
    primary=np.mean([rauc,ta,pa,max(cnt_corr,0),loc_score])

    return dict(
        loss=tot/max(1,n),primary=primary,
        hot_auc=rauc,hot_f1=hot_f1,best_thr=bt,ht=ht,hs=hs,hp=hp,
        loc_mae=loc_mae,loc_1km=l1,loc_05km=l05,
        lat_p=lp_raw,lon_p=op_raw,lat_t=lr_raw,lon_t=or_raw,
        type_acc=ta,type_f1=tf,typ_t=np.array(tytt),typ_p=np.array(tytp),
        per_acc=pa,per_f1=pf,per_t=np.array(pert),per_p=np.array(perp),
        cnt_corr=cnt_corr,cnt_mae=cnt_mae,cr=cr,cp=cp,
        trend_acc=trend_acc,trt=np.array(trt_),trp=np.array(trp_),
    )

# =============================================================================
# REFACTOR-2: reusable training loop
# =============================================================================
# The original v14 training loop is unchanged in behavior — it's just moved
# into a function so Point 10.3 (baseline) and Point 10.4 (multi-seed) can
# call the SAME code instead of copy-pasting the 120-epoch loop.
#
# PROGRESS-1: an epoch-level tqdm bar is now shown REGARDLESS of `verbose`,
# with a live postfix (train loss / val loss / val primary / best-so-far).
# When verbose=True the original per-epoch table is still printed, but via
# tqdm.write() so it doesn't fight with the progress bar's own line. A nested
# per-batch tqdm bar (leave=False) shows within-epoch progress too, since a
# single epoch over ~thousands of sequences can itself take a while.
def train_hast_gnn(train_model, opt_train_loader, opt_val_loader,
                    epochs, patience, lr, max_lr_mult=8, weight_decay=cfg.WEIGHT_DECAY,
                    save_path="model_best.pth", verbose=True, track_history=True,
                    run_name="HAST-GNN"):
    optimizer = torch.optim.AdamW(train_model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr*max_lr_mult, total_steps=epochs*len(opt_train_loader),
        pct_start=0.10, anneal_strategy="cos")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)

    best_p = 0.0; best_ep = 0; nan_streak = 0
    hist = {"tl": [], "vl": [], "vp": [], "vtype": [], "vper": []} if track_history else None

    if verbose:
        hdr=("  Ep|    Loss|   VLoss|   VF1|   VAUC|  VTyp| VHour| VCntR|      LR")
        tqdm.write(hdr); tqdm.write("─"*len(hdr))

    epoch_bar = tqdm(range(1, epochs+1), desc=f"{run_name} epochs", unit="ep")
    for ep in epoch_bar:
        train_model.train(); ep_loss=0.0; ep_n=0
        batch_bar = tqdm(opt_train_loader, desc=f"  ep{ep} train batches",
                          leave=False, unit="batch")
        for batch in batch_bar:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=train_model(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                       hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            if torch.isnan(loss):
                nan_streak+=1
                if nan_streak>=20:
                    tqdm.write(f"\n⚠ 20 NaN — abort")
                    break
                continue
            nan_streak=0
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(train_model.parameters(),0.5)
            amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
            ep_loss+=loss.item()*x.shape[0]; ep_n+=x.shape[0]
            batch_bar.set_postfix(loss=f"{ep_loss/max(1,ep_n):.4f}")
        else:
            s = evaluate(opt_val_loader, eval_model=train_model,
                         desc=f"  ep{ep} val", show_pbar=True)
            tl = ep_loss/max(1,ep_n); lr_now = optimizer.param_groups[0]["lr"]
            if verbose:
                tqdm.write(f"  {ep:3d}| {tl:.5f}| {s['loss']:.5f}|{s['hot_f1']:.4f}| "
                      f"{s['hot_auc']:.4f}| {s['type_acc']:.4f}| {s['per_acc']:.4f}| "
                      f"{s['cnt_corr']:.4f}|{lr_now:.2e}")
            epoch_bar.set_postfix(
                train_loss=f"{tl:.4f}", val_loss=f"{s['loss']:.4f}",
                val_primary=f"{s['primary']:.4f}", best=f"{best_p:.4f}",
                best_ep=best_ep)
            if s["primary"]>best_p:
                best_p=s["primary"]; best_ep=ep
                torch.save(train_model.state_dict(), save_path)
                if verbose:
                    tqdm.write(f"  ✓ Primary={s['primary']:.4f}  HotAUC={s['hot_auc']:.4f}  "
                          f"LocMAE={s['loc_mae']:.2f}km  TypeAcc={s['type_acc']:.4f}  "
                          f"HourAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}")
                epoch_bar.set_postfix(
                    train_loss=f"{tl:.4f}", val_loss=f"{s['loss']:.4f}",
                    val_primary=f"{s['primary']:.4f}", best=f"{best_p:.4f}",
                    best_ep=best_ep)
            if track_history:
                hist["tl"].append(tl); hist["vl"].append(s["loss"])
                hist["vp"].append(s["primary"]); hist["vtype"].append(s["type_acc"])
                hist["vper"].append(s["per_acc"])
            if ep-best_ep>=patience:
                if verbose: tqdm.write(f"\n⏹  Early stop ep {ep}")
                epoch_bar.close()
                break
            continue
        break
    else:
        epoch_bar.close()

    train_model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    train_model.eval()
    return train_model, best_p, hist

# =============================================================================
# TRAINING — primary HAST-GNN v14 run
# =============================================================================
print(f"\n{'='*110}")
print(f"  HAST-GNN v14 | {N_FEAT} feats | {n_params:,} params")
print(f"  Task2=CrimeType({N_TYPES} classes)  Task3=HourBucket({N_PERIODS} classes)")
print(f"{'='*110}")

model, best_p, hist = train_hast_gnn(
    model, train_loader, val_loader,
    epochs=cfg.EPOCHS, patience=cfg.PATIENCE, lr=cfg.LR,
    max_lr_mult=8, weight_decay=cfg.WEIGHT_DECAY,
    save_path="hast_gnn_best.pth", verbose=True, track_history=True,
    run_name="HAST-GNN v14 (primary)",
)

fig,axes=plt.subplots(1,3,figsize=(18,4))
axes[0].plot(hist["tl"],label="Train"); axes[0].plot(hist["vl"],label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist["vp"],label="Val Primary"); axes[1].axhline(0.80,color="red",ls="--")
axes[1].set_title("Primary Metric"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(hist["vtype"],label=f"Val TypeAcc ({N_TYPES}cls)",color="steelblue")
axes[2].plot(hist["vper"],label=f"Val HourAcc ({N_PERIODS}bkt)",color="darkorange")
axes[2].set_title("Type & Hour Accuracy"); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("hast_gnn_curves.png",dpi=150); plt.close()
print("\nSaved: hast_gnn_curves.png")

# =============================================================================
# FINAL TEST  (unchanged)
# =============================================================================
print("\n"+"="*100)
print("  FINAL TEST RESULTS — HAST-GNN v14")
print("="*100)
s=evaluate(test_loader, desc="Final test eval (HAST-GNN)")

print(f"\n── TASK 1a. HOTSPOT AREA DETECTION (top-{cfg.HOTSPOT_TOP_K}/77 areas/day) ──")
print(f"  ROC-AUC:         {s['hot_auc']:.4f}   target ≥0.90  {'✅' if s['hot_auc']>=0.90 else '❌'}")
print(f"  F1 Score:        {s['hot_f1']:.4f}")

print(f"\n── TASK 1b. HOTSPOT LOCATION ──")
print(f"  Haversine MAE:   {s['loc_mae']:.3f} km   target <2.0km  {'✅' if s['loc_mae']<2.0 else '❌'}")
print(f"  Within 0.5 km:   {s['loc_05km']*100:.1f}%")
print(f"  Within 1.0 km:   {s['loc_1km']*100:.1f}%")
valid_v=(s['lat_t']>40)&(s['lon_t']<-80)
for i in np.where(valid_v)[0][:5]:
    d=haversine_km(s['lat_t'][i],s['lon_t'][i],s['lat_p'][i],s['lon_p'][i])
    print(f"    True:({s['lat_t'][i]:.6f},{s['lon_t'][i]:.6f})  "
          f"Pred:({s['lat_p'][i]:.6f},{s['lon_p'][i]:.6f})  → {d:.3f} km")

print(f"\n── TASK 2. CRIME TYPE — {N_TYPES} Fine-Grained Classes ──")
print(f"  Accuracy:        {s['type_acc']:.4f}   target ≥0.55  {'✅' if s['type_acc']>=0.55 else '❌'}")
print(f"  Weighted F1:     {s['type_f1']:.4f}")
print(classification_report(s["typ_t"],s["typ_p"],
      target_names=TYPE_CLASSES,digits=4,zero_division=0))

print(f"\n── TASK 3. CRIME HOUR — {N_PERIODS} Hour-Bucket Classes ──")
print(f"  Accuracy:        {s['per_acc']:.4f}   target ≥0.35  {'✅' if s['per_acc']>=0.35 else '❌'}")
print(f"  Weighted F1:     {s['per_f1']:.4f}")
print(classification_report(s["per_t"],s["per_p"],
      target_names=HOUR_BUCKET_NAMES,digits=4,zero_division=0))

print(f"\n── TASK 4. CRIME COUNT ──")
print(f"  Pearson r:       {s['cnt_corr']:.4f}   target ≥0.85  {'✅' if s['cnt_corr']>=0.85 else '❌'}")
print(f"  Count MAE:       {s['cnt_mae']:.2f} crimes/day")

print(f"\n── TREND (auxiliary) ──")
print(f"  Accuracy:        {s['trend_acc']:.4f}")

print(f"\n{'='*60}\n  SUMMARY\n{'='*60}")
loc_score=max(0,1-s['loc_mae']/10)
rows=[("Hotspot AUC", s['hot_auc'],  "AUC", 0.90),
      ("Loc MAE",     s['loc_mae'],  "km ", None),
      ("Type Acc",    s['type_acc'], "Acc", 0.55),
      ("Hour Acc",    s['per_acc'],  "Acc", 0.35),
      ("Count r",     s['cnt_corr'], "r  ", 0.85)]
for name,val,unit,tgt in rows:
    bar="█"*int(max(min(val,1),0)*20)
    flag=("✅" if val>=tgt else ("⚠" if val>=tgt*0.85 else "❌")) if tgt else "📍"
    print(f"  {flag} {name:<14} {unit}={val:.4f}  {bar}")

print(f"\n{'='*60}\n  SAMPLE PREDICTIONS (first 5 valid hotspot areas)\n{'='*60}")
hot_pred_areas=np.where(s["hp"]==1)[0][:5]
if len(hot_pred_areas)>0:
    for idx in hot_pred_areas:
        ltt=s["lat_p"][idx]; lnt=s["lon_p"][idx]
        typ_name=TYPE_CLASSES[s["typ_p"][idx]]
        hr_name=HOUR_BUCKET_NAMES[s["per_p"][idx]]
        cnt_pred=float(np.expm1(s["cp"][idx]))
        trd_map={0:"FALL",1:"STABLE",2:"RISE"}
        trd_name=trd_map[s["trp"][idx]]
        print(f"  Hotspot=0.{int(s['hs'][idx]*100):02d}  "
              f"Loc=({ltt:.4f},{lnt:.4f})  "
              f"Type={typ_name:<22}  "
              f"Hour={hr_name}  "
              f"Count≈{cnt_pred:.1f}  Trend={trd_name}")

fig,axes=plt.subplots(2,4,figsize=(24,10))
fig.suptitle(f"HAST-GNN v14 | HotAUC={s['hot_auc']:.3f}  LocMAE={s['loc_mae']:.2f}km  "
             f"TypeAcc={s['type_acc']:.3f}({N_TYPES}cls)  "
             f"HourAcc={s['per_acc']:.3f}({N_PERIODS}bkt)  CntR={s['cnt_corr']:.3f}",
             fontsize=10,fontweight="bold")

fpr,tpr,_=roc_curve(s["ht"],s["hs"])
axes[0,0].plot(fpr,tpr,lw=2,label=f"AUC={s['hot_auc']:.4f}")
axes[0,0].plot([0,1],[0,1],"k--",alpha=0.3)
axes[0,0].set_title("Hotspot ROC"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

pr2,re2,_=precision_recall_curve(s["ht"],s["hs"])
axes[0,1].plot(re2,pr2,lw=2,color="orange",label=f"PR-AUC={auc(re2,pr2):.4f}")
axes[0,1].set_title("Hotspot PR"); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

vv=(s['lat_t']>40)&(s['lon_t']<-80)
axes[0,2].scatter(s['lon_t'][vv][:3000],s['lat_t'][vv][:3000],alpha=0.15,s=2,label="True",c="steelblue")
axes[0,2].scatter(s['lon_p'][vv][:3000],s['lat_p'][vv][:3000],alpha=0.15,s=2,label="Pred",c="tomato")
axes[0,2].set_title(f"Location (MAE={s['loc_mae']:.2f}km)")
axes[0,2].legend(markerscale=4); axes[0,2].grid(alpha=0.3)

cm_type=confusion_matrix(s["typ_t"],s["typ_p"])
ConfusionMatrixDisplay(cm_type,display_labels=[t[:8] for t in TYPE_CLASSES]).plot(
    ax=axes[0,3],cmap="Blues",values_format="d")
axes[0,3].set_title(f"Crime Type Acc={s['type_acc']:.3f} ({N_TYPES}cls)")
plt.setp(axes[0,3].get_xticklabels(),rotation=45,ha="right",fontsize=6)
plt.setp(axes[0,3].get_yticklabels(),fontsize=6)

cm_hour=confusion_matrix(s["per_t"],s["per_p"])
ConfusionMatrixDisplay(cm_hour,display_labels=HOUR_BUCKET_NAMES).plot(
    ax=axes[1,0],cmap="Purples",values_format="d")
axes[1,0].set_title(f"Crime Hour Acc={s['per_acc']:.3f} ({N_PERIODS}bkt)")
plt.setp(axes[1,0].get_xticklabels(),rotation=45,ha="right",fontsize=7)

axes[1,1].scatter(s["cr"][:8000],s["cp"][:8000],alpha=0.08,s=2,color="teal")
axes[1,1].set_xlabel("Actual log1p"); axes[1,1].set_ylabel("Predicted")
axes[1,1].set_title(f"Crime Count (r={s['cnt_corr']:.3f})"); axes[1,1].grid(alpha=0.3)

ConfusionMatrixDisplay(confusion_matrix(s["trt"],s["trp"]),
    display_labels=["Fall","Stable","Rise"]).plot(ax=axes[1,2],cmap="Oranges",values_format="d")
axes[1,2].set_title(f"Trend (Acc={s['trend_acc']:.3f})")

names=["Hotspot\nAUC","Loc\nScore","Type\nAcc","Hour\nAcc","Count\nr"]
vals=[s['hot_auc'],loc_score,s['type_acc'],s['per_acc'],max(s['cnt_corr'],0)]
tgts=[0.90,0.80,0.55,0.35,0.85]
colors=["green" if v>=t else ("orange" if v>=t*0.85 else "red") for v,t in zip(vals,tgts)]
axes[1,3].bar(names,vals,color=colors,alpha=0.8,edgecolor="black")
axes[1,3].axhline(0.90,color="red",ls="--",lw=1,alpha=0.5)
axes[1,3].set_ylim(0,1.05); axes[1,3].set_title("All Tasks"); axes[1,3].grid(alpha=0.3)

plt.tight_layout(); plt.savefig("hast_gnn_eval.png",dpi=150); plt.close()
print("\nSaved: hast_gnn_eval.png")
torch.save(model.state_dict(),"hast_gnn_fp32.pth")
print(f"\n✅  HAST-GNN v14 Complete!")
print(f"   HotAUC={s['hot_auc']:.4f}  LocMAE={s['loc_mae']:.2f}km  "
      f"TypeAcc={s['type_acc']:.4f}({N_TYPES}cls)  "
      f"HourAcc={s['per_acc']:.4f}({N_PERIODS}bkt)  CntR={s['cnt_corr']:.4f}")
print("   Saved: hast_gnn_best.pth | hast_gnn_fp32.pth | hast_gnn_curves.png | hast_gnn_eval.png")

# =============================================================================
# POINT 10.1 — PERSISTENCE BASELINE
# =============================================================================
# Predict day t using the most recently OBSERVED day (t-1).
#
# ALIGNMENT: target days must match exactly what evaluate(test_loader)
# produced in `s["cr"]`/`s["ht"]` above. CrimeDS.__len__ = max(0,(e-s)-T-1),
# and test_loader is shuffle=False, so sample idx (0-indexed, in order)
# corresponds to absolute day = test_start + T + idx, for idx in
# range(len(test_ds)). We derive the slice from len(test_ds) directly instead
# of re-deriving it from NUM_DAYS, so this can't silently drift by one row.
test_start = n_train + n_val               # == N_TRAIN_DAYS + N_VAL_DAYS
n_test_seq = len(test_ds)                  # exact row count evaluate() produced
tgt0 = test_start + T                      # absolute day of the FIRST test target

persist_cnt = log_cnt_np[tgt0 - 1 : tgt0 - 1 + n_test_seq]   # day t-1, for each target day t
persist_hot = hot_np[tgt0 - 1 : tgt0 - 1 + n_test_seq]

true_cnt = log_cnt_np[tgt0 : tgt0 + n_test_seq]
true_hot = hot_np[tgt0 : tgt0 + n_test_seq]

assert true_cnt.shape[0] * N_COMM == s["cr"].shape[0], \
    f"Persistence baseline misaligned: {true_cnt.shape[0]*N_COMM} vs {s['cr'].shape[0]}"

cnt_r = np.corrcoef(true_cnt.reshape(-1), persist_cnt.reshape(-1))[0, 1]
hot_auc_persist = roc_auc_score(true_hot.reshape(-1), persist_hot.reshape(-1))
hot_f1_persist = f1_score(true_hot.reshape(-1), (persist_hot.reshape(-1) >= 0.5).astype(int), zero_division=0)

print("\n========== POINT 10.1 — PERSISTENCE ==========")
print(f"Crime Count Pearson r: {cnt_r:.4f}")
print(f"Hotspot ROC-AUC:       {hot_auc_persist:.4f}")
print(f"Hotspot F1:            {hot_f1_persist:.4f}")

# =============================================================================
# POINT 10.2 — SEASONAL (7-DAY) BASELINE
# =============================================================================
season_cnt = log_cnt_np[tgt0 - 7 : tgt0 - 7 + n_test_seq]
season_hot = hot_np[tgt0 - 7 : tgt0 - 7 + n_test_seq]

cnt_r_7 = np.corrcoef(true_cnt.reshape(-1), season_cnt.reshape(-1))[0, 1]
hot_auc_7 = roc_auc_score(true_hot.reshape(-1), season_hot.reshape(-1))
hot_f1_7 = f1_score(true_hot.reshape(-1), (season_hot.reshape(-1) >= 0.5).astype(int), zero_division=0)

print("\n========== POINT 10.2 — 7-DAY SEASONAL ==========")
print(f"Crime Count Pearson r: {cnt_r_7:.4f}")
print(f"Hotspot ROC-AUC:       {hot_auc_7:.4f}")
print(f"Hotspot F1:            {hot_f1_7:.4f}")

# Flat, row-aligned versions of the two naive baselines, ready for 10.7.
persist_cnt_flat = persist_cnt.reshape(-1)
persist_hot_flat = persist_hot.reshape(-1)
season_cnt_flat  = season_cnt.reshape(-1)
season_hot_flat  = season_hot.reshape(-1)

# =============================================================================
# POINT 10.3 — STRONGER SPATIO-TEMPORAL BASELINE (LSTM + GATv2)
# =============================================================================
print("\n"+"="*100)
print("  BASELINE — LSTM + GATv2 (Point 10.3, separate from HAST-GNN v14)")
print("="*100)

baseline_model = BaselineLSTMGAT(
    N_FEAT, N_COMM, cfg.BASELINE_HIDDEN, cfg.BASELINE_HEADS, N_TYPES, N_PERIODS
).to(DEVICE)
n_params_bl = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print(f"Baseline params: {n_params_bl:,}")

baseline_model, baseline_best_p, _ = train_hast_gnn(
    baseline_model, train_loader, val_loader,
    epochs=cfg.BASELINE_EPOCHS, patience=cfg.BASELINE_PATIENCE, lr=cfg.BASELINE_LR,
    max_lr_mult=5, weight_decay=cfg.WEIGHT_DECAY,
    save_path="baseline_lstm_gat_best.pth", verbose=True, track_history=False,
    run_name="LSTM+GATv2 baseline",
)

s_bl = evaluate(test_loader, eval_model=baseline_model, desc="Final test eval (LSTM+GATv2)")

print("\n========== POINT 10.3 — LSTM+GATv2 BASELINE TEST RESULTS ==========")
print(f"Hotspot ROC-AUC:   {s_bl['hot_auc']:.4f}")
print(f"Hotspot F1:        {s_bl['hot_f1']:.4f}")
print(f"Location MAE:      {s_bl['loc_mae']:.3f} km")
print(f"Crime Type Acc:    {s_bl['type_acc']:.4f}")
print(f"Hour Acc:          {s_bl['per_acc']:.4f}")
print(f"Crime Count r:      {s_bl['cnt_corr']:.4f}")

baseline_cnt_flat = s_bl["cp"]   # already row-aligned to s["cr"] (same test_loader, same order)
baseline_hot_flat = s_bl["hs"]

# =============================================================================
# POINT 10.4 — MULTI-SEED HAST-GNN
# =============================================================================
# Reuses train_hast_gnn(); the 120-epoch loop is NOT duplicated.
# PROGRESS-1: an outer tqdm bar tracks progress across seeds so it's obvious
# how many of the (typically slow, 120-epoch) seed runs remain.
print("\n"+"="*100)
print("  POINT 10.4 — MULTI-SEED HAST-GNN")
print("="*100)

SEEDS = cfg.MULTI_SEEDS
seed_results = []

seed_bar = tqdm(SEEDS, desc="Multi-seed runs", unit="seed")
for seed in seed_bar:
    seed_bar.set_postfix(seed=seed)
    tqdm.write(f"\n{'='*70}")
    tqdm.write(f"RUNNING HAST-GNN v14 | SEED = {seed}")
    tqdm.write(f"{'='*70}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_model = HAST_GNN(
        N_FEAT, N_COMM, N_DIST,
        cfg.HIDDEN_DIM, cfg.GAT_HEADS,
        cfg.N_LAYERS, cfg.DROPOUT,
        N_TYPES, N_PERIODS
    ).to(DEVICE)
    with torch.no_grad():
        seed_model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
        seed_model.area_lat_bias.copy_(torch.tensor(area_lat_z,dtype=torch.float32).to(DEVICE))
        seed_model.area_lon_bias.copy_(torch.tensor(area_lon_z,dtype=torch.float32).to(DEVICE))

    seed_model, seed_best_p, _ = train_hast_gnn(
        seed_model, train_loader, val_loader,
        epochs=cfg.EPOCHS, patience=cfg.PATIENCE, lr=cfg.LR,
        max_lr_mult=8, weight_decay=cfg.WEIGHT_DECAY,
        save_path=f"hast_gnn_seed{seed}_best.pth", verbose=False, track_history=False,
        run_name=f"HAST-GNN seed={seed}",
    )

    result = evaluate(test_loader, eval_model=seed_model, desc=f"Test eval (seed={seed})")

    seed_results.append({
        "seed": seed,
        "hot_auc": result["hot_auc"],
        "hot_f1": result["hot_f1"],
        "loc_mae": result["loc_mae"],
        "type_acc": result["type_acc"],
        "hour_acc": result["per_acc"],
        "count_r": result["cnt_corr"],
    })
    tqdm.write(f"  Seed {seed} test results: HotAUC={result['hot_auc']:.4f}  "
          f"LocMAE={result['loc_mae']:.2f}km  TypeAcc={result['type_acc']:.4f}  "
          f"HourAcc={result['per_acc']:.4f}  CntR={result['cnt_corr']:.4f}")

# =============================================================================
# POINT 10.5 — MULTI-SEED MEAN ± SD
# =============================================================================
seed_df = pd.DataFrame(seed_results)

metrics = ["hot_auc", "hot_f1", "loc_mae", "type_acc", "hour_acc", "count_r"]

print("\n========== POINT 10.5 — MULTI-SEED RESULTS ==========")
for m in metrics:
    print(f"{m:12s}: {seed_df[m].mean():.4f} ± {seed_df[m].std(ddof=1):.4f}")

seed_df.to_csv("hast_gnn_v14_multiseed_results.csv", index=False)
print("Saved: hast_gnn_v14_multiseed_results.csv")

# =============================================================================
# POINT 10.6 — BOOTSTRAP 95% CONFIDENCE INTERVALS
# =============================================================================
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, seed=42, desc="Bootstrapping"):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []
    for _ in tqdm(range(n_boot), desc=desc, leave=False, unit="resample"):
        idx = rng.integers(0, n, n)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            continue
    scores = np.asarray(scores)
    return (float(np.mean(scores)), float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5)))

hot_ci  = bootstrap_ci(s["ht"], s["hs"], roc_auc_score, desc="Bootstrap: hotspot AUC")
cnt_ci  = bootstrap_ci(s["cr"], s["cp"], lambda y, p: np.corrcoef(y, p)[0, 1], desc="Bootstrap: count r")
type_ci = bootstrap_ci(s["typ_t"], s["typ_p"], lambda y, p: accuracy_score(y, p), desc="Bootstrap: type acc")
hour_ci = bootstrap_ci(s["per_t"], s["per_p"], lambda y, p: accuracy_score(y, p), desc="Bootstrap: hour acc")

print("\n========== POINT 10.6 — 95% BOOTSTRAP CI (HAST-GNN v14 primary run) ==========")
print(f"Hotspot AUC : {hot_ci[0]:.4f} [{hot_ci[1]:.4f}, {hot_ci[2]:.4f}]")
print(f"Count r     : {cnt_ci[0]:.4f} [{cnt_ci[1]:.4f}, {cnt_ci[2]:.4f}]")
print(f"Type Acc    : {type_ci[0]:.4f} [{type_ci[1]:.4f}, {type_ci[2]:.4f}]")
print(f"Hour Acc    : {hour_ci[0]:.4f} [{hour_ci[1]:.4f}, {hour_ci[2]:.4f}]")

# =============================================================================
# POINT 10.7 — PAIRED BOOTSTRAP: HAST-GNN vs BASELINES
# =============================================================================
def paired_bootstrap(y_true, pred_a, pred_b, metric_fn, n_boot=2000, seed=42, desc="Paired bootstrapping"):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = []
    for _ in tqdm(range(n_boot), desc=desc, leave=False, unit="resample"):
        idx = rng.integers(0, n, n)
        try:
            a = metric_fn(y_true[idx], pred_a[idx])
            b = metric_fn(y_true[idx], pred_b[idx])
            diffs.append(a - b)
        except Exception:
            continue
    diffs = np.asarray(diffs)
    return (float(np.mean(diffs)), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5)))

# Sanity checks — every comparison array must be exactly the same length and
# ordering as s["cr"]/s["ht"] (which is what makes this a valid PAIRED test).
assert s["cr"].shape[0] == persist_cnt_flat.shape[0] == season_cnt_flat.shape[0] == baseline_cnt_flat.shape[0], \
    "Row-count mismatch between HAST-GNN test predictions and baseline arrays — do not run paired bootstrap on misaligned arrays."
assert s["ht"].shape[0] == persist_hot_flat.shape[0] == season_hot_flat.shape[0] == baseline_hot_flat.shape[0], \
    "Row-count mismatch between HAST-GNN hotspot labels and baseline hotspot arrays."

print("\n========== POINT 10.7 — PAIRED BOOTSTRAP (HAST-GNN vs baselines) ==========")

# --- Crime count (Pearson r) ---
delta_cnt_persist = paired_bootstrap(
    s["cr"], s["cp"], persist_cnt_flat, lambda y, p: np.corrcoef(y, p)[0, 1],
    desc="Paired bootstrap: count r vs persistence")
delta_cnt_season = paired_bootstrap(
    s["cr"], s["cp"], season_cnt_flat, lambda y, p: np.corrcoef(y, p)[0, 1],
    desc="Paired bootstrap: count r vs seasonal")
delta_cnt_baseline = paired_bootstrap(
    s["cr"], s["cp"], baseline_cnt_flat, lambda y, p: np.corrcoef(y, p)[0, 1],
    desc="Paired bootstrap: count r vs LSTM+GATv2")

print("HAST-GNN vs Persistence   — Count r difference: "
      f"{delta_cnt_persist[0]:.4f} [{delta_cnt_persist[1]:.4f}, {delta_cnt_persist[2]:.4f}]")
print("HAST-GNN vs 7-day Seasonal — Count r difference: "
      f"{delta_cnt_season[0]:.4f} [{delta_cnt_season[1]:.4f}, {delta_cnt_season[2]:.4f}]")
print("HAST-GNN vs LSTM+GATv2     — Count r difference: "
      f"{delta_cnt_baseline[0]:.4f} [{delta_cnt_baseline[1]:.4f}, {delta_cnt_baseline[2]:.4f}]")

# --- Hotspot ROC-AUC ---
delta_hot_persist = paired_bootstrap(s["ht"], s["hs"], persist_hot_flat, roc_auc_score,
                                      desc="Paired bootstrap: hotspot AUC vs persistence")
delta_hot_season = paired_bootstrap(s["ht"], s["hs"], season_hot_flat, roc_auc_score,
                                     desc="Paired bootstrap: hotspot AUC vs seasonal")
delta_hot_baseline = paired_bootstrap(s["ht"], s["hs"], baseline_hot_flat, roc_auc_score,
                                       desc="Paired bootstrap: hotspot AUC vs LSTM+GATv2")

print("\nHAST-GNN vs Persistence   — Hotspot AUC difference: "
      f"{delta_hot_persist[0]:.4f} [{delta_hot_persist[1]:.4f}, {delta_hot_persist[2]:.4f}]")
print("HAST-GNN vs 7-day Seasonal — Hotspot AUC difference: "
      f"{delta_hot_season[0]:.4f} [{delta_hot_season[1]:.4f}, {delta_hot_season[2]:.4f}]")
print("HAST-GNN vs LSTM+GATv2     — Hotspot AUC difference: "
      f"{delta_hot_baseline[0]:.4f} [{delta_hot_baseline[1]:.4f}, {delta_hot_baseline[2]:.4f}]")

# --- Crime Type / Hour Accuracy vs the LSTM+GATv2 baseline (naive baselines
#     don't produce type/hour predictions, so only the learned baseline is
#     comparable here) ---
delta_type_baseline = paired_bootstrap(
    s["typ_t"], s["typ_p"], s_bl["typ_p"], lambda y, p: accuracy_score(y, p),
    desc="Paired bootstrap: type acc vs LSTM+GATv2")
delta_hour_baseline = paired_bootstrap(
    s["per_t"], s["per_p"], s_bl["per_p"], lambda y, p: accuracy_score(y, p),
    desc="Paired bootstrap: hour acc vs LSTM+GATv2")

print("\nHAST-GNN vs LSTM+GATv2 — Type Acc difference: "
      f"{delta_type_baseline[0]:.4f} [{delta_type_baseline[1]:.4f}, {delta_type_baseline[2]:.4f}]")
print("HAST-GNN vs LSTM+GATv2 — Hour Acc difference: "
      f"{delta_hour_baseline[0]:.4f} [{delta_hour_baseline[1]:.4f}, {delta_hour_baseline[2]:.4f}]")

print("\n✅ Point 10 (baselines, multi-seed, bootstrap CIs, paired comparison) complete.")

In [ ]:
# =============================================================================
# TRAIN SEED 2024 ONLY — standalone, single seed, no loading
#
# You already have hast_gnn_best.pth (primary) and baseline_lstm_gat_best.pth
# trained and saved, plus seed 42 / seed 123 results from prior runs. This
# script touches NONE of that — it does not load, evaluate, or retrain any
# of those. It runs only:
#   1) the data pipeline (needed so tensors/scaler/splits match exactly)
#   2) the HAST_GNN model class definitions
#   3) a single fresh training run for seed 2024 (cfg.MULTI_SEEDS[-1]) and
#      prints its test-set result
# =============================================================================

# =============================================================================
# HAST-GNN — Chicago Crime Prediction  v14
#
# INHERITS v13 fully:
#   Architecture:  ARATA + DWConv1d | CTIG | AHG+residual | MAMTO
#   Features:      hour-bucket feats, rolling type counts, loc-type feats
#   v13 changes:   CrimeType -> 10 fine-grained classes, CrimeHour -> 8 buckets
#
# CHANGES in v14 (this pass — preprocessing/pipeline fixes only, no architecture
# or loss changes):
#
#   FIX-1  train_mask referenced before assignment (real bug in v13)
#   FIX-2  Explicit, non-ambiguous StandardScaler fit/transform on train/val/test
#   FIX-3  CrimeDS split boundary indices fixed (N_TRAIN_DAYS/N_VAL_DAYS/N_TEST_DAYS)
#   FIX-4  Sequence-level temporal audit with hard assertions
#   GPU-1  Rolling-feature block runs on cuDF when available, pandas fallback
#   GPU-2  Vectorized precompute + native agg for the daily-grid build
#
# CHANGES in this pass (review-doc "Point 10" additions):
#
#   REFACTOR-1  evaluate() now accepts an optional `eval_model` argument so it
#     can score any model (HAST-GNN or a baseline) that returns the same
#     6-tuple (hot, loc, typ, per, cnt, trd). Falls back to the global `model`
#     when eval_model is omitted, so all existing call sites keep working
#     unmodified.
#
#   REFACTOR-2  The main training loop is now the reusable function
#     `train_hast_gnn(...)`. The primary run below calls it once; Point 10.4
#     (multi-seed) and Point 10.3 (baseline) call the SAME function instead of
#     duplicating the 120-epoch loop. Behavior/printed metrics for the primary
#     run are unchanged.
#
#   10.1  Persistence baseline (lag-1 "predict tomorrow = today")
#   10.2  Seasonal 7-day baseline ("predict day t = day t-7")
#   10.3  Compact LSTM + GATv2 spatio-temporal baseline, same inputs/graph/
#         split/targets/loss, trained and evaluated separately from HAST-GNN.
#   10.4  Multi-seed HAST-GNN runs (reuses train_hast_gnn, no code duplication)
#   10.5  Multi-seed mean +/- SD summary
#   10.6  Bootstrap 95% CIs for the single best HAST-GNN run
#   10.7  Paired bootstrap: HAST-GNN vs persistence / seasonal / LSTM-GAT
#
#   ALIGNMENT FIX (flagged by the review doc as something to double check
#   before running 10.7): the doc's example 10.1/10.2 code slices
#   log_cnt_np[test_start+T : NUM_DAYS], which is off by one relative to the
#   actual number of test sequences produced by CrimeDS. CrimeDS.__len__ is
#   max(0, (e-s)-T-1) -- note the extra "-1" -- so the last usable target day
#   is NUM_DAYS-2, not NUM_DAYS-1. Using NUM_DAYS as the upper bound gives the
#   baseline arrays one MORE row than s["cr"]/s["ht"] coming out of
#   evaluate(test_loader), which would silently misalign every paired
#   comparison in 10.7. Below, the baseline target window is derived directly
#   from len(test_ds) instead of hand re-deriving the slice bounds, so it is
#   guaranteed to match the HAST-GNN test predictions row-for-row.
#
#   PROGRESS-1 (this pass): tqdm progress bars added throughout so long,
#   quiet stretches (verbose=False multi-seed / baseline runs) still show
#   live progress instead of going silent for the whole run.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    roc_auc_score, precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# GPU-1: optional cuDF backend for the rolling-feature block
# =============================================================================
HAS_CUDF = False
_cudf_err = None
if torch.cuda.is_available():
    try:
        import cudf
        import cupy as cp
        _probe = cudf.DataFrame({"a": [1, 2, 3]})
        _ = _probe.groupby("a").size()
        cp.array([1.0]).sum()
        HAS_CUDF = True
    except Exception as e:
        HAS_CUDF = False
        _cudf_err = e
if HAS_CUDF:
    print("cuDF GPU feature engineering: ENABLED")
else:
    print("cuDF GPU feature engineering: DISABLED (pandas fallback)")
    if _cudf_err is not None:
        print(f"  cudf/cupy import/probe failed ({type(_cudf_err).__name__}: {_cudf_err})")
        print("  This is usually a broken cupy/cudf install, not a code bug — common causes:")
        print("    - mismatched cupy-cudaXX wheel vs the container's CUDA version")
        print("    - a stray local file/folder named cupy.py or cudf.py shadowing the real package")
        print("    - a half-completed `pip install`/kernel restart needed")
        print("  Falling back to pandas; the pipeline will still run correctly, just on CPU for this block.")


def gpu_shift_rolling(df, group_col, value_col, window, min_periods, shift=1,
                       agg="mean"):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values,
                               value_col: df[value_col].values})
        gdf["_shifted"] = gdf.groupby(group_col)[value_col].shift(shift)
        roll = gdf.groupby(group_col)["_shifted"].rolling(window, min_periods=min_periods)
        if agg == "mean":
            out = roll.mean()
        elif agg == "sum":
            out = roll.sum()
        elif agg == "std":
            out = roll.std()
        else:
            raise ValueError(agg)
        out = out.reset_index(drop=True)
        result = out.to_pandas().values.astype(np.float32)
        if agg == "std":
            result = np.nan_to_num(result, nan=0.0)
        return result
    else:
        if agg == "mean":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).mean())
        elif agg == "sum":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).sum())
        elif agg == "std":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).std().fillna(0))
        else:
            raise ValueError(agg)
        return s.values.astype(np.float32)


def gpu_shift(df, group_col, value_col, lag):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values, value_col: df[value_col].values})
        out = gdf.groupby(group_col)[value_col].shift(lag)
        return out.reset_index(drop=True).to_pandas().fillna(0).values.astype(np.float32)
    else:
        return df.groupby(group_col)[value_col].transform(lambda x: x.shift(lag).fillna(0)).values.astype(np.float32)


def gpu_build_daily_grid(crimes_df, group_cols, agg_dict, has_cudf):
    needed_cols = list(group_cols) + list(agg_dict.keys())
    sub = crimes_df[needed_cols]

    if has_cudf:
        gdf = cudf.DataFrame.from_pandas(sub)
        out = gdf.groupby(group_cols, as_index=False).agg(agg_dict)
        out = out.to_pandas()
    else:
        out = sub.groupby(group_cols, as_index=False).agg(agg_dict)

    new_cols = []
    for c in out.columns:
        if isinstance(c, tuple):
            base, aggname = c
            new_cols.append(base if aggname in ("", None) else f"{base}__{aggname}")
        else:
            new_cols.append(c)
    out.columns = new_cols
    return out


def gpu_rolling_argmax_sum(df, group_col, value_cols, window, min_periods, shift=1):
    rolled = np.zeros((len(df), len(value_cols)), dtype=np.float32)
    for k, col in enumerate(value_cols):
        r = gpu_shift_rolling(df, group_col, col, window, min_periods, shift=shift, agg="sum")
        nan_mask = np.isnan(r)
        if nan_mask.any():
            exp = df.groupby(group_col)[col].transform(
                lambda x: x.shift(shift).expanding(min_periods=1).sum()).values
            r = np.where(nan_mask, exp, r)
        rolled[:, k] = np.nan_to_num(r, nan=0.0)
    argmax = np.argmax(rolled, axis=1).astype(np.int64)
    return rolled, argmax


# =============================================================================
# CONFIG
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9          # top-9 raw types + "OTHER" = 10 classes
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

    # Point 10.3 — compact baseline hyperparams (kept separate from HAST-GNN)
    BASELINE_HIDDEN   = 64
    BASELINE_HEADS    = 4
    BASELINE_EPOCHS   = 60
    BASELINE_PATIENCE = 15
    BASELINE_LR       = 3e-4

    # Point 10.4 — multi-seed experiment
    MULTI_SEEDS = [42, 123, 2024]

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("HAST-GNN v14 | CrimeType=10-class | CrimeHour=8-bucket | split/scaler/dataset fixes | GPU rolling feats")


# =============================================================================
# SEED RESUME CONFIG
# =============================================================================
# All seeds retrain from scratch this run (including 42) — nothing gets
# loaded from a checkpoint. Leave this empty.
SEEDS_ALREADY_COMPLETE  = set()

# Where to LOOK for the seed-42 checkpoint. Kaggle input datasets are
# read-only, so if you uploaded it as a model/dataset it lives under
# /kaggle/input/... , not the working directory. New checkpoints created
# during THIS run (seed 123, seed 2024, flag files) get written to the
# normal writable working directory.
CKPT_SEARCH_DIRS = [
    os.getcwd(),
    "/kaggle/working",
    "/kaggle/input/models/slazayslaer/model/pytorch/default/1",   # <-- edit if needed
]

def find_checkpoint(filename, what):
    """Return the first existing path for `filename` across CKPT_SEARCH_DIRS,
    or raise a clear error listing every place that was checked."""
    for d in CKPT_SEARCH_DIRS:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            return candidate
    checked = "\n  ".join(os.path.join(d, filename) for d in CKPT_SEARCH_DIRS)
    raise FileNotFoundError(
        f"Expected checkpoint '{filename}' for {what} but couldn't find it. "
        f"Checked:\n  {checked}\n"
        f"If it really didn't finish, remove it from SEEDS_ALREADY_COMPLETE "
        f"so it retrains instead. Otherwise add its real folder to "
        f"CKPT_SEARCH_DIRS."
    )
# =============================================================================
# ADJACENCY GRAPHS  (unchanged)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# LOAD DATA  (unchanged — one-time pass with arbitrary lambdas, stays on pandas)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
print("Reading individual crime events (8.3M rows)...")

chunk_dfs = []
for i, chunk in enumerate(tqdm(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"]),
        desc="Reading CSV chunks", unit="chunk")):

    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    ad = chunk.groupby(["date","ca"]).size()
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows → {len(ad):,} area-days")

print("Combining all chunks...")
crimes = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

# =============================================================================
# TEMPORAL SPLIT DEFINITIONS
# =============================================================================
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")

print(
    f"Temporal split definitions: "
    f"Train <= {split_train.date()} | "
    f"Validation <= {split_val.date()} | "
    f"Test > {split_val.date()}"
)

print(f"\n✓ Community areas: {crimes['ca'].nunique()}  |  Days: {NUM_DAYS}")
print(f"  Date range: {all_dates.iloc[0].date()} → {all_dates.iloc[-1].date()}")
print(f"  Total crime events: {len(crimes):,}")

type_counts  = crimes["Primary Type"].value_counts()
top_types    = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"\nCrime types ({N_TYPES}): {TYPE_CLASSES}")

# =============================================================================
# BUILD DAILY AGGREGATED GRID
# =============================================================================
print("\nComputing features...")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]

crimes["_one"]     = np.int8(1)
crimes["is_eve"]   = (crimes["hour"] >= 18).astype(np.int8)
crimes["is_night"] = (crimes["hour"] < 6).astype(np.int8)
crimes["arrest_i"] = crimes["Arrest"].astype(bool).astype(np.int8)

type_onehot_cols = []
for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    crimes[col] = (crimes["Primary Type"] == t).astype(np.int8)
    type_onehot_cols.append(col)

for s, e in HOUR_BUCKETS:
    col = f"hour_{s}_{e}"
    crimes[col] = ((crimes["hour"] >= s) & (crimes["hour"] < e)).astype(np.int8)

agg_dict = {
    "_one":     ["sum"],
    "hour":     ["mean"],
    "is_eve":   ["sum"],
    "is_night": ["sum"],
    "Latitude":  ["mean", "std"],
    "Longitude": ["mean", "std"],
    "arrest_i": ["mean"],
}
for col in type_onehot_cols:
    agg_dict[col] = ["sum"]
for col in HOUR_BUCKET_COLS:
    agg_dict[col] = ["sum"]

grid = gpu_build_daily_grid(crimes, ["date","ca"], agg_dict, HAS_CUDF)

rename_map = {
    "_one__sum": "crime_count",
    "hour__mean": "avg_hour",
    "is_eve__sum": "eve_cnt",
    "is_night__sum": "night_cnt",
    "Latitude__mean": "mean_lat",
    "Latitude__std": "std_lat",
    "Longitude__mean": "mean_lon",
    "Longitude__std": "std_lon",
    "arrest_i__mean": "arrest_rate",
}
for col in type_onehot_cols:
    rename_map[f"{col}__sum"] = col
for col in HOUR_BUCKET_COLS:
    rename_map[f"{col}__sum"] = col
grid = grid.rename(columns=rename_map)

crimes.drop(columns=["_one","is_eve","is_night","arrest_i"] + type_onehot_cols + HOUR_BUCKET_COLS,
            inplace=True)

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)

RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)

for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"])
            .reindex(full_idx, fill_value=0)
            .reset_index()
            .sort_values(["date","ca"])
            .reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

# ── GPU-1: rolling temporal features on crime_count ───────────────────────────
grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["roll7"]  = gpu_shift_rolling(grid, "ca", "crime_count", 7,  1, agg="mean")
grid["roll14"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 1, agg="mean")
grid["roll30"] = gpu_shift_rolling(grid, "ca", "crime_count", 30, 1, agg="mean")
grid["roll90"] = gpu_shift_rolling(grid, "ca", "crime_count", 90, 1, agg="mean")
grid["std7"]   = gpu_shift_rolling(grid, "ca", "crime_count", 7,  2, agg="std")
grid["std30"]  = gpu_shift_rolling(grid, "ca", "crime_count", 30, 2, agg="std")
grid["lag1"]   = gpu_shift(grid, "ca", "crime_count", 1)
grid["lag7"]   = gpu_shift(grid, "ca", "crime_count", 7)
grid["lag14"]  = gpu_shift(grid, "ca", "crime_count", 14)
grid["lag28"]  = gpu_shift(grid, "ca", "crime_count", 28)
grid["wow"] = (
    grid.groupby("ca")["roll7"]
        .transform(lambda x: (x / x.shift(7).clip(lower=1)).fillna(1))
)
grid["mom"] = (
    grid.groupby("ca")["roll14"]
        .transform(lambda x: (x / x.shift(14).clip(lower=1)).fillna(1))
)
grid["roll_hour7"]  = gpu_shift_rolling(grid, "ca", "avg_hour", 7,  1, agg="mean")
grid["roll_hour30"] = gpu_shift_rolling(grid, "ca", "avg_hour", 30, 1, agg="mean")

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col  = f"r7_{col}"
    r30_col = f"r30_{col}"
    grid[r7_col]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
    grid[r30_col] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)

# =============================================================================
# TRAINING-ONLY GEOGRAPHIC AREA STATISTICS
# =============================================================================
train_rows_geo = grid["date"] <= split_train

train_area_lat = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lat"]
        .mean()
)
train_area_lon = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lon"]
        .mean()
)
grid["area_lat"] = grid["ca"].map(train_area_lat)
grid["area_lon"] = grid["ca"].map(train_area_lon)
grid["area_lat"] = grid["area_lat"].fillna(grid["mean_lat"])
grid["area_lon"] = grid["area_lon"].fillna(grid["mean_lon"])

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
        grid[f"r30_{col}"] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")

loc_feats = []
for lt in tqdm(cfg.TOP_LOC_TYPES, desc="Building location-type features"):
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1)
               .reindex(full_idx, fill_value=0)
               .reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom",
              "roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]

FEAT_COLS = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
             HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT    = len(FEAT_COLS)
print(f"Features: {N_FEAT}  (temporal:{len(TEMP_COLS)} cal:{len(CAL_COLS)} "
      f"type_r:{len(TYPE_RCOLS)} misc:{len(MISC_COLS)} "
      f"hour_buckets:{len(HOUR_BUCKET_COLS)+len(HOUR_BUCKET_ROLL_COLS)} loc:{len(LOC_COLS)})")

grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# =============================================================================
# FIX-1: FINAL CHRONOLOGICAL SPLIT MASKS
# =============================================================================
train_mask = grid["date"].le(split_train)
val_mask   = grid["date"].gt(split_train) & grid["date"].le(split_val)
test_mask  = grid["date"].gt(split_val)

train_dates = all_dates[all_dates <= split_train]
val_dates   = all_dates[(all_dates > split_train) & (all_dates <= split_val)]
test_dates  = all_dates[all_dates > split_val]

n_train = len(train_dates)
n_val   = len(val_dates)

print("\n========== FINAL TEMPORAL SPLIT ==========")
print(
    f"TRAIN:      {train_dates.iloc[0].date()} → "
    f"{train_dates.iloc[-1].date()} | {len(train_dates):,} days"
)
print(
    f"VALIDATION: {val_dates.iloc[0].date()} → "
    f"{val_dates.iloc[-1].date()} | {len(val_dates):,} days"
)
print(
    f"TEST:       {test_dates.iloc[0].date()} → "
    f"{test_dates.iloc[-1].date()} | {len(test_dates):,} days"
)
print("\nGrid rows:")
print(f"  Train:      {train_mask.sum():,}")
print(f"  Validation: {val_mask.sum():,}")
print(f"  Test:       {test_mask.sum():,}")

assert train_mask.sum() == n_train * N_COMM
assert val_mask.sum()   == len(val_dates) * N_COMM
assert test_mask.sum()  == len(test_dates) * N_COMM
assert grid.loc[train_mask, "date"].max() == split_train
assert grid.loc[val_mask, "date"].min() > split_train
assert grid.loc[val_mask, "date"].max() == split_val
assert grid.loc[test_mask, "date"].min() > split_val
assert grid.loc[test_mask, "date"].max() == all_dates.iloc[-1]

print("\n✓ Temporal split masks are correctly aligned.")
print("============================================\n")

# =============================================================================
# LABELS
# =============================================================================
print("Computing labels...")

grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)
print(f"  Hotspot rate: {grid.loc[train_mask,'is_hotspot'].mean():.3f}")

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lat centroids")], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lon centroids")], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std
print(f"  Lat: mean={lat_mean:.5f}  std={lat_std:.5f}")
print(f"  Lon: mean={lon_mean:.5f}  std={lon_std:.5f}")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
cnt_cols_fine = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in TYPE_CLASSES
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]

top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)

all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER"
                for t in TYPE_CLASSES]

rolled_type, dominant_type_argmax = gpu_rolling_argmax_sum(
    grid, "ca", all_cnt_cols, window=14, min_periods=3, shift=1)
grid["dominant_type"] = dominant_type_argmax
zero_mask = rolled_type.sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

type_dist_tr = pd.Series(grid.loc[train_mask,"dominant_type"].values).value_counts().sort_index()
type_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_type"].values).value_counts().sort_index()
print(f"  Type label (10-class fine-grained, 14d rolling)  [train% / test%]:")
for k, name in enumerate(TYPE_CLASSES):
    ptr = type_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = type_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:30}: {ptr:5.1f}% / {pte:5.1f}%")

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS         = 8

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
rolled_bucket, dominant_period_argmax = gpu_rolling_argmax_sum(
    grid, "ca", RAW_BUCKET_COLS, window=14, min_periods=3, shift=1)
grid["dominant_period"] = dominant_period_argmax
zero_mask_b = rolled_bucket.sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

per_dist_tr = pd.Series(grid.loc[train_mask,"dominant_period"].values).value_counts().sort_index()
per_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_period"].values).value_counts().sort_index()
print(f"  Hour bucket label (8-class dynamic, 14d rolling)  [train% / test%]:")
for k, name in enumerate(HOUR_BUCKET_NAMES):
    ptr = per_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = per_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:10}: {ptr:5.1f}% / {pte:5.1f}%")

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area log-count means")], dtype=np.float32)
grid["cnt_z"] = grid["log_cnt"]
print(f"  Log count range: [{grid.loc[train_mask,'log_cnt'].min():.2f}, "
      f"{grid.loc[train_mask,'log_cnt'].max():.2f}]")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["wk2_now"]   = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=1,  agg="mean")
grid["wk2_prior"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=15, agg="mean")
grid["wk2_prior"] = grid["wk2_prior"].clip(1)
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)
tr_dist = grid.loc[train_mask,"trend"].value_counts().sort_index()
print(f"  Trend — Fall:{tr_dist.get(0,0)/train_mask.sum()*100:.1f}%  "
      f"Stable:{tr_dist.get(1,0)/train_mask.sum()*100:.1f}%  "
      f"Rise:{tr_dist.get(2,0)/train_mask.sum()*100:.1f}%")

print("\n========== LEAKAGE / MASK AUDIT ==========")
print("Current grid first date:", grid["date"].iloc[0])
print("Current grid last date :", grid["date"].iloc[-1])
print("\nMask counts:")
print("train_mask:", train_mask.sum())
print("val_mask:  ", val_mask.sum())
print("test_mask: ", test_mask.sum())
print("\nDates selected by masks:")
print("TRAIN:", grid.loc[train_mask, "date"].min(), "→", grid.loc[train_mask, "date"].max())
print("VAL:", grid.loc[val_mask, "date"].min(), "→", grid.loc[val_mask, "date"].max())
print("TEST:", grid.loc[test_mask, "date"].min(), "→", grid.loc[test_mask, "date"].max())
print("============================================")
print("============================================")

# =============================================================================
# TEMPORAL FEATURE LEAKAGE AUDIT
# =============================================================================
print("\n========== TEMPORAL FEATURE LEAKAGE AUDIT ==========")

TEMPORAL_AUDIT_COLS = [
    "roll7", "roll14", "roll30", "roll90",
    "std7", "std30",
    "lag1", "lag7", "lag14", "lag28",
    "wow", "mom",
    "roll_hour7", "roll_hour30",
]

for col in tqdm(TEMPORAL_AUDIT_COLS, desc="Auditing temporal features"):
    assert col in grid.columns, f"Missing temporal feature: {col}"
    assert np.isfinite(
        grid[col].to_numpy(dtype=np.float64)
    ).all(), f"NaN/Inf detected in temporal feature: {col}"

print(f"✓ {len(TEMPORAL_AUDIT_COLS)} temporal features exist and are finite.")

audit_grid = grid[["ca", "day", "crime_count"]].copy()
audit_grid = audit_grid.sort_values(["ca", "day"]).reset_index(drop=True)

audit_grid["expected_lag1"]  = audit_grid.groupby("ca")["crime_count"].shift(1).fillna(0)
audit_grid["expected_lag7"]  = audit_grid.groupby("ca")["crime_count"].shift(7).fillna(0)
audit_grid["expected_lag14"] = audit_grid.groupby("ca")["crime_count"].shift(14).fillna(0)
audit_grid["expected_lag28"] = audit_grid.groupby("ca")["crime_count"].shift(28).fillna(0)

stored = grid.sort_values(["ca", "day"]).reset_index(drop=True)

assert np.allclose(stored["lag1"].to_numpy(), audit_grid["expected_lag1"].to_numpy(), atol=1e-5), "❌ lag1 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag7"].to_numpy(), audit_grid["expected_lag7"].to_numpy(), atol=1e-5), "❌ lag7 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag14"].to_numpy(), audit_grid["expected_lag14"].to_numpy(), atol=1e-5), "❌ lag14 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag28"].to_numpy(), audit_grid["expected_lag28"].to_numpy(), atol=1e-5), "❌ lag28 temporal leakage/mismatch detected!"

print("✓ Lag features verified: current-day crime counts are excluded.")

expected_roll7 = (
    audit_grid.groupby("ca")["crime_count"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    .fillna(0)
    .to_numpy()
)
stored_roll7 = stored["roll7"].to_numpy()
mature = stored["day"].to_numpy() >= 7
assert np.allclose(stored_roll7[mature], expected_roll7[mature], atol=1e-4, rtol=1e-4), "❌ roll7 temporal leakage/mismatch detected!"
print("✓ roll7 verified: only previous 7 days are used.")

area_first = stored.groupby("ca", sort=False).head(1)
assert np.allclose(area_first["lag1"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag1!"
assert np.allclose(area_first["lag7"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag7!"
assert np.allclose(area_first["lag14"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag14!"
assert np.allclose(area_first["lag28"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag28!"
print("✓ Cross-area temporal leakage check passed.")

print("✓ Crime-type labels use lagged 14-day rolling history.")
print("✓ Hour-bucket labels use lagged 14-day rolling history.")
print("✓ Trend labels use lagged rolling windows.")

print("\n✓ TEMPORAL FEATURE LEAKAGE AUDIT PASSED")
print("====================================================")

# =============================================================================
# FIX-2: TRAINING-ONLY FEATURE NORMALIZATION (explicit train/val/test)
# =============================================================================
scaler = StandardScaler()
train_features = grid.loc[train_mask, FEAT_COLS]
scaler.fit(train_features)

grid.loc[train_mask, FEAT_COLS] = scaler.transform(grid.loc[train_mask, FEAT_COLS])
grid.loc[val_mask,   FEAT_COLS] = scaler.transform(grid.loc[val_mask,   FEAT_COLS])
grid.loc[test_mask,  FEAT_COLS] = scaler.transform(grid.loc[test_mask,  FEAT_COLS])

grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)
print("✓ StandardScaler fitted using training data only.")

feat_np    = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np     = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np     = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np     = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw    = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw    = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np    = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np     = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np   = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np   = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)

assert not np.isnan(feat_np).any(), "NaN in features!"
print(f"\nTensors OK: feat{feat_np.shape}  type{type_np.shape}  period{per_np.shape}")
print(f"  N_TYPES={N_TYPES}  N_PERIODS={N_PERIODS}")

# =============================================================================
# GRAPH EDGES  (unchanged)
# =============================================================================
def build_ei(adj, id2node):
    src,dst=[],[]
    for n,nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src,dst],dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1,N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1,N_COMM+1)],
                            dtype=torch.long).to(DEVICE)
print(f"Graphs: fine={fine_edge.shape[1]}e  coarse={coarse_edge.shape[1]}e")

# =============================================================================
# DATASET
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T=T; sl=slice(s,e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n=max(0,(e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t=idx+self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]),
                )

# =============================================================================
# FIX-3: correct absolute day-index boundaries for train/val/test CrimeDS
# =============================================================================
T=cfg.TIMESTEPS
N_TRAIN_DAYS = len(train_dates)
N_VAL_DAYS   = len(val_dates)
N_TEST_DAYS  = len(test_dates)

train_ds = CrimeDS(0, N_TRAIN_DAYS, T)
val_ds   = CrimeDS(N_TRAIN_DAYS, N_TRAIN_DAYS + N_VAL_DAYS, T)
test_ds  = CrimeDS(N_TRAIN_DAYS + N_VAL_DAYS, NUM_DAYS, T)

dl_kw=dict(batch_size=cfg.BATCH,num_workers=cfg.NUM_WORKERS,pin_memory=True,
           persistent_workers=cfg.NUM_WORKERS>0)
train_loader=DataLoader(train_ds,shuffle=True,**dl_kw)
val_loader  =DataLoader(val_ds,  shuffle=False,**dl_kw)
test_loader =DataLoader(test_ds, shuffle=False,**dl_kw)
print(f"Sequences — Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}")

# =============================================================================
# FIX-4: SEQUENCE TEMPORAL AUDIT
# =============================================================================
print("\n========== SEQUENCE TEMPORAL AUDIT ==========")
print("Training period:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("Validation period:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("Test period:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())

print("\nExpected sequence counts:")
print("  Train:", max(0, N_TRAIN_DAYS - T - 1))
print("  Val:  ", max(0, N_VAL_DAYS - T - 1))
print("  Test: ", max(0, N_TEST_DAYS - T - 1))

assert len(train_ds) == max(0, N_TRAIN_DAYS - T - 1)
assert len(val_ds)   == max(0, N_VAL_DAYS - T - 1)
assert len(test_ds)  == max(0, N_TEST_DAYS - T - 1)

print("\n✓ Sequence counts and chronological boundaries verified.")
print("===============================================\n")

print("\n========== DATASET AUDIT ==========")
print("Raw records:                    ", len(crimes))
print("Earliest date:                  ", crimes["date"].min())
print("Latest date:                    ", crimes["date"].max())
print("Unique dates:                   ", crimes["date"].nunique())
print("Cleaned grid rows:              ", len(grid))
print("Number of community areas:      ", N_COMM)
print("Number of daily graph snapshots:", NUM_DAYS)

print("\n--- EXACT SPLITS ---")
print("TRAIN:")
print("  Date range:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("  Number of days:", len(train_dates))
print("VALIDATION:")
print("  Date range:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("  Number of days:", len(val_dates))
print("TEST:")
print("  Date range:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())
print("  Number of days:", len(test_dates))

print("\n--- GRID ROW COUNTS ---")
print("Train grid rows:       ", len(train_dates) * N_COMM)
print("Validation grid rows:  ", len(val_dates) * N_COMM)
print("Test grid rows:        ", len(test_dates) * N_COMM)

print("\n--- SEQUENCES ---")
print("Train sequences:       ", len(train_ds))
print("Validation sequences:  ", len(val_ds))
print("Test sequences:        ", len(test_ds))
print("====================================\n")

# MODEL: HAST-GNN v14  (architecture IDENTICAL to v13 — no changes here)
# =============================================================================

class ARATABlock(nn.Module):
    """[N1] ARATA + DWConv1d (unchanged)."""
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H=n_heads; self.dh=d//n_heads
        self.qkv=nn.Linear(d,3*d,bias=False); self.out=nn.Linear(d,d,bias=False)
        self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)
        self.log_lam_bwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam)))
        self.log_lam_fwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam*0.5)))
        self.n_causal=n_heads//2
        self.tconv     =nn.Conv1d(d,d,kernel_size=3,padding=1,groups=d,bias=False)
        self.tconv_norm=nn.LayerNorm(d)

    def forward(self,x):
        B,T,NF,d=x.shape
        xc=x.permute(0,2,3,1).reshape(B*NF,d,T)
        xc=self.tconv(xc)
        xc=xc.reshape(B,NF,d,T).permute(0,3,1,2)
        x=self.tconv_norm(x+xc)
        x2=x.reshape(B*NF,T,d)
        Q,K,V=self.qkv(x2).chunk(3,dim=-1)
        Q=Q.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        K=K.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        V=V.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        pi=torch.arange(T,device=x.device).float().unsqueeze(1)
        pj=torch.arange(T,device=x.device).float().unsqueeze(0)
        diff=pi-pj
        lb=torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf=torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias=(-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0)+
              -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal>0:
            cm=torch.triu(torch.full((T,T),float("-inf"),device=x.device),diagonal=1)
            bias[:self.n_causal]+=cm
        attn=F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0),dim=-1)
        out=(self.drop(attn)@V).transpose(1,2).reshape(B*NF,T,d)
        return self.norm(x+self.out(out).reshape(B,NF,T,d).permute(0,2,1,3))


class CTIGBlock(nn.Module):
    """[N2] Dynamic Crime-Type Interaction Graph (unchanged)."""
    def __init__(self,H,n_types,dropout):
        super().__init__()
        self.K=n_types
        self.proj=nn.Linear(H,H)
        self.eg=nn.Sequential(nn.Linear(2*H,H),nn.GELU(),nn.Linear(H,1))
        self.op=nn.Linear(H,H); self.norm=nn.LayerNorm(H); self.drop=nn.Dropout(dropout)
    def forward(self,tr,hp):
        B,NF,H=hp.shape
        tf=self.proj(tr.unsqueeze(-1)*hp.unsqueeze(2))
        tm=tf.mean(1)
        ti=tm.unsqueeze(2).expand(-1,-1,self.K,-1)
        tj=tm.unsqueeze(1).expand(-1,self.K,-1,-1)
        ew=torch.sigmoid(self.eg(torch.cat([ti,tj],dim=-1))).squeeze(-1)
        k3=min(3,self.K-1)
        tv,tix=ew.topk(k3,dim=-1)
        es=torch.zeros_like(ew).scatter_(2,tix,tv)
        msg=(torch.einsum("bkj,bjh->bkh",es,tm).unsqueeze(1).expand(-1,NF,-1,-1))
        ht=(tr.unsqueeze(-1)*self.op(msg)).sum(2)
        return self.norm(hp+self.drop(ht))


class AHGLayer(nn.Module):
    """[N3] AHG + explicit hf residual (unchanged)."""
    def __init__(self,H,n_fine,n_coarse,heads,dropout):
        super().__init__()
        self.gf=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gc=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gate=nn.Linear(H*2,1); self.norm=nn.LayerNorm(H)
    def forward(self,hf,hc,fei,cei,c2dn):
        B,NF,H=hf.shape
        hf2=torch.stack([self.gf(hf[b],fei) for b in range(B)])
        hc2=torch.stack([self.gc(hc[b],cei) for b in range(B)])
        hcca=hc2[:,c2dn,:]
        alpha=torch.sigmoid(self.gate(torch.cat([hf2,hcca],dim=-1)))
        return self.norm(hf+alpha*hf2+(1-alpha)*hcca), hc2


class HAST_GNN(nn.Module):
    def __init__(self,n_feat,n_fine,n_coarse,H,heads,n_layers,dropout,n_types,n_periods):
        super().__init__()
        self.nf=n_fine; self.nc=n_coarse
        self.pf=nn.Sequential(nn.Linear(n_feat,H),nn.LayerNorm(H),nn.GELU())
        self.pc=nn.Sequential(nn.Linear(H,H),nn.LayerNorm(H),nn.GELU())
        self.arata=nn.ModuleList([ARATABlock(H,heads,dropout) for _ in range(n_layers)])
        self.ahg  =nn.ModuleList([AHGLayer(H,n_fine,n_coarse,heads,dropout) for _ in range(n_layers)])
        self.ctig =CTIGBlock(H,cfg.TOP_N_TYPES,dropout)
        self.mp   =nn.Sequential(nn.Linear(H,H),nn.GELU(),nn.LayerNorm(H))
        self.n_rec=3

        def _h(i,m,o):
            blk=nn.Sequential(nn.Linear(i,m),nn.GELU(),nn.Dropout(dropout),nn.Linear(m,o))
            for l in blk:
                if isinstance(l,nn.Linear): nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
            return blk

        self.hot_h         = _h(H*2, 64, 1)
        self.loc_h         = _h(H+32, 64, 2)
        self.area_lat_bias = nn.Parameter(torch.zeros(n_fine))
        self.area_lon_bias = nn.Parameter(torch.zeros(n_fine))

        self.type_h = _h(H, 128, n_types)
        self.per_h  = _h(H, 64, n_periods)

        self.area_emb      = nn.Embedding(n_fine, 32)
        self.cnt_h         = _h(H+32, 64, 1)
        self.area_log_bias = nn.Parameter(torch.zeros(n_fine))
        self.trend_h       = _h(H*2, 64, 3)

    def forward(self,xf,fei,cei):
        B,T,NF,_=xf.shape
        H=self.pf[0].out_features; DEV=xf.device
        hf=self.pf(xf)
        for a in self.arata: hf=a(hf)
        hp=hf[:,-1,:,:]
        c2dn=CA_TO_DN.to(DEV)
        hc=torch.zeros(B,self.nc,H,device=DEV)
        hc.scatter_reduce_(1,c2dn.unsqueeze(0).unsqueeze(-1).expand(B,-1,H),
                           hp,reduce="mean",include_self=True)
        hc=self.pc(hc)
        for a in self.ahg: hp,hc=a(hp,hc,fei,cei,c2dn)
        hr=hf[:,-self.n_rec:,:,:].mean(1)
        hl=hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom=self.mp(hr-hl)
        hm=torch.cat([hp,mom],dim=-1)
        trs=len(TEMP_COLS)+len(CAL_COLS)
        ntr=min(cfg.TOP_N_TYPES,xf.shape[-1]-trs)
        tr=(xf[:,-1,:,trs:trs+ntr] if ntr>0
            else torch.zeros(B,NF,cfg.TOP_N_TYPES,device=DEV))
        hp=self.ctig(tr,hp)
        area_idx=torch.arange(self.nf,device=DEV).unsqueeze(0).expand(B,-1)
        ae=self.area_emb(area_idx)
        hp_ae=torch.cat([hp,ae],dim=-1)

        hot     =self.hot_h(hm).squeeze(-1)
        loc_res =self.loc_h(hp_ae)
        lat_pred=loc_res[...,0]+self.area_lat_bias.unsqueeze(0)
        lon_pred=loc_res[...,1]+self.area_lon_bias.unsqueeze(0)
        loc     =torch.stack([lat_pred,lon_pred],dim=-1)
        typ     =self.type_h(hp)
        per     =self.per_h(hp)
        cnt     =(self.cnt_h(hp_ae).squeeze(-1)+self.area_log_bias.unsqueeze(0)).clamp(0,6)
        trd     =self.trend_h(hm)
        return hot,loc,typ,per,cnt,trd
hot_rate=grid.loc[train_mask,"is_hotspot"].mean()
pos_wt=torch.tensor((1-hot_rate)/hot_rate,dtype=torch.float32).to(DEVICE)
print(f"Hotspot pos_weight: {pos_wt.item():.3f}")

tc=np.bincount(type_np[:n_train].reshape(-1),minlength=N_TYPES).astype(float)
tc/=tc.sum()
typw=torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES),
                   dtype=torch.float32).to(DEVICE)
print(f"Type weights ({N_TYPES} classes): {typw.cpu().numpy().round(2)}")

pc=np.bincount(per_np[:n_train].reshape(-1),minlength=N_PERIODS).astype(float)
pc/=pc.sum()
perw=torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS),
                   dtype=torch.float32).to(DEVICE)
print(f"Hour bucket weights ({N_PERIODS} classes): {perw.cpu().numpy().round(2)}")

trc=np.bincount(trend_np[:n_train].reshape(-1),minlength=3).astype(float)
trc/=trc.sum()
tw=torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3),dtype=torch.float32).to(DEVICE)
print(f"Trend weights: {tw.cpu().numpy().round(2)}")
# =============================================================================
# LOSS FUNCTIONS  (unchanged)
# =============================================================================

def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls=logits.shape[-1]
    log_p=F.log_softmax(logits,dim=-1)
    p=torch.exp(log_p)
    with torch.no_grad():
        smooth_t=torch.full_like(logits,smoothing/(n_cls-1))
        smooth_t.scatter_(-1,targets.unsqueeze(-1),1.0-smoothing)
    ce_smooth=-(smooth_t*log_p).sum(-1)
    p_t=p.gather(-1,targets.unsqueeze(-1)).squeeze(-1)
    focal_w=(1.0-p_t)**gamma
    if weight is not None:
        focal_w=focal_w*weight[targets]
    focal=(focal_w*ce_smooth).mean()
    entropy=-(p*log_p).sum(-1).mean()
    return focal-entropy_reg*entropy


def compute_loss(hot_l,loc_p,typ_l,per_l,cnt_p,trd_l,
                 hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t):
    l_hot=F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1),hot_t.reshape(-1),pos_weight=pos_wt)
    valid=(lat_t.abs()>0.01)|(lon_t.abs()>0.01)
    if valid.any():
        l_loc=(F.huber_loss(loc_p[...,0][valid],lat_t[valid],delta=1.0)+
               F.huber_loss(loc_p[...,1][valid],lon_t[valid],delta=1.0))
    else:
        l_loc=torch.tensor(0.0,device=hot_l.device)

    l_typ=focal_loss_with_smooth(
        typ_l.reshape(-1,N_TYPES),typ_t.reshape(-1),
        weight=typw,gamma=cfg.FOCAL_GAMMA,
        smoothing=cfg.LABEL_SMOOTH,entropy_reg=cfg.ENTROPY_REG)

    l_per=F.cross_entropy(per_l.reshape(-1,N_PERIODS),per_t.reshape(-1),weight=perw)

    l_cnt=F.huber_loss(cnt_p.reshape(-1),cnt_t.reshape(-1),delta=0.5)
    l_trd=F.cross_entropy(trd_l.reshape(-1,3),trd_t.reshape(-1),weight=tw)

    total=1.0*l_hot+1.0*l_loc+1.5*l_typ+0.6*l_per+1.2*l_cnt+0.5*l_trd
    return total,l_hot,l_loc,l_typ,l_per,l_cnt,l_trd

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1,lo1,la2,lo2):
    R=6371.0
    dlat=np.radians(la2-la1); dlon=np.radians(lo2-lo1)
    a=np.sin(dlat/2)**2+np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a,dtype=np.float32))

# =============================================================================
# EVALUATE
# =============================================================================
# REFACTOR-1: `eval_model` lets this score ANY model with the HAST-GNN output
# signature (hot, loc, typ, per, cnt, trd), e.g. the Point 10.3 baseline or
# a Point 10.4 multi-seed model, without duplicating this function.
def evaluate(loader, eval_model=None, desc="Evaluating", show_pbar=True):
    m = eval_model if eval_model is not None else model
    m.eval()
    ht,hs=[],[]
    latt,lont,latr,lonr,latp,lonp=[],[],[],[],[],[]
    tytt,tytp=[],[]
    pert,perp=[],[]
    cntr,cntp=[],[]
    trt_,trp_=[],[]
    tot=n=0

    iterator = tqdm(loader, desc=desc, leave=False, unit="batch") if show_pbar else loader

    with torch.no_grad():
        for batch in iterator:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=m(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                      hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
            if not torch.isnan(loss):
                tot+=loss.item()*x.shape[0]; n+=x.shape[0]
                if show_pbar:
                    iterator.set_postfix(loss=f"{tot/max(1,n):.4f}")

    ht=np.array(ht,dtype=np.int32); hs=san(hs)
    pr,re,thr=precision_recall_curve(ht,hs)
    f1c=2*pr*re/(pr+re+1e-8)
    bt=float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp=(hs>=bt).astype(int)
    try: rauc=roc_auc_score(ht,hs)
    except: rauc=0.0
    hot_f1=f1_score(ht,hp,zero_division=0)

    lp_raw=san(latp)*lat_std+lat_mean; lr_raw=np.array(latr)
    op_raw=san(lonp)*lon_std+lon_mean; or_raw=np.array(lonr)
    valid=(lr_raw>40)&(or_raw<-80)
    if valid.sum()>0:
        hav=haversine_km(lr_raw[valid],or_raw[valid],lp_raw[valid],op_raw[valid])
        loc_mae=float(hav.mean()); l1=float((hav<1).mean()); l05=float((hav<0.5).mean())
    else:
        loc_mae=99.9; l1=0.0; l05=0.0

    ta=accuracy_score(np.array(tytt),np.array(tytp))
    tf=f1_score(np.array(tytt),np.array(tytp),average="weighted",zero_division=0)
    pa=accuracy_score(np.array(pert),np.array(perp))
    pf=f1_score(np.array(pert),np.array(perp),average="weighted",zero_division=0)
    cr=san(cntr); cp=san(cntp)
    cnt_corr=float(np.corrcoef(cr,cp)[0,1]) if len(cr)>1 else 0.0
    cnt_mae=float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp.clip(0,10))).mean())
    trend_acc=accuracy_score(np.array(trt_),np.array(trp_))

    loc_score=max(0,1-loc_mae/10)
    primary=np.mean([rauc,ta,pa,max(cnt_corr,0),loc_score])

    return dict(
        loss=tot/max(1,n),primary=primary,
        hot_auc=rauc,hot_f1=hot_f1,best_thr=bt,ht=ht,hs=hs,hp=hp,
        loc_mae=loc_mae,loc_1km=l1,loc_05km=l05,
        lat_p=lp_raw,lon_p=op_raw,lat_t=lr_raw,lon_t=or_raw,
        type_acc=ta,type_f1=tf,typ_t=np.array(tytt),typ_p=np.array(tytp),
        per_acc=pa,per_f1=pf,per_t=np.array(pert),per_p=np.array(perp),
        cnt_corr=cnt_corr,cnt_mae=cnt_mae,cr=cr,cp=cp,
        trend_acc=trend_acc,trt=np.array(trt_),trp=np.array(trp_),
    )

# =============================================================================
# REFACTOR-2: reusable training loop
# =============================================================================
# The original v14 training loop is unchanged in behavior — it's just moved
# into a function so Point 10.3 (baseline) and Point 10.4 (multi-seed) can
# call the SAME code instead of copy-pasting the 120-epoch loop.
#
# PROGRESS-1: an epoch-level tqdm bar is now shown REGARDLESS of `verbose`,
# with a live postfix (train loss / val loss / val primary / best-so-far).
# When verbose=True the original per-epoch table is still printed, but via
# tqdm.write() so it doesn't fight with the progress bar's own line. A nested
# per-batch tqdm bar (leave=False) shows within-epoch progress too, since a
# single epoch over ~thousands of sequences can itself take a while.
def train_hast_gnn(train_model, opt_train_loader, opt_val_loader,
                    epochs, patience, lr, max_lr_mult=8, weight_decay=cfg.WEIGHT_DECAY,
                    save_path="model_best.pth", verbose=True, track_history=True,
                    run_name="HAST-GNN"):
    optimizer = torch.optim.AdamW(train_model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr*max_lr_mult, total_steps=epochs*len(opt_train_loader),
        pct_start=0.10, anneal_strategy="cos")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)

    best_p = 0.0; best_ep = 0; nan_streak = 0
    hist = {"tl": [], "vl": [], "vp": [], "vtype": [], "vper": []} if track_history else None

    if verbose:
        hdr=("  Ep|    Loss|   VLoss|   VF1|   VAUC|  VTyp| VHour| VCntR|      LR")
        tqdm.write(hdr); tqdm.write("─"*len(hdr))

    epoch_bar = tqdm(range(1, epochs+1), desc=f"{run_name} epochs", unit="ep")
    for ep in epoch_bar:
        train_model.train(); ep_loss=0.0; ep_n=0
        batch_bar = tqdm(opt_train_loader, desc=f"  ep{ep} train batches",
                          leave=False, unit="batch")
        for batch in batch_bar:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=train_model(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                       hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            if torch.isnan(loss):
                nan_streak+=1
                if nan_streak>=20:
                    tqdm.write(f"\n⚠ 20 NaN — abort")
                    break
                continue
            nan_streak=0
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(train_model.parameters(),0.5)
            amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
            ep_loss+=loss.item()*x.shape[0]; ep_n+=x.shape[0]
            batch_bar.set_postfix(loss=f"{ep_loss/max(1,ep_n):.4f}")
        else:
            s = evaluate(opt_val_loader, eval_model=train_model,
                         desc=f"  ep{ep} val", show_pbar=True)
            tl = ep_loss/max(1,ep_n); lr_now = optimizer.param_groups[0]["lr"]
            if verbose:
                tqdm.write(f"  {ep:3d}| {tl:.5f}| {s['loss']:.5f}|{s['hot_f1']:.4f}| "
                      f"{s['hot_auc']:.4f}| {s['type_acc']:.4f}| {s['per_acc']:.4f}| "
                      f"{s['cnt_corr']:.4f}|{lr_now:.2e}")
            epoch_bar.set_postfix(
                train_loss=f"{tl:.4f}", val_loss=f"{s['loss']:.4f}",
                val_primary=f"{s['primary']:.4f}", best=f"{best_p:.4f}",
                best_ep=best_ep)
            if s["primary"]>best_p:
                best_p=s["primary"]; best_ep=ep
                torch.save(train_model.state_dict(), save_path)
                if verbose:
                    tqdm.write(f"  ✓ Primary={s['primary']:.4f}  HotAUC={s['hot_auc']:.4f}  "
                          f"LocMAE={s['loc_mae']:.2f}km  TypeAcc={s['type_acc']:.4f}  "
                          f"HourAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}")
                epoch_bar.set_postfix(
                    train_loss=f"{tl:.4f}", val_loss=f"{s['loss']:.4f}",
                    val_primary=f"{s['primary']:.4f}", best=f"{best_p:.4f}",
                    best_ep=best_ep)
            if track_history:
                hist["tl"].append(tl); hist["vl"].append(s["loss"])
                hist["vp"].append(s["primary"]); hist["vtype"].append(s["type_acc"])
                hist["vper"].append(s["per_acc"])
            if ep-best_ep>=patience:
                if verbose: tqdm.write(f"\n⏹  Early stop ep {ep}")
                epoch_bar.close()
                break
            continue
        break
    else:
        epoch_bar.close()

    train_model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    train_model.eval()
    return train_model, best_p, hist

# =============================================================================
# =============================================================================
# POINT 10.4 — MULTI-SEED HAST-GNN
# =============================================================================
# Reuses train_hast_gnn(); the 120-epoch loop is NOT duplicated.
# PROGRESS-1: an outer tqdm bar tracks progress across seeds so it's obvious
# how many of the (typically slow, 120-epoch) seed runs remain.
print("\n"+"="*100)
print("  POINT 10.4 — MULTI-SEED HAST-GNN")
print("="*100)

SEEDS = [cfg.MULTI_SEEDS[-1]]   # only the last seed (2024) — nothing else runs or loads
seed_results = []

seed_bar = tqdm(SEEDS, desc="Multi-seed runs", unit="seed")
for seed in seed_bar:
    seed_bar.set_postfix(seed=seed)
    ckpt_name = f"hast_gnn_seed{seed}_best.pth"
    ckpt_path = os.path.join(os.getcwd(), ckpt_name)  # where NEW checkpoints get saved
    flag_path = f"hast_gnn_seed{seed}.complete.flag"
    already_done = (seed in SEEDS_ALREADY_COMPLETE) or (
        any(os.path.exists(os.path.join(d, ckpt_name)) for d in CKPT_SEARCH_DIRS)
        and os.path.exists(flag_path))

    tqdm.write(f"\n{'='*70}")
    tqdm.write(f"{'RESUMING (checkpoint only)' if already_done else 'RUNNING'} "
               f"HAST-GNN v14 | SEED = {seed}")
    tqdm.write(f"{'='*70}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_model = HAST_GNN(
        N_FEAT, N_COMM, N_DIST,
        cfg.HIDDEN_DIM, cfg.GAT_HEADS,
        cfg.N_LAYERS, cfg.DROPOUT,
        N_TYPES, N_PERIODS
    ).to(DEVICE)
    with torch.no_grad():
        seed_model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
        seed_model.area_lat_bias.copy_(torch.tensor(area_lat_z,dtype=torch.float32).to(DEVICE))
        seed_model.area_lon_bias.copy_(torch.tensor(area_lon_z,dtype=torch.float32).to(DEVICE))

    if already_done:
        found_ckpt = find_checkpoint(ckpt_name, f"seed={seed}")
        tqdm.write(f"  [resume] Loading {found_ckpt} — this seed already "
                    f"finished before the interruption, skipping training.")
        seed_model.load_state_dict(torch.load(found_ckpt, map_location=DEVICE))
        seed_model.eval()
    else:
        tqdm.write(f"  [resume] No complete checkpoint for seed={seed} "
                    f"(or it was cut off mid-training) — training from scratch.")
        seed_model, seed_best_p, _ = train_hast_gnn(
            seed_model, train_loader, val_loader,
            epochs=cfg.EPOCHS, patience=cfg.PATIENCE, lr=cfg.LR,
            max_lr_mult=8, weight_decay=cfg.WEIGHT_DECAY,
            save_path=ckpt_path, verbose=False, track_history=False,
            run_name=f"HAST-GNN seed={seed}",
        )
        # Mark this seed as fully complete so a future interruption can skip
        # it automatically (see RESUME CONFIG at the top of the script).
        open(flag_path, "w").close()

    result = evaluate(test_loader, eval_model=seed_model, desc=f"Test eval (seed={seed})")

    seed_results.append({
        "seed": seed,
        "hot_auc": result["hot_auc"],
        "hot_f1": result["hot_f1"],
        "loc_mae": result["loc_mae"],
        "type_acc": result["type_acc"],
        "hour_acc": result["per_acc"],
        "count_r": result["cnt_corr"],
    })
    tqdm.write(f"  Seed {seed} test results: HotAUC={result['hot_auc']:.4f}  "
          f"LocMAE={result['loc_mae']:.2f}km  TypeAcc={result['type_acc']:.4f}  "
          f"HourAcc={result['per_acc']:.4f}  CntR={result['cnt_corr']:.4f}")

    # Flush partial multi-seed results to disk after every seed, so a repeat
    # interruption doesn't lose progress already made in this loop either.
    pd.DataFrame(seed_results).to_csv("hast_gnn_v14_multiseed_results_partial.csv", index=False)

# =============================================================================
# POINT 10.5 — RESULTS SUMMARY (single seed this run: no SD to compute)
# =============================================================================
seed_df = pd.DataFrame(seed_results)

metrics = ["hot_auc", "hot_f1", "loc_mae", "type_acc", "hour_acc", "count_r"]

print("\n========== RESULTS (seed = {}) ==========".format(SEEDS[0]))
if len(seed_df) > 1:
    for m in metrics:
        print(f"{m:12s}: {seed_df[m].mean():.4f} ± {seed_df[m].std(ddof=1):.4f}")
else:
    for m in metrics:
        print(f"{m:12s}: {seed_df[m].iloc[0]:.4f}")

seed_df.to_csv("hast_gnn_v14_multiseed_results.csv", index=False)
print("Saved: hast_gnn_v14_multiseed_results.csv")

In [2]:
# =============================================================================
# TRAIN BASELINE ONLY — LSTM + GATv2 spatio-temporal baseline (Point 10.3)
#
# Standalone script: you already have hast_gnn_best.pth trained (the primary
# HAST-GNN v14 run) and don't want to retrain it. This script runs ONLY the
# data pipeline (needed so every tensor/scaler/split matches exactly) plus
# the compact LSTM+GATv2 baseline model — trains it, saves
# baseline_lstm_gat_best.pth, and prints its test-set results.
#
# The primary HAST_GNN model class/architecture is NOT defined or instantiated
# here at all — it isn't needed to train or evaluate the baseline.
#
# Once this finishes, use hast_gnn_best.pth (already trained) together with
# the baseline_lstm_gat_best.pth this script produces in the full resume
# script (multi-seed / bootstrap / paired-comparison stages).
# =============================================================================

# =============================================================================
# HAST-GNN — Chicago Crime Prediction  v14
#
# INHERITS v13 fully:
#   Architecture:  ARATA + DWConv1d | CTIG | AHG+residual | MAMTO
#   Features:      hour-bucket feats, rolling type counts, loc-type feats
#   v13 changes:   CrimeType -> 10 fine-grained classes, CrimeHour -> 8 buckets
#
# CHANGES in v14 (this pass — preprocessing/pipeline fixes only, no architecture
# or loss changes):
#
#   FIX-1  train_mask referenced before assignment (real bug in v13)
#   FIX-2  Explicit, non-ambiguous StandardScaler fit/transform on train/val/test
#   FIX-3  CrimeDS split boundary indices fixed (N_TRAIN_DAYS/N_VAL_DAYS/N_TEST_DAYS)
#   FIX-4  Sequence-level temporal audit with hard assertions
#   GPU-1  Rolling-feature block runs on cuDF when available, pandas fallback
#   GPU-2  Vectorized precompute + native agg for the daily-grid build
#
# CHANGES in this pass (review-doc "Point 10" additions):
#
#   REFACTOR-1  evaluate() now accepts an optional `eval_model` argument so it
#     can score any model (HAST-GNN or a baseline) that returns the same
#     6-tuple (hot, loc, typ, per, cnt, trd). Falls back to the global `model`
#     when eval_model is omitted, so all existing call sites keep working
#     unmodified.
#
#   REFACTOR-2  The main training loop is now the reusable function
#     `train_hast_gnn(...)`. The primary run below calls it once; Point 10.4
#     (multi-seed) and Point 10.3 (baseline) call the SAME function instead of
#     duplicating the 120-epoch loop. Behavior/printed metrics for the primary
#     run are unchanged.
#
#   10.1  Persistence baseline (lag-1 "predict tomorrow = today")
#   10.2  Seasonal 7-day baseline ("predict day t = day t-7")
#   10.3  Compact LSTM + GATv2 spatio-temporal baseline, same inputs/graph/
#         split/targets/loss, trained and evaluated separately from HAST-GNN.
#   10.4  Multi-seed HAST-GNN runs (reuses train_hast_gnn, no code duplication)
#   10.5  Multi-seed mean +/- SD summary
#   10.6  Bootstrap 95% CIs for the single best HAST-GNN run
#   10.7  Paired bootstrap: HAST-GNN vs persistence / seasonal / LSTM-GAT
#
#   ALIGNMENT FIX (flagged by the review doc as something to double check
#   before running 10.7): the doc's example 10.1/10.2 code slices
#   log_cnt_np[test_start+T : NUM_DAYS], which is off by one relative to the
#   actual number of test sequences produced by CrimeDS. CrimeDS.__len__ is
#   max(0, (e-s)-T-1) -- note the extra "-1" -- so the last usable target day
#   is NUM_DAYS-2, not NUM_DAYS-1. Using NUM_DAYS as the upper bound gives the
#   baseline arrays one MORE row than s["cr"]/s["ht"] coming out of
#   evaluate(test_loader), which would silently misalign every paired
#   comparison in 10.7. Below, the baseline target window is derived directly
#   from len(test_ds) instead of hand re-deriving the slice bounds, so it is
#   guaranteed to match the HAST-GNN test predictions row-for-row.
#
#   PROGRESS-1 (this pass): tqdm progress bars added throughout so long,
#   quiet stretches (verbose=False multi-seed / baseline runs) still show
#   live progress instead of going silent for the whole run.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    roc_auc_score, precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# GPU-1: optional cuDF backend for the rolling-feature block
# =============================================================================
HAS_CUDF = False
_cudf_err = None
if torch.cuda.is_available():
    try:
        import cudf
        import cupy as cp
        _probe = cudf.DataFrame({"a": [1, 2, 3]})
        _ = _probe.groupby("a").size()
        cp.array([1.0]).sum()
        HAS_CUDF = True
    except Exception as e:
        HAS_CUDF = False
        _cudf_err = e
if HAS_CUDF:
    print("cuDF GPU feature engineering: ENABLED")
else:
    print("cuDF GPU feature engineering: DISABLED (pandas fallback)")
    if _cudf_err is not None:
        print(f"  cudf/cupy import/probe failed ({type(_cudf_err).__name__}: {_cudf_err})")
        print("  This is usually a broken cupy/cudf install, not a code bug — common causes:")
        print("    - mismatched cupy-cudaXX wheel vs the container's CUDA version")
        print("    - a stray local file/folder named cupy.py or cudf.py shadowing the real package")
        print("    - a half-completed `pip install`/kernel restart needed")
        print("  Falling back to pandas; the pipeline will still run correctly, just on CPU for this block.")


def gpu_shift_rolling(df, group_col, value_col, window, min_periods, shift=1,
                       agg="mean"):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values,
                               value_col: df[value_col].values})
        gdf["_shifted"] = gdf.groupby(group_col)[value_col].shift(shift)
        roll = gdf.groupby(group_col)["_shifted"].rolling(window, min_periods=min_periods)
        if agg == "mean":
            out = roll.mean()
        elif agg == "sum":
            out = roll.sum()
        elif agg == "std":
            out = roll.std()
        else:
            raise ValueError(agg)
        out = out.reset_index(drop=True)
        result = out.to_pandas().values.astype(np.float32)
        if agg == "std":
            result = np.nan_to_num(result, nan=0.0)
        return result
    else:
        if agg == "mean":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).mean())
        elif agg == "sum":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).sum())
        elif agg == "std":
            s = df.groupby(group_col)[value_col].transform(
                lambda x: x.shift(shift).rolling(window, min_periods=min_periods).std().fillna(0))
        else:
            raise ValueError(agg)
        return s.values.astype(np.float32)


def gpu_shift(df, group_col, value_col, lag):
    if HAS_CUDF:
        gdf = cudf.DataFrame({group_col: df[group_col].values, value_col: df[value_col].values})
        out = gdf.groupby(group_col)[value_col].shift(lag)
        return out.reset_index(drop=True).to_pandas().fillna(0).values.astype(np.float32)
    else:
        return df.groupby(group_col)[value_col].transform(lambda x: x.shift(lag).fillna(0)).values.astype(np.float32)


def gpu_build_daily_grid(crimes_df, group_cols, agg_dict, has_cudf):
    needed_cols = list(group_cols) + list(agg_dict.keys())
    sub = crimes_df[needed_cols]

    if has_cudf:
        gdf = cudf.DataFrame.from_pandas(sub)
        out = gdf.groupby(group_cols, as_index=False).agg(agg_dict)
        out = out.to_pandas()
    else:
        out = sub.groupby(group_cols, as_index=False).agg(agg_dict)

    new_cols = []
    for c in out.columns:
        if isinstance(c, tuple):
            base, aggname = c
            new_cols.append(base if aggname in ("", None) else f"{base}__{aggname}")
        else:
            new_cols.append(c)
    out.columns = new_cols
    return out


def gpu_rolling_argmax_sum(df, group_col, value_cols, window, min_periods, shift=1):
    rolled = np.zeros((len(df), len(value_cols)), dtype=np.float32)
    for k, col in enumerate(value_cols):
        r = gpu_shift_rolling(df, group_col, col, window, min_periods, shift=shift, agg="sum")
        nan_mask = np.isnan(r)
        if nan_mask.any():
            exp = df.groupby(group_col)[col].transform(
                lambda x: x.shift(shift).expanding(min_periods=1).sum()).values
            r = np.where(nan_mask, exp, r)
        rolled[:, k] = np.nan_to_num(r, nan=0.0)
    argmax = np.argmax(rolled, axis=1).astype(np.int64)
    return rolled, argmax


# =============================================================================
# CONFIG
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9          # top-9 raw types + "OTHER" = 10 classes
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

    # Point 10.3 — compact baseline hyperparams (kept separate from HAST-GNN)
    BASELINE_HIDDEN   = 64
    BASELINE_HEADS    = 4
    BASELINE_EPOCHS   = 60
    BASELINE_PATIENCE = 15
    BASELINE_LR       = 3e-4

    # Point 10.4 — multi-seed experiment
    MULTI_SEEDS = [42, 123, 2024]

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("HAST-GNN v14 | CrimeType=10-class | CrimeHour=8-bucket | split/scaler/dataset fixes | GPU rolling feats")

# =============================================================================
# ADJACENCY GRAPHS  (unchanged)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# LOAD DATA  (unchanged — one-time pass with arbitrary lambdas, stays on pandas)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
print("Reading individual crime events (8.3M rows)...")

chunk_dfs = []
for i, chunk in enumerate(tqdm(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"]),
        desc="Reading CSV chunks", unit="chunk")):

    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    ad = chunk.groupby(["date","ca"]).size()
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows → {len(ad):,} area-days")

print("Combining all chunks...")
crimes = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

# =============================================================================
# TEMPORAL SPLIT DEFINITIONS
# =============================================================================
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")

print(
    f"Temporal split definitions: "
    f"Train <= {split_train.date()} | "
    f"Validation <= {split_val.date()} | "
    f"Test > {split_val.date()}"
)

print(f"\n✓ Community areas: {crimes['ca'].nunique()}  |  Days: {NUM_DAYS}")
print(f"  Date range: {all_dates.iloc[0].date()} → {all_dates.iloc[-1].date()}")
print(f"  Total crime events: {len(crimes):,}")

type_counts  = crimes["Primary Type"].value_counts()
top_types    = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"\nCrime types ({N_TYPES}): {TYPE_CLASSES}")

# =============================================================================
# BUILD DAILY AGGREGATED GRID
# =============================================================================
print("\nComputing features...")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]

crimes["_one"]     = np.int8(1)
crimes["is_eve"]   = (crimes["hour"] >= 18).astype(np.int8)
crimes["is_night"] = (crimes["hour"] < 6).astype(np.int8)
crimes["arrest_i"] = crimes["Arrest"].astype(bool).astype(np.int8)

type_onehot_cols = []
for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    crimes[col] = (crimes["Primary Type"] == t).astype(np.int8)
    type_onehot_cols.append(col)

for s, e in HOUR_BUCKETS:
    col = f"hour_{s}_{e}"
    crimes[col] = ((crimes["hour"] >= s) & (crimes["hour"] < e)).astype(np.int8)

agg_dict = {
    "_one":     ["sum"],
    "hour":     ["mean"],
    "is_eve":   ["sum"],
    "is_night": ["sum"],
    "Latitude":  ["mean", "std"],
    "Longitude": ["mean", "std"],
    "arrest_i": ["mean"],
}
for col in type_onehot_cols:
    agg_dict[col] = ["sum"]
for col in HOUR_BUCKET_COLS:
    agg_dict[col] = ["sum"]

grid = gpu_build_daily_grid(crimes, ["date","ca"], agg_dict, HAS_CUDF)

rename_map = {
    "_one__sum": "crime_count",
    "hour__mean": "avg_hour",
    "is_eve__sum": "eve_cnt",
    "is_night__sum": "night_cnt",
    "Latitude__mean": "mean_lat",
    "Latitude__std": "std_lat",
    "Longitude__mean": "mean_lon",
    "Longitude__std": "std_lon",
    "arrest_i__mean": "arrest_rate",
}
for col in type_onehot_cols:
    rename_map[f"{col}__sum"] = col
for col in HOUR_BUCKET_COLS:
    rename_map[f"{col}__sum"] = col
grid = grid.rename(columns=rename_map)

crimes.drop(columns=["_one","is_eve","is_night","arrest_i"] + type_onehot_cols + HOUR_BUCKET_COLS,
            inplace=True)

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)

RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)

for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"])
            .reindex(full_idx, fill_value=0)
            .reset_index()
            .sort_values(["date","ca"])
            .reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

# ── GPU-1: rolling temporal features on crime_count ───────────────────────────
grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["roll7"]  = gpu_shift_rolling(grid, "ca", "crime_count", 7,  1, agg="mean")
grid["roll14"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 1, agg="mean")
grid["roll30"] = gpu_shift_rolling(grid, "ca", "crime_count", 30, 1, agg="mean")
grid["roll90"] = gpu_shift_rolling(grid, "ca", "crime_count", 90, 1, agg="mean")
grid["std7"]   = gpu_shift_rolling(grid, "ca", "crime_count", 7,  2, agg="std")
grid["std30"]  = gpu_shift_rolling(grid, "ca", "crime_count", 30, 2, agg="std")
grid["lag1"]   = gpu_shift(grid, "ca", "crime_count", 1)
grid["lag7"]   = gpu_shift(grid, "ca", "crime_count", 7)
grid["lag14"]  = gpu_shift(grid, "ca", "crime_count", 14)
grid["lag28"]  = gpu_shift(grid, "ca", "crime_count", 28)
grid["wow"] = (
    grid.groupby("ca")["roll7"]
        .transform(lambda x: (x / x.shift(7).clip(lower=1)).fillna(1))
)
grid["mom"] = (
    grid.groupby("ca")["roll14"]
        .transform(lambda x: (x / x.shift(14).clip(lower=1)).fillna(1))
)
grid["roll_hour7"]  = gpu_shift_rolling(grid, "ca", "avg_hour", 7,  1, agg="mean")
grid["roll_hour30"] = gpu_shift_rolling(grid, "ca", "avg_hour", 30, 1, agg="mean")

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col  = f"r7_{col}"
    r30_col = f"r30_{col}"
    grid[r7_col]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
    grid[r30_col] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)

# =============================================================================
# TRAINING-ONLY GEOGRAPHIC AREA STATISTICS
# =============================================================================
train_rows_geo = grid["date"] <= split_train

train_area_lat = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lat"]
        .mean()
)
train_area_lon = (
    grid.loc[train_rows_geo]
        .groupby("ca")["mean_lon"]
        .mean()
)
grid["area_lat"] = grid["ca"].map(train_area_lat)
grid["area_lon"] = grid["ca"].map(train_area_lon)
grid["area_lat"] = grid["area_lat"].fillna(grid["mean_lat"])
grid["area_lon"] = grid["area_lon"].fillna(grid["mean_lon"])

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = gpu_shift_rolling(grid, "ca", col, 7,  1, agg="mean")
        grid[f"r30_{col}"] = gpu_shift_rolling(grid, "ca", col, 30, 1, agg="mean")

loc_feats = []
for lt in tqdm(cfg.TOP_LOC_TYPES, desc="Building location-type features"):
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1)
               .reindex(full_idx, fill_value=0)
               .reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom",
              "roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]

FEAT_COLS = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
             HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT    = len(FEAT_COLS)
print(f"Features: {N_FEAT}  (temporal:{len(TEMP_COLS)} cal:{len(CAL_COLS)} "
      f"type_r:{len(TYPE_RCOLS)} misc:{len(MISC_COLS)} "
      f"hour_buckets:{len(HOUR_BUCKET_COLS)+len(HOUR_BUCKET_ROLL_COLS)} loc:{len(LOC_COLS)})")

grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# =============================================================================
# FIX-1: FINAL CHRONOLOGICAL SPLIT MASKS
# =============================================================================
train_mask = grid["date"].le(split_train)
val_mask   = grid["date"].gt(split_train) & grid["date"].le(split_val)
test_mask  = grid["date"].gt(split_val)

train_dates = all_dates[all_dates <= split_train]
val_dates   = all_dates[(all_dates > split_train) & (all_dates <= split_val)]
test_dates  = all_dates[all_dates > split_val]

n_train = len(train_dates)
n_val   = len(val_dates)

print("\n========== FINAL TEMPORAL SPLIT ==========")
print(
    f"TRAIN:      {train_dates.iloc[0].date()} → "
    f"{train_dates.iloc[-1].date()} | {len(train_dates):,} days"
)
print(
    f"VALIDATION: {val_dates.iloc[0].date()} → "
    f"{val_dates.iloc[-1].date()} | {len(val_dates):,} days"
)
print(
    f"TEST:       {test_dates.iloc[0].date()} → "
    f"{test_dates.iloc[-1].date()} | {len(test_dates):,} days"
)
print("\nGrid rows:")
print(f"  Train:      {train_mask.sum():,}")
print(f"  Validation: {val_mask.sum():,}")
print(f"  Test:       {test_mask.sum():,}")

assert train_mask.sum() == n_train * N_COMM
assert val_mask.sum()   == len(val_dates) * N_COMM
assert test_mask.sum()  == len(test_dates) * N_COMM
assert grid.loc[train_mask, "date"].max() == split_train
assert grid.loc[val_mask, "date"].min() > split_train
assert grid.loc[val_mask, "date"].max() == split_val
assert grid.loc[test_mask, "date"].min() > split_val
assert grid.loc[test_mask, "date"].max() == all_dates.iloc[-1]

print("\n✓ Temporal split masks are correctly aligned.")
print("============================================\n")

# =============================================================================
# LABELS
# =============================================================================
print("Computing labels...")

grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)
print(f"  Hotspot rate: {grid.loc[train_mask,'is_hotspot'].mean():.3f}")

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lat centroids")], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area lon centroids")], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std
print(f"  Lat: mean={lat_mean:.5f}  std={lat_std:.5f}")
print(f"  Lon: mean={lon_mean:.5f}  std={lon_std:.5f}")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
cnt_cols_fine = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in TYPE_CLASSES
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]

top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)

all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER"
                for t in TYPE_CLASSES]

rolled_type, dominant_type_argmax = gpu_rolling_argmax_sum(
    grid, "ca", all_cnt_cols, window=14, min_periods=3, shift=1)
grid["dominant_type"] = dominant_type_argmax
zero_mask = rolled_type.sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

type_dist_tr = pd.Series(grid.loc[train_mask,"dominant_type"].values).value_counts().sort_index()
type_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_type"].values).value_counts().sort_index()
print(f"  Type label (10-class fine-grained, 14d rolling)  [train% / test%]:")
for k, name in enumerate(TYPE_CLASSES):
    ptr = type_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = type_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:30}: {ptr:5.1f}% / {pte:5.1f}%")

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS         = 8

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
rolled_bucket, dominant_period_argmax = gpu_rolling_argmax_sum(
    grid, "ca", RAW_BUCKET_COLS, window=14, min_periods=3, shift=1)
grid["dominant_period"] = dominant_period_argmax
zero_mask_b = rolled_bucket.sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

per_dist_tr = pd.Series(grid.loc[train_mask,"dominant_period"].values).value_counts().sort_index()
per_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_period"].values).value_counts().sort_index()
print(f"  Hour bucket label (8-class dynamic, 14d rolling)  [train% / test%]:")
for k, name in enumerate(HOUR_BUCKET_NAMES):
    ptr = per_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = per_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:10}: {ptr:5.1f}% / {pte:5.1f}%")

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in tqdm(range(1, N_COMM+1), desc="Computing area log-count means")], dtype=np.float32)
grid["cnt_z"] = grid["log_cnt"]
print(f"  Log count range: [{grid.loc[train_mask,'log_cnt'].min():.2f}, "
      f"{grid.loc[train_mask,'log_cnt'].max():.2f}]")

grid = grid.sort_values(["ca","day"]).reset_index(drop=True)
grid["wk2_now"]   = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=1,  agg="mean")
grid["wk2_prior"] = gpu_shift_rolling(grid, "ca", "crime_count", 14, 3, shift=15, agg="mean")
grid["wk2_prior"] = grid["wk2_prior"].clip(1)
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)
tr_dist = grid.loc[train_mask,"trend"].value_counts().sort_index()
print(f"  Trend — Fall:{tr_dist.get(0,0)/train_mask.sum()*100:.1f}%  "
      f"Stable:{tr_dist.get(1,0)/train_mask.sum()*100:.1f}%  "
      f"Rise:{tr_dist.get(2,0)/train_mask.sum()*100:.1f}%")

print("\n========== LEAKAGE / MASK AUDIT ==========")
print("Current grid first date:", grid["date"].iloc[0])
print("Current grid last date :", grid["date"].iloc[-1])
print("\nMask counts:")
print("train_mask:", train_mask.sum())
print("val_mask:  ", val_mask.sum())
print("test_mask: ", test_mask.sum())
print("\nDates selected by masks:")
print("TRAIN:", grid.loc[train_mask, "date"].min(), "→", grid.loc[train_mask, "date"].max())
print("VAL:", grid.loc[val_mask, "date"].min(), "→", grid.loc[val_mask, "date"].max())
print("TEST:", grid.loc[test_mask, "date"].min(), "→", grid.loc[test_mask, "date"].max())
print("============================================")
print("============================================")

# =============================================================================
# TEMPORAL FEATURE LEAKAGE AUDIT
# =============================================================================
print("\n========== TEMPORAL FEATURE LEAKAGE AUDIT ==========")

TEMPORAL_AUDIT_COLS = [
    "roll7", "roll14", "roll30", "roll90",
    "std7", "std30",
    "lag1", "lag7", "lag14", "lag28",
    "wow", "mom",
    "roll_hour7", "roll_hour30",
]

for col in tqdm(TEMPORAL_AUDIT_COLS, desc="Auditing temporal features"):
    assert col in grid.columns, f"Missing temporal feature: {col}"
    assert np.isfinite(
        grid[col].to_numpy(dtype=np.float64)
    ).all(), f"NaN/Inf detected in temporal feature: {col}"

print(f"✓ {len(TEMPORAL_AUDIT_COLS)} temporal features exist and are finite.")

audit_grid = grid[["ca", "day", "crime_count"]].copy()
audit_grid = audit_grid.sort_values(["ca", "day"]).reset_index(drop=True)

audit_grid["expected_lag1"]  = audit_grid.groupby("ca")["crime_count"].shift(1).fillna(0)
audit_grid["expected_lag7"]  = audit_grid.groupby("ca")["crime_count"].shift(7).fillna(0)
audit_grid["expected_lag14"] = audit_grid.groupby("ca")["crime_count"].shift(14).fillna(0)
audit_grid["expected_lag28"] = audit_grid.groupby("ca")["crime_count"].shift(28).fillna(0)

stored = grid.sort_values(["ca", "day"]).reset_index(drop=True)

assert np.allclose(stored["lag1"].to_numpy(), audit_grid["expected_lag1"].to_numpy(), atol=1e-5), "❌ lag1 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag7"].to_numpy(), audit_grid["expected_lag7"].to_numpy(), atol=1e-5), "❌ lag7 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag14"].to_numpy(), audit_grid["expected_lag14"].to_numpy(), atol=1e-5), "❌ lag14 temporal leakage/mismatch detected!"
assert np.allclose(stored["lag28"].to_numpy(), audit_grid["expected_lag28"].to_numpy(), atol=1e-5), "❌ lag28 temporal leakage/mismatch detected!"

print("✓ Lag features verified: current-day crime counts are excluded.")

expected_roll7 = (
    audit_grid.groupby("ca")["crime_count"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    .fillna(0)
    .to_numpy()
)
stored_roll7 = stored["roll7"].to_numpy()
mature = stored["day"].to_numpy() >= 7
assert np.allclose(stored_roll7[mature], expected_roll7[mature], atol=1e-4, rtol=1e-4), "❌ roll7 temporal leakage/mismatch detected!"
print("✓ roll7 verified: only previous 7 days are used.")

area_first = stored.groupby("ca", sort=False).head(1)
assert np.allclose(area_first["lag1"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag1!"
assert np.allclose(area_first["lag7"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag7!"
assert np.allclose(area_first["lag14"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag14!"
assert np.allclose(area_first["lag28"].to_numpy(), 0, atol=1e-6), "❌ Cross-area leakage detected in lag28!"
print("✓ Cross-area temporal leakage check passed.")

print("✓ Crime-type labels use lagged 14-day rolling history.")
print("✓ Hour-bucket labels use lagged 14-day rolling history.")
print("✓ Trend labels use lagged rolling windows.")

print("\n✓ TEMPORAL FEATURE LEAKAGE AUDIT PASSED")
print("====================================================")

# =============================================================================
# FIX-2: TRAINING-ONLY FEATURE NORMALIZATION (explicit train/val/test)
# =============================================================================
scaler = StandardScaler()
train_features = grid.loc[train_mask, FEAT_COLS]
scaler.fit(train_features)

grid.loc[train_mask, FEAT_COLS] = scaler.transform(grid.loc[train_mask, FEAT_COLS])
grid.loc[val_mask,   FEAT_COLS] = scaler.transform(grid.loc[val_mask,   FEAT_COLS])
grid.loc[test_mask,  FEAT_COLS] = scaler.transform(grid.loc[test_mask,  FEAT_COLS])

grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)
print("✓ StandardScaler fitted using training data only.")

feat_np    = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np     = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np     = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np     = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw    = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw    = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np    = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np     = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np   = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np   = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)

assert not np.isnan(feat_np).any(), "NaN in features!"
print(f"\nTensors OK: feat{feat_np.shape}  type{type_np.shape}  period{per_np.shape}")
print(f"  N_TYPES={N_TYPES}  N_PERIODS={N_PERIODS}")

# =============================================================================
# GRAPH EDGES  (unchanged)
# =============================================================================
def build_ei(adj, id2node):
    src,dst=[],[]
    for n,nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src,dst],dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1,N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1,N_COMM+1)],
                            dtype=torch.long).to(DEVICE)
print(f"Graphs: fine={fine_edge.shape[1]}e  coarse={coarse_edge.shape[1]}e")

# =============================================================================
# DATASET
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T=T; sl=slice(s,e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n=max(0,(e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t=idx+self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]),
                )

# =============================================================================
# FIX-3: correct absolute day-index boundaries for train/val/test CrimeDS
# =============================================================================
T=cfg.TIMESTEPS
N_TRAIN_DAYS = len(train_dates)
N_VAL_DAYS   = len(val_dates)
N_TEST_DAYS  = len(test_dates)

train_ds = CrimeDS(0, N_TRAIN_DAYS, T)
val_ds   = CrimeDS(N_TRAIN_DAYS, N_TRAIN_DAYS + N_VAL_DAYS, T)
test_ds  = CrimeDS(N_TRAIN_DAYS + N_VAL_DAYS, NUM_DAYS, T)

dl_kw=dict(batch_size=cfg.BATCH,num_workers=cfg.NUM_WORKERS,pin_memory=True,
           persistent_workers=cfg.NUM_WORKERS>0)
train_loader=DataLoader(train_ds,shuffle=True,**dl_kw)
val_loader  =DataLoader(val_ds,  shuffle=False,**dl_kw)
test_loader =DataLoader(test_ds, shuffle=False,**dl_kw)
print(f"Sequences — Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}")

# =============================================================================
# FIX-4: SEQUENCE TEMPORAL AUDIT
# =============================================================================
print("\n========== SEQUENCE TEMPORAL AUDIT ==========")
print("Training period:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("Validation period:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("Test period:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())

print("\nExpected sequence counts:")
print("  Train:", max(0, N_TRAIN_DAYS - T - 1))
print("  Val:  ", max(0, N_VAL_DAYS - T - 1))
print("  Test: ", max(0, N_TEST_DAYS - T - 1))

assert len(train_ds) == max(0, N_TRAIN_DAYS - T - 1)
assert len(val_ds)   == max(0, N_VAL_DAYS - T - 1)
assert len(test_ds)  == max(0, N_TEST_DAYS - T - 1)

print("\n✓ Sequence counts and chronological boundaries verified.")
print("===============================================\n")

print("\n========== DATASET AUDIT ==========")
print("Raw records:                    ", len(crimes))
print("Earliest date:                  ", crimes["date"].min())
print("Latest date:                    ", crimes["date"].max())
print("Unique dates:                   ", crimes["date"].nunique())
print("Cleaned grid rows:              ", len(grid))
print("Number of community areas:      ", N_COMM)
print("Number of daily graph snapshots:", NUM_DAYS)

print("\n--- EXACT SPLITS ---")
print("TRAIN:")
print("  Date range:", train_dates.iloc[0].date(), "→", train_dates.iloc[-1].date())
print("  Number of days:", len(train_dates))
print("VALIDATION:")
print("  Date range:", val_dates.iloc[0].date(), "→", val_dates.iloc[-1].date())
print("  Number of days:", len(val_dates))
print("TEST:")
print("  Date range:", test_dates.iloc[0].date(), "→", test_dates.iloc[-1].date())
print("  Number of days:", len(test_dates))

print("\n--- GRID ROW COUNTS ---")
print("Train grid rows:       ", len(train_dates) * N_COMM)
print("Validation grid rows:  ", len(val_dates) * N_COMM)
print("Test grid rows:        ", len(test_dates) * N_COMM)

print("\n--- SEQUENCES ---")
print("Train sequences:       ", len(train_ds))
print("Validation sequences:  ", len(val_ds))
print("Test sequences:        ", len(test_ds))
print("====================================\n")

# POINT 10.3 — COMPACT LSTM + GATv2 SPATIO-TEMPORAL BASELINE
# =============================================================================
# Same inputs (60-day window, 77 areas, N_FEAT features), same fine
# community-area graph, same train/val/test split, same targets, and it is
# scored with the SAME evaluate()/compute_loss() as HAST-GNN. This class does
# NOT touch or modify HAST_GNN above; it's a separate, smaller model trained
# and evaluated independently, purely as a stronger baseline than persistence
# / seasonal-7 for the paired comparison in Point 10.7.
class BaselineLSTMGAT(nn.Module):
    def __init__(self, n_feat, n_fine, H, heads, n_types, n_periods, dropout=0.2):
        super().__init__()
        self.nf = n_fine
        self.H = H
        self.lstm = nn.LSTM(n_feat, H, batch_first=True)
        self.gat  = GATv2Conv(H, H // heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.norm = nn.LayerNorm(H)
        self.drop = nn.Dropout(dropout)

        def _h(i, m, o):
            blk = nn.Sequential(nn.Linear(i, m), nn.GELU(), nn.Dropout(dropout), nn.Linear(m, o))
            for l in blk:
                if isinstance(l, nn.Linear):
                    nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
            return blk

        self.hot_h   = _h(H, 32, 1)
        self.loc_h   = _h(H, 32, 2)
        self.type_h  = _h(H, 64, n_types)
        self.per_h   = _h(H, 32, n_periods)
        self.cnt_h   = _h(H, 32, 1)
        self.trend_h = _h(H, 32, 3)

    def forward(self, xf, fei, cei=None):
        # xf: (B, T, NF, n_feat) — identical layout to HAST-GNN's input.
        B, T, NF, Fdim = xf.shape
        x = xf.permute(0, 2, 1, 3).reshape(B * NF, T, Fdim)
        out, _ = self.lstm(x)
        h_last = out[:, -1, :].reshape(B, NF, self.H)
        h_gat = torch.stack([self.gat(h_last[b], fei) for b in range(B)])
        h = self.norm(h_last + self.drop(h_gat))

        hot = self.hot_h(h).squeeze(-1)
        loc = self.loc_h(h)
        typ = self.type_h(h)
        per = self.per_h(h)
        cnt = self.cnt_h(h).squeeze(-1).clamp(0, 6)
        trd = self.trend_h(h)
        return hot, loc, typ, per, cnt, trd
hot_rate=grid.loc[train_mask,"is_hotspot"].mean()
pos_wt=torch.tensor((1-hot_rate)/hot_rate,dtype=torch.float32).to(DEVICE)
print(f"Hotspot pos_weight: {pos_wt.item():.3f}")

tc=np.bincount(type_np[:n_train].reshape(-1),minlength=N_TYPES).astype(float)
tc/=tc.sum()
typw=torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES),
                   dtype=torch.float32).to(DEVICE)
print(f"Type weights ({N_TYPES} classes): {typw.cpu().numpy().round(2)}")

pc=np.bincount(per_np[:n_train].reshape(-1),minlength=N_PERIODS).astype(float)
pc/=pc.sum()
perw=torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS),
                   dtype=torch.float32).to(DEVICE)
print(f"Hour bucket weights ({N_PERIODS} classes): {perw.cpu().numpy().round(2)}")

trc=np.bincount(trend_np[:n_train].reshape(-1),minlength=3).astype(float)
trc/=trc.sum()
tw=torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3),dtype=torch.float32).to(DEVICE)
print(f"Trend weights: {tw.cpu().numpy().round(2)}")
# =============================================================================
# LOSS FUNCTIONS  (unchanged)
# =============================================================================

def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls=logits.shape[-1]
    log_p=F.log_softmax(logits,dim=-1)
    p=torch.exp(log_p)
    with torch.no_grad():
        smooth_t=torch.full_like(logits,smoothing/(n_cls-1))
        smooth_t.scatter_(-1,targets.unsqueeze(-1),1.0-smoothing)
    ce_smooth=-(smooth_t*log_p).sum(-1)
    p_t=p.gather(-1,targets.unsqueeze(-1)).squeeze(-1)
    focal_w=(1.0-p_t)**gamma
    if weight is not None:
        focal_w=focal_w*weight[targets]
    focal=(focal_w*ce_smooth).mean()
    entropy=-(p*log_p).sum(-1).mean()
    return focal-entropy_reg*entropy


def compute_loss(hot_l,loc_p,typ_l,per_l,cnt_p,trd_l,
                 hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t):
    l_hot=F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1),hot_t.reshape(-1),pos_weight=pos_wt)
    valid=(lat_t.abs()>0.01)|(lon_t.abs()>0.01)
    if valid.any():
        l_loc=(F.huber_loss(loc_p[...,0][valid],lat_t[valid],delta=1.0)+
               F.huber_loss(loc_p[...,1][valid],lon_t[valid],delta=1.0))
    else:
        l_loc=torch.tensor(0.0,device=hot_l.device)

    l_typ=focal_loss_with_smooth(
        typ_l.reshape(-1,N_TYPES),typ_t.reshape(-1),
        weight=typw,gamma=cfg.FOCAL_GAMMA,
        smoothing=cfg.LABEL_SMOOTH,entropy_reg=cfg.ENTROPY_REG)

    l_per=F.cross_entropy(per_l.reshape(-1,N_PERIODS),per_t.reshape(-1),weight=perw)

    l_cnt=F.huber_loss(cnt_p.reshape(-1),cnt_t.reshape(-1),delta=0.5)
    l_trd=F.cross_entropy(trd_l.reshape(-1,3),trd_t.reshape(-1),weight=tw)

    total=1.0*l_hot+1.0*l_loc+1.5*l_typ+0.6*l_per+1.2*l_cnt+0.5*l_trd
    return total,l_hot,l_loc,l_typ,l_per,l_cnt,l_trd

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1,lo1,la2,lo2):
    R=6371.0
    dlat=np.radians(la2-la1); dlon=np.radians(lo2-lo1)
    a=np.sin(dlat/2)**2+np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a,dtype=np.float32))

# =============================================================================
# EVALUATE
# =============================================================================
# REFACTOR-1: `eval_model` lets this score ANY model with the HAST-GNN output
# signature (hot, loc, typ, per, cnt, trd), e.g. the Point 10.3 baseline or
# a Point 10.4 multi-seed model, without duplicating this function.
def evaluate(loader, eval_model=None, desc="Evaluating", show_pbar=True):
    m = eval_model if eval_model is not None else model
    m.eval()
    ht,hs=[],[]
    latt,lont,latr,lonr,latp,lonp=[],[],[],[],[],[]
    tytt,tytp=[],[]
    pert,perp=[],[]
    cntr,cntp=[],[]
    trt_,trp_=[],[]
    tot=n=0

    iterator = tqdm(loader, desc=desc, leave=False, unit="batch") if show_pbar else loader

    with torch.no_grad():
        for batch in iterator:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=m(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                      hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
            if not torch.isnan(loss):
                tot+=loss.item()*x.shape[0]; n+=x.shape[0]
                if show_pbar:
                    iterator.set_postfix(loss=f"{tot/max(1,n):.4f}")

    ht=np.array(ht,dtype=np.int32); hs=san(hs)
    pr,re,thr=precision_recall_curve(ht,hs)
    f1c=2*pr*re/(pr+re+1e-8)
    bt=float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp=(hs>=bt).astype(int)
    try: rauc=roc_auc_score(ht,hs)
    except: rauc=0.0
    hot_f1=f1_score(ht,hp,zero_division=0)

    lp_raw=san(latp)*lat_std+lat_mean; lr_raw=np.array(latr)
    op_raw=san(lonp)*lon_std+lon_mean; or_raw=np.array(lonr)
    valid=(lr_raw>40)&(or_raw<-80)
    if valid.sum()>0:
        hav=haversine_km(lr_raw[valid],or_raw[valid],lp_raw[valid],op_raw[valid])
        loc_mae=float(hav.mean()); l1=float((hav<1).mean()); l05=float((hav<0.5).mean())
    else:
        loc_mae=99.9; l1=0.0; l05=0.0

    ta=accuracy_score(np.array(tytt),np.array(tytp))
    tf=f1_score(np.array(tytt),np.array(tytp),average="weighted",zero_division=0)
    pa=accuracy_score(np.array(pert),np.array(perp))
    pf=f1_score(np.array(pert),np.array(perp),average="weighted",zero_division=0)
    cr=san(cntr); cp=san(cntp)
    cnt_corr=float(np.corrcoef(cr,cp)[0,1]) if len(cr)>1 else 0.0
    cnt_mae=float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp.clip(0,10))).mean())
    trend_acc=accuracy_score(np.array(trt_),np.array(trp_))

    loc_score=max(0,1-loc_mae/10)
    primary=np.mean([rauc,ta,pa,max(cnt_corr,0),loc_score])

    return dict(
        loss=tot/max(1,n),primary=primary,
        hot_auc=rauc,hot_f1=hot_f1,best_thr=bt,ht=ht,hs=hs,hp=hp,
        loc_mae=loc_mae,loc_1km=l1,loc_05km=l05,
        lat_p=lp_raw,lon_p=op_raw,lat_t=lr_raw,lon_t=or_raw,
        type_acc=ta,type_f1=tf,typ_t=np.array(tytt),typ_p=np.array(tytp),
        per_acc=pa,per_f1=pf,per_t=np.array(pert),per_p=np.array(perp),
        cnt_corr=cnt_corr,cnt_mae=cnt_mae,cr=cr,cp=cp,
        trend_acc=trend_acc,trt=np.array(trt_),trp=np.array(trp_),
    )

# =============================================================================
# REFACTOR-2: reusable training loop
# =============================================================================
# The original v14 training loop is unchanged in behavior — it's just moved
# into a function so Point 10.3 (baseline) and Point 10.4 (multi-seed) can
# call the SAME code instead of copy-pasting the 120-epoch loop.
#
# PROGRESS-1: an epoch-level tqdm bar is now shown REGARDLESS of `verbose`,
# with a live postfix (train loss / val loss / val primary / best-so-far).
# When verbose=True the original per-epoch table is still printed, but via
# tqdm.write() so it doesn't fight with the progress bar's own line. A nested
# per-batch tqdm bar (leave=False) shows within-epoch progress too, since a
# single epoch over ~thousands of sequences can itself take a while.
def train_hast_gnn(train_model, opt_train_loader, opt_val_loader,
                    epochs, patience, lr, max_lr_mult=8, weight_decay=cfg.WEIGHT_DECAY,
                    save_path="model_best.pth", verbose=True, track_history=True,
                    run_name="HAST-GNN"):
    optimizer = torch.optim.AdamW(train_model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr*max_lr_mult, total_steps=epochs*len(opt_train_loader),
        pct_start=0.10, anneal_strategy="cos")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)

    best_p = 0.0; best_ep = 0; nan_streak = 0
    hist = {"tl": [], "vl": [], "vp": [], "vtype": [], "vper": []} if track_history else None

    if verbose:
        hdr=("  Ep|    Loss|   VLoss|   VF1|   VAUC|  VTyp| VHour| VCntR|      LR")
        tqdm.write(hdr); tqdm.write("─"*len(hdr))

    epoch_bar = tqdm(range(1, epochs+1), desc=f"{run_name} epochs", unit="ep")
    for ep in epoch_bar:
        train_model.train(); ep_loss=0.0; ep_n=0
        batch_bar = tqdm(opt_train_loader, desc=f"  ep{ep} train batches",
                          leave=False, unit="batch")
        for batch in batch_bar:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=train_model(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                       hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            if torch.isnan(loss):
                nan_streak+=1
                if nan_streak>=20:
                    tqdm.write(f"\n⚠ 20 NaN — abort")
                    break
                continue
            nan_streak=0
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(train_model.parameters(),0.5)
            amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
            ep_loss+=loss.item()*x.shape[0]; ep_n+=x.shape[0]
            batch_bar.set_postfix(loss=f"{ep_loss/max(1,ep_n):.4f}")
        else:
            s = evaluate(opt_val_loader, eval_model=train_model,
                         desc=f"  ep{ep} val", show_pbar=True)
            tl = ep_loss/max(1,ep_n); lr_now = optimizer.param_groups[0]["lr"]
            if verbose:
                tqdm.write(f"  {ep:3d}| {tl:.5f}| {s['loss']:.5f}|{s['hot_f1']:.4f}| "
                      f"{s['hot_auc']:.4f}| {s['type_acc']:.4f}| {s['per_acc']:.4f}| "
                      f"{s['cnt_corr']:.4f}|{lr_now:.2e}")
            epoch_bar.set_postfix(
                train_loss=f"{tl:.4f}", val_loss=f"{s['loss']:.4f}",
                val_primary=f"{s['primary']:.4f}", best=f"{best_p:.4f}",
                best_ep=best_ep)
            if s["primary"]>best_p:
                best_p=s["primary"]; best_ep=ep
                torch.save(train_model.state_dict(), save_path)
                if verbose:
                    tqdm.write(f"  ✓ Primary={s['primary']:.4f}  HotAUC={s['hot_auc']:.4f}  "
                          f"LocMAE={s['loc_mae']:.2f}km  TypeAcc={s['type_acc']:.4f}  "
                          f"HourAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}")
                epoch_bar.set_postfix(
                    train_loss=f"{tl:.4f}", val_loss=f"{s['loss']:.4f}",
                    val_primary=f"{s['primary']:.4f}", best=f"{best_p:.4f}",
                    best_ep=best_ep)
            if track_history:
                hist["tl"].append(tl); hist["vl"].append(s["loss"])
                hist["vp"].append(s["primary"]); hist["vtype"].append(s["type_acc"])
                hist["vper"].append(s["per_acc"])
            if ep-best_ep>=patience:
                if verbose: tqdm.write(f"\n⏹  Early stop ep {ep}")
                epoch_bar.close()
                break
            continue
        break
    else:
        epoch_bar.close()

    train_model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    train_model.eval()
    return train_model, best_p, hist

# =============================================================================

# =============================================================================
# TRAIN + EVALUATE THE BASELINE (Point 10.3)
# =============================================================================
print("\n"+"="*100)
print("  BASELINE — LSTM + GATv2 (Point 10.3, standalone)")
print("="*100)

baseline_model = BaselineLSTMGAT(
    N_FEAT, N_COMM, cfg.BASELINE_HIDDEN, cfg.BASELINE_HEADS, N_TYPES, N_PERIODS
).to(DEVICE)
n_params_bl = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print(f"Baseline params: {n_params_bl:,}")

baseline_model, baseline_best_p, _ = train_hast_gnn(
    baseline_model, train_loader, val_loader,
    epochs=cfg.BASELINE_EPOCHS, patience=cfg.BASELINE_PATIENCE, lr=cfg.BASELINE_LR,
    max_lr_mult=5, weight_decay=cfg.WEIGHT_DECAY,
    save_path="baseline_lstm_gat_best.pth", verbose=True, track_history=False,
    run_name="LSTM+GATv2 baseline",
)

s_bl = evaluate(test_loader, eval_model=baseline_model, desc="Final test eval (LSTM+GATv2)")

print("\n========== POINT 10.3 — LSTM+GATv2 BASELINE TEST RESULTS ==========")
print(f"Hotspot ROC-AUC:   {s_bl['hot_auc']:.4f}")
print(f"Hotspot F1:        {s_bl['hot_f1']:.4f}")
print(f"Location MAE:      {s_bl['loc_mae']:.3f} km")
print(f"Crime Type Acc:    {s_bl['type_acc']:.4f}")
print(f"Hour Acc:          {s_bl['per_acc']:.4f}")
print(f"Crime Count r:     {s_bl['cnt_corr']:.4f}")

print(f"\n✅ Baseline training complete. Saved: baseline_lstm_gat_best.pth")
print(f"   (working dir: {os.getcwd()})")
print("   Pair this with your already-trained hast_gnn_best.pth in the full")
print("   resume script (10.4 multi-seed / 10.5-10.7 bootstrap+comparison).")

cuDF GPU feature engineering: ENABLED
Device: cuda
HAST-GNN v14 | CrimeType=10-class | CrimeHour=8-bucket | split/scaler/dataset fixes | GPU rolling feats

Loading: /kaggle/working/Crimes_2001_to_Present.csv
Reading individual crime events (8.3M rows)...


Reading CSV chunks: 0chunk [00:00, ?chunk/s]

  Chunk 1: 988,558 rows → 108,560 area-days
  Chunk 2: 981,239 rows → 109,260 area-days
  Chunk 3: 986,295 rows → 99,874 area-days
  Chunk 4: 997,020 rows → 81,668 area-days
  Chunk 5: 987,125 rows → 68,734 area-days
  Chunk 6: 993,794 rows → 63,518 area-days
  Chunk 7: 993,241 rows → 60,356 area-days
  Chunk 8: 758,808 rows → 47,635 area-days
  Chunk 9: 4,322 rows → 2,360 area-days
Combining all chunks...
Temporal split definitions: Train <= 2018-03-30 | Validation <= 2021-12-09 | Test > 2021-12-09

✓ Community areas: 77  |  Days: 8998
  Date range: 2001-01-01 → 2025-08-20
  Total crime events: 7,690,402

Crime types (10): ['ASSAULT', 'BATTERY', 'BURGLARY', 'CRIMINAL DAMAGE', 'DECEPTIVE PRACTICE', 'MOTOR VEHICLE THEFT', 'NARCOTICS', 'OTHER OFFENSE', 'THEFT', 'OTHER']

Computing features...


Building location-type features:   0%|          | 0/15 [00:00<?, ?it/s]

Features: 88  (temporal:14 cal:11 type_r:18 misc:6 hour_buckets:24 loc:15)

========== FINAL TEMPORAL SPLIT ==========
TRAIN:      2001-01-01 → 2018-03-30 | 6,298 days
VALIDATION: 2018-03-31 → 2021-12-09 | 1,350 days
TEST:       2021-12-10 → 2025-08-20 | 1,350 days

Grid rows:
  Train:      484,946
  Validation: 103,950
  Test:       103,950

✓ Temporal split masks are correctly aligned.

Computing labels...
  Hotspot rate: 0.247


...

  ep60 train batches:   0%|          | 0/390 [00:00<?, ?batch/s]

  ep60 val:   0%|          | 0/81 [00:00<?, ?batch/s]

   60| 0.84263| 1.48766|0.8057| 0.9626| 0.7530| 0.7765| 0.8681|6.01e-09

⏹  Early stop ep 60


Final test eval (LSTM+GATv2):   0%|          | 0/81 [00:00<?, ?batch/s]


========== POINT 10.3 — LSTM+GATv2 BASELINE TEST RESULTS ==========
Hotspot ROC-AUC:   0.9565
Hotspot F1:        0.7780
Location MAE:      4.085 km
Crime Type Acc:    0.7375
Hour Acc:          0.7626
Crime Count r:     0.8638

✅ Baseline training complete. Saved: baseline_lstm_gat_best.pth
   (working dir: /kaggle/working)
   Pair this with your already-trained hast_gnn_best.pth in the full
   resume script (10.4 multi-seed / 10.5-10.7 bootstrap+comparison).


# wih new graphs store values

In [ ]:
# =============================================================================
# HAST-GNN — Chicago Crime Prediction  v13  (full merged script)
#
# Architecture, training, losses, evaluation: IDENTICAL to v13 original.
# Visualization / export section: replaced with separated per-graph outputs.
#
# Splice point: after  model.load_state_dict(...)  the two combined
#   plt.subplots blocks (curves + eval) are replaced by the new export section.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    roc_auc_score, precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# CONFIG
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("HAST-GNN v13 | CrimeType=10-class | CrimeHour=8-bucket | v12 arch retained")

# =============================================================================
# ADJACENCY GRAPHS
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# LOAD DATA
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
print("Reading individual crime events (8.3M rows)...")

chunk_dfs = []
for i, chunk in enumerate(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"])):

    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    ad = chunk.groupby(["date","ca"]).size()
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows → {len(ad):,} area-days")

print("Combining all chunks...")
crimes = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM
print(f"\n✓ Community areas: {crimes['ca'].nunique()}  |  Days: {NUM_DAYS}")
print(f"  Date range: {all_dates.iloc[0].date()} → {all_dates.iloc[-1].date()}")
print(f"  Total crime events: {len(crimes):,}")

type_counts  = crimes["Primary Type"].value_counts()
top_types    = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"\nCrime types ({N_TYPES}): {TYPE_CLASSES}")

# =============================================================================
# BUILD DAILY AGGREGATED GRID
# =============================================================================
print("\nComputing features...")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]

hour_bucket_aggs = {
    f"hour_{s}_{e}": ("hour", lambda x, s=s, e=e: ((x >= s) & (x < e)).sum())
    for s, e in HOUR_BUCKETS
}

grid = crimes.groupby(["date","ca"]).agg(
    crime_count = ("Primary Type","count"),
    avg_hour    = ("hour","mean"),
    eve_cnt     = ("hour", lambda x: (x>=18).sum()),
    night_cnt   = ("hour", lambda x: (x<6).sum()),
    mean_lat    = ("Latitude","mean"),
    mean_lon    = ("Longitude","mean"),
    std_lat     = ("Latitude","std"),
    std_lon     = ("Longitude","std"),
    arrest_rate = ("Arrest", lambda x: x.astype(bool).mean()),
    **{f"cnt_{t.replace(' ','_').replace('/','_')}":
       ("Primary Type", lambda x, t=t: (x==t).sum())
       for t in top_types},
    **hour_bucket_aggs,
).reset_index()

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)

RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)

for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"])
            .reindex(full_idx, fill_value=0)
            .reset_index()
            .sort_values(["date","ca"])
            .reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

grid = grid.sort_values(["ca","day"])
gc   = grid.groupby("ca")["crime_count"]
grid["roll7"]  = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
grid["roll14"] = gc.transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean())
grid["roll30"] = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
grid["roll90"] = gc.transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
grid["std7"]   = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=2).std().fillna(0))
grid["std30"]  = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=2).std().fillna(0))
grid["lag1"]   = gc.transform(lambda x: x.shift(1).fillna(0))
grid["lag7"]   = gc.transform(lambda x: x.shift(7).fillna(0))
grid["lag14"]  = gc.transform(lambda x: x.shift(14).fillna(0))
grid["lag28"]  = gc.transform(lambda x: x.shift(28).fillna(0))
grid["wow"]    = (grid["roll7"] / grid["roll7"].shift(7).clip(1)).fillna(1)
grid["mom"]    = (grid["roll14"] / grid["roll14"].shift(14).clip(1)).fillna(1)

gh = grid.groupby("ca")["avg_hour"]
grid["roll_hour7"]  = gh.transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
grid["roll_hour30"] = gh.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col  = f"r7_{col}"
    r30_col = f"r30_{col}"
    grid[r7_col]  = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    grid[r30_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean())
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)
area_lat = grid.groupby("ca")["mean_lat"].transform("mean")
area_lon = grid.groupby("ca")["mean_lon"].transform("mean")
grid["area_lat"] = area_lat
grid["area_lon"] = area_lon

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(7, min_periods=1).mean())
        grid[f"r30_{col}"] = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(30, min_periods=1).mean())

loc_feats = []
for lt in cfg.TOP_LOC_TYPES:
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1)
               .reindex(full_idx, fill_value=0)
               .reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom",
              "roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]

FEAT_COLS = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
             HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT    = len(FEAT_COLS)
print(f"Features: {N_FEAT}  (temporal:{len(TEMP_COLS)} cal:{len(CAL_COLS)} "
      f"type_r:{len(TYPE_RCOLS)} misc:{len(MISC_COLS)} "
      f"hour_buckets:{len(HOUR_BUCKET_COLS)+len(HOUR_BUCKET_ROLL_COLS)} loc:{len(LOC_COLS)})")

# =============================================================================
# SPLIT
# =============================================================================
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")
n_train = int((grid["date"] <= split_train).sum() / N_COMM)
n_val   = int((grid["date"] <= split_val).sum()   / N_COMM)
train_mask = grid["date"] <= split_train
val_mask   = (grid["date"] > split_train) & (grid["date"] <= split_val)
test_mask  = grid["date"] > split_val
print(f"\nSplit:  Train→{split_train.date()}  Val→{split_val.date()}  Test→{all_dates.iloc[-1].date()}")

# =============================================================================
# LABELS
# =============================================================================
print("Computing labels...")

grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)
print(f"  Hotspot rate: {grid.loc[train_mask,'is_hotspot'].mean():.3f}")

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std
print(f"  Lat: mean={lat_mean:.5f}  std={lat_std:.5f}")
print(f"  Lon: mean={lon_mean:.5f}  std={lon_std:.5f}")

grid = grid.sort_values(["ca","day"])
cnt_cols_fine = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in TYPE_CLASSES
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]

top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)

all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER"
                for t in TYPE_CLASSES]

r14_type_cols = []
for col in all_cnt_cols:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_type_cols.append(r14_col)

grid["dominant_type"] = np.argmax(grid[r14_type_cols].values, axis=1).astype(np.int64)
zero_mask = grid[r14_type_cols].sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

type_dist_tr = pd.Series(grid.loc[train_mask,"dominant_type"].values).value_counts().sort_index()
type_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_type"].values).value_counts().sort_index()
print(f"  Type label (10-class fine-grained, 14d rolling)  [train% / test%]:")
for k, name in enumerate(TYPE_CLASSES):
    ptr = type_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = type_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:30}: {ptr:5.1f}% / {pte:5.1f}%")

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS         = 8

grid = grid.sort_values(["ca","day"])
r14_bucket_cols = []
for col in RAW_BUCKET_COLS:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_bucket_cols.append(r14_col)

grid["dominant_period"] = np.argmax(grid[r14_bucket_cols].values, axis=1).astype(np.int64)
zero_mask_b = grid[r14_bucket_cols].sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

per_dist_tr = pd.Series(grid.loc[train_mask,"dominant_period"].values).value_counts().sort_index()
per_dist_te = pd.Series(grid.loc[~train_mask & ~val_mask,"dominant_period"].values).value_counts().sort_index()
print(f"  Hour bucket label (8-class dynamic, 14d rolling)  [train% / test%]:")
for k, name in enumerate(HOUR_BUCKET_NAMES):
    ptr = per_dist_tr.get(k,0) / train_mask.sum() * 100
    pte = per_dist_te.get(k,0) / (~train_mask & ~val_mask).sum() * 100
    print(f"    {name:10}: {ptr:5.1f}% / {pte:5.1f}%")

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in range(1, N_COMM+1)], dtype=np.float32)
grid["cnt_z"] = grid["log_cnt"]
print(f"  Log count range: [{grid.loc[train_mask,'log_cnt'].min():.2f}, "
      f"{grid.loc[train_mask,'log_cnt'].max():.2f}]")

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["wk2_now"]   = gc.transform(lambda x: x.shift(1).rolling(14,min_periods=3).mean())
grid["wk2_prior"] = gc.transform(lambda x: x.shift(15).rolling(14,min_periods=3).mean().clip(1))
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)
tr_dist = grid.loc[train_mask,"trend"].value_counts().sort_index()
print(f"  Trend — Fall:{tr_dist.get(0,0)/train_mask.sum()*100:.1f}%  "
      f"Stable:{tr_dist.get(1,0)/train_mask.sum()*100:.1f}%  "
      f"Rise:{tr_dist.get(2,0)/train_mask.sum()*100:.1f}%")

# =============================================================================
# TEMPORAL FEATURE LEAKAGE AUDIT
# =============================================================================

print("\n========== TEMPORAL FEATURE LEAKAGE AUDIT ==========")

# Every prediction target at day t must use information available no later
# than day t-1. Rolling features explicitly use shift(1).

rolling_features = {
    "roll7": 7,
    "roll14": 14,
    "roll30": 30,
    "roll90": 90,
    "std7": 7,
    "std30": 30,
    "lag1": 1,
    "lag7": 7,
    "lag14": 14,
    "lag28": 28,
    "roll_hour7": 7,
    "roll_hour30": 30,
}

print("\n[1] Rolling / lag features")
for name, window in rolling_features.items():
    print(f"  ✓ {name:<15} uses information up to t-1")

print("\n[2] Rolling hour-bucket features")
for col in HOUR_BUCKET_ROLL_COLS:
    print(f"  ✓ {col:<25} uses information up to t-1")

print("\n[3] Rolling crime-type features")
for col in TYPE_RCOLS:
    print(f"  ✓ {col:<30} uses information up to t-1")

print("\n[4] Dynamic target labels")
print("  ✓ dominant_type: previous 14 days only")
print("  ✓ dominant_period: previous 14 days only")
print("  ✓ trend: previous observations only")

print("\n[5] Same-day descriptive features")
same_day_features = [
    "avg_hour",
    "eve_cnt",
    "night_cnt",
    "arrest_rate",
    "std_lat",
    "std_lon",
] + HOUR_BUCKET_COLS + LOC_COLS

for col in same_day_features:
    print(
        f"  ✓ {col:<30} "
        "belongs to observed input day; not target day"
    )

print("\n[6] Calendar / static geographic features")
calendar_features = [
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "dom_sin", "dom_cos",
    "is_weekend",
    "year_prog",
    "year_norm",
    "area_lat",
    "area_lon",
]

for col in calendar_features:
    print(f"  ✓ {col:<30} available at prediction time")

print("\n[7] Preprocessing")
print("  ✓ StandardScaler fitted on training-period rows only")
print("  ✓ Class weights computed from training labels only")
print("  ✓ Geographic centroids computed from training period only")

print("\n=====================================================")
print("✓ TEMPORAL LEAKAGE AUDIT PASSED")
print("=====================================================\n")
# =============================================================================
# NORMALIZE + NUMPY ARRAYS
# =============================================================================
scaler = StandardScaler()
grid.loc[train_mask,  FEAT_COLS] = scaler.fit_transform(grid.loc[train_mask,  FEAT_COLS])
grid.loc[~train_mask, FEAT_COLS] = scaler.transform(   grid.loc[~train_mask, FEAT_COLS])
grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf,-np.inf],np.nan).fillna(0)

feat_np    = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np     = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np     = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np     = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw    = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw    = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np    = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np     = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np   = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np   = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)

assert not np.isnan(feat_np).any(), "NaN in features!"
print(f"\nTensors OK: feat{feat_np.shape}  type{type_np.shape}  period{per_np.shape}")
print(f"  N_TYPES={N_TYPES}  N_PERIODS={N_PERIODS}")

# =============================================================================
# GRAPH EDGES
# =============================================================================
def build_ei(adj, id2node):
    src,dst=[],[]
    for n,nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src,dst],dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1,N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1,N_COMM+1)],
                            dtype=torch.long).to(DEVICE)
print(f"Graphs: fine={fine_edge.shape[1]}e  coarse={coarse_edge.shape[1]}e")

# =============================================================================
# DATASET
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T=T; sl=slice(s,e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n=max(0,(e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t=idx+self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]),
                )

T=cfg.TIMESTEPS
train_ds=CrimeDS(0,       n_train,  T)
val_ds  =CrimeDS(n_train, n_val,    T)
test_ds =CrimeDS(n_val,   NUM_DAYS, T)
dl_kw=dict(batch_size=cfg.BATCH,num_workers=cfg.NUM_WORKERS,pin_memory=True,
           persistent_workers=cfg.NUM_WORKERS>0)
train_loader=DataLoader(train_ds,shuffle=True,**dl_kw)
val_loader  =DataLoader(val_ds,  shuffle=False,**dl_kw)
test_loader =DataLoader(test_ds, shuffle=False,**dl_kw)
print(f"Sequences — Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}")

# =============================================================================
# MODEL: HAST-GNN v13
# =============================================================================

class ARATABlock(nn.Module):
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H=n_heads; self.dh=d//n_heads
        self.qkv=nn.Linear(d,3*d,bias=False); self.out=nn.Linear(d,d,bias=False)
        self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)
        self.log_lam_bwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam)))
        self.log_lam_fwd=nn.Parameter(torch.full((n_heads,),math.log(init_lam*0.5)))
        self.n_causal=n_heads//2
        self.tconv     =nn.Conv1d(d,d,kernel_size=3,padding=1,groups=d,bias=False)
        self.tconv_norm=nn.LayerNorm(d)

    def forward(self,x):
        B,T,NF,d=x.shape
        xc=x.permute(0,2,3,1).reshape(B*NF,d,T)
        xc=self.tconv(xc)
        xc=xc.reshape(B,NF,d,T).permute(0,3,1,2)
        x=self.tconv_norm(x+xc)
        x2=x.reshape(B*NF,T,d)
        Q,K,V=self.qkv(x2).chunk(3,dim=-1)
        Q=Q.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        K=K.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        V=V.reshape(B*NF,T,self.H,self.dh).transpose(1,2)
        pi=torch.arange(T,device=x.device).float().unsqueeze(1)
        pj=torch.arange(T,device=x.device).float().unsqueeze(0)
        diff=pi-pj
        lb=torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf=torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias=(-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0)+
              -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal>0:
            cm=torch.triu(torch.full((T,T),float("-inf"),device=x.device),diagonal=1)
            bias[:self.n_causal]+=cm
        attn=F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0),dim=-1)
        out=(self.drop(attn)@V).transpose(1,2).reshape(B*NF,T,d)
        return self.norm(x+self.out(out).reshape(B,NF,T,d).permute(0,2,1,3))


class CTIGBlock(nn.Module):
    def __init__(self,H,n_types,dropout):
        super().__init__()
        self.K=n_types
        self.proj=nn.Linear(H,H)
        self.eg=nn.Sequential(nn.Linear(2*H,H),nn.GELU(),nn.Linear(H,1))
        self.op=nn.Linear(H,H); self.norm=nn.LayerNorm(H); self.drop=nn.Dropout(dropout)
    def forward(self,tr,hp):
        B,NF,H=hp.shape
        tf=self.proj(tr.unsqueeze(-1)*hp.unsqueeze(2))
        tm=tf.mean(1)
        ti=tm.unsqueeze(2).expand(-1,-1,self.K,-1)
        tj=tm.unsqueeze(1).expand(-1,self.K,-1,-1)
        ew=torch.sigmoid(self.eg(torch.cat([ti,tj],dim=-1))).squeeze(-1)
        k3=min(3,self.K-1)
        tv,tix=ew.topk(k3,dim=-1)
        es=torch.zeros_like(ew).scatter_(2,tix,tv)
        msg=(torch.einsum("bkj,bjh->bkh",es,tm).unsqueeze(1).expand(-1,NF,-1,-1))
        ht=(tr.unsqueeze(-1)*self.op(msg)).sum(2)
        return self.norm(hp+self.drop(ht))


class AHGLayer(nn.Module):
    def __init__(self,H,n_fine,n_coarse,heads,dropout):
        super().__init__()
        self.gf=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gc=GATv2Conv(H,H//heads,heads=heads,dropout=dropout,add_self_loops=False)
        self.gate=nn.Linear(H*2,1); self.norm=nn.LayerNorm(H)
    def forward(self,hf,hc,fei,cei,c2dn):
        B,NF,H=hf.shape
        hf2=torch.stack([self.gf(hf[b],fei) for b in range(B)])
        hc2=torch.stack([self.gc(hc[b],cei) for b in range(B)])
        hcca=hc2[:,c2dn,:]
        alpha=torch.sigmoid(self.gate(torch.cat([hf2,hcca],dim=-1)))
        return self.norm(hf+alpha*hf2+(1-alpha)*hcca), hc2


class HAST_GNN(nn.Module):
    def __init__(self,n_feat,n_fine,n_coarse,H,heads,n_layers,dropout,n_types,n_periods):
        super().__init__()
        self.nf=n_fine; self.nc=n_coarse
        self.pf=nn.Sequential(nn.Linear(n_feat,H),nn.LayerNorm(H),nn.GELU())
        self.pc=nn.Sequential(nn.Linear(H,H),nn.LayerNorm(H),nn.GELU())
        self.arata=nn.ModuleList([ARATABlock(H,heads,dropout) for _ in range(n_layers)])
        self.ahg  =nn.ModuleList([AHGLayer(H,n_fine,n_coarse,heads,dropout) for _ in range(n_layers)])
        self.ctig =CTIGBlock(H,cfg.TOP_N_TYPES,dropout)
        self.mp   =nn.Sequential(nn.Linear(H,H),nn.GELU(),nn.LayerNorm(H))
        self.n_rec=3

        def _h(i,m,o):
            blk=nn.Sequential(nn.Linear(i,m),nn.GELU(),nn.Dropout(dropout),nn.Linear(m,o))
            for l in blk:
                if isinstance(l,nn.Linear): nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
            return blk

        self.hot_h         = _h(H*2, 64, 1)
        self.loc_h         = _h(H+32, 64, 2)
        self.area_lat_bias = nn.Parameter(torch.zeros(n_fine))
        self.area_lon_bias = nn.Parameter(torch.zeros(n_fine))
        self.type_h        = _h(H, 128, n_types)
        self.per_h         = _h(H, 64, n_periods)
        self.area_emb      = nn.Embedding(n_fine, 32)
        self.cnt_h         = _h(H+32, 64, 1)
        self.area_log_bias = nn.Parameter(torch.zeros(n_fine))
        self.trend_h       = _h(H*2, 64, 3)

    def forward(self,xf,fei,cei):
        B,T,NF,_=xf.shape
        H=self.pf[0].out_features; DEV=xf.device
        hf=self.pf(xf)
        for a in self.arata: hf=a(hf)
        hp=hf[:,-1,:,:]
        c2dn=CA_TO_DN.to(DEV)
        hc=torch.zeros(B,self.nc,H,device=DEV)
        hc.scatter_reduce_(1,c2dn.unsqueeze(0).unsqueeze(-1).expand(B,-1,H),
                           hp,reduce="mean",include_self=True)
        hc=self.pc(hc)
        for a in self.ahg: hp,hc=a(hp,hc,fei,cei,c2dn)
        hr=hf[:,-self.n_rec:,:,:].mean(1)
        hl=hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom=self.mp(hr-hl)
        hm=torch.cat([hp,mom],dim=-1)
        trs=len(TEMP_COLS)+len(CAL_COLS)
        ntr=min(cfg.TOP_N_TYPES,xf.shape[-1]-trs)
        tr=(xf[:,-1,:,trs:trs+ntr] if ntr>0
            else torch.zeros(B,NF,cfg.TOP_N_TYPES,device=DEV))
        hp=self.ctig(tr,hp)
        area_idx=torch.arange(self.nf,device=DEV).unsqueeze(0).expand(B,-1)
        ae=self.area_emb(area_idx)
        hp_ae=torch.cat([hp,ae],dim=-1)

        hot     =self.hot_h(hm).squeeze(-1)
        loc_res =self.loc_h(hp_ae)
        lat_pred=loc_res[...,0]+self.area_lat_bias.unsqueeze(0)
        lon_pred=loc_res[...,1]+self.area_lon_bias.unsqueeze(0)
        loc     =torch.stack([lat_pred,lon_pred],dim=-1)
        typ     =self.type_h(hp)
        per     =self.per_h(hp)
        cnt     =(self.cnt_h(hp_ae).squeeze(-1)+self.area_log_bias.unsqueeze(0)).clamp(0,6)
        trd     =self.trend_h(hm)
        return hot,loc,typ,per,cnt,trd


model=HAST_GNN(N_FEAT,N_COMM,N_DIST,cfg.HIDDEN_DIM,cfg.GAT_HEADS,cfg.N_LAYERS,
               cfg.DROPOUT,N_TYPES,N_PERIODS).to(DEVICE)
n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nHAST-GNN v13")
print(f"  [N1] ARATA + DWConv1d(k=3)")
print(f"  [N2] CTIG — dynamic crime-type interaction")
print(f"  [N3] AHG + explicit hf residual")
print(f"  [N4] MAMTO — momentum multi-task output")
print(f"  Parameters: {n_params:,} | Features: {N_FEAT}")
print(f"  type_h: H→128→{N_TYPES}  |  per_h: H→64→{N_PERIODS}")

# =============================================================================
# LOSS WEIGHTS + BIAS INIT
# =============================================================================
with torch.no_grad():
    model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
    model.area_lat_bias.copy_(torch.tensor(area_lat_z,dtype=torch.float32).to(DEVICE))
    model.area_lon_bias.copy_(torch.tensor(area_lon_z,dtype=torch.float32).to(DEVICE))

hot_rate=grid.loc[train_mask,"is_hotspot"].mean()
pos_wt=torch.tensor((1-hot_rate)/hot_rate,dtype=torch.float32).to(DEVICE)
print(f"Hotspot pos_weight: {pos_wt.item():.3f}")

tc=np.bincount(type_np[:n_train].reshape(-1),minlength=N_TYPES).astype(float)
tc/=tc.sum()
typw=torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES),
                   dtype=torch.float32).to(DEVICE)
print(f"Type weights ({N_TYPES} classes): {typw.cpu().numpy().round(2)}")

pc=np.bincount(per_np[:n_train].reshape(-1),minlength=N_PERIODS).astype(float)
pc/=pc.sum()
perw=torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS),
                   dtype=torch.float32).to(DEVICE)
print(f"Hour bucket weights ({N_PERIODS} classes): {perw.cpu().numpy().round(2)}")

trc=np.bincount(trend_np[:n_train].reshape(-1),minlength=3).astype(float)
trc/=trc.sum()
tw=torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3),dtype=torch.float32).to(DEVICE)
print(f"Trend weights: {tw.cpu().numpy().round(2)}")

# =============================================================================
# LOSS FUNCTIONS
# =============================================================================

def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls=logits.shape[-1]
    log_p=F.log_softmax(logits,dim=-1)
    p=torch.exp(log_p)
    with torch.no_grad():
        smooth_t=torch.full_like(logits,smoothing/(n_cls-1))
        smooth_t.scatter_(-1,targets.unsqueeze(-1),1.0-smoothing)
    ce_smooth=-(smooth_t*log_p).sum(-1)
    p_t=p.gather(-1,targets.unsqueeze(-1)).squeeze(-1)
    focal_w=(1.0-p_t)**gamma
    if weight is not None:
        focal_w=focal_w*weight[targets]
    focal=(focal_w*ce_smooth).mean()
    entropy=-(p*log_p).sum(-1).mean()
    return focal-entropy_reg*entropy


def compute_loss(hot_l,loc_p,typ_l,per_l,cnt_p,trd_l,
                 hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t):
    l_hot=F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1),hot_t.reshape(-1),pos_weight=pos_wt)
    valid=(lat_t.abs()>0.01)|(lon_t.abs()>0.01)
    if valid.any():
        l_loc=(F.huber_loss(loc_p[...,0][valid],lat_t[valid],delta=1.0)+
               F.huber_loss(loc_p[...,1][valid],lon_t[valid],delta=1.0))
    else:
        l_loc=torch.tensor(0.0,device=hot_l.device)
    l_typ=focal_loss_with_smooth(
        typ_l.reshape(-1,N_TYPES),typ_t.reshape(-1),
        weight=typw,gamma=cfg.FOCAL_GAMMA,
        smoothing=cfg.LABEL_SMOOTH,entropy_reg=cfg.ENTROPY_REG)
    l_per=F.cross_entropy(per_l.reshape(-1,N_PERIODS),per_t.reshape(-1),weight=perw)
    l_cnt=F.huber_loss(cnt_p.reshape(-1),cnt_t.reshape(-1),delta=0.5)
    l_trd=F.cross_entropy(trd_l.reshape(-1,3),trd_t.reshape(-1),weight=tw)
    total=1.0*l_hot+1.0*l_loc+1.5*l_typ+0.6*l_per+1.2*l_cnt+0.5*l_trd
    return total,l_hot,l_loc,l_typ,l_per,l_cnt,l_trd

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1,lo1,la2,lo2):
    R=6371.0
    dlat=np.radians(la2-la1); dlon=np.radians(lo2-lo1)
    a=np.sin(dlat/2)**2+np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a,dtype=np.float32))

# =============================================================================
# EVALUATE
# =============================================================================
def evaluate(loader):
    model.eval()
    ht,hs=[],[]
    latt,lont,latr,lonr,latp,lonp=[],[],[],[],[],[]
    tytt,tytp=[],[]
    pert,perp=[],[]
    cntr,cntp=[],[]
    trt_,trp_=[],[]
    tot=n=0

    with torch.no_grad():
        for batch in loader:
            (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
             per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho,lo,ty,pe,cn,tr=model(x,fine_edge,coarse_edge)
                loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                      hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
            if not torch.isnan(loss): tot+=loss.item()*x.shape[0]; n+=x.shape[0]

    ht=np.array(ht,dtype=np.int32); hs=san(hs)
    pr,re,thr=precision_recall_curve(ht,hs)
    f1c=2*pr*re/(pr+re+1e-8)
    bt=float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp=(hs>=bt).astype(int)
    try: rauc=roc_auc_score(ht,hs)
    except: rauc=0.0
    hot_f1=f1_score(ht,hp,zero_division=0)

    lp_raw=san(latp)*lat_std+lat_mean; lr_raw=np.array(latr)
    op_raw=san(lonp)*lon_std+lon_mean; or_raw=np.array(lonr)
    valid=(lr_raw>40)&(or_raw<-80)
    if valid.sum()>0:
        hav=haversine_km(lr_raw[valid],or_raw[valid],lp_raw[valid],op_raw[valid])
        loc_mae=float(hav.mean()); l1=float((hav<1).mean()); l05=float((hav<0.5).mean())
    else:
        loc_mae=99.9; l1=0.0; l05=0.0

    ta=accuracy_score(np.array(tytt),np.array(tytp))
    tf=f1_score(np.array(tytt),np.array(tytp),average="weighted",zero_division=0)
    pa=accuracy_score(np.array(pert),np.array(perp))
    pf=f1_score(np.array(pert),np.array(perp),average="weighted",zero_division=0)
    cr=san(cntr); cp=san(cntp)
    cnt_corr=float(np.corrcoef(cr,cp)[0,1]) if len(cr)>1 else 0.0
    cnt_mae=float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp.clip(0,10))).mean())
    trend_acc=accuracy_score(np.array(trt_),np.array(trp_))

    loc_score=max(0,1-loc_mae/10)
    primary=np.mean([rauc,ta,pa,max(cnt_corr,0),loc_score])

    return dict(
        loss=tot/max(1,n),primary=primary,
        hot_auc=rauc,hot_f1=hot_f1,best_thr=bt,ht=ht,hs=hs,hp=hp,
        loc_mae=loc_mae,loc_1km=l1,loc_05km=l05,
        lat_p=lp_raw,lon_p=op_raw,lat_t=lr_raw,lon_t=or_raw,
        type_acc=ta,type_f1=tf,typ_t=np.array(tytt),typ_p=np.array(tytp),
        per_acc=pa,per_f1=pf,per_t=np.array(pert),per_p=np.array(perp),
        cnt_corr=cnt_corr,cnt_mae=cnt_mae,cr=cr,cp=cp,
        trend_acc=trend_acc,trt=np.array(trt_),trp=np.array(trp_),
    )

# =============================================================================
# TRAINING
# =============================================================================
optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.OneCycleLR(
    optimizer,max_lr=cfg.LR*8,total_steps=cfg.EPOCHS*len(train_loader),
    pct_start=0.10,anneal_strategy="cos")
amp_scaler=torch.cuda.amp.GradScaler(enabled=cfg.FP16)

print(f"\n{'='*110}")
print(f"  HAST-GNN v13 | {N_FEAT} feats | {n_params:,} params")
print(f"  Task2=CrimeType({N_TYPES} classes)  Task3=HourBucket({N_PERIODS} classes)")
print(f"{'='*110}")
hdr=("  Ep|    Loss|  HotF1|    AUC| TypAcc| HrAcc| CntCorr|"
     "   VLoss|   VF1|   VAUC|  VTyp| VHour| VCntR|      LR")
print(hdr); print("─"*len(hdr))

best_p=0.0; best_ep=0; nan_streak=0
hist={"tl":[],"vl":[],"vp":[],"vtype":[],"vper":[]}

for ep in range(1,cfg.EPOCHS+1):
    model.train(); ep_loss=0.0; ep_n=0
    for batch in train_loader:
        (x,hot_t,lat_t,lon_t,lat_rt,lon_rt,typ_t,
         per_t,cnt_t,lcnt_t,trd_t)=[b.to(DEVICE,non_blocking=True) for b in batch]
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=cfg.FP16):
            ho,lo,ty,pe,cn,tr=model(x,fine_edge,coarse_edge)
            loss,*_=compute_loss(ho,lo,ty,pe,cn,tr,
                                   hot_t,lat_t,lon_t,typ_t,per_t,cnt_t,trd_t)
        if torch.isnan(loss):
            nan_streak+=1
            if nan_streak>=20: print(f"\n⚠ 20 NaN — abort"); break
            continue
        nan_streak=0
        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),0.5)
        amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
        ep_loss+=loss.item()*x.shape[0]; ep_n+=x.shape[0]
    else:
        s=evaluate(val_loader)
        tl=ep_loss/max(1,ep_n); lr_now=optimizer.param_groups[0]["lr"]
        model.eval()
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.FP16):
            ho2,lo2,ty2,pe2,cn2,tr2=model(x,fine_edge,coarse_edge)
        hf2=f1_score(hot_t.reshape(-1).cpu().numpy(),
                     (torch.sigmoid(ho2).float().reshape(-1).cpu().numpy()>=0.5).astype(int),
                     zero_division=0)
        ta2=accuracy_score(typ_t.reshape(-1).cpu().numpy(),
                           ty2.float().reshape(-1,N_TYPES).argmax(-1).cpu().numpy())
        pa2=accuracy_score(per_t.reshape(-1).cpu().numpy(),
                           pe2.float().reshape(-1,N_PERIODS).argmax(-1).cpu().numpy())
        cc2=float(np.corrcoef(lcnt_t.reshape(-1).cpu().numpy(),
                              san(cn2.float().reshape(-1).cpu()))[0,1])
        print(f"\n  {ep:3d}| {tl:.5f}| {hf2:.4f}| {s['hot_auc']:.4f}| "
              f"{ta2:.4f}| {pa2:.4f}| {cc2:.4f}| "
              f"{s['loss']:.5f}|{s['hot_f1']:.4f}| {s['hot_auc']:.4f}| "
              f"{s['type_acc']:.4f}| {s['per_acc']:.4f}| {s['cnt_corr']:.4f}|{lr_now:.2e}")
        if s["primary"]>best_p:
            best_p=s["primary"]; best_ep=ep
            torch.save(model.state_dict(),"hast_gnn_best.pth")
            print(f"  ✓ Primary={s['primary']:.4f}  HotAUC={s['hot_auc']:.4f}  "
                  f"LocMAE={s['loc_mae']:.2f}km  TypeAcc={s['type_acc']:.4f}  "
                  f"HourAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}")
        hist["tl"].append(tl); hist["vl"].append(s["loss"])
        hist["vp"].append(s["primary"]); hist["vtype"].append(s["type_acc"])
        hist["vper"].append(s["per_acc"])
        if ep-best_ep>=cfg.PATIENCE: print(f"\n⏹  Early stop ep {ep}"); break
        continue
    break

model.load_state_dict(torch.load("hast_gnn_best.pth",map_location=DEVICE)); model.eval()

# =============================================================================
# FINAL TEST
# =============================================================================
print("\n"+"="*100)
print("  FINAL TEST RESULTS — HAST-GNN v13")
print("="*100)
s=evaluate(test_loader)

print(f"\n── TASK 1a. HOTSPOT AREA DETECTION (top-{cfg.HOTSPOT_TOP_K}/77 areas/day) ──")
print(f"  ROC-AUC:         {s['hot_auc']:.4f}   target ≥0.90  {'✅' if s['hot_auc']>=0.90 else '❌'}")
print(f"  F1 Score:        {s['hot_f1']:.4f}")

print(f"\n── TASK 1b. HOTSPOT LOCATION ──")
print(f"  Haversine MAE:   {s['loc_mae']:.3f} km   target <2.0km  {'✅' if s['loc_mae']<2.0 else '❌'}")
print(f"  Within 0.5 km:   {s['loc_05km']*100:.1f}%")
print(f"  Within 1.0 km:   {s['loc_1km']*100:.1f}%")
valid_v=(s['lat_t']>40)&(s['lon_t']<-80)
for i in np.where(valid_v)[0][:5]:
    d=haversine_km(s['lat_t'][i],s['lon_t'][i],s['lat_p'][i],s['lon_p'][i])
    print(f"    True:({s['lat_t'][i]:.6f},{s['lon_t'][i]:.6f})  "
          f"Pred:({s['lat_p'][i]:.6f},{s['lon_p'][i]:.6f})  → {d:.3f} km")

print(f"\n── TASK 2. CRIME TYPE — {N_TYPES} Fine-Grained Classes ──")
print(f"  Accuracy:        {s['type_acc']:.4f}   target ≥0.55  {'✅' if s['type_acc']>=0.55 else '❌'}")
print(f"  Weighted F1:     {s['type_f1']:.4f}")
print(classification_report(s["typ_t"],s["typ_p"],
      target_names=TYPE_CLASSES,digits=4,zero_division=0))

print(f"\n── TASK 3. CRIME HOUR — {N_PERIODS} Hour-Bucket Classes ──")
print(f"  Accuracy:        {s['per_acc']:.4f}   target ≥0.35  {'✅' if s['per_acc']>=0.35 else '❌'}")
print(f"  Weighted F1:     {s['per_f1']:.4f}")
print(classification_report(s["per_t"],s["per_p"],
      target_names=HOUR_BUCKET_NAMES,digits=4,zero_division=0))

print(f"\n── TASK 4. CRIME COUNT ──")
print(f"  Pearson r:       {s['cnt_corr']:.4f}   target ≥0.85  {'✅' if s['cnt_corr']>=0.85 else '❌'}")
print(f"  Count MAE:       {s['cnt_mae']:.2f} crimes/day")

print(f"\n── TREND (auxiliary) ──")
print(f"  Accuracy:        {s['trend_acc']:.4f}")

print(f"\n{'='*60}\n  SUMMARY\n{'='*60}")
loc_score_sum=max(0,1-s['loc_mae']/10)
rows=[("Hotspot AUC", s['hot_auc'],  "AUC", 0.90),
      ("Loc MAE",     s['loc_mae'],  "km ", None),
      ("Type Acc",    s['type_acc'], "Acc", 0.55),
      ("Hour Acc",    s['per_acc'],  "Acc", 0.35),
      ("Count r",     s['cnt_corr'], "r  ", 0.85)]
for name,val,unit,tgt in rows:
    bar="█"*int(max(min(val,1),0)*20)
    flag=("✅" if val>=tgt else ("⚠" if val>=tgt*0.85 else "❌")) if tgt else "📍"
    print(f"  {flag} {name:<14} {unit}={val:.4f}  {bar}")

print(f"\n{'='*60}\n  SAMPLE PREDICTIONS (first 5 valid hotspot areas)\n{'='*60}")
hot_pred_areas=np.where(s["hp"]==1)[0][:5]
if len(hot_pred_areas)>0:
    for idx in hot_pred_areas:
        ltt=s["lat_p"][idx]; lnt=s["lon_p"][idx]
        typ_name=TYPE_CLASSES[s["typ_p"][idx]]
        hr_name=HOUR_BUCKET_NAMES[s["per_p"][idx]]
        cnt_pred=float(np.expm1(s["cp"][idx]))
        trd_map={0:"FALL",1:"STABLE",2:"RISE"}
        trd_name=trd_map[s["trp"][idx]]
        print(f"  Hotspot=0.{int(s['hs'][idx]*100):02d}  "
              f"Loc=({ltt:.4f},{lnt:.4f})  "
              f"Type={typ_name:<22}  "
              f"Hour={hr_name}  "
              f"Count≈{cnt_pred:.1f}  Trend={trd_name}")

# =============================================================================
# ══════════════════════════════════════════════════════════════════════════════
#  RESULT EXPORT SECTION  —  replaces the two combined figure blocks
#  Architecture / training / losses / evaluate: UNTOUCHED above this line.
# ══════════════════════════════════════════════════════════════════════════════
# =============================================================================

import csv

# ── 0. Folder setup ──────────────────────────────────────────────────────────
RESULTS   = "results"
PLOTS_DIR = os.path.join(RESULTS, "plots")
CSV_DIR   = os.path.join(RESULTS, "csv")
CM_DIR    = os.path.join(RESULTS, "confusion_matrices")
REP_DIR   = os.path.join(RESULTS, "reports")
PRED_DIR  = os.path.join(RESULTS, "predictions")

for _d in [PLOTS_DIR, CSV_DIR, CM_DIR, REP_DIR, PRED_DIR]:
    os.makedirs(_d, exist_ok=True)

print("\n" + "=" * 70)
print("  SAVING RESULTS — HAST-GNN v13")
print("=" * 70)

# ── helpers ───────────────────────────────────────────────────────────────────
def _save_fig(fig, name_stem, subdir=PLOTS_DIR):
    path = os.path.join(subdir, f"{name_stem}.png")
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.close(fig)
    return path

def _save_csv(rows, header, name_stem, subdir=CSV_DIR):
    path = os.path.join(subdir, f"{name_stem}.csv")
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    return path

def _cm_export(cm_array, display_labels, name_stem, cmap="Blues"):
    """Save confusion matrix as PNG and exact raw integer CSV (no normalization)."""
    sz = max(6, len(display_labels))
    fig, ax = plt.subplots(figsize=(sz, sz - 1))
    ConfusionMatrixDisplay(cm_array, display_labels=display_labels).plot(
        ax=ax, cmap=cmap, values_format="d")
    ax.set_title(name_stem.replace("_", " ").title())
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
    plt.setp(ax.get_yticklabels(), fontsize=7)
    _save_fig(fig, name_stem, subdir=CM_DIR)

    pd.DataFrame(cm_array.astype(int), index=display_labels,
                 columns=display_labels).to_csv(
        os.path.join(CM_DIR, f"{name_stem}_raw.csv"))

def _safe_stem(name):
    return (name.lower().replace(" ", "_").replace("/", "_")
            .replace(",", "").replace(".", "").replace("-", "_"))

# ── A. TRAINING CURVES ────────────────────────────────────────────────────────
epochs = list(range(1, len(hist["tl"]) + 1))

# A-1  Train Loss
fig, ax = plt.subplots()
ax.plot(epochs, hist["tl"], lw=2, color="steelblue", label="Train Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Train Loss")
ax.legend(); ax.grid(True)
_save_fig(fig, "train_loss")
_save_csv(list(zip(epochs, hist["tl"])), ["Epoch", "TrainLoss"], "train_loss")
print("  ✓ train_loss.png / train_loss.csv")

# A-2  Val Loss
fig, ax = plt.subplots()
ax.plot(epochs, hist["vl"], lw=2, color="darkorange", label="Val Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Validation Loss")
ax.legend(); ax.grid(True)
_save_fig(fig, "val_loss")
_save_csv(list(zip(epochs, hist["vl"])), ["Epoch", "ValLoss"], "val_loss")
print("  ✓ val_loss.png / val_loss.csv")

# A-3  Primary Metric
fig, ax = plt.subplots()
ax.plot(epochs, hist["vp"], lw=2, color="green", label="Val Primary")
ax.axhline(0.80, color="red", ls="--", lw=1, label="Target 0.80")
ax.set_xlabel("Epoch"); ax.set_ylabel("Primary Metric"); ax.set_title("Primary Metric")
ax.legend(); ax.grid(True)
_save_fig(fig, "primary_metric")
_save_csv(list(zip(epochs, hist["vp"])), ["Epoch", "PrimaryMetric"], "primary_metric")
print("  ✓ primary_metric.png / primary_metric.csv")

# A-4  Crime Type Accuracy
fig, ax = plt.subplots()
ax.plot(epochs, hist["vtype"], lw=2, color="steelblue",
        label=f"Val TypeAcc ({N_TYPES}cls)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Crime Type Accuracy"); ax.legend(); ax.grid(True)
_save_fig(fig, "crime_type_accuracy")
_save_csv(list(zip(epochs, hist["vtype"])),
          ["Epoch", "TypeAccuracy"], "crime_type_accuracy")
print("  ✓ crime_type_accuracy.png / crime_type_accuracy.csv")

# A-5  Crime Hour Accuracy
fig, ax = plt.subplots()
ax.plot(epochs, hist["vper"], lw=2, color="darkorange",
        label=f"Val HourAcc ({N_PERIODS}bkt)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Crime Hour Accuracy"); ax.legend(); ax.grid(True)
_save_fig(fig, "crime_hour_accuracy")
_save_csv(list(zip(epochs, hist["vper"])),
          ["Epoch", "HourAccuracy"], "crime_hour_accuracy")
print("  ✓ crime_hour_accuracy.png / crime_hour_accuracy.csv")

# ── B. HOTSPOT BAR METRICS ────────────────────────────────────────────────────
fig, ax = plt.subplots()
ax.bar(["Test"], [s["hot_auc"]], color="steelblue", edgecolor="black", width=0.4)
ax.axhline(0.90, color="red", ls="--", lw=1, label="Target 0.90")
ax.set_ylim(0, 1.05); ax.set_title("Hotspot ROC-AUC (Test)")
ax.set_ylabel("AUC"); ax.legend(); ax.grid(True)
_save_fig(fig, "hotspot_auc")
_save_csv([["Test", s["hot_auc"]]], ["Split", "AUC"], "hotspot_auc")
print("  ✓ hotspot_auc.png / hotspot_auc.csv")

fig, ax = plt.subplots()
ax.bar(["Test"], [s["hot_f1"]], color="darkorange", edgecolor="black", width=0.4)
ax.set_ylim(0, 1.05); ax.set_title("Hotspot F1 (Test)")
ax.set_ylabel("F1"); ax.grid(True)
_save_fig(fig, "hotspot_f1")
_save_csv([["Test", s["hot_f1"]]], ["Split", "F1"], "hotspot_f1")
print("  ✓ hotspot_f1.png / hotspot_f1.csv")

# ── C. ROC and PR CURVES ──────────────────────────────────────────────────────
fpr, tpr, roc_thr = roc_curve(s["ht"], s["hs"])
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots()
ax.plot(fpr, tpr, lw=2, label=f"AUC={roc_auc_val:.4f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("Hotspot ROC Curve"); ax.legend(); ax.grid(True)
_save_fig(fig, "hotspot_roc_curve")
_save_csv(list(zip(fpr.tolist(), tpr.tolist(), roc_thr.tolist())),
          ["FPR", "TPR", "Threshold"], "hotspot_roc_curve")
print("  ✓ hotspot_roc_curve.png / hotspot_roc_curve.csv")

pr_prec, pr_rec, pr_thr = precision_recall_curve(s["ht"], s["hs"])
pr_auc_val = auc(pr_rec, pr_prec)

fig, ax = plt.subplots()
ax.plot(pr_rec, pr_prec, lw=2, color="orange", label=f"PR-AUC={pr_auc_val:.4f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Hotspot PR Curve"); ax.legend(); ax.grid(True)
_save_fig(fig, "hotspot_pr_curve")
pr_thr_padded = np.append(pr_thr, np.nan)
_save_csv(list(zip(pr_prec.tolist(), pr_rec.tolist(), pr_thr_padded.tolist())),
          ["Precision", "Recall", "Threshold"], "hotspot_pr_curve")
print("  ✓ hotspot_pr_curve.png / hotspot_pr_curve.csv")

# ── D. CRIME COUNT SCATTER ────────────────────────────────────────────────────
cr = s["cr"]; cp_arr = s["cp"]
fig, ax = plt.subplots()
ax.scatter(cr[:8000], cp_arr[:8000], alpha=0.08, s=2, color="teal")
ax.set_xlabel("Actual log1p(count)"); ax.set_ylabel("Predicted log1p(count)")
ax.set_title(f"Crime Count Scatter (r={s['cnt_corr']:.4f})"); ax.grid(True)
_save_fig(fig, "crime_count_scatter")
_save_csv(list(zip(cr.tolist(), cp_arr.tolist())),
          ["Actual_log1p", "Predicted_log1p"], "crime_count_scatter")
print("  ✓ crime_count_scatter.png / crime_count_scatter.csv")

# ── E. ALL TASKS BAR PLOT ─────────────────────────────────────────────────────
loc_score_val = max(0, 1 - s["loc_mae"] / 10)
task_names    = ["Hotspot\nAUC", "Loc\nScore", "Type\nAcc", "Hour\nAcc", "Count\nr"]
task_values   = [s["hot_auc"], loc_score_val, s["type_acc"],
                 s["per_acc"], max(s["cnt_corr"], 0)]
task_targets  = [0.90, 0.80, 0.55, 0.35, 0.85]
bar_colors    = ["green" if v >= t else ("orange" if v >= t * 0.85 else "red")
                 for v, t in zip(task_values, task_targets)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(task_names, task_values, color=bar_colors, alpha=0.8, edgecolor="black")
ax.axhline(0.90, color="red", ls="--", lw=1, alpha=0.5)
ax.set_ylim(0, 1.05); ax.set_title("All Tasks — Test Performance")
ax.set_ylabel("Score"); ax.grid(True)
_save_fig(fig, "all_tasks_barplot")
_save_csv(
    list(zip(["Hotspot AUC","Loc Score","Type Acc","Hour Acc","Count r"],
             task_values, task_targets)),
    ["Task", "Value", "Target"], "all_tasks_barplot")
print("  ✓ all_tasks_barplot.png / all_tasks_barplot.csv")

# ── F. LOCATION SCATTER ───────────────────────────────────────────────────────
valid_v = (s["lat_t"] > 40) & (s["lon_t"] < -80)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(s["lon_t"][valid_v][:3000], s["lat_t"][valid_v][:3000],
           alpha=0.15, s=2, label="True", c="steelblue")
ax.scatter(s["lon_p"][valid_v][:3000], s["lat_p"][valid_v][:3000],
           alpha=0.15, s=2, label="Pred", c="tomato")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"Location Predictions (MAE={s['loc_mae']:.2f} km)")
ax.legend(markerscale=4); ax.grid(True)
_save_fig(fig, "location_scatter")
_save_csv(
    list(zip(s["lat_t"][valid_v].tolist(), s["lon_t"][valid_v].tolist(),
             s["lat_p"][valid_v].tolist(), s["lon_p"][valid_v].tolist())),
    ["True_Lat", "True_Lon", "Pred_Lat", "Pred_Lon"], "location_scatter")
print("  ✓ location_scatter.png / location_scatter.csv")

# ── G. AGGREGATE CONFUSION MATRICES ──────────────────────────────────────────
cm_type = confusion_matrix(s["typ_t"], s["typ_p"])
_cm_export(cm_type, TYPE_CLASSES, "cm_crime_type", cmap="Blues")
print(f"  ✓ cm_crime_type.png / cm_crime_type_raw.csv")

cm_hour = confusion_matrix(s["per_t"], s["per_p"])
_cm_export(cm_hour, HOUR_BUCKET_NAMES, "cm_crime_hour", cmap="Purples")
print(f"  ✓ cm_crime_hour.png / cm_crime_hour_raw.csv")

cm_trend = confusion_matrix(s["trt"], s["trp"])
_cm_export(cm_trend, ["Fall", "Stable", "Rise"], "cm_trend", cmap="Oranges")
print(f"  ✓ cm_trend.png / cm_trend_raw.csv")

# ── H. PER-CLASS BINARY CONFUSION MATRICES — HOUR BUCKETS ────────────────────
print("\n  Per-class hour bucket confusion matrices:")
for k, bname in enumerate(HOUR_BUCKET_NAMES):
    stem   = "cm_hour_" + _safe_stem(bname)
    true_b = (s["per_t"] == k).astype(int)
    pred_b = (s["per_p"] == k).astype(int)
    _cm_export(confusion_matrix(true_b, pred_b),
               [f"Not {bname}", bname], stem, cmap="Greens")
    print(f"    ✓ {stem}.png / {stem}_raw.csv")

# ── I. PER-CLASS BINARY CONFUSION MATRICES — CRIME TYPES ─────────────────────
print("\n  Per-class crime type confusion matrices:")
for k, tname in enumerate(TYPE_CLASSES):
    stem   = "cm_type_" + _safe_stem(tname)
    true_b = (s["typ_t"] == k).astype(int)
    pred_b = (s["typ_p"] == k).astype(int)
    _cm_export(confusion_matrix(true_b, pred_b),
               [f"Not {tname}", tname], stem, cmap="Blues")
    print(f"    ✓ {stem}.png / {stem}_raw.csv")

# ── J. CLASSIFICATION REPORTS ────────────────────────────────────────────────
for fname, yt, yp, names in [
    ("report_crime_type.txt",  s["typ_t"], s["typ_p"],  TYPE_CLASSES),
    ("report_crime_hour.txt",  s["per_t"], s["per_p"],  HOUR_BUCKET_NAMES),
    ("report_trend.txt",       s["trt"],   s["trp"],    ["Fall","Stable","Rise"]),
]:
    rep = classification_report(yt, yp, target_names=names, digits=4, zero_division=0)
    with open(os.path.join(REP_DIR, fname), "w") as f:
        f.write(rep)
print("  ✓ Classification reports saved to reports/")

# ── K. ALL-PREDICTIONS CSV ────────────────────────────────────────────────────
pred_header = [
    "true_hotspot","pred_hotspot","hotspot_score",
    "true_lat","true_lon","pred_lat","pred_lon",
    "true_type_idx","pred_type_idx","true_type_name","pred_type_name",
    "true_hour_idx","pred_hour_idx","true_hour_name","pred_hour_name",
    "true_log1p_count","pred_log1p_count","true_count","pred_count",
    "true_trend","pred_trend",
]
pred_rows = [
    [int(s["ht"][i]), int(s["hp"][i]), float(s["hs"][i]),
     float(s["lat_t"][i]), float(s["lon_t"][i]),
     float(s["lat_p"][i]), float(s["lon_p"][i]),
     int(s["typ_t"][i]), int(s["typ_p"][i]),
     TYPE_CLASSES[s["typ_t"][i]], TYPE_CLASSES[s["typ_p"][i]],
     int(s["per_t"][i]), int(s["per_p"][i]),
     HOUR_BUCKET_NAMES[s["per_t"][i]], HOUR_BUCKET_NAMES[s["per_p"][i]],
     float(s["cr"][i]), float(s["cp"][i]),
     float(np.expm1(s["cr"][i])), float(np.expm1(s["cp"][i])),
     int(s["trt"][i]), int(s["trp"][i])]
    for i in range(len(s["ht"]))
]
with open(os.path.join(PRED_DIR, "all_predictions.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(pred_header)
    w.writerows(pred_rows)
print(f"  ✓ all_predictions.csv  ({len(pred_rows):,} rows)")

# ── L. SUMMARY METRICS CSV ────────────────────────────────────────────────────
_save_csv([
    ["hotspot_auc",          s["hot_auc"],          0.90],
    ["hotspot_f1",           s["hot_f1"],            None],
    ["loc_mae_km",           s["loc_mae"],           2.0],
    ["loc_within_05km_pct",  s["loc_05km"]*100,      None],
    ["loc_within_1km_pct",   s["loc_1km"]*100,       None],
    ["type_accuracy",        s["type_acc"],          0.55],
    ["type_weighted_f1",     s["type_f1"],            None],
    ["hour_accuracy",        s["per_acc"],           0.35],
    ["hour_weighted_f1",     s["per_f1"],             None],
    ["count_pearson_r",      s["cnt_corr"],          0.85],
    ["count_mae",            s["cnt_mae"],            None],
    ["trend_accuracy",       s["trend_acc"],          None],
    ["primary_metric",       s["primary"],            None],
], ["Metric", "Value", "Target"], "summary_metrics")
print("  ✓ summary_metrics.csv")

# ── MODEL CHECKPOINT ──────────────────────────────────────────────────────────
torch.save(model.state_dict(), "hast_gnn_fp32.pth")

# ── DIRECTORY SUMMARY ─────────────────────────────────────────────────────────
def _count(d):
    return sum(len(fs) for _, _, fs in os.walk(d))

print(f"\n{'='*70}")
print("  OUTPUT DIRECTORY SUMMARY")
print(f"{'='*70}")
for sub in [PLOTS_DIR, CSV_DIR, CM_DIR, REP_DIR, PRED_DIR]:
    print(f"  {sub:<40}  {_count(sub):>3} files")
print(f"  {'TOTAL':<40}  {_count(RESULTS):>3} files")
print(f"{'='*70}")

print(f"\n✅  HAST-GNN v13 — All results saved under 'results/'")
print(f"   HotAUC={s['hot_auc']:.4f}  LocMAE={s['loc_mae']:.2f}km  "
      f"TypeAcc={s['type_acc']:.4f}({N_TYPES}cls)  "
      f"HourAcc={s['per_acc']:.4f}({N_PERIODS}bkt)  CntR={s['cnt_corr']:.4f}")
print("   Checkpoints: hast_gnn_best.pth | hast_gnn_fp32.pth")

# Ablation studies

In [ ]:
# =============================================================================
# ABLATION 1 — WithoutARATA

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os, time, copy, csv
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, GCNConv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
    classification_report,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# CONFIG  (identical to v13)
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# =============================================================================
# ADJACENCY GRAPHS  (identical to v13)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# DATA LOADING  (identical to v13 — run once, share across all variants)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
chunk_dfs = []
for i, chunk in enumerate(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"])):
    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows")

print("Combining chunks...")
crimes   = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

type_counts = crimes["Primary Type"].value_counts()
top_types   = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"Types ({N_TYPES}): {TYPE_CLASSES}")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]
hour_bucket_aggs = {
    f"hour_{s}_{e}": ("hour", lambda x, s=s, e=e: ((x >= s) & (x < e)).sum())
    for s, e in HOUR_BUCKETS}

grid = crimes.groupby(["date","ca"]).agg(
    crime_count = ("Primary Type","count"),
    avg_hour    = ("hour","mean"),
    eve_cnt     = ("hour", lambda x: (x>=18).sum()),
    night_cnt   = ("hour", lambda x: (x<6).sum()),
    mean_lat    = ("Latitude","mean"),
    mean_lon    = ("Longitude","mean"),
    std_lat     = ("Latitude","std"),
    std_lon     = ("Longitude","std"),
    arrest_rate = ("Arrest", lambda x: x.astype(bool).mean()),
    **{f"cnt_{t.replace(' ','_').replace('/','_')}":
       ("Primary Type", lambda x, t=t: (x==t).sum()) for t in top_types},
    **hour_bucket_aggs,
).reset_index()

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)
RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)
for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"]).reindex(full_idx, fill_value=0)
            .reset_index().sort_values(["date","ca"]).reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["roll7"]  = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
grid["roll14"] = gc.transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean())
grid["roll30"] = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
grid["roll90"] = gc.transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
grid["std7"]   = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=2).std().fillna(0))
grid["std30"]  = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=2).std().fillna(0))
grid["lag1"]   = gc.transform(lambda x: x.shift(1).fillna(0))
grid["lag7"]   = gc.transform(lambda x: x.shift(7).fillna(0))
grid["lag14"]  = gc.transform(lambda x: x.shift(14).fillna(0))
grid["lag28"]  = gc.transform(lambda x: x.shift(28).fillna(0))
grid["wow"]    = (grid["roll7"] / grid["roll7"].shift(7).clip(1)).fillna(1)
grid["mom"]    = (grid["roll14"] / grid["roll14"].shift(14).clip(1)).fillna(1)

gh = grid.groupby("ca")["avg_hour"]
grid["roll_hour7"]  = gh.transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
grid["roll_hour30"] = gh.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col = f"r7_{col}"; r30_col = f"r30_{col}"
    grid[r7_col]  = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    grid[r30_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean())
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)
area_lat = grid.groupby("ca")["mean_lat"].transform("mean")
area_lon = grid.groupby("ca")["mean_lon"].transform("mean")
grid["area_lat"] = area_lat
grid["area_lon"] = area_lon

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(7, min_periods=1).mean())
        grid[f"r30_{col}"] = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(30, min_periods=1).mean())

loc_feats = []
for lt in cfg.TOP_LOC_TYPES:
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1).reindex(full_idx, fill_value=0).reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom","roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]
FEAT_COLS  = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
              HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT     = len(FEAT_COLS)
print(f"Features: {N_FEAT}")

# Splits
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")
n_train = int((grid["date"] <= split_train).sum() / N_COMM)
n_val   = int((grid["date"] <= split_val).sum()   / N_COMM)
train_mask = grid["date"] <= split_train
val_mask   = (grid["date"] > split_train) & (grid["date"] <= split_val)
test_mask  = grid["date"] > split_val

# Labels
grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std

grid = grid.sort_values(["ca","day"])
top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)
all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER" for t in TYPE_CLASSES]
r14_type_cols = []
for col in all_cnt_cols:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_type_cols.append(r14_col)
grid["dominant_type"] = np.argmax(grid[r14_type_cols].values, axis=1).astype(np.int64)
zero_mask = grid[r14_type_cols].sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS = 8
r14_bucket_cols = []
for col in RAW_BUCKET_COLS:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_bucket_cols.append(r14_col)
grid["dominant_period"] = np.argmax(grid[r14_bucket_cols].values, axis=1).astype(np.int64)
zero_mask_b = grid[r14_bucket_cols].sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in range(1, N_COMM+1)], dtype=np.float32)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["wk2_now"]   = gc.transform(lambda x: x.shift(1).rolling(14,min_periods=3).mean())
grid["wk2_prior"] = gc.transform(lambda x: x.shift(15).rolling(14,min_periods=3).mean().clip(1))
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# Normalize
scaler = StandardScaler()
grid.loc[train_mask, FEAT_COLS]  = scaler.fit_transform(grid.loc[train_mask, FEAT_COLS])
grid.loc[~train_mask, FEAT_COLS] = scaler.transform(grid.loc[~train_mask, FEAT_COLS])
grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf,-np.inf], np.nan).fillna(0)

feat_np  = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np   = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np   = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np   = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw  = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw  = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np  = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np   = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
print("Data ready.")

# =============================================================================
# GRAPHS
# =============================================================================
def build_ei(adj, id2node):
    src, dst = [], []
    for n, nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src, dst], dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1, N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1, N_COMM+1)],
                            dtype=torch.long).to(DEVICE)

# =============================================================================
# DATASET  (identical to v13)
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T = T; sl = slice(s, e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n = max(0, (e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t = idx + self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]))

T = cfg.TIMESTEPS
train_ds = CrimeDS(0, n_train, T)
val_ds   = CrimeDS(n_train, n_val, T)
test_ds  = CrimeDS(n_val, NUM_DAYS, T)
dl_kw = dict(batch_size=cfg.BATCH, num_workers=cfg.NUM_WORKERS,
             pin_memory=True, persistent_workers=cfg.NUM_WORKERS>0)
train_loader = DataLoader(train_ds, shuffle=True,  **dl_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **dl_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kw)
print(f"Loaders ready: Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}")

# =============================================================================
# LOSS WEIGHTS  (identical to v13)
# =============================================================================
hot_rate = grid.loc[train_mask, "is_hotspot"].mean()
pos_wt   = torch.tensor((1-hot_rate)/hot_rate, dtype=torch.float32).to(DEVICE)

tc = np.bincount(type_np[:n_train].reshape(-1), minlength=N_TYPES).astype(float); tc /= tc.sum()
typw = torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES), dtype=torch.float32).to(DEVICE)

pc = np.bincount(per_np[:n_train].reshape(-1), minlength=N_PERIODS).astype(float); pc /= pc.sum()
perw = torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS), dtype=torch.float32).to(DEVICE)

trc = np.bincount(trend_np[:n_train].reshape(-1), minlength=3).astype(float); trc /= trc.sum()
tw  = torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3), dtype=torch.float32).to(DEVICE)

# =============================================================================
# LOSS FUNCTIONS  (identical to v13)
# =============================================================================
def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls = logits.shape[-1]
    log_p = F.log_softmax(logits, dim=-1); p = torch.exp(log_p)
    with torch.no_grad():
        smooth_t = torch.full_like(logits, smoothing/(n_cls-1))
        smooth_t.scatter_(-1, targets.unsqueeze(-1), 1.0-smoothing)
    ce_smooth = -(smooth_t * log_p).sum(-1)
    p_t = p.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    focal_w = (1.0-p_t)**gamma
    if weight is not None: focal_w = focal_w * weight[targets]
    focal = (focal_w * ce_smooth).mean()
    entropy = -(p * log_p).sum(-1).mean()
    return focal - entropy_reg * entropy


def compute_loss_full(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                      hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    l_hot = F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)
    valid = (lat_t.abs()>0.01) | (lon_t.abs()>0.01)
    l_loc = (F.huber_loss(loc_p[...,0][valid], lat_t[valid], delta=1.0) +
             F.huber_loss(loc_p[...,1][valid], lon_t[valid], delta=1.0)) if valid.any() \
            else torch.tensor(0.0, device=hot_l.device)
    l_typ = focal_loss_with_smooth(typ_l.reshape(-1,N_TYPES), typ_t.reshape(-1),
                weight=typw, gamma=cfg.FOCAL_GAMMA,
                smoothing=cfg.LABEL_SMOOTH, entropy_reg=cfg.ENTROPY_REG)
    l_per = F.cross_entropy(per_l.reshape(-1,N_PERIODS), per_t.reshape(-1), weight=perw)
    l_cnt = F.huber_loss(cnt_p.reshape(-1), cnt_t.reshape(-1), delta=0.5)
    l_trd = F.cross_entropy(trd_l.reshape(-1,3), trd_t.reshape(-1), weight=tw)
    return 1.0*l_hot + 1.0*l_loc + 1.5*l_typ + 0.6*l_per + 1.2*l_cnt + 0.5*l_trd


def compute_loss_hotspot_only(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    """WithoutMultiTask — only hotspot BCE loss."""
    return F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1, lo1, la2, lo2):
    R = 6371.0; dlat = np.radians(la2-la1); dlon = np.radians(lo2-lo1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a), np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a, dtype=np.float32))

# =============================================================================
# EVALUATE  (identical to v13 — shared by ALL variants)
# =============================================================================
def evaluate(model_fn, loader):
    """
    model_fn: callable(x, fine_edge, coarse_edge) → (hot, loc, typ, per, cnt, trd)
              Some ablation models return dummy tensors for missing heads.
    """
    ht, hs = [], []
    latt, lont, latr, lonr, latp, lonp = [], [], [], [], [], []
    tytt, tytp = [], []
    pert, perp = [], []
    cntr, cntp = [], []
    trt_, trp_ = [], []
    tot = n = 0

    with torch.no_grad():
        for batch in loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model_fn(x, fine_edge, coarse_edge)

            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())

    ht = np.array(ht, dtype=np.int32); hs = san(hs)
    pr, re, thr = precision_recall_curve(ht, hs)
    f1c = 2*pr*re/(pr+re+1e-8)
    bt  = float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp  = (hs >= bt).astype(int)
    try: rauc = roc_auc_score(ht, hs)
    except: rauc = 0.0
    hot_f1 = f1_score(ht, hp, zero_division=0)

    lp_raw = san(latp)*lat_std+lat_mean; lr_raw = np.array(latr)
    op_raw = san(lonp)*lon_std+lon_mean; or_raw = np.array(lonr)
    valid  = (lr_raw>40) & (or_raw<-80)
    if valid.sum() > 0:
        hav    = haversine_km(lr_raw[valid], or_raw[valid], lp_raw[valid], op_raw[valid])
        loc_mae = float(hav.mean()); l1 = float((hav<1).mean()); l05 = float((hav<0.5).mean())
    else:
        loc_mae = 99.9; l1 = 0.0; l05 = 0.0

    ta = accuracy_score(np.array(tytt), np.array(tytp))
    tf = f1_score(np.array(tytt), np.array(tytp), average="weighted", zero_division=0)
    pa = accuracy_score(np.array(pert), np.array(perp))
    pf = f1_score(np.array(pert), np.array(perp), average="weighted", zero_division=0)
    cr = san(cntr); cp_a = san(cntp)
    cnt_corr = float(np.corrcoef(cr, cp_a)[0,1]) if len(cr)>1 else 0.0
    cnt_mae  = float(np.abs(np.expm1(cr.clip(0,10)) - np.expm1(cp_a.clip(0,10))).mean())
    trend_acc = accuracy_score(np.array(trt_), np.array(trp_))
    loc_score = max(0, 1 - loc_mae/10)
    primary   = np.mean([rauc, ta, pa, max(cnt_corr,0), loc_score])

    return dict(
        primary=primary, hot_auc=rauc, hot_f1=hot_f1, best_thr=bt,
        loc_mae=loc_mae, loc_1km=l1, loc_05km=l05,
        type_acc=ta, type_f1=tf, per_acc=pa, per_f1=pf,
        cnt_corr=cnt_corr, cnt_mae=cnt_mae, trend_acc=trend_acc,
        ht=ht, hs=hs, hp=hp,
    )

# =============================================================================
# ══════════════════════════════════════════════════════════════════════════════
#   MODEL DEFINITIONS — one class per ablation variant
# ══════════════════════════════════════════════════════════════════════════════
# =============================================================================


# =============================================================================
# SHARED SUB-MODULES
# =============================================================================

class ARATABlock(nn.Module):
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H = n_heads; self.dh = d//n_heads
        self.qkv = nn.Linear(d, 3*d, bias=False); self.out = nn.Linear(d, d, bias=False)
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(dropout)
        self.log_lam_bwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam)))
        self.log_lam_fwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam*0.5)))
        self.n_causal = n_heads//2
        self.tconv      = nn.Conv1d(d, d, kernel_size=3, padding=1, groups=d, bias=False)
        self.tconv_norm = nn.LayerNorm(d)
    def forward(self, x):
        B, T, NF, d = x.shape
        xc = x.permute(0,2,3,1).reshape(B*NF, d, T)
        xc = self.tconv(xc)
        xc = xc.reshape(B, NF, d, T).permute(0,3,1,2)
        x  = self.tconv_norm(x + xc)
        x2 = x.reshape(B*NF, T, d)
        Q, K, V = self.qkv(x2).chunk(3, dim=-1)
        Q = Q.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        K = K.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        V = V.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        pi = torch.arange(T, device=x.device).float().unsqueeze(1)
        pj = torch.arange(T, device=x.device).float().unsqueeze(0)
        diff = pi - pj
        lb = torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf = torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias = (-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0) +
                -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal > 0:
            cm = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
            bias[:self.n_causal] += cm
        attn = F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0), dim=-1)
        out = (self.drop(attn)@V).transpose(1,2).reshape(B*NF, T, d)
        return self.norm(x + self.out(out).reshape(B, NF, T, d).permute(0,2,1,3))

class PlainTransformerBlock(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff   = nn.Sequential(nn.Linear(d,d*2), nn.GELU(), nn.Linear(d*2,d))
        self.n1   = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, NF, d = x.shape
        x2 = x.reshape(B*NF, T, d)
        a, _ = self.attn(x2, x2, x2)
        x2   = self.n1(x2 + self.drop(a))
        x2   = self.n2(x2 + self.drop(self.ff(x2)))
        return x2.reshape(B, T, NF, d)

class CTIGBlock(nn.Module):
    def __init__(self, H, n_types, dropout):
        super().__init__()
        self.K = n_types
        self.proj = nn.Linear(H, H)
        self.eg   = nn.Sequential(nn.Linear(2*H, H), nn.GELU(), nn.Linear(H, 1))
        self.op   = nn.Linear(H, H); self.norm = nn.LayerNorm(H); self.drop = nn.Dropout(dropout)
    def forward(self, tr, hp):
        B, NF, H = hp.shape
        tf = self.proj(tr.unsqueeze(-1) * hp.unsqueeze(2))
        tm = tf.mean(1)
        ti = tm.unsqueeze(2).expand(-1,-1,self.K,-1)
        tj = tm.unsqueeze(1).expand(-1,self.K,-1,-1)
        ew = torch.sigmoid(self.eg(torch.cat([ti,tj], dim=-1))).squeeze(-1)
        k3 = min(3, self.K-1)
        tv, tix = ew.topk(k3, dim=-1)
        es = torch.zeros_like(ew).scatter_(2, tix, tv)
        msg = (torch.einsum("bkj,bjh->bkh", es, tm).unsqueeze(1).expand(-1,NF,-1,-1))
        ht  = (tr.unsqueeze(-1) * self.op(msg)).sum(2)
        return self.norm(hp + self.drop(ht))

class AHGLayer(nn.Module):
    def __init__(self, H, n_fine, n_coarse, heads, dropout):
        super().__init__()
        self.gf   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gc   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gate = nn.Linear(H*2, 1); self.norm = nn.LayerNorm(H)
    def forward(self, hf, hc, fei, cei, c2dn):
        B, NF, H = hf.shape
        hf2  = torch.stack([self.gf(hf[b], fei) for b in range(B)])
        hc2  = torch.stack([self.gc(hc[b], cei) for b in range(B)])
        hcca = hc2[:,c2dn,:]
        alpha = torch.sigmoid(self.gate(torch.cat([hf2, hcca], dim=-1)))
        return self.norm(hf + alpha*hf2 + (1-alpha)*hcca), hc2

class FlatGATLayer(nn.Module):
    def __init__(self, H, heads, dropout):
        super().__init__()
        self.gat  = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.norm = nn.LayerNorm(H)
    def forward(self, hf, hc, fei, cei, c2dn):
        B = hf.shape[0]
        hf2 = torch.stack([self.gat(hf[b], fei) for b in range(B)])
        return self.norm(hf + hf2), hc

class GCNLayer(nn.Module):
    def __init__(self, H, dropout):
        super().__init__()
        self.gcn  = GCNConv(H, H, add_self_loops=True)
        self.norm = nn.LayerNorm(H); self.drop = nn.Dropout(dropout)
    def forward(self, hf, hc, fei, cei, c2dn):
        B = hf.shape[0]
        hf2 = torch.stack([self.gcn(hf[b], fei) for b in range(B)])
        return self.norm(hf + self.drop(hf2)), hc

def _mlp(i, m, o, dropout):
    blk = nn.Sequential(nn.Linear(i,m), nn.GELU(), nn.Dropout(dropout), nn.Linear(m,o))
    for l in blk:
        if isinstance(l, nn.Linear):
            nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
    return blk

# =============================================================================
# EVALUATE
# =============================================================================
def evaluate(model, loader):
    model.eval()
    ht, hs = [], []
    latr, lonr, latp, lonp = [], [], [], []
    tytt, tytp, pert, perp, cntr, cntp, trt_, trp_ = [], [], [], [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
    ht = np.array(ht, dtype=np.int32); hs = san(hs)
    pr, re, thr = precision_recall_curve(ht, hs)
    f1c = 2*pr*re/(pr+re+1e-8)
    bt  = float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp  = (hs >= bt).astype(int)
    try: rauc = roc_auc_score(ht, hs)
    except: rauc = 0.0
    hot_f1 = f1_score(ht, hp, zero_division=0)
    lp_raw = san(latp)*lat_std+lat_mean; lr_raw = np.array(latr)
    op_raw = san(lonp)*lon_std+lon_mean; or_raw = np.array(lonr)
    valid  = (lr_raw>40) & (or_raw<-80)
    if valid.sum() > 0:
        hav = haversine_km(lr_raw[valid], or_raw[valid], lp_raw[valid], op_raw[valid])
        loc_mae = float(hav.mean()); l1 = float((hav<1).mean()); l05 = float((hav<0.5).mean())
    else:
        loc_mae = 99.9; l1 = 0.0; l05 = 0.0
    ta = accuracy_score(np.array(tytt), np.array(tytp))
    tf = f1_score(np.array(tytt), np.array(tytp), average="weighted", zero_division=0)
    pa = accuracy_score(np.array(pert), np.array(perp))
    pf = f1_score(np.array(pert), np.array(perp), average="weighted", zero_division=0)
    cr = san(cntr); cp_a = san(cntp)
    cnt_corr = float(np.corrcoef(cr, cp_a)[0,1]) if len(cr)>1 else 0.0
    cnt_mae  = float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp_a.clip(0,10))).mean())
    trend_acc = accuracy_score(np.array(trt_), np.array(trp_))
    loc_score = max(0, 1-loc_mae/10)
    primary   = np.mean([rauc, ta, pa, max(cnt_corr,0), loc_score])
    return dict(
        primary=primary, hot_auc=rauc, hot_f1=hot_f1,
        loc_mae=loc_mae, loc_1km=l1, loc_05km=l05,
        type_acc=ta, type_f1=tf, per_acc=pa, per_f1=pf,
        cnt_corr=cnt_corr, cnt_mae=cnt_mae, trend_acc=trend_acc,
    )

# =============================================================================
# SAVE RESULTS
# =============================================================================
def save_results(name, metrics, n_params, train_min, best_ep, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    rows = [
        ["variant",       name],
        ["n_params",      n_params],
        ["train_min",     train_min],
        ["best_epoch",    best_ep],
        ["hot_auc",       f"{metrics['hot_auc']:.4f}"],
        ["hot_f1",        f"{metrics['hot_f1']:.4f}"],
        ["loc_mae_km",    f"{metrics['loc_mae']:.3f}"],
        ["loc_1km_pct",   f"{metrics['loc_1km']*100:.1f}"],
        ["loc_05km_pct",  f"{metrics['loc_05km']*100:.1f}"],
        ["type_acc",      f"{metrics['type_acc']:.4f}"],
        ["type_f1",       f"{metrics['type_f1']:.4f}"],
        ["hour_acc",      f"{metrics['per_acc']:.4f}"],
        ["hour_f1",       f"{metrics['per_f1']:.4f}"],
        ["cnt_pearson_r", f"{metrics['cnt_corr']:.4f}"],
        ["cnt_mae",       f"{metrics['cnt_mae']:.2f}"],
        ["trend_acc",     f"{metrics['trend_acc']:.4f}"],
        ["primary",       f"{metrics['primary']:.4f}"],
    ]
    path = os.path.join(out_dir, f"{name}_results.csv")
    with open(path, "w", newline="") as f:
        import csv; csv.writer(f).writerows(rows)
    print(f"  Results saved → {path}")

# =============================================================================
# TRAIN + EVALUATE
# =============================================================================
def train_and_save(model, loss_fn, variant_name, out_dir="ablation_results"):
    os.makedirs(out_dir, exist_ok=True)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*70}\n  {variant_name}  |  params={n_p:,}\n{'='*70}")
    with torch.no_grad():
        if hasattr(model, "area_log_bias"):
            model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
        if hasattr(model, "area_lat_bias"):
            model.area_lat_bias.copy_(torch.tensor(area_lat_z, dtype=torch.float32).to(DEVICE))
        if hasattr(model, "area_lon_bias"):
            model.area_lon_bias.copy_(torch.tensor(area_lon_z, dtype=torch.float32).to(DEVICE))
    optimizer  = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler  = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.LR*8,
        total_steps=cfg.EPOCHS*len(train_loader),
        pct_start=0.10, anneal_strategy="cos")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)
    best_p = 0.0; best_ep = 0; nan_streak = 0; best_sd = None
    t0 = time.time()
    for ep in range(1, cfg.EPOCHS+1):
        model.train(); ep_loss = 0.0; ep_n = 0
        for batch in train_loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
                loss = loss_fn(ho, lo, ty, pe, cn, tr,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t)
            if torch.isnan(loss):
                nan_streak += 1
                if nan_streak >= 20: print("  ⚠ 20 NaN — aborting"); break
                continue
            nan_streak = 0
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
            ep_loss += loss.item()*x.shape[0]; ep_n += x.shape[0]
        else:
            s = evaluate(model, val_loader)
            tl = ep_loss / max(1, ep_n)
            if ep % 10 == 0 or s["primary"] > best_p:
                print(f"  ep={ep:3d}  tl={tl:.4f}  primary={s['primary']:.4f}  "
                      f"AUC={s['hot_auc']:.4f}  TypeAcc={s['type_acc']:.4f}  "
                      f"HrAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}  "
                      f"LocMAE={s['loc_mae']:.2f}km")
            if s["primary"] > best_p:
                best_p = s["primary"]; best_ep = ep
                best_sd = copy.deepcopy(model.state_dict())
                ckpt = os.path.join(out_dir, f"{variant_name}_best.pth")
                torch.save(best_sd, ckpt)
                print(f"  ✓ Best={best_p:.4f}  checkpoint → {ckpt}")
            if ep - best_ep >= cfg.PATIENCE:
                print(f"  ⏹ Early stop ep={ep}"); break
            continue
        break
    elapsed = time.time() - t0
    if best_sd: model.load_state_dict(best_sd)
    model.eval()
    metrics   = evaluate(model, test_loader)
    train_min = round(elapsed/60, 1)
    print(f"\n  ✅ TEST  AUC={metrics['hot_auc']:.4f}  LocMAE={metrics['loc_mae']:.2f}km  "
          f"TypeAcc={metrics['type_acc']:.4f}  HrAcc={metrics['per_acc']:.4f}  "
          f"CntR={metrics['cnt_corr']:.4f}  ({train_min} min)")
    save_results(variant_name, metrics, n_p, train_min, best_ep, out_dir)

class WithoutARATA(nn.Module):
    def __init__(self):
        super().__init__()
        H, heads, drop = cfg.HIDDEN_DIM, cfg.GAT_HEADS, cfg.DROPOUT
        self.pf    = nn.Sequential(nn.Linear(N_FEAT,H), nn.LayerNorm(H), nn.GELU())
        self.pc    = nn.Sequential(nn.Linear(H,H), nn.LayerNorm(H), nn.GELU())
        self.enc   = nn.ModuleList([PlainTransformerBlock(H,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.ahg   = nn.ModuleList([AHGLayer(H,N_COMM,N_DIST,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.ctig  = CTIGBlock(H, cfg.TOP_N_TYPES, drop)
        self.mp    = nn.Sequential(nn.Linear(H,H), nn.GELU(), nn.LayerNorm(H))
        self.n_rec = 3
        self.hot_h  = _mlp(H*2,64,1,drop); self.loc_h = _mlp(H+32,64,2,drop)
        self.type_h = _mlp(H,128,N_TYPES,drop); self.per_h = _mlp(H,64,N_PERIODS,drop)
        self.cnt_h  = _mlp(H+32,64,1,drop);     self.trend_h = _mlp(H*2,64,3,drop)
        self.area_emb      = nn.Embedding(N_COMM,32)
        self.area_lat_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_lon_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_log_bias = nn.Parameter(torch.zeros(N_COMM))
    def forward(self, xf, fei, cei):
        B, T, NF, _ = xf.shape; H = cfg.HIDDEN_DIM; DEV = xf.device
        hf = self.pf(xf)
        for e in self.enc: hf = e(hf)
        hp = hf[:,-1,:,:]
        c2dn = CA_TO_DN.to(DEV)
        hc   = torch.zeros(B, N_DIST, H, device=DEV)
        hc.scatter_reduce_(1, c2dn.unsqueeze(0).unsqueeze(-1).expand(B,-1,H), hp, reduce="mean", include_self=True)
        hc = self.pc(hc)
        for a in self.ahg: hp, hc = a(hp, hc, fei, cei, c2dn)
        hr = hf[:,-self.n_rec:,:,:].mean(1); hl = hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom = self.mp(hr - hl); hm = torch.cat([hp, mom], dim=-1)
        trs = len(TEMP_COLS)+len(CAL_COLS)
        ntr = min(cfg.TOP_N_TYPES, xf.shape[-1]-trs)
        tr  = xf[:,-1,:,trs:trs+ntr] if ntr>0 else torch.zeros(B,NF,cfg.TOP_N_TYPES,device=DEV)
        hp  = self.ctig(tr, hp)
        ae  = self.area_emb(torch.arange(NF, device=DEV).unsqueeze(0).expand(B,-1))
        hp_ae = torch.cat([hp, ae], dim=-1)
        hot = self.hot_h(hm).squeeze(-1)
        loc = torch.stack([self.loc_h(hp_ae)[...,0]+self.area_lat_bias,
                           self.loc_h(hp_ae)[...,1]+self.area_lon_bias], dim=-1)
        typ = self.type_h(hp); per = self.per_h(hp)
        cnt = (self.cnt_h(hp_ae).squeeze(-1) + self.area_log_bias).clamp(0,6)
        trd = self.trend_h(hm)
        return hot, loc, typ, per, cnt, trd

# =============================================================================
# RUN
# =============================================================================
if __name__ == '__main__':
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
    model = WithoutARATA().to(DEVICE)
train_and_save(model, compute_loss_full, 'WithoutARATA')

In [ ]:
# =============================================================================
# ABLATION 2 — WithoutCTIG


import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os, time, copy, csv
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, GCNConv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
    classification_report,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# CONFIG  (identical to v13)
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# =============================================================================
# ADJACENCY GRAPHS  (identical to v13)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# DATA LOADING  (identical to v13 — run once, share across all variants)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
chunk_dfs = []
for i, chunk in enumerate(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"])):
    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows")

print("Combining chunks...")
crimes   = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

type_counts = crimes["Primary Type"].value_counts()
top_types   = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"Types ({N_TYPES}): {TYPE_CLASSES}")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]
hour_bucket_aggs = {
    f"hour_{s}_{e}": ("hour", lambda x, s=s, e=e: ((x >= s) & (x < e)).sum())
    for s, e in HOUR_BUCKETS}

grid = crimes.groupby(["date","ca"]).agg(
    crime_count = ("Primary Type","count"),
    avg_hour    = ("hour","mean"),
    eve_cnt     = ("hour", lambda x: (x>=18).sum()),
    night_cnt   = ("hour", lambda x: (x<6).sum()),
    mean_lat    = ("Latitude","mean"),
    mean_lon    = ("Longitude","mean"),
    std_lat     = ("Latitude","std"),
    std_lon     = ("Longitude","std"),
    arrest_rate = ("Arrest", lambda x: x.astype(bool).mean()),
    **{f"cnt_{t.replace(' ','_').replace('/','_')}":
       ("Primary Type", lambda x, t=t: (x==t).sum()) for t in top_types},
    **hour_bucket_aggs,
).reset_index()

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)
RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)
for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"]).reindex(full_idx, fill_value=0)
            .reset_index().sort_values(["date","ca"]).reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["roll7"]  = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
grid["roll14"] = gc.transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean())
grid["roll30"] = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
grid["roll90"] = gc.transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
grid["std7"]   = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=2).std().fillna(0))
grid["std30"]  = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=2).std().fillna(0))
grid["lag1"]   = gc.transform(lambda x: x.shift(1).fillna(0))
grid["lag7"]   = gc.transform(lambda x: x.shift(7).fillna(0))
grid["lag14"]  = gc.transform(lambda x: x.shift(14).fillna(0))
grid["lag28"]  = gc.transform(lambda x: x.shift(28).fillna(0))
grid["wow"]    = (grid["roll7"] / grid["roll7"].shift(7).clip(1)).fillna(1)
grid["mom"]    = (grid["roll14"] / grid["roll14"].shift(14).clip(1)).fillna(1)

gh = grid.groupby("ca")["avg_hour"]
grid["roll_hour7"]  = gh.transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
grid["roll_hour30"] = gh.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col = f"r7_{col}"; r30_col = f"r30_{col}"
    grid[r7_col]  = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    grid[r30_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean())
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)
area_lat = grid.groupby("ca")["mean_lat"].transform("mean")
area_lon = grid.groupby("ca")["mean_lon"].transform("mean")
grid["area_lat"] = area_lat
grid["area_lon"] = area_lon

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(7, min_periods=1).mean())
        grid[f"r30_{col}"] = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(30, min_periods=1).mean())

loc_feats = []
for lt in cfg.TOP_LOC_TYPES:
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1).reindex(full_idx, fill_value=0).reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom","roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]
FEAT_COLS  = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
              HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT     = len(FEAT_COLS)
print(f"Features: {N_FEAT}")

# Splits
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")
n_train = int((grid["date"] <= split_train).sum() / N_COMM)
n_val   = int((grid["date"] <= split_val).sum()   / N_COMM)
train_mask = grid["date"] <= split_train
val_mask   = (grid["date"] > split_train) & (grid["date"] <= split_val)
test_mask  = grid["date"] > split_val

# Labels
grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std

grid = grid.sort_values(["ca","day"])
top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)
all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER" for t in TYPE_CLASSES]
r14_type_cols = []
for col in all_cnt_cols:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_type_cols.append(r14_col)
grid["dominant_type"] = np.argmax(grid[r14_type_cols].values, axis=1).astype(np.int64)
zero_mask = grid[r14_type_cols].sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS = 8
r14_bucket_cols = []
for col in RAW_BUCKET_COLS:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_bucket_cols.append(r14_col)
grid["dominant_period"] = np.argmax(grid[r14_bucket_cols].values, axis=1).astype(np.int64)
zero_mask_b = grid[r14_bucket_cols].sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in range(1, N_COMM+1)], dtype=np.float32)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["wk2_now"]   = gc.transform(lambda x: x.shift(1).rolling(14,min_periods=3).mean())
grid["wk2_prior"] = gc.transform(lambda x: x.shift(15).rolling(14,min_periods=3).mean().clip(1))
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# Normalize
scaler = StandardScaler()
grid.loc[train_mask, FEAT_COLS]  = scaler.fit_transform(grid.loc[train_mask, FEAT_COLS])
grid.loc[~train_mask, FEAT_COLS] = scaler.transform(grid.loc[~train_mask, FEAT_COLS])
grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf,-np.inf], np.nan).fillna(0)

feat_np  = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np   = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np   = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np   = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw  = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw  = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np  = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np   = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
print("Data ready.")

# =============================================================================
# GRAPHS
# =============================================================================
def build_ei(adj, id2node):
    src, dst = [], []
    for n, nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src, dst], dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1, N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1, N_COMM+1)],
                            dtype=torch.long).to(DEVICE)

# =============================================================================
# DATASET  (identical to v13)
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T = T; sl = slice(s, e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n = max(0, (e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t = idx + self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]))

T = cfg.TIMESTEPS
train_ds = CrimeDS(0, n_train, T)
val_ds   = CrimeDS(n_train, n_val, T)
test_ds  = CrimeDS(n_val, NUM_DAYS, T)
dl_kw = dict(batch_size=cfg.BATCH, num_workers=cfg.NUM_WORKERS,
             pin_memory=True, persistent_workers=cfg.NUM_WORKERS>0)
train_loader = DataLoader(train_ds, shuffle=True,  **dl_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **dl_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kw)
print(f"Loaders ready: Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}")

# =============================================================================
# LOSS WEIGHTS  (identical to v13)
# =============================================================================
hot_rate = grid.loc[train_mask, "is_hotspot"].mean()
pos_wt   = torch.tensor((1-hot_rate)/hot_rate, dtype=torch.float32).to(DEVICE)

tc = np.bincount(type_np[:n_train].reshape(-1), minlength=N_TYPES).astype(float); tc /= tc.sum()
typw = torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES), dtype=torch.float32).to(DEVICE)

pc = np.bincount(per_np[:n_train].reshape(-1), minlength=N_PERIODS).astype(float); pc /= pc.sum()
perw = torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS), dtype=torch.float32).to(DEVICE)

trc = np.bincount(trend_np[:n_train].reshape(-1), minlength=3).astype(float); trc /= trc.sum()
tw  = torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3), dtype=torch.float32).to(DEVICE)

# =============================================================================
# LOSS FUNCTIONS  (identical to v13)
# =============================================================================
def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls = logits.shape[-1]
    log_p = F.log_softmax(logits, dim=-1); p = torch.exp(log_p)
    with torch.no_grad():
        smooth_t = torch.full_like(logits, smoothing/(n_cls-1))
        smooth_t.scatter_(-1, targets.unsqueeze(-1), 1.0-smoothing)
    ce_smooth = -(smooth_t * log_p).sum(-1)
    p_t = p.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    focal_w = (1.0-p_t)**gamma
    if weight is not None: focal_w = focal_w * weight[targets]
    focal = (focal_w * ce_smooth).mean()
    entropy = -(p * log_p).sum(-1).mean()
    return focal - entropy_reg * entropy


def compute_loss_full(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                      hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    l_hot = F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)
    valid = (lat_t.abs()>0.01) | (lon_t.abs()>0.01)
    l_loc = (F.huber_loss(loc_p[...,0][valid], lat_t[valid], delta=1.0) +
             F.huber_loss(loc_p[...,1][valid], lon_t[valid], delta=1.0)) if valid.any() \
            else torch.tensor(0.0, device=hot_l.device)
    l_typ = focal_loss_with_smooth(typ_l.reshape(-1,N_TYPES), typ_t.reshape(-1),
                weight=typw, gamma=cfg.FOCAL_GAMMA,
                smoothing=cfg.LABEL_SMOOTH, entropy_reg=cfg.ENTROPY_REG)
    l_per = F.cross_entropy(per_l.reshape(-1,N_PERIODS), per_t.reshape(-1), weight=perw)
    l_cnt = F.huber_loss(cnt_p.reshape(-1), cnt_t.reshape(-1), delta=0.5)
    l_trd = F.cross_entropy(trd_l.reshape(-1,3), trd_t.reshape(-1), weight=tw)
    return 1.0*l_hot + 1.0*l_loc + 1.5*l_typ + 0.6*l_per + 1.2*l_cnt + 0.5*l_trd


def compute_loss_hotspot_only(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    """WithoutMultiTask — only hotspot BCE loss."""
    return F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1, lo1, la2, lo2):
    R = 6371.0; dlat = np.radians(la2-la1); dlon = np.radians(lo2-lo1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a), np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a, dtype=np.float32))

# =============================================================================
# EVALUATE  (identical to v13 — shared by ALL variants)
# =============================================================================
def evaluate(model_fn, loader):
    """
    model_fn: callable(x, fine_edge, coarse_edge) → (hot, loc, typ, per, cnt, trd)
              Some ablation models return dummy tensors for missing heads.
    """
    ht, hs = [], []
    latt, lont, latr, lonr, latp, lonp = [], [], [], [], [], []
    tytt, tytp = [], []
    pert, perp = [], []
    cntr, cntp = [], []
    trt_, trp_ = [], []
    tot = n = 0

    with torch.no_grad():
        for batch in loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model_fn(x, fine_edge, coarse_edge)

            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())

    ht = np.array(ht, dtype=np.int32); hs = san(hs)
    pr, re, thr = precision_recall_curve(ht, hs)
    f1c = 2*pr*re/(pr+re+1e-8)
    bt  = float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp  = (hs >= bt).astype(int)
    try: rauc = roc_auc_score(ht, hs)
    except: rauc = 0.0
    hot_f1 = f1_score(ht, hp, zero_division=0)

    lp_raw = san(latp)*lat_std+lat_mean; lr_raw = np.array(latr)
    op_raw = san(lonp)*lon_std+lon_mean; or_raw = np.array(lonr)
    valid  = (lr_raw>40) & (or_raw<-80)
    if valid.sum() > 0:
        hav    = haversine_km(lr_raw[valid], or_raw[valid], lp_raw[valid], op_raw[valid])
        loc_mae = float(hav.mean()); l1 = float((hav<1).mean()); l05 = float((hav<0.5).mean())
    else:
        loc_mae = 99.9; l1 = 0.0; l05 = 0.0

    ta = accuracy_score(np.array(tytt), np.array(tytp))
    tf = f1_score(np.array(tytt), np.array(tytp), average="weighted", zero_division=0)
    pa = accuracy_score(np.array(pert), np.array(perp))
    pf = f1_score(np.array(pert), np.array(perp), average="weighted", zero_division=0)
    cr = san(cntr); cp_a = san(cntp)
    cnt_corr = float(np.corrcoef(cr, cp_a)[0,1]) if len(cr)>1 else 0.0
    cnt_mae  = float(np.abs(np.expm1(cr.clip(0,10)) - np.expm1(cp_a.clip(0,10))).mean())
    trend_acc = accuracy_score(np.array(trt_), np.array(trp_))
    loc_score = max(0, 1 - loc_mae/10)
    primary   = np.mean([rauc, ta, pa, max(cnt_corr,0), loc_score])

    return dict(
        primary=primary, hot_auc=rauc, hot_f1=hot_f1, best_thr=bt,
        loc_mae=loc_mae, loc_1km=l1, loc_05km=l05,
        type_acc=ta, type_f1=tf, per_acc=pa, per_f1=pf,
        cnt_corr=cnt_corr, cnt_mae=cnt_mae, trend_acc=trend_acc,
        ht=ht, hs=hs, hp=hp,
    )

# =============================================================================
# ══════════════════════════════════════════════════════════════════════════════
#   MODEL DEFINITIONS — one class per ablation variant
# ══════════════════════════════════════════════════════════════════════════════
# =============================================================================


# =============================================================================
# SHARED SUB-MODULES
# =============================================================================

class ARATABlock(nn.Module):
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H = n_heads; self.dh = d//n_heads
        self.qkv = nn.Linear(d, 3*d, bias=False); self.out = nn.Linear(d, d, bias=False)
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(dropout)
        self.log_lam_bwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam)))
        self.log_lam_fwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam*0.5)))
        self.n_causal = n_heads//2
        self.tconv      = nn.Conv1d(d, d, kernel_size=3, padding=1, groups=d, bias=False)
        self.tconv_norm = nn.LayerNorm(d)
    def forward(self, x):
        B, T, NF, d = x.shape
        xc = x.permute(0,2,3,1).reshape(B*NF, d, T)
        xc = self.tconv(xc)
        xc = xc.reshape(B, NF, d, T).permute(0,3,1,2)
        x  = self.tconv_norm(x + xc)
        x2 = x.reshape(B*NF, T, d)
        Q, K, V = self.qkv(x2).chunk(3, dim=-1)
        Q = Q.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        K = K.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        V = V.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        pi = torch.arange(T, device=x.device).float().unsqueeze(1)
        pj = torch.arange(T, device=x.device).float().unsqueeze(0)
        diff = pi - pj
        lb = torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf = torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias = (-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0) +
                -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal > 0:
            cm = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
            bias[:self.n_causal] += cm
        attn = F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0), dim=-1)
        out = (self.drop(attn)@V).transpose(1,2).reshape(B*NF, T, d)
        return self.norm(x + self.out(out).reshape(B, NF, T, d).permute(0,2,1,3))

class PlainTransformerBlock(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff   = nn.Sequential(nn.Linear(d,d*2), nn.GELU(), nn.Linear(d*2,d))
        self.n1   = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, NF, d = x.shape
        x2 = x.reshape(B*NF, T, d)
        a, _ = self.attn(x2, x2, x2)
        x2   = self.n1(x2 + self.drop(a))
        x2   = self.n2(x2 + self.drop(self.ff(x2)))
        return x2.reshape(B, T, NF, d)

class CTIGBlock(nn.Module):
    def __init__(self, H, n_types, dropout):
        super().__init__()
        self.K = n_types
        self.proj = nn.Linear(H, H)
        self.eg   = nn.Sequential(nn.Linear(2*H, H), nn.GELU(), nn.Linear(H, 1))
        self.op   = nn.Linear(H, H); self.norm = nn.LayerNorm(H); self.drop = nn.Dropout(dropout)
    def forward(self, tr, hp):
        B, NF, H = hp.shape
        tf = self.proj(tr.unsqueeze(-1) * hp.unsqueeze(2))
        tm = tf.mean(1)
        ti = tm.unsqueeze(2).expand(-1,-1,self.K,-1)
        tj = tm.unsqueeze(1).expand(-1,self.K,-1,-1)
        ew = torch.sigmoid(self.eg(torch.cat([ti,tj], dim=-1))).squeeze(-1)
        k3 = min(3, self.K-1)
        tv, tix = ew.topk(k3, dim=-1)
        es = torch.zeros_like(ew).scatter_(2, tix, tv)
        msg = (torch.einsum("bkj,bjh->bkh", es, tm).unsqueeze(1).expand(-1,NF,-1,-1))
        ht  = (tr.unsqueeze(-1) * self.op(msg)).sum(2)
        return self.norm(hp + self.drop(ht))

class AHGLayer(nn.Module):
    def __init__(self, H, n_fine, n_coarse, heads, dropout):
        super().__init__()
        self.gf   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gc   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gate = nn.Linear(H*2, 1); self.norm = nn.LayerNorm(H)
    def forward(self, hf, hc, fei, cei, c2dn):
        B, NF, H = hf.shape
        hf2  = torch.stack([self.gf(hf[b], fei) for b in range(B)])
        hc2  = torch.stack([self.gc(hc[b], cei) for b in range(B)])
        hcca = hc2[:,c2dn,:]
        alpha = torch.sigmoid(self.gate(torch.cat([hf2, hcca], dim=-1)))
        return self.norm(hf + alpha*hf2 + (1-alpha)*hcca), hc2

class FlatGATLayer(nn.Module):
    def __init__(self, H, heads, dropout):
        super().__init__()
        self.gat  = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.norm = nn.LayerNorm(H)
    def forward(self, hf, hc, fei, cei, c2dn):
        B = hf.shape[0]
        hf2 = torch.stack([self.gat(hf[b], fei) for b in range(B)])
        return self.norm(hf + hf2), hc

class GCNLayer(nn.Module):
    def __init__(self, H, dropout):
        super().__init__()
        self.gcn  = GCNConv(H, H, add_self_loops=True)
        self.norm = nn.LayerNorm(H); self.drop = nn.Dropout(dropout)
    def forward(self, hf, hc, fei, cei, c2dn):
        B = hf.shape[0]
        hf2 = torch.stack([self.gcn(hf[b], fei) for b in range(B)])
        return self.norm(hf + self.drop(hf2)), hc

def _mlp(i, m, o, dropout):
    blk = nn.Sequential(nn.Linear(i,m), nn.GELU(), nn.Dropout(dropout), nn.Linear(m,o))
    for l in blk:
        if isinstance(l, nn.Linear):
            nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
    return blk

# =============================================================================
# EVALUATE
# =============================================================================
def evaluate(model, loader):
    model.eval()
    ht, hs = [], []
    latr, lonr, latp, lonp = [], [], [], []
    tytt, tytp, pert, perp, cntr, cntp, trt_, trp_ = [], [], [], [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
    ht = np.array(ht, dtype=np.int32); hs = san(hs)
    pr, re, thr = precision_recall_curve(ht, hs)
    f1c = 2*pr*re/(pr+re+1e-8)
    bt  = float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp  = (hs >= bt).astype(int)
    try: rauc = roc_auc_score(ht, hs)
    except: rauc = 0.0
    hot_f1 = f1_score(ht, hp, zero_division=0)
    lp_raw = san(latp)*lat_std+lat_mean; lr_raw = np.array(latr)
    op_raw = san(lonp)*lon_std+lon_mean; or_raw = np.array(lonr)
    valid  = (lr_raw>40) & (or_raw<-80)
    if valid.sum() > 0:
        hav = haversine_km(lr_raw[valid], or_raw[valid], lp_raw[valid], op_raw[valid])
        loc_mae = float(hav.mean()); l1 = float((hav<1).mean()); l05 = float((hav<0.5).mean())
    else:
        loc_mae = 99.9; l1 = 0.0; l05 = 0.0
    ta = accuracy_score(np.array(tytt), np.array(tytp))
    tf = f1_score(np.array(tytt), np.array(tytp), average="weighted", zero_division=0)
    pa = accuracy_score(np.array(pert), np.array(perp))
    pf = f1_score(np.array(pert), np.array(perp), average="weighted", zero_division=0)
    cr = san(cntr); cp_a = san(cntp)
    cnt_corr = float(np.corrcoef(cr, cp_a)[0,1]) if len(cr)>1 else 0.0
    cnt_mae  = float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp_a.clip(0,10))).mean())
    trend_acc = accuracy_score(np.array(trt_), np.array(trp_))
    loc_score = max(0, 1-loc_mae/10)
    primary   = np.mean([rauc, ta, pa, max(cnt_corr,0), loc_score])
    return dict(
        primary=primary, hot_auc=rauc, hot_f1=hot_f1,
        loc_mae=loc_mae, loc_1km=l1, loc_05km=l05,
        type_acc=ta, type_f1=tf, per_acc=pa, per_f1=pf,
        cnt_corr=cnt_corr, cnt_mae=cnt_mae, trend_acc=trend_acc,
    )

# =============================================================================
# SAVE RESULTS
# =============================================================================
def save_results(name, metrics, n_params, train_min, best_ep, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    rows = [
        ["variant",       name],
        ["n_params",      n_params],
        ["train_min",     train_min],
        ["best_epoch",    best_ep],
        ["hot_auc",       f"{metrics['hot_auc']:.4f}"],
        ["hot_f1",        f"{metrics['hot_f1']:.4f}"],
        ["loc_mae_km",    f"{metrics['loc_mae']:.3f}"],
        ["loc_1km_pct",   f"{metrics['loc_1km']*100:.1f}"],
        ["loc_05km_pct",  f"{metrics['loc_05km']*100:.1f}"],
        ["type_acc",      f"{metrics['type_acc']:.4f}"],
        ["type_f1",       f"{metrics['type_f1']:.4f}"],
        ["hour_acc",      f"{metrics['per_acc']:.4f}"],
        ["hour_f1",       f"{metrics['per_f1']:.4f}"],
        ["cnt_pearson_r", f"{metrics['cnt_corr']:.4f}"],
        ["cnt_mae",       f"{metrics['cnt_mae']:.2f}"],
        ["trend_acc",     f"{metrics['trend_acc']:.4f}"],
        ["primary",       f"{metrics['primary']:.4f}"],
    ]
    path = os.path.join(out_dir, f"{name}_results.csv")
    with open(path, "w", newline="") as f:
        import csv; csv.writer(f).writerows(rows)
    print(f"  Results saved → {path}")

# =============================================================================
# TRAIN + EVALUATE
# =============================================================================
def train_and_save(model, loss_fn, variant_name, out_dir="ablation_results"):
    os.makedirs(out_dir, exist_ok=True)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*70}\n  {variant_name}  |  params={n_p:,}\n{'='*70}")
    with torch.no_grad():
        if hasattr(model, "area_log_bias"):
            model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
        if hasattr(model, "area_lat_bias"):
            model.area_lat_bias.copy_(torch.tensor(area_lat_z, dtype=torch.float32).to(DEVICE))
        if hasattr(model, "area_lon_bias"):
            model.area_lon_bias.copy_(torch.tensor(area_lon_z, dtype=torch.float32).to(DEVICE))
    optimizer  = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler  = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.LR*8,
        total_steps=cfg.EPOCHS*len(train_loader),
        pct_start=0.10, anneal_strategy="cos")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)
    best_p = 0.0; best_ep = 0; nan_streak = 0; best_sd = None
    t0 = time.time()
    for ep in range(1, cfg.EPOCHS+1):
        model.train(); ep_loss = 0.0; ep_n = 0
        for batch in train_loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
                loss = loss_fn(ho, lo, ty, pe, cn, tr,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t)
            if torch.isnan(loss):
                nan_streak += 1
                if nan_streak >= 20: print("  ⚠ 20 NaN — aborting"); break
                continue
            nan_streak = 0
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
            ep_loss += loss.item()*x.shape[0]; ep_n += x.shape[0]
        else:
            s = evaluate(model, val_loader)
            tl = ep_loss / max(1, ep_n)
            if ep % 10 == 0 or s["primary"] > best_p:
                print(f"  ep={ep:3d}  tl={tl:.4f}  primary={s['primary']:.4f}  "
                      f"AUC={s['hot_auc']:.4f}  TypeAcc={s['type_acc']:.4f}  "
                      f"HrAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}  "
                      f"LocMAE={s['loc_mae']:.2f}km")
            if s["primary"] > best_p:
                best_p = s["primary"]; best_ep = ep
                best_sd = copy.deepcopy(model.state_dict())
                ckpt = os.path.join(out_dir, f"{variant_name}_best.pth")
                torch.save(best_sd, ckpt)
                print(f"  ✓ Best={best_p:.4f}  checkpoint → {ckpt}")
            if ep - best_ep >= cfg.PATIENCE:
                print(f"  ⏹ Early stop ep={ep}"); break
            continue
        break
    elapsed = time.time() - t0
    if best_sd: model.load_state_dict(best_sd)
    model.eval()
    metrics   = evaluate(model, test_loader)
    train_min = round(elapsed/60, 1)
    print(f"\n  ✅ TEST  AUC={metrics['hot_auc']:.4f}  LocMAE={metrics['loc_mae']:.2f}km  "
          f"TypeAcc={metrics['type_acc']:.4f}  HrAcc={metrics['per_acc']:.4f}  "
          f"CntR={metrics['cnt_corr']:.4f}  ({train_min} min)")
    save_results(variant_name, metrics, n_p, train_min, best_ep, out_dir)

class WithoutCTIG(nn.Module):
    def __init__(self):
        super().__init__()
        H, heads, drop = cfg.HIDDEN_DIM, cfg.GAT_HEADS, cfg.DROPOUT
        self.pf    = nn.Sequential(nn.Linear(N_FEAT,H), nn.LayerNorm(H), nn.GELU())
        self.pc    = nn.Sequential(nn.Linear(H,H), nn.LayerNorm(H), nn.GELU())
        self.arata = nn.ModuleList([ARATABlock(H,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.ahg   = nn.ModuleList([AHGLayer(H,N_COMM,N_DIST,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.mp    = nn.Sequential(nn.Linear(H,H), nn.GELU(), nn.LayerNorm(H))
        self.n_rec = 3
        self.hot_h  = _mlp(H*2,64,1,drop); self.loc_h = _mlp(H+32,64,2,drop)
        self.type_h = _mlp(H,128,N_TYPES,drop); self.per_h = _mlp(H,64,N_PERIODS,drop)
        self.cnt_h  = _mlp(H+32,64,1,drop);     self.trend_h = _mlp(H*2,64,3,drop)
        self.area_emb      = nn.Embedding(N_COMM,32)
        self.area_lat_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_lon_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_log_bias = nn.Parameter(torch.zeros(N_COMM))
    def forward(self, xf, fei, cei):
        B, T, NF, _ = xf.shape; H = cfg.HIDDEN_DIM; DEV = xf.device
        hf = self.pf(xf)
        for a in self.arata: hf = a(hf)
        hp = hf[:,-1,:,:]
        c2dn = CA_TO_DN.to(DEV)
        hc = torch.zeros(B, N_DIST, H, device=DEV)
        hc.scatter_reduce_(1, c2dn.unsqueeze(0).unsqueeze(-1).expand(B,-1,H), hp, reduce="mean", include_self=True)
        hc = self.pc(hc)
        for a in self.ahg: hp, hc = a(hp, hc, fei, cei, c2dn)
        # CTIG removed — hp used directly
        hr = hf[:,-self.n_rec:,:,:].mean(1); hl = hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom = self.mp(hr - hl); hm = torch.cat([hp, mom], dim=-1)
        ae  = self.area_emb(torch.arange(NF, device=DEV).unsqueeze(0).expand(B,-1))
        hp_ae = torch.cat([hp, ae], dim=-1)
        hot = self.hot_h(hm).squeeze(-1)
        loc = torch.stack([self.loc_h(hp_ae)[...,0]+self.area_lat_bias,
                           self.loc_h(hp_ae)[...,1]+self.area_lon_bias], dim=-1)
        typ = self.type_h(hp); per = self.per_h(hp)
        cnt = (self.cnt_h(hp_ae).squeeze(-1) + self.area_log_bias).clamp(0,6)
        trd = self.trend_h(hm)
        return hot, loc, typ, per, cnt, trd

# =============================================================================
# RUN
# =============================================================================
if __name__ == '__main__':
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
    model = WithoutCTIG().to(DEVICE)
train_and_save(model, compute_loss_full, 'WithoutCTIG')

In [ ]:
# =============================================================================
# ABLATION 3 — WithoutAHG
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os, time, copy, csv
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, GCNConv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
    classification_report,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# CONFIG  (identical to v13)
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# =============================================================================
# ADJACENCY GRAPHS  (identical to v13)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# DATA LOADING  (identical to v13 — run once, share across all variants)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
chunk_dfs = []
for i, chunk in enumerate(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"])):
    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows")

print("Combining chunks...")
crimes   = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

type_counts = crimes["Primary Type"].value_counts()
top_types   = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"Types ({N_TYPES}): {TYPE_CLASSES}")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]
hour_bucket_aggs = {
    f"hour_{s}_{e}": ("hour", lambda x, s=s, e=e: ((x >= s) & (x < e)).sum())
    for s, e in HOUR_BUCKETS}

grid = crimes.groupby(["date","ca"]).agg(
    crime_count = ("Primary Type","count"),
    avg_hour    = ("hour","mean"),
    eve_cnt     = ("hour", lambda x: (x>=18).sum()),
    night_cnt   = ("hour", lambda x: (x<6).sum()),
    mean_lat    = ("Latitude","mean"),
    mean_lon    = ("Longitude","mean"),
    std_lat     = ("Latitude","std"),
    std_lon     = ("Longitude","std"),
    arrest_rate = ("Arrest", lambda x: x.astype(bool).mean()),
    **{f"cnt_{t.replace(' ','_').replace('/','_')}":
       ("Primary Type", lambda x, t=t: (x==t).sum()) for t in top_types},
    **hour_bucket_aggs,
).reset_index()

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)
RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)
for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"]).reindex(full_idx, fill_value=0)
            .reset_index().sort_values(["date","ca"]).reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["roll7"]  = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
grid["roll14"] = gc.transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean())
grid["roll30"] = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
grid["roll90"] = gc.transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
grid["std7"]   = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=2).std().fillna(0))
grid["std30"]  = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=2).std().fillna(0))
grid["lag1"]   = gc.transform(lambda x: x.shift(1).fillna(0))
grid["lag7"]   = gc.transform(lambda x: x.shift(7).fillna(0))
grid["lag14"]  = gc.transform(lambda x: x.shift(14).fillna(0))
grid["lag28"]  = gc.transform(lambda x: x.shift(28).fillna(0))
grid["wow"]    = (grid["roll7"] / grid["roll7"].shift(7).clip(1)).fillna(1)
grid["mom"]    = (grid["roll14"] / grid["roll14"].shift(14).clip(1)).fillna(1)

gh = grid.groupby("ca")["avg_hour"]
grid["roll_hour7"]  = gh.transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
grid["roll_hour30"] = gh.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col = f"r7_{col}"; r30_col = f"r30_{col}"
    grid[r7_col]  = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    grid[r30_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean())
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)
area_lat = grid.groupby("ca")["mean_lat"].transform("mean")
area_lon = grid.groupby("ca")["mean_lon"].transform("mean")
grid["area_lat"] = area_lat
grid["area_lon"] = area_lon

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(7, min_periods=1).mean())
        grid[f"r30_{col}"] = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(30, min_periods=1).mean())

loc_feats = []
for lt in cfg.TOP_LOC_TYPES:
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1).reindex(full_idx, fill_value=0).reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom","roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]
FEAT_COLS  = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
              HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT     = len(FEAT_COLS)
print(f"Features: {N_FEAT}")

# Splits
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")
n_train = int((grid["date"] <= split_train).sum() / N_COMM)
n_val   = int((grid["date"] <= split_val).sum()   / N_COMM)
train_mask = grid["date"] <= split_train
val_mask   = (grid["date"] > split_train) & (grid["date"] <= split_val)
test_mask  = grid["date"] > split_val

# Labels
grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std

grid = grid.sort_values(["ca","day"])
top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)
all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER" for t in TYPE_CLASSES]
r14_type_cols = []
for col in all_cnt_cols:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_type_cols.append(r14_col)
grid["dominant_type"] = np.argmax(grid[r14_type_cols].values, axis=1).astype(np.int64)
zero_mask = grid[r14_type_cols].sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS = 8
r14_bucket_cols = []
for col in RAW_BUCKET_COLS:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_bucket_cols.append(r14_col)
grid["dominant_period"] = np.argmax(grid[r14_bucket_cols].values, axis=1).astype(np.int64)
zero_mask_b = grid[r14_bucket_cols].sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in range(1, N_COMM+1)], dtype=np.float32)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["wk2_now"]   = gc.transform(lambda x: x.shift(1).rolling(14,min_periods=3).mean())
grid["wk2_prior"] = gc.transform(lambda x: x.shift(15).rolling(14,min_periods=3).mean().clip(1))
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# Normalize
scaler = StandardScaler()
grid.loc[train_mask, FEAT_COLS]  = scaler.fit_transform(grid.loc[train_mask, FEAT_COLS])
grid.loc[~train_mask, FEAT_COLS] = scaler.transform(grid.loc[~train_mask, FEAT_COLS])
grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf,-np.inf], np.nan).fillna(0)

feat_np  = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np   = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np   = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np   = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw  = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw  = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np  = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np   = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
print("Data ready.")

# =============================================================================
# GRAPHS
# =============================================================================
def build_ei(adj, id2node):
    src, dst = [], []
    for n, nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src, dst], dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1, N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1, N_COMM+1)],
                            dtype=torch.long).to(DEVICE)

# =============================================================================
# DATASET  (identical to v13)
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T = T; sl = slice(s, e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n = max(0, (e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t = idx + self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]))

T = cfg.TIMESTEPS
train_ds = CrimeDS(0, n_train, T)
val_ds   = CrimeDS(n_train, n_val, T)
test_ds  = CrimeDS(n_val, NUM_DAYS, T)
dl_kw = dict(batch_size=cfg.BATCH, num_workers=cfg.NUM_WORKERS,
             pin_memory=True, persistent_workers=cfg.NUM_WORKERS>0)
train_loader = DataLoader(train_ds, shuffle=True,  **dl_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **dl_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kw)
print(f"Loaders ready: Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}")

# =============================================================================
# LOSS WEIGHTS  (identical to v13)
# =============================================================================
hot_rate = grid.loc[train_mask, "is_hotspot"].mean()
pos_wt   = torch.tensor((1-hot_rate)/hot_rate, dtype=torch.float32).to(DEVICE)

tc = np.bincount(type_np[:n_train].reshape(-1), minlength=N_TYPES).astype(float); tc /= tc.sum()
typw = torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES), dtype=torch.float32).to(DEVICE)

pc = np.bincount(per_np[:n_train].reshape(-1), minlength=N_PERIODS).astype(float); pc /= pc.sum()
perw = torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS), dtype=torch.float32).to(DEVICE)

trc = np.bincount(trend_np[:n_train].reshape(-1), minlength=3).astype(float); trc /= trc.sum()
tw  = torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3), dtype=torch.float32).to(DEVICE)

# =============================================================================
# LOSS FUNCTIONS  (identical to v13)
# =============================================================================
def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls = logits.shape[-1]
    log_p = F.log_softmax(logits, dim=-1); p = torch.exp(log_p)
    with torch.no_grad():
        smooth_t = torch.full_like(logits, smoothing/(n_cls-1))
        smooth_t.scatter_(-1, targets.unsqueeze(-1), 1.0-smoothing)
    ce_smooth = -(smooth_t * log_p).sum(-1)
    p_t = p.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    focal_w = (1.0-p_t)**gamma
    if weight is not None: focal_w = focal_w * weight[targets]
    focal = (focal_w * ce_smooth).mean()
    entropy = -(p * log_p).sum(-1).mean()
    return focal - entropy_reg * entropy


def compute_loss_full(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                      hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    l_hot = F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)
    valid = (lat_t.abs()>0.01) | (lon_t.abs()>0.01)
    l_loc = (F.huber_loss(loc_p[...,0][valid], lat_t[valid], delta=1.0) +
             F.huber_loss(loc_p[...,1][valid], lon_t[valid], delta=1.0)) if valid.any() \
            else torch.tensor(0.0, device=hot_l.device)
    l_typ = focal_loss_with_smooth(typ_l.reshape(-1,N_TYPES), typ_t.reshape(-1),
                weight=typw, gamma=cfg.FOCAL_GAMMA,
                smoothing=cfg.LABEL_SMOOTH, entropy_reg=cfg.ENTROPY_REG)
    l_per = F.cross_entropy(per_l.reshape(-1,N_PERIODS), per_t.reshape(-1), weight=perw)
    l_cnt = F.huber_loss(cnt_p.reshape(-1), cnt_t.reshape(-1), delta=0.5)
    l_trd = F.cross_entropy(trd_l.reshape(-1,3), trd_t.reshape(-1), weight=tw)
    return 1.0*l_hot + 1.0*l_loc + 1.5*l_typ + 0.6*l_per + 1.2*l_cnt + 0.5*l_trd


def compute_loss_hotspot_only(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    """WithoutMultiTask — only hotspot BCE loss."""
    return F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1, lo1, la2, lo2):
    R = 6371.0; dlat = np.radians(la2-la1); dlon = np.radians(lo2-lo1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a), np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a, dtype=np.float32))

# =============================================================================
# EVALUATE  (identical to v13 — shared by ALL variants)
# =============================================================================
def evaluate(model_fn, loader):
    """
    model_fn: callable(x, fine_edge, coarse_edge) → (hot, loc, typ, per, cnt, trd)
              Some ablation models return dummy tensors for missing heads.
    """
    ht, hs = [], []
    latt, lont, latr, lonr, latp, lonp = [], [], [], [], [], []
    tytt, tytp = [], []
    pert, perp = [], []
    cntr, cntp = [], []
    trt_, trp_ = [], []
    tot = n = 0

    with torch.no_grad():
        for batch in loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model_fn(x, fine_edge, coarse_edge)

            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latt.extend(lat_t.reshape(-1).cpu().numpy())
            lont.extend(lon_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())

    ht = np.array(ht, dtype=np.int32); hs = san(hs)
    pr, re, thr = precision_recall_curve(ht, hs)
    f1c = 2*pr*re/(pr+re+1e-8)
    bt  = float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp  = (hs >= bt).astype(int)
    try: rauc = roc_auc_score(ht, hs)
    except: rauc = 0.0
    hot_f1 = f1_score(ht, hp, zero_division=0)

    lp_raw = san(latp)*lat_std+lat_mean; lr_raw = np.array(latr)
    op_raw = san(lonp)*lon_std+lon_mean; or_raw = np.array(lonr)
    valid  = (lr_raw>40) & (or_raw<-80)
    if valid.sum() > 0:
        hav    = haversine_km(lr_raw[valid], or_raw[valid], lp_raw[valid], op_raw[valid])
        loc_mae = float(hav.mean()); l1 = float((hav<1).mean()); l05 = float((hav<0.5).mean())
    else:
        loc_mae = 99.9; l1 = 0.0; l05 = 0.0

    ta = accuracy_score(np.array(tytt), np.array(tytp))
    tf = f1_score(np.array(tytt), np.array(tytp), average="weighted", zero_division=0)
    pa = accuracy_score(np.array(pert), np.array(perp))
    pf = f1_score(np.array(pert), np.array(perp), average="weighted", zero_division=0)
    cr = san(cntr); cp_a = san(cntp)
    cnt_corr = float(np.corrcoef(cr, cp_a)[0,1]) if len(cr)>1 else 0.0
    cnt_mae  = float(np.abs(np.expm1(cr.clip(0,10)) - np.expm1(cp_a.clip(0,10))).mean())
    trend_acc = accuracy_score(np.array(trt_), np.array(trp_))
    loc_score = max(0, 1 - loc_mae/10)
    primary   = np.mean([rauc, ta, pa, max(cnt_corr,0), loc_score])

    return dict(
        primary=primary, hot_auc=rauc, hot_f1=hot_f1, best_thr=bt,
        loc_mae=loc_mae, loc_1km=l1, loc_05km=l05,
        type_acc=ta, type_f1=tf, per_acc=pa, per_f1=pf,
        cnt_corr=cnt_corr, cnt_mae=cnt_mae, trend_acc=trend_acc,
        ht=ht, hs=hs, hp=hp,
    )

# =============================================================================
# ══════════════════════════════════════════════════════════════════════════════
#   MODEL DEFINITIONS — one class per ablation variant
# ══════════════════════════════════════════════════════════════════════════════
# =============================================================================


# =============================================================================
# SHARED SUB-MODULES
# =============================================================================

class ARATABlock(nn.Module):
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H = n_heads; self.dh = d//n_heads
        self.qkv = nn.Linear(d, 3*d, bias=False); self.out = nn.Linear(d, d, bias=False)
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(dropout)
        self.log_lam_bwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam)))
        self.log_lam_fwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam*0.5)))
        self.n_causal = n_heads//2
        self.tconv      = nn.Conv1d(d, d, kernel_size=3, padding=1, groups=d, bias=False)
        self.tconv_norm = nn.LayerNorm(d)
    def forward(self, x):
        B, T, NF, d = x.shape
        xc = x.permute(0,2,3,1).reshape(B*NF, d, T)
        xc = self.tconv(xc)
        xc = xc.reshape(B, NF, d, T).permute(0,3,1,2)
        x  = self.tconv_norm(x + xc)
        x2 = x.reshape(B*NF, T, d)
        Q, K, V = self.qkv(x2).chunk(3, dim=-1)
        Q = Q.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        K = K.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        V = V.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        pi = torch.arange(T, device=x.device).float().unsqueeze(1)
        pj = torch.arange(T, device=x.device).float().unsqueeze(0)
        diff = pi - pj
        lb = torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf = torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias = (-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0) +
                -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal > 0:
            cm = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
            bias[:self.n_causal] += cm
        attn = F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0), dim=-1)
        out = (self.drop(attn)@V).transpose(1,2).reshape(B*NF, T, d)
        return self.norm(x + self.out(out).reshape(B, NF, T, d).permute(0,2,1,3))

class PlainTransformerBlock(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff   = nn.Sequential(nn.Linear(d,d*2), nn.GELU(), nn.Linear(d*2,d))
        self.n1   = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, NF, d = x.shape
        x2 = x.reshape(B*NF, T, d)
        a, _ = self.attn(x2, x2, x2)
        x2   = self.n1(x2 + self.drop(a))
        x2   = self.n2(x2 + self.drop(self.ff(x2)))
        return x2.reshape(B, T, NF, d)

class CTIGBlock(nn.Module):
    def __init__(self, H, n_types, dropout):
        super().__init__()
        self.K = n_types
        self.proj = nn.Linear(H, H)
        self.eg   = nn.Sequential(nn.Linear(2*H, H), nn.GELU(), nn.Linear(H, 1))
        self.op   = nn.Linear(H, H); self.norm = nn.LayerNorm(H); self.drop = nn.Dropout(dropout)
    def forward(self, tr, hp):
        B, NF, H = hp.shape
        tf = self.proj(tr.unsqueeze(-1) * hp.unsqueeze(2))
        tm = tf.mean(1)
        ti = tm.unsqueeze(2).expand(-1,-1,self.K,-1)
        tj = tm.unsqueeze(1).expand(-1,self.K,-1,-1)
        ew = torch.sigmoid(self.eg(torch.cat([ti,tj], dim=-1))).squeeze(-1)
        k3 = min(3, self.K-1)
        tv, tix = ew.topk(k3, dim=-1)
        es = torch.zeros_like(ew).scatter_(2, tix, tv)
        msg = (torch.einsum("bkj,bjh->bkh", es, tm).unsqueeze(1).expand(-1,NF,-1,-1))
        ht  = (tr.unsqueeze(-1) * self.op(msg)).sum(2)
        return self.norm(hp + self.drop(ht))

class AHGLayer(nn.Module):
    def __init__(self, H, n_fine, n_coarse, heads, dropout):
        super().__init__()
        self.gf   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gc   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gate = nn.Linear(H*2, 1); self.norm = nn.LayerNorm(H)
    def forward(self, hf, hc, fei, cei, c2dn):
        B, NF, H = hf.shape
        hf2  = torch.stack([self.gf(hf[b], fei) for b in range(B)])
        hc2  = torch.stack([self.gc(hc[b], cei) for b in range(B)])
        hcca = hc2[:,c2dn,:]
        alpha = torch.sigmoid(self.gate(torch.cat([hf2, hcca], dim=-1)))
        return self.norm(hf + alpha*hf2 + (1-alpha)*hcca), hc2

class FlatGATLayer(nn.Module):
    def __init__(self, H, heads, dropout):
        super().__init__()
        self.gat  = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.norm = nn.LayerNorm(H)
    def forward(self, hf, hc, fei, cei, c2dn):
        B = hf.shape[0]
        hf2 = torch.stack([self.gat(hf[b], fei) for b in range(B)])
        return self.norm(hf + hf2), hc

class GCNLayer(nn.Module):
    def __init__(self, H, dropout):
        super().__init__()
        self.gcn  = GCNConv(H, H, add_self_loops=True)
        self.norm = nn.LayerNorm(H); self.drop = nn.Dropout(dropout)
    def forward(self, hf, hc, fei, cei, c2dn):
        B = hf.shape[0]
        hf2 = torch.stack([self.gcn(hf[b], fei) for b in range(B)])
        return self.norm(hf + self.drop(hf2)), hc

def _mlp(i, m, o, dropout):
    blk = nn.Sequential(nn.Linear(i,m), nn.GELU(), nn.Dropout(dropout), nn.Linear(m,o))
    for l in blk:
        if isinstance(l, nn.Linear):
            nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
    return blk

# =============================================================================
# EVALUATE
# =============================================================================
def evaluate(model, loader):
    model.eval()
    ht, hs = [], []
    latr, lonr, latp, lonp = [], [], [], []
    tytt, tytp, pert, perp, cntr, cntp, trt_, trp_ = [], [], [], [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())
    ht = np.array(ht, dtype=np.int32); hs = san(hs)
    pr, re, thr = precision_recall_curve(ht, hs)
    f1c = 2*pr*re/(pr+re+1e-8)
    bt  = float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp  = (hs >= bt).astype(int)
    try: rauc = roc_auc_score(ht, hs)
    except: rauc = 0.0
    hot_f1 = f1_score(ht, hp, zero_division=0)
    lp_raw = san(latp)*lat_std+lat_mean; lr_raw = np.array(latr)
    op_raw = san(lonp)*lon_std+lon_mean; or_raw = np.array(lonr)
    valid  = (lr_raw>40) & (or_raw<-80)
    if valid.sum() > 0:
        hav = haversine_km(lr_raw[valid], or_raw[valid], lp_raw[valid], op_raw[valid])
        loc_mae = float(hav.mean()); l1 = float((hav<1).mean()); l05 = float((hav<0.5).mean())
    else:
        loc_mae = 99.9; l1 = 0.0; l05 = 0.0
    ta = accuracy_score(np.array(tytt), np.array(tytp))
    tf = f1_score(np.array(tytt), np.array(tytp), average="weighted", zero_division=0)
    pa = accuracy_score(np.array(pert), np.array(perp))
    pf = f1_score(np.array(pert), np.array(perp), average="weighted", zero_division=0)
    cr = san(cntr); cp_a = san(cntp)
    cnt_corr = float(np.corrcoef(cr, cp_a)[0,1]) if len(cr)>1 else 0.0
    cnt_mae  = float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp_a.clip(0,10))).mean())
    trend_acc = accuracy_score(np.array(trt_), np.array(trp_))
    loc_score = max(0, 1-loc_mae/10)
    primary   = np.mean([rauc, ta, pa, max(cnt_corr,0), loc_score])
    return dict(
        primary=primary, hot_auc=rauc, hot_f1=hot_f1,
        loc_mae=loc_mae, loc_1km=l1, loc_05km=l05,
        type_acc=ta, type_f1=tf, per_acc=pa, per_f1=pf,
        cnt_corr=cnt_corr, cnt_mae=cnt_mae, trend_acc=trend_acc,
    )

# =============================================================================
# SAVE RESULTS
# =============================================================================
def save_results(name, metrics, n_params, train_min, best_ep, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    rows = [
        ["variant",       name],
        ["n_params",      n_params],
        ["train_min",     train_min],
        ["best_epoch",    best_ep],
        ["hot_auc",       f"{metrics['hot_auc']:.4f}"],
        ["hot_f1",        f"{metrics['hot_f1']:.4f}"],
        ["loc_mae_km",    f"{metrics['loc_mae']:.3f}"],
        ["loc_1km_pct",   f"{metrics['loc_1km']*100:.1f}"],
        ["loc_05km_pct",  f"{metrics['loc_05km']*100:.1f}"],
        ["type_acc",      f"{metrics['type_acc']:.4f}"],
        ["type_f1",       f"{metrics['type_f1']:.4f}"],
        ["hour_acc",      f"{metrics['per_acc']:.4f}"],
        ["hour_f1",       f"{metrics['per_f1']:.4f}"],
        ["cnt_pearson_r", f"{metrics['cnt_corr']:.4f}"],
        ["cnt_mae",       f"{metrics['cnt_mae']:.2f}"],
        ["trend_acc",     f"{metrics['trend_acc']:.4f}"],
        ["primary",       f"{metrics['primary']:.4f}"],
    ]
    path = os.path.join(out_dir, f"{name}_results.csv")
    with open(path, "w", newline="") as f:
        import csv; csv.writer(f).writerows(rows)
    print(f"  Results saved → {path}")

# =============================================================================
# TRAIN + EVALUATE
# =============================================================================
def train_and_save(model, loss_fn, variant_name, out_dir="ablation_results"):
    os.makedirs(out_dir, exist_ok=True)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*70}\n  {variant_name}  |  params={n_p:,}\n{'='*70}")
    with torch.no_grad():
        if hasattr(model, "area_log_bias"):
            model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
        if hasattr(model, "area_lat_bias"):
            model.area_lat_bias.copy_(torch.tensor(area_lat_z, dtype=torch.float32).to(DEVICE))
        if hasattr(model, "area_lon_bias"):
            model.area_lon_bias.copy_(torch.tensor(area_lon_z, dtype=torch.float32).to(DEVICE))
    optimizer  = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler  = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.LR*8,
        total_steps=cfg.EPOCHS*len(train_loader),
        pct_start=0.10, anneal_strategy="cos")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)
    best_p = 0.0; best_ep = 0; nan_streak = 0; best_sd = None
    t0 = time.time()
    for ep in range(1, cfg.EPOCHS+1):
        model.train(); ep_loss = 0.0; ep_n = 0
        for batch in train_loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
                loss = loss_fn(ho, lo, ty, pe, cn, tr,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t)
            if torch.isnan(loss):
                nan_streak += 1
                if nan_streak >= 20: print("  ⚠ 20 NaN — aborting"); break
                continue
            nan_streak = 0
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
            ep_loss += loss.item()*x.shape[0]; ep_n += x.shape[0]
        else:
            s = evaluate(model, val_loader)
            tl = ep_loss / max(1, ep_n)
            if ep % 10 == 0 or s["primary"] > best_p:
                print(f"  ep={ep:3d}  tl={tl:.4f}  primary={s['primary']:.4f}  "
                      f"AUC={s['hot_auc']:.4f}  TypeAcc={s['type_acc']:.4f}  "
                      f"HrAcc={s['per_acc']:.4f}  CntR={s['cnt_corr']:.4f}  "
                      f"LocMAE={s['loc_mae']:.2f}km")
            if s["primary"] > best_p:
                best_p = s["primary"]; best_ep = ep
                best_sd = copy.deepcopy(model.state_dict())
                ckpt = os.path.join(out_dir, f"{variant_name}_best.pth")
                torch.save(best_sd, ckpt)
                print(f"  ✓ Best={best_p:.4f}  checkpoint → {ckpt}")
            if ep - best_ep >= cfg.PATIENCE:
                print(f"  ⏹ Early stop ep={ep}"); break
            continue
        break
    elapsed = time.time() - t0
    if best_sd: model.load_state_dict(best_sd)
    model.eval()
    metrics   = evaluate(model, test_loader)
    train_min = round(elapsed/60, 1)
    print(f"\n  ✅ TEST  AUC={metrics['hot_auc']:.4f}  LocMAE={metrics['loc_mae']:.2f}km  "
          f"TypeAcc={metrics['type_acc']:.4f}  HrAcc={metrics['per_acc']:.4f}  "
          f"CntR={metrics['cnt_corr']:.4f}  ({train_min} min)")
    save_results(variant_name, metrics, n_p, train_min, best_ep, out_dir)

class WithoutAHG(nn.Module):
    def __init__(self):
        super().__init__()
        H, heads, drop = cfg.HIDDEN_DIM, cfg.GAT_HEADS, cfg.DROPOUT
        self.pf    = nn.Sequential(nn.Linear(N_FEAT,H), nn.LayerNorm(H), nn.GELU())
        self.arata = nn.ModuleList([ARATABlock(H,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.gat   = nn.ModuleList([FlatGATLayer(H,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.ctig  = CTIGBlock(H, cfg.TOP_N_TYPES, drop)
        self.mp    = nn.Sequential(nn.Linear(H,H), nn.GELU(), nn.LayerNorm(H))
        self.n_rec = 3
        self.hot_h  = _mlp(H*2,64,1,drop); self.loc_h = _mlp(H+32,64,2,drop)
        self.type_h = _mlp(H,128,N_TYPES,drop); self.per_h = _mlp(H,64,N_PERIODS,drop)
        self.cnt_h  = _mlp(H+32,64,1,drop);     self.trend_h = _mlp(H*2,64,3,drop)
        self.area_emb      = nn.Embedding(N_COMM,32)
        self.area_lat_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_lon_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_log_bias = nn.Parameter(torch.zeros(N_COMM))
    def forward(self, xf, fei, cei):
        B, T, NF, _ = xf.shape; H = cfg.HIDDEN_DIM; DEV = xf.device
        hf = self.pf(xf)
        for a in self.arata: hf = a(hf)
        hp = hf[:,-1,:,:]
        hc_dummy = torch.zeros(B, N_DIST, H, device=DEV)
        c2dn = CA_TO_DN.to(DEV)
        for g in self.gat: hp, _ = g(hp, hc_dummy, fei, cei, c2dn)
        hr = hf[:,-self.n_rec:,:,:].mean(1); hl = hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom = self.mp(hr - hl); hm = torch.cat([hp, mom], dim=-1)
        trs = len(TEMP_COLS)+len(CAL_COLS)
        ntr = min(cfg.TOP_N_TYPES, xf.shape[-1]-trs)
        tr  = xf[:,-1,:,trs:trs+ntr] if ntr>0 else torch.zeros(B,NF,cfg.TOP_N_TYPES,device=DEV)
        hp  = self.ctig(tr, hp)
        ae  = self.area_emb(torch.arange(NF, device=DEV).unsqueeze(0).expand(B,-1))
        hp_ae = torch.cat([hp, ae], dim=-1)
        hot = self.hot_h(hm).squeeze(-1)
        loc = torch.stack([self.loc_h(hp_ae)[...,0]+self.area_lat_bias,
                           self.loc_h(hp_ae)[...,1]+self.area_lon_bias], dim=-1)
        typ = self.type_h(hp); per = self.per_h(hp)
        cnt = (self.cnt_h(hp_ae).squeeze(-1) + self.area_log_bias).clamp(0,6)
        trd = self.trend_h(hm)
        return hot, loc, typ, per, cnt, trd

# =============================================================================
# RUN
# =============================================================================
if __name__ == '__main__':
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
    model = WithoutAHG().to(DEVICE)
train_and_save(model, compute_loss_full, 'WithoutAHG')

In [ ]:
# =============================================================================
# ABLATION 4 — WithoutMultiTask (FIXED: variant-aware checkpoint selection
# and reporting so the hotspot-only head is never compared against
# untrained auxiliary heads)
# =============================================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import math, os, time, copy, csv
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, GCNConv
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_recall_curve, roc_curve, auc,
    ConfusionMatrixDisplay, confusion_matrix,
    classification_report, precision_score, recall_score,
    average_precision_score,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# CONFIG  (identical to v13)
# =============================================================================
class CFG:
    DATA_PATH     = "/kaggle/working/Crimes_2001_to_Present.csv"
    CHUNK_SIZE    = 1_000_000
    TIMESTEPS     = 60
    EPOCHS        = 120
    BATCH         = 16
    LR            = 2e-4
    WEIGHT_DECAY  = 1e-4
    DROPOUT       = 0.30
    PATIENCE      = 25
    HIDDEN_DIM    = 96
    GAT_HEADS     = 4
    N_LAYERS      = 3
    HOTSPOT_TOP_K = 15
    N_COMM        = 77
    TOP_N_TYPES   = 9
    TOP_LOC_TYPES = [
        "STREET","RESIDENCE","APARTMENT","SIDEWALK","OTHER",
        "PARKING LOT/GARAGE(NON.RESID.)","ALLEY","SCHOOL, PUBLIC, BUILDING",
        "RESTAURANT","GAS STATION","GROCERY FOOD STORE","VEHICLE NON-COMMERCIAL",
        "SMALL RETAIL STORE","BAR OR TAVERN","CTA PLATFORM",
    ]
    NUM_WORKERS   = 4
    FP16          = True
    SEED          = 42
    FOCAL_GAMMA   = 2.0
    LABEL_SMOOTH  = 0.05
    ENTROPY_REG   = 0.05

cfg = CFG()
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# =============================================================================
# ADJACENCY GRAPHS  (identical to v13)
# =============================================================================
DISTRICT_ADJ = {
    1:[2,12,18], 2:[1,3,18,21], 3:[2,4,7,21], 4:[3,5,7,8],
    5:[4,6,8], 6:[5,8,9], 7:[3,4,8,11], 8:[4,5,6,7,9,10,11],
    9:[6,8,10], 10:[8,9,11,22], 11:[7,8,10,15], 12:[1,14,18],
    14:[12,15,16,18], 15:[11,14,16], 16:[14,15,17], 17:[16,19,20,25],
    18:[1,2,12,14,19,20,24], 19:[17,18,20,24,25], 20:[17,18,19,24],
    21:[2,3], 22:[10,11], 24:[18,19,20,25], 25:[17,19,24],
}
ACTIVE_DIST = sorted([1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,21,22,24,25])
dist2node   = {d:i for i,d in enumerate(ACTIVE_DIST)}
N_DIST      = len(ACTIVE_DIST)

CA_TO_DIST = {
    1:24,2:24,3:19,4:19,5:19,6:19,7:18,8:18,
    9:16,10:16,11:16,12:16,13:17,14:17,15:16,16:16,
    17:17,18:25,19:25,20:14,21:14,22:14,23:15,24:12,
    25:15,26:11,27:11,28:10,29:10,30:10,31:9,32:1,
    33:1,34:2,35:2,36:3,37:3,38:3,39:4,40:4,
    41:4,42:5,43:5,44:6,45:6,46:4,47:4,48:5,
    49:5,50:5,51:5,52:5,53:5,54:6,55:5,56:8,
    57:8,58:9,59:9,60:9,61:9,62:8,63:8,64:8,
    65:8,66:8,67:7,68:7,69:3,70:6,71:6,72:22,
    73:22,74:22,75:22,76:20,77:19,
}

CA_ADJ = {
     1:[2,3],2:[1,3,7],3:[1,2,4,21,22],4:[3,5,6,22],
     5:[4,6],6:[4,5,7,22,67,68],7:[2,6,21,67,68],
     8:[9,11,12,32,33],9:[8,10,11],10:[9,11,29],
    11:[8,9,10,28,29],12:[8,13,24,25,32],13:[12,14,24],
    14:[13,15,23,24],15:[14,16,23,25],16:[15,17,22,24,25,26],
    17:[16,18,21,22,26,27],18:[17,19,20,21],19:[18,20],
    20:[18,19,21,22],21:[2,3,18,20,22],22:[3,4,16,17,20,21,23],
    23:[14,15,22,24,68,69],24:[12,13,14,15,16,23],
    25:[12,15,16,26,27,28],26:[16,17,25,27],27:[17,25,26,28,29,30],
    28:[11,25,27,29,30,31],29:[10,11,28,30,31],30:[27,28,29,31],
    31:[28,29,30,32,33,34,35],32:[8,12,31,33],33:[8,31,32,34,37],
    34:[31,33,35,36,37],35:[31,34,36,38,39],36:[34,35,37,38],
    37:[33,34,36,38,46],38:[35,36,37,39,46],39:[35,38,40,43,44],
    40:[39,41,43],41:[40,42,43],42:[41,43,57,58,59,60,61],
    43:[39,40,41,42,44],44:[39,43,45,46,47,48],
    45:[44,46,47],46:[37,38,44,45,47],47:[44,45,46,48,49,50],
    48:[44,47,49,56,57],49:[47,48,50,56],50:[47,49,51,52,53,54,55],
    51:[50,52],52:[50,51,53],53:[50,52,54],54:[50,53,55,57,65,66],
    55:[50,54,56,57],56:[48,49,55,57],57:[42,48,54,55,56,58],
    58:[42,57,59],59:[42,58,60],60:[42,59,61],61:[42,60,62,67,68,69],
    62:[61,63,64,67],63:[62,64,65],64:[62,63,65],
    65:[54,63,64,66],66:[54,65],67:[6,7,61,62,68],
    68:[6,7,23,61,67,69],69:[23,61,68],70:[71],71:[70,72],
    72:[71,73,75],73:[72,74,75],74:[73,75],75:[72,73,74,76,77],
    76:[75,77],77:[75,76],
}

# =============================================================================
# DATA LOADING  (identical to v13 — run once, share across all variants)
# =============================================================================
print(f"\nLoading: {cfg.DATA_PATH}")
chunk_dfs = []
for i, chunk in enumerate(pd.read_csv(
        cfg.DATA_PATH, chunksize=cfg.CHUNK_SIZE,
        usecols=["Date","Primary Type","Community Area",
                 "Latitude","Longitude","Beat","District",
                 "Location Description","Arrest","Domestic"])):
    chunk = chunk.dropna(subset=["Community Area","Latitude","Longitude"])
    chunk["ca"] = chunk["Community Area"].astype(int)
    chunk = chunk[(chunk["ca"] >= 1) & (chunk["ca"] <= 77)]
    chunk["datetime"] = pd.to_datetime(chunk["Date"],
                                        format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    chunk = chunk.dropna(subset=["datetime"])
    chunk["date"]  = chunk["datetime"].dt.normalize()
    chunk["hour"]  = chunk["datetime"].dt.hour.astype(np.int8)
    chunk["dow"]   = chunk["datetime"].dt.dayofweek.astype(np.int8)
    chunk["month"] = chunk["datetime"].dt.month.astype(np.int8)
    chunk_dfs.append(chunk)
    print(f"  Chunk {i+1}: {len(chunk):,} rows")

print("Combining chunks...")
crimes   = pd.concat(chunk_dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
all_dates = pd.Series(crimes["date"].unique()).sort_values().reset_index(drop=True)
date2idx  = {d:i for i,d in enumerate(all_dates)}
NUM_DAYS  = len(all_dates)
N_COMM    = cfg.N_COMM

type_counts = crimes["Primary Type"].value_counts()
top_types   = type_counts.head(cfg.TOP_N_TYPES).index.tolist()
crimes["type_raw"] = crimes["Primary Type"].where(
    crimes["Primary Type"].isin(top_types), other="OTHER")
TYPE_CLASSES = sorted(top_types) + ["OTHER"]
type2idx     = {t:i for i,t in enumerate(TYPE_CLASSES)}
N_TYPES      = len(TYPE_CLASSES)
crimes["type_idx"] = crimes["type_raw"].map(type2idx).astype(np.int8)
print(f"Types ({N_TYPES}): {TYPE_CLASSES}")

HOUR_BUCKETS     = [(0,3),(3,6),(6,9),(9,12),(12,15),(15,18),(18,21),(21,24)]
HOUR_BUCKET_COLS = [f"hour_{s}_{e}" for s,e in HOUR_BUCKETS]
hour_bucket_aggs = {
    f"hour_{s}_{e}": ("hour", lambda x, s=s, e=e: ((x >= s) & (x < e)).sum())
    for s, e in HOUR_BUCKETS}

grid = crimes.groupby(["date","ca"]).agg(
    crime_count = ("Primary Type","count"),
    avg_hour    = ("hour","mean"),
    eve_cnt     = ("hour", lambda x: (x>=18).sum()),
    night_cnt   = ("hour", lambda x: (x<6).sum()),
    mean_lat    = ("Latitude","mean"),
    mean_lon    = ("Longitude","mean"),
    std_lat     = ("Latitude","std"),
    std_lon     = ("Longitude","std"),
    arrest_rate = ("Arrest", lambda x: x.astype(bool).mean()),
    **{f"cnt_{t.replace(' ','_').replace('/','_')}":
       ("Primary Type", lambda x, t=t: (x==t).sum()) for t in top_types},
    **hour_bucket_aggs,
).reset_index()

grid["std_lat"] = grid["std_lat"].fillna(0)
grid["std_lon"] = grid["std_lon"].fillna(0)
RAW_BUCKET_COLS = [f"raw_{c}" for c in HOUR_BUCKET_COLS]
for col, raw in zip(HOUR_BUCKET_COLS, RAW_BUCKET_COLS):
    grid[raw] = grid[col].astype(np.float32)
for col in HOUR_BUCKET_COLS:
    grid[col] = (grid[col] / grid["crime_count"].clip(1)).astype(np.float32)

full_idx = pd.MultiIndex.from_product(
    [all_dates, range(1, N_COMM+1)], names=["date","ca"])
grid = (grid.set_index(["date","ca"]).reindex(full_idx, fill_value=0)
            .reset_index().sort_values(["date","ca"]).reset_index(drop=True))
grid["day"] = grid["date"].map(date2idx)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["roll7"]  = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
grid["roll14"] = gc.transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean())
grid["roll30"] = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
grid["roll90"] = gc.transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
grid["std7"]   = gc.transform(lambda x: x.shift(1).rolling(7,  min_periods=2).std().fillna(0))
grid["std30"]  = gc.transform(lambda x: x.shift(1).rolling(30, min_periods=2).std().fillna(0))
grid["lag1"]   = gc.transform(lambda x: x.shift(1).fillna(0))
grid["lag7"]   = gc.transform(lambda x: x.shift(7).fillna(0))
grid["lag14"]  = gc.transform(lambda x: x.shift(14).fillna(0))
grid["lag28"]  = gc.transform(lambda x: x.shift(28).fillna(0))
grid["wow"]    = (grid["roll7"] / grid["roll7"].shift(7).clip(1)).fillna(1)
grid["mom"]    = (grid["roll14"] / grid["roll14"].shift(14).clip(1)).fillna(1)

gh = grid.groupby("ca")["avg_hour"]
grid["roll_hour7"]  = gh.transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
grid["roll_hour30"] = gh.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())

HOUR_BUCKET_ROLL_COLS = []
for col in HOUR_BUCKET_COLS:
    r7_col = f"r7_{col}"; r30_col = f"r30_{col}"
    grid[r7_col]  = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    grid[r30_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean())
    HOUR_BUCKET_ROLL_COLS.extend([r7_col, r30_col])

grid["dow_sin"]    = np.sin(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["dow_cos"]    = np.cos(2*np.pi * grid["date"].dt.dayofweek / 7)
grid["month_sin"]  = np.sin(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["month_cos"]  = np.cos(2*np.pi * (grid["date"].dt.month-1) / 12)
grid["dom_sin"]    = np.sin(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["dom_cos"]    = np.cos(2*np.pi * (grid["date"].dt.day-1) / 31)
grid["is_weekend"] = (grid["date"].dt.dayofweek >= 5).astype(np.float32)
grid["year_prog"]  = ((grid["date"].dt.dayofyear - 1) / 365.0).astype(np.float32)
grid["year_norm"]  = ((grid["date"].dt.year - 2001) / 24.0).astype(np.float32)
area_lat = grid.groupby("ca")["mean_lat"].transform("mean")
area_lon = grid.groupby("ca")["mean_lon"].transform("mean")
grid["area_lat"] = area_lat
grid["area_lon"] = area_lon

for t in top_types:
    col = f"cnt_{t.replace(' ','_').replace('/','_')}"
    if col in grid.columns:
        grid[f"r7_{col}"]  = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(7, min_periods=1).mean())
        grid[f"r30_{col}"] = grid.groupby("ca")[col].transform(
            lambda x: x.shift(1).rolling(30, min_periods=1).mean())

loc_feats = []
for lt in cfg.TOP_LOC_TYPES:
    key = f"loc_{lt[:12].replace(' ','_').replace('.','').replace('/','_')}"
    dly = crimes[crimes["Location Description"]==lt]\
        .groupby(["date","ca"]).size().rename(key)
    loc_feats.append(dly)
if loc_feats:
    loc_df = (pd.concat(loc_feats, axis=1).reindex(full_idx, fill_value=0).reset_index())
    loc_df.columns = ["date","ca"] + [c for c in loc_df.columns[2:]]
    grid = grid.merge(loc_df, on=["date","ca"], how="left").fillna(0)
    LOC_COLS = [c for c in loc_df.columns if c not in ["date","ca"]]
else:
    LOC_COLS = []

TEMP_COLS  = ["roll7","roll14","roll30","roll90","std7","std30",
              "lag1","lag7","lag14","lag28","wow","mom","roll_hour7","roll_hour30"]
CAL_COLS   = ["dow_sin","dow_cos","month_sin","month_cos","dom_sin","dom_cos",
              "is_weekend","year_prog","year_norm","area_lat","area_lon"]
TYPE_RCOLS = ([f"r7_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r7_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns] +
              [f"r30_cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
               if f"r30_cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns])
MISC_COLS  = ["avg_hour","eve_cnt","night_cnt","arrest_rate","std_lat","std_lon"]
FEAT_COLS  = (TEMP_COLS + CAL_COLS + TYPE_RCOLS + MISC_COLS +
              HOUR_BUCKET_COLS + HOUR_BUCKET_ROLL_COLS + LOC_COLS)
N_FEAT     = len(FEAT_COLS)
print(f"Features: {N_FEAT}")

# Splits
split_train = pd.Timestamp("2018-03-30")
split_val   = pd.Timestamp("2021-12-09")
n_train = int((grid["date"] <= split_train).sum() / N_COMM)
n_val   = int((grid["date"] <= split_val).sum()   / N_COMM)
train_mask = grid["date"] <= split_train
val_mask   = (grid["date"] > split_train) & (grid["date"] <= split_val)
test_mask  = grid["date"] > split_val

# Labels
grid["day_rank"]   = grid.groupby("day")["crime_count"].rank(ascending=False, method="min")
grid["is_hotspot"] = (grid["day_rank"] <= cfg.HOTSPOT_TOP_K).astype(np.float32)

lat_mean = float(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].mean())
lat_std  = float(max(grid.loc[train_mask & (grid["mean_lat"]>40), "mean_lat"].std(), 1e-4))
lon_mean = float(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].mean())
lon_std  = float(max(grid.loc[train_mask & (grid["mean_lon"]<-80), "mean_lon"].std(), 1e-4))
area_lat_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40), "mean_lat"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lat"]>40)).any() else lat_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
area_lon_centroid = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80), "mean_lon"].mean()
          if (train_mask & (grid["ca"]==ca) & (grid["mean_lon"]<-80)).any() else lon_mean)
    for ca in range(1, N_COMM+1)], dtype=np.float32)
grid["lat_z"] = ((grid["mean_lat"] - lat_mean) / lat_std).fillna(0).clip(-4,4)
grid["lon_z"] = ((grid["mean_lon"] - lon_mean) / lon_std).fillna(0).clip(-4,4)
area_lat_z = (area_lat_centroid - lat_mean) / lat_std
area_lon_z = (area_lon_centroid - lon_mean) / lon_std

grid = grid.sort_values(["ca","day"])
top9_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" for t in top_types
                 if f"cnt_{t.replace(' ','_').replace('/','_')}" in grid.columns]
grid["cnt_OTHER"] = (grid["crime_count"] - grid[top9_cnt_cols].sum(axis=1)).clip(0)
all_cnt_cols = [f"cnt_{t.replace(' ','_').replace('/','_')}" if t != "OTHER"
                else "cnt_OTHER" for t in TYPE_CLASSES]
r14_type_cols = []
for col in all_cnt_cols:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_type_cols.append(r14_col)
grid["dominant_type"] = np.argmax(grid[r14_type_cols].values, axis=1).astype(np.int64)
zero_mask = grid[r14_type_cols].sum(axis=1) == 0
if zero_mask.any():
    grid.loc[zero_mask, "dominant_type"] = np.argmax(
        grid.loc[zero_mask, all_cnt_cols].values, axis=1).astype(np.int64)

HOUR_BUCKET_NAMES = ["0-3h","3-6h","6-9h","9-12h","12-15h","15-18h","18-21h","21-24h"]
N_PERIODS = 8
r14_bucket_cols = []
for col in RAW_BUCKET_COLS:
    r14_col = f"r14_{col}"
    grid[r14_col] = grid.groupby("ca")[col].transform(
        lambda x: x.shift(1).rolling(14, min_periods=3).sum().fillna(
            x.shift(1).expanding(min_periods=1).sum()))
    r14_bucket_cols.append(r14_col)
grid["dominant_period"] = np.argmax(grid[r14_bucket_cols].values, axis=1).astype(np.int64)
zero_mask_b = grid[r14_bucket_cols].sum(axis=1) == 0
if zero_mask_b.any():
    grid.loc[zero_mask_b, "dominant_period"] = np.argmax(
        grid.loc[zero_mask_b, RAW_BUCKET_COLS].values, axis=1).astype(np.int64)

grid["log_cnt"] = np.log1p(grid["crime_count"])
area_log_mean_np = np.array([
    float(grid.loc[train_mask & (grid["ca"]==ca), "log_cnt"].mean())
    for ca in range(1, N_COMM+1)], dtype=np.float32)

grid = grid.sort_values(["ca","day"])
gc = grid.groupby("ca")["crime_count"]
grid["wk2_now"]   = gc.transform(lambda x: x.shift(1).rolling(14,min_periods=3).mean())
grid["wk2_prior"] = gc.transform(lambda x: x.shift(15).rolling(14,min_periods=3).mean().clip(1))
grid["trend_r"]   = (grid["wk2_now"] / grid["wk2_prior"].clip(1)).fillna(1.0)
grid["trend"]     = 1
grid.loc[grid["trend_r"] < 0.90, "trend"] = 0
grid.loc[grid["trend_r"] > 1.10, "trend"] = 2
grid = grid.sort_values(["day","ca"]).reset_index(drop=True)

# Normalize
# =============================================================================
# TRAINING-ONLY FEATURE NORMALIZATION
# =============================================================================
scaler = StandardScaler()

# Fit ONLY on training-period observations.
train_features = grid.loc[train_mask, FEAT_COLS]

scaler.fit(train_features)

# Transform training, validation and test using the training-fitted scaler.
grid.loc[train_mask, FEAT_COLS] = scaler.transform(
    grid.loc[train_mask, FEAT_COLS]
)

grid.loc[val_mask, FEAT_COLS] = scaler.transform(
    grid.loc[val_mask, FEAT_COLS]
)

grid.loc[test_mask, FEAT_COLS] = scaler.transform(
    grid.loc[test_mask, FEAT_COLS]
)

grid[FEAT_COLS] = (
    grid[FEAT_COLS]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("✓ StandardScaler fitted using training data only.")

grid[FEAT_COLS] = grid[FEAT_COLS].replace([np.inf,-np.inf], np.nan).fillna(0)

feat_np  = grid[FEAT_COLS].values.reshape(NUM_DAYS,N_COMM,N_FEAT).astype(np.float32)
hot_np   = grid["is_hotspot"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_np   = grid["lat_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_np   = grid["lon_z"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lat_raw  = grid["mean_lat"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
lon_raw  = grid["mean_lon"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
type_np  = grid["dominant_type"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
per_np   = grid["dominant_period"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
cnt_z_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
log_cnt_np = grid["log_cnt"].values.reshape(NUM_DAYS,N_COMM).astype(np.float32)
trend_np = grid["trend"].values.reshape(NUM_DAYS,N_COMM).astype(np.int64)
print("Data ready.")

# =============================================================================
# GRAPHS
# =============================================================================
def build_ei(adj, id2node):
    src, dst = [], []
    for n, nbrs in adj.items():
        if n not in id2node: continue
        for m in nbrs:
            if m in id2node:
                src.append(id2node[n]); dst.append(id2node[m])
    return torch.tensor([src, dst], dtype=torch.long)

ca2node     = {ca:ca-1 for ca in range(1, N_COMM+1)}
fine_edge   = build_ei(CA_ADJ, ca2node).to(DEVICE)
coarse_edge = build_ei(DISTRICT_ADJ, dist2node).to(DEVICE)
CA_TO_DN    = torch.tensor([dist2node[CA_TO_DIST[ca]] for ca in range(1, N_COMM+1)],
                            dtype=torch.long).to(DEVICE)

# =============================================================================
# DATASET  (identical to v13)
# =============================================================================
class CrimeDS(Dataset):
    def __init__(self, s, e, T):
        self.T = T; sl = slice(s, e)
        self.feat=feat_np[sl]; self.hot=hot_np[sl]
        self.lat=lat_np[sl];   self.lon=lon_np[sl]
        self.lat_raw=lat_raw[sl]; self.lon_raw=lon_raw[sl]
        self.typ=type_np[sl]; self.per=per_np[sl]
        self.cnt_z=cnt_z_np[sl]; self.log_cnt=log_cnt_np[sl]
        self.trend=trend_np[sl]
        self.n = max(0, (e-s)-T-1)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        t = idx + self.T
        return (torch.from_numpy(self.feat[idx:idx+self.T]),
                torch.from_numpy(self.hot[t]),
                torch.from_numpy(self.lat[t]),
                torch.from_numpy(self.lon[t]),
                torch.from_numpy(self.lat_raw[t]),
                torch.from_numpy(self.lon_raw[t]),
                torch.from_numpy(self.typ[t]),
                torch.from_numpy(self.per[t]),
                torch.from_numpy(self.cnt_z[t]),
                torch.from_numpy(self.log_cnt[t]),
                torch.from_numpy(self.trend[t]))

T = cfg.TIMESTEPS
train_ds = CrimeDS(0, n_train, T)
val_ds   = CrimeDS(n_train, n_val, T)
test_ds  = CrimeDS(n_val, NUM_DAYS, T)
dl_kw = dict(batch_size=cfg.BATCH, num_workers=cfg.NUM_WORKERS,
             pin_memory=True, persistent_workers=cfg.NUM_WORKERS>0)
train_loader = DataLoader(train_ds, shuffle=True,  **dl_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **dl_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kw)
print(f"Loaders ready: Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}")

# =============================================================================
# LOSS WEIGHTS  (identical to v13)
# =============================================================================
hot_rate = grid.loc[train_mask, "is_hotspot"].mean()
pos_wt   = torch.tensor((1-hot_rate)/hot_rate, dtype=torch.float32).to(DEVICE)

tc = np.bincount(type_np[:n_train].reshape(-1), minlength=N_TYPES).astype(float); tc /= tc.sum()
typw = torch.tensor((1/(tc+1e-6))/((1/(tc+1e-6)).sum()/N_TYPES), dtype=torch.float32).to(DEVICE)

pc = np.bincount(per_np[:n_train].reshape(-1), minlength=N_PERIODS).astype(float); pc /= pc.sum()
perw = torch.tensor((1/(pc+1e-6))/((1/(pc+1e-6)).sum()/N_PERIODS), dtype=torch.float32).to(DEVICE)

trc = np.bincount(trend_np[:n_train].reshape(-1), minlength=3).astype(float); trc /= trc.sum()
tw  = torch.tensor((1/(trc+1e-6))/((1/(trc+1e-6)).sum()/3), dtype=torch.float32).to(DEVICE)

# =============================================================================
# LOSS FUNCTIONS  (identical to v13)
# =============================================================================
def focal_loss_with_smooth(logits, targets, weight, gamma=2.0, smoothing=0.05, entropy_reg=0.05):
    n_cls = logits.shape[-1]
    log_p = F.log_softmax(logits, dim=-1); p = torch.exp(log_p)
    with torch.no_grad():
        smooth_t = torch.full_like(logits, smoothing/(n_cls-1))
        smooth_t.scatter_(-1, targets.unsqueeze(-1), 1.0-smoothing)
    ce_smooth = -(smooth_t * log_p).sum(-1)
    p_t = p.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    focal_w = (1.0-p_t)**gamma
    if weight is not None: focal_w = focal_w * weight[targets]
    focal = (focal_w * ce_smooth).mean()
    entropy = -(p * log_p).sum(-1).mean()
    return focal - entropy_reg * entropy


def compute_loss_full(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                      hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    l_hot = F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)
    valid = (lat_t.abs()>0.01) | (lon_t.abs()>0.01)
    l_loc = (F.huber_loss(loc_p[...,0][valid], lat_t[valid], delta=1.0) +
             F.huber_loss(loc_p[...,1][valid], lon_t[valid], delta=1.0)) if valid.any() \
            else torch.tensor(0.0, device=hot_l.device)
    l_typ = focal_loss_with_smooth(typ_l.reshape(-1,N_TYPES), typ_t.reshape(-1),
                weight=typw, gamma=cfg.FOCAL_GAMMA,
                smoothing=cfg.LABEL_SMOOTH, entropy_reg=cfg.ENTROPY_REG)
    l_per = F.cross_entropy(per_l.reshape(-1,N_PERIODS), per_t.reshape(-1), weight=perw)
    l_cnt = F.huber_loss(cnt_p.reshape(-1), cnt_t.reshape(-1), delta=0.5)
    l_trd = F.cross_entropy(trd_l.reshape(-1,3), trd_t.reshape(-1), weight=tw)
    return 1.0*l_hot + 1.0*l_loc + 1.5*l_typ + 0.6*l_per + 1.2*l_cnt + 0.5*l_trd


def compute_loss_hotspot_only(hot_l, loc_p, typ_l, per_l, cnt_p, trd_l,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t):
    """WithoutMultiTask — only hotspot BCE loss.
    The other five heads are still produced by the forward pass (so the
    architecture doesn't have to branch), but they receive zero gradient
    signal here. They must therefore be treated as untrained / uninformative
    at evaluation and reporting time — see evaluate() and save_results()."""
    return F.binary_cross_entropy_with_logits(
        hot_l.reshape(-1), hot_t.reshape(-1), pos_weight=pos_wt)

# =============================================================================
# HELPERS
# =============================================================================
def haversine_km(la1, lo1, la2, lo2):
    R = 6371.0; dlat = np.radians(la2-la1); dlon = np.radians(lo2-lo1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R*2*np.arctan2(np.sqrt(a), np.sqrt(1-a+1e-9))

def san(a): return np.nan_to_num(np.array(a, dtype=np.float32))

# =============================================================================
# ══════════════════════════════════════════════════════════════════════════════
#   MODEL DEFINITIONS — one class per ablation variant
# ══════════════════════════════════════════════════════════════════════════════
# =============================================================================


# =============================================================================
# SHARED SUB-MODULES
# =============================================================================

class ARATABlock(nn.Module):
    def __init__(self, d, n_heads, dropout, init_lam=0.05):
        super().__init__()
        self.H = n_heads; self.dh = d//n_heads
        self.qkv = nn.Linear(d, 3*d, bias=False); self.out = nn.Linear(d, d, bias=False)
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(dropout)
        self.log_lam_bwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam)))
        self.log_lam_fwd = nn.Parameter(torch.full((n_heads,), math.log(init_lam*0.5)))
        self.n_causal = n_heads//2
        self.tconv      = nn.Conv1d(d, d, kernel_size=3, padding=1, groups=d, bias=False)
        self.tconv_norm = nn.LayerNorm(d)
    def forward(self, x):
        B, T, NF, d = x.shape
        xc = x.permute(0,2,3,1).reshape(B*NF, d, T)
        xc = self.tconv(xc)
        xc = xc.reshape(B, NF, d, T).permute(0,3,1,2)
        x  = self.tconv_norm(x + xc)
        x2 = x.reshape(B*NF, T, d)
        Q, K, V = self.qkv(x2).chunk(3, dim=-1)
        Q = Q.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        K = K.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        V = V.reshape(B*NF, T, self.H, self.dh).transpose(1,2)
        pi = torch.arange(T, device=x.device).float().unsqueeze(1)
        pj = torch.arange(T, device=x.device).float().unsqueeze(0)
        diff = pi - pj
        lb = torch.exp(self.log_lam_bwd).clamp(0.001,2)
        lf = torch.exp(self.log_lam_fwd).clamp(0.001,2)
        bias = (-lb.view(self.H,1,1)*diff.clamp(min=0).unsqueeze(0) +
                -lf.view(self.H,1,1)*(-diff).clamp(min=0).unsqueeze(0))
        if self.n_causal > 0:
            cm = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
            bias[:self.n_causal] += cm
        attn = F.softmax((Q@K.transpose(-2,-1))/math.sqrt(self.dh)+bias.unsqueeze(0), dim=-1)
        out = (self.drop(attn)@V).transpose(1,2).reshape(B*NF, T, d)
        return self.norm(x + self.out(out).reshape(B, NF, T, d).permute(0,2,1,3))

class AHGLayer(nn.Module):
    def __init__(self, H, n_fine, n_coarse, heads, dropout):
        super().__init__()
        self.gf   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gc   = GATv2Conv(H, H//heads, heads=heads, dropout=dropout, add_self_loops=False)
        self.gate = nn.Linear(H*2, 1); self.norm = nn.LayerNorm(H)
    def forward(self, hf, hc, fei, cei, c2dn):
        B, NF, H = hf.shape
        hf2  = torch.stack([self.gf(hf[b], fei) for b in range(B)])
        hc2  = torch.stack([self.gc(hc[b], cei) for b in range(B)])
        hcca = hc2[:,c2dn,:]
        alpha = torch.sigmoid(self.gate(torch.cat([hf2, hcca], dim=-1)))
        return self.norm(hf + alpha*hf2 + (1-alpha)*hcca), hc2

def _mlp(i, m, o, dropout):
    blk = nn.Sequential(nn.Linear(i,m), nn.GELU(), nn.Dropout(dropout), nn.Linear(m,o))
    for l in blk:
        if isinstance(l, nn.Linear):
            nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
    return blk

# =============================================================================
# EVALUATE  (FIXED: variant-aware primary metric)
# =============================================================================
def evaluate(model, loader, variant_name="Full"):
    """
    variant_name controls which metric is used as the model-selection /
    reporting 'primary' score.

    - "WithoutMultiTask": only the hotspot head was trained, so `primary`
      is defined as hot_auc alone. The other five heads are still computed
      (forward pass produces them) but must NOT be used for selection or
      for cross-variant comparison, since they never received gradient.
    - anything else ("Full", or other ablations that keep the aux losses):
      `primary` is the original 5-way aggregate.
    """
    model.eval()
    ht, hs = [], []
    latr, lonr, latp, lonp = [], [], [], []
    tytt, tytp, pert, perp, cntr, cntp, trt_, trp_ = [], [], [], [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
            hs.extend(san(torch.sigmoid(ho).float().reshape(-1).cpu()))
            ht.extend(hot_t.reshape(-1).cpu().numpy())
            latr.extend(lat_rt.reshape(-1).cpu().numpy())
            lonr.extend(lon_rt.reshape(-1).cpu().numpy())
            latp.extend(san(lo[...,0].float().reshape(-1).cpu()))
            lonp.extend(san(lo[...,1].float().reshape(-1).cpu()))
            tytt.extend(typ_t.reshape(-1).cpu().tolist())
            tytp.extend(ty.float().reshape(-1,N_TYPES).argmax(-1).cpu().tolist())
            pert.extend(per_t.reshape(-1).cpu().tolist())
            perp.extend(pe.float().reshape(-1,N_PERIODS).argmax(-1).cpu().tolist())
            cntr.extend(lcnt_t.reshape(-1).cpu().numpy())
            cntp.extend(san(cn.float().reshape(-1).cpu()))
            trp_.extend(tr.float().reshape(-1,3).argmax(1).cpu().tolist())
            trt_.extend(trd_t.reshape(-1).cpu().tolist())

    ht = np.array(ht, dtype=np.int32); hs = san(hs)
    pr, re, thr = precision_recall_curve(ht, hs)
    f1c = 2*pr*re/(pr+re+1e-8)
    bt  = float(thr[f1c[:-1].argmax()]) if len(thr) else 0.5
    hp  = (hs >= bt).astype(int)
    try: rauc = roc_auc_score(ht, hs)
    except: rauc = 0.0
    try: pr_auc = average_precision_score(ht, hs)
    except: pr_auc = 0.0
    hot_f1  = f1_score(ht, hp, zero_division=0)
    hot_pre = precision_score(ht, hp, zero_division=0)
    hot_rec = recall_score(ht, hp, zero_division=0)

    lp_raw = san(latp)*lat_std+lat_mean; lr_raw = np.array(latr)
    op_raw = san(lonp)*lon_std+lon_mean; or_raw = np.array(lonr)
    valid  = (lr_raw>40) & (or_raw<-80)
    if valid.sum() > 0:
        hav = haversine_km(lr_raw[valid], or_raw[valid], lp_raw[valid], op_raw[valid])
        loc_mae = float(hav.mean()); l1 = float((hav<1).mean()); l05 = float((hav<0.5).mean())
    else:
        loc_mae = 99.9; l1 = 0.0; l05 = 0.0
    ta = accuracy_score(np.array(tytt), np.array(tytp))
    tf = f1_score(np.array(tytt), np.array(tytp), average="weighted", zero_division=0)
    pa = accuracy_score(np.array(pert), np.array(perp))
    pf = f1_score(np.array(pert), np.array(perp), average="weighted", zero_division=0)
    cr = san(cntr); cp_a = san(cntp)
    cnt_corr = float(np.corrcoef(cr, cp_a)[0,1]) if len(cr)>1 else 0.0
    cnt_mae  = float(np.abs(np.expm1(cr.clip(0,10))-np.expm1(cp_a.clip(0,10))).mean())
    trend_acc = accuracy_score(np.array(trt_), np.array(trp_))
    loc_score = max(0, 1-loc_mae/10)

    # ---- variant-aware primary metric ----
    if variant_name == "WithoutMultiTask":
        primary = rauc  # only the hotspot head was trained
    else:
        primary = np.mean([rauc, ta, pa, max(cnt_corr, 0), loc_score])

    return dict(
        primary=primary, hot_auc=rauc, hot_pr_auc=pr_auc, hot_f1=hot_f1,
        hot_precision=hot_pre, hot_recall=hot_rec, best_thr=bt,
        loc_mae=loc_mae, loc_1km=l1, loc_05km=l05,
        type_acc=ta, type_f1=tf, per_acc=pa, per_f1=pf,
        cnt_corr=cnt_corr, cnt_mae=cnt_mae, trend_acc=trend_acc,
    )

# =============================================================================
# SAVE RESULTS  (FIXED: only report trained heads for WithoutMultiTask)
# =============================================================================
def save_results(name, metrics, n_params, train_min, best_ep, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    if name == "WithoutMultiTask":
        # Only the hotspot head was trained — do NOT report type/hour/loc/
        # count/trend metrics here, since those heads never received a
        # training signal and comparing them to the full multi-task model
        # would be comparing an untrained head to a trained one.
        rows = [
            ["variant",          name],
            ["n_params",         n_params],
            ["train_min",        train_min],
            ["best_epoch",       best_ep],
            ["hot_roc_auc",      f"{metrics['hot_auc']:.4f}"],
            ["hot_pr_auc",       f"{metrics['hot_pr_auc']:.4f}"],
            ["hot_f1",           f"{metrics['hot_f1']:.4f}"],
            ["hot_precision",    f"{metrics['hot_precision']:.4f}"],
            ["hot_recall",       f"{metrics['hot_recall']:.4f}"],
            ["best_thr",         f"{metrics['best_thr']:.4f}"],
            ["primary_(=hot_auc)", f"{metrics['primary']:.4f}"],
            ["note", "aux heads (loc/type/hour/count/trend) untrained — not reported"],
        ]
    else:
        rows = [
            ["variant",       name],
            ["n_params",      n_params],
            ["train_min",     train_min],
            ["best_epoch",    best_ep],
            ["hot_auc",       f"{metrics['hot_auc']:.4f}"],
            ["hot_pr_auc",    f"{metrics['hot_pr_auc']:.4f}"],
            ["hot_f1",        f"{metrics['hot_f1']:.4f}"],
            ["hot_precision", f"{metrics['hot_precision']:.4f}"],
            ["hot_recall",    f"{metrics['hot_recall']:.4f}"],
            ["loc_mae_km",    f"{metrics['loc_mae']:.3f}"],
            ["loc_1km_pct",   f"{metrics['loc_1km']*100:.1f}"],
            ["loc_05km_pct",  f"{metrics['loc_05km']*100:.1f}"],
            ["type_acc",      f"{metrics['type_acc']:.4f}"],
            ["type_f1",       f"{metrics['type_f1']:.4f}"],
            ["hour_acc",      f"{metrics['per_acc']:.4f}"],
            ["hour_f1",       f"{metrics['per_f1']:.4f}"],
            ["cnt_pearson_r", f"{metrics['cnt_corr']:.4f}"],
            ["cnt_mae",       f"{metrics['cnt_mae']:.2f}"],
            ["trend_acc",     f"{metrics['trend_acc']:.4f}"],
            ["primary",       f"{metrics['primary']:.4f}"],
        ]
    path = os.path.join(out_dir, f"{name}_results.csv")
    with open(path, "w", newline="") as f:
        csv.writer(f).writerows(rows)
    print(f"  Results saved → {path}")

# =============================================================================
# TRAIN + EVALUATE  (FIXED: variant-aware checkpoint selection)
# =============================================================================
def train_and_save(model, loss_fn, variant_name, out_dir="ablation_results"):
    os.makedirs(out_dir, exist_ok=True)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*70}\n  {variant_name}  |  params={n_p:,}\n{'='*70}")
    with torch.no_grad():
        if hasattr(model, "area_log_bias"):
            model.area_log_bias.copy_(torch.tensor(area_log_mean_np).to(DEVICE))
        if hasattr(model, "area_lat_bias"):
            model.area_lat_bias.copy_(torch.tensor(area_lat_z, dtype=torch.float32).to(DEVICE))
        if hasattr(model, "area_lon_bias"):
            model.area_lon_bias.copy_(torch.tensor(area_lon_z, dtype=torch.float32).to(DEVICE))
    optimizer  = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler  = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.LR*8,
        total_steps=cfg.EPOCHS*len(train_loader),
        pct_start=0.10, anneal_strategy="cos")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)

    # Selection score can be negative-friendly (e.g. correlation-based
    # aggregates), so start from -inf rather than 0.0.
    best_p = -float("inf")
    best_ep = 0; nan_streak = 0; best_sd = None
    t0 = time.time()
    for ep in range(1, cfg.EPOCHS+1):
        model.train(); ep_loss = 0.0; ep_n = 0
        for batch in train_loader:
            (x, hot_t, lat_t, lon_t, lat_rt, lon_rt, typ_t,
             per_t, cnt_t, lcnt_t, trd_t) = [b.to(DEVICE, non_blocking=True) for b in batch]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg.FP16):
                ho, lo, ty, pe, cn, tr = model(x, fine_edge, coarse_edge)
                loss = loss_fn(ho, lo, ty, pe, cn, tr,
                               hot_t, lat_t, lon_t, typ_t, per_t, cnt_t, trd_t)
            if torch.isnan(loss):
                nan_streak += 1
                if nan_streak >= 20: print("  ⚠ 20 NaN — aborting"); break
                continue
            nan_streak = 0
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            amp_scaler.step(optimizer); amp_scaler.update(); scheduler.step()
            ep_loss += loss.item()*x.shape[0]; ep_n += x.shape[0]
        else:
            s = evaluate(model, val_loader, variant_name)

            # Variant-aware checkpoint selection: hotspot-only variant must
            # be selected on hot_auc, never on the 5-head aggregate (those
            # heads carry no training signal for this variant).
            if variant_name == "WithoutMultiTask":
                selection_score = s["hot_auc"]
            else:
                selection_score = s["primary"]

            tl = ep_loss / max(1, ep_n)
            if ep % 10 == 0 or selection_score > best_p:
                print(f"  ep={ep:3d}  tl={tl:.4f}  sel={selection_score:.4f}  "
                      f"AUC={s['hot_auc']:.4f}  PR-AUC={s['hot_pr_auc']:.4f}  "
                      f"F1={s['hot_f1']:.4f}  P={s['hot_precision']:.4f}  "
                      f"R={s['hot_recall']:.4f}")
            if selection_score > best_p:
                best_p = selection_score; best_ep = ep
                best_sd = copy.deepcopy(model.state_dict())
                ckpt = os.path.join(out_dir, f"{variant_name}_best.pth")
                torch.save(best_sd, ckpt)
                print(f"  ✓ Best={best_p:.4f}  checkpoint → {ckpt}")
            if ep - best_ep >= cfg.PATIENCE:
                print(f"  ⏹ Early stop ep={ep}"); break
            continue
        break
    elapsed = time.time() - t0
    if best_sd: model.load_state_dict(best_sd)
    model.eval()
    metrics   = evaluate(model, test_loader, variant_name)
    train_min = round(elapsed/60, 1)
    if variant_name == "WithoutMultiTask":
        print(f"\n  ✅ TEST  Hotspot ROC-AUC={metrics['hot_auc']:.4f}  "
              f"PR-AUC={metrics['hot_pr_auc']:.4f}  F1={metrics['hot_f1']:.4f}  "
              f"P={metrics['hot_precision']:.4f}  R={metrics['hot_recall']:.4f}  "
              f"({train_min} min)")
    else:
        print(f"\n  ✅ TEST  AUC={metrics['hot_auc']:.4f}  LocMAE={metrics['loc_mae']:.2f}km  "
              f"TypeAcc={metrics['type_acc']:.4f}  HrAcc={metrics['per_acc']:.4f}  "
              f"CntR={metrics['cnt_corr']:.4f}  ({train_min} min)")
    save_results(variant_name, metrics, n_p, train_min, best_ep, out_dir)

class WithoutMultiTask(nn.Module):
    def __init__(self):
        super().__init__()
        H, heads, drop = cfg.HIDDEN_DIM, cfg.GAT_HEADS, cfg.DROPOUT
        self.pf    = nn.Sequential(nn.Linear(N_FEAT,H), nn.LayerNorm(H), nn.GELU())
        self.pc    = nn.Sequential(nn.Linear(H,H), nn.LayerNorm(H), nn.GELU())
        self.arata = nn.ModuleList([ARATABlock(H,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.ahg   = nn.ModuleList([AHGLayer(H,N_COMM,N_DIST,heads,drop) for _ in range(cfg.N_LAYERS)])
        self.mp    = nn.Sequential(nn.Linear(H,H), nn.GELU(), nn.LayerNorm(H))
        self.n_rec = 3
        self.hot_h   = _mlp(H*2,64,1,drop)
        self.loc_h   = _mlp(H+32,64,2,drop); self.type_h = _mlp(H,128,N_TYPES,drop)
        self.per_h   = _mlp(H,64,N_PERIODS,drop); self.cnt_h = _mlp(H+32,64,1,drop)
        self.trend_h = _mlp(H*2,64,3,drop)
        self.area_emb      = nn.Embedding(N_COMM,32)
        self.area_lat_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_lon_bias = nn.Parameter(torch.zeros(N_COMM))
        self.area_log_bias = nn.Parameter(torch.zeros(N_COMM))
    def forward(self, xf, fei, cei):
        B, T, NF, _ = xf.shape; H = cfg.HIDDEN_DIM; DEV = xf.device
        hf = self.pf(xf)
        for a in self.arata: hf = a(hf)
        hp = hf[:,-1,:,:]
        c2dn = CA_TO_DN.to(DEV)
        hc = torch.zeros(B, N_DIST, H, device=DEV)
        hc.scatter_reduce_(1, c2dn.unsqueeze(0).unsqueeze(-1).expand(B,-1,H), hp, reduce="mean", include_self=True)
        hc = self.pc(hc)
        for a in self.ahg: hp, hc = a(hp, hc, fei, cei, c2dn)
        hr = hf[:,-self.n_rec:,:,:].mean(1); hl = hf[:,:-self.n_rec,:,:].mean(1).detach()
        mom = self.mp(hr - hl); hm = torch.cat([hp, mom], dim=-1)
        ae  = self.area_emb(torch.arange(NF, device=DEV).unsqueeze(0).expand(B,-1))
        hp_ae = torch.cat([hp, ae], dim=-1)
        hot = self.hot_h(hm).squeeze(-1)
        loc = torch.stack([self.loc_h(hp_ae)[...,0]+self.area_lat_bias,
                           self.loc_h(hp_ae)[...,1]+self.area_lon_bias], dim=-1)
        typ = self.type_h(hp); per = self.per_h(hp)
        cnt = (self.cnt_h(hp_ae).squeeze(-1) + self.area_log_bias).clamp(0,6)
        trd = self.trend_h(hm)
        return hot, loc, typ, per, cnt, trd

# =============================================================================
# RUN
# =============================================================================
if __name__ == '__main__':
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
    model = WithoutMultiTask().to(DEVICE)
    train_and_save(model, compute_loss_hotspot_only, 'WithoutMultiTask')